# Prediksi Tinggi Muka Air (TMA) — BBWS Bengawan Solo


# Deskripsi Dataset
---

Dataset yang merupakan time series:
1. Memiliki rentang waktu
2. Memiliki korelasi antar fitur
3. Memiliki pola berulang

Target yang diinginkan adalah memprediksi rentang tingginya muka air.
<br>
<b>
Algoritma yang digunakan (Regression)</b>

NOTE: <br>
Diketahui bahwa bangawan solo ada di antara lokasi perbatasan jawa tengah dengan jawa timur


**File**

| **File** | **Deskripsi** |
|:---------|:--------------|
| `train.csv` | Dataset pelatihan berisi data historis TMA |
| `test.csv` | Dataset pengujian untuk prediksi (kolom `id`) |
| `sample_submission.csv` | File contoh format submisi |
| `data_pendukung/data_lingkungan.csv` | Data fitur eksogen (cuaca, tanah, iklim) resolusi per jam |
| `data_pendukung/koordinat_pos.csv` | Koordinat 30 pos pemantauan |
| `data_pendukung/HydroRIVERS_v10_au_shp/` | Data sungai (shapefile) |

**Target**
| Variable   | Tipe     | Deskripsi              |
|------------|----------|------------------------|
| `datetime` | datetime | Waktu Observasi        |
| `nama_pos` | str      | Nama pos pemantauan    |
| `tma_mdpl` | float    | Tinggi Muka Air (mdpl) |
<br>

**Fitur Pendukung**
| Variable                   | Tipe  | Deskripsi                               |
|----------------------------|-------|-----------------------------------------|
| `rainfall_mm`              | float | Curah hujan (mm)                        |
| `humidity_pct`             | float | Kelembapan udara (%)                    |
| `wind_direction_deg`       | float | Arah mata angin (o)                     |
| `dew_point_c`              | float | Titik embun (Celsius)                   |
| `cloud_cover_pct`          | float | Tutupan awan (%)                        |
| `temperature_c`            | float | Suhu udara 2m (Celsius)                 |
| `wind_speed_kmh`           | float | Kecepatan angin 10m (km/h)              |
| `rainfall_openmeteo_mm`    | float | Curah hujan (mm)                        |
| `rainfall_max_24h_mm`      | float | Curah hujan maksimum 24 jam roling      |
| `solar_radiation_mj_m2`    | float | Radiasi Surya ALLSKU_SFC_SW_SWN (MJ/m2) |
| `soil_moisture_0_7cm`      | float | Kelembapan tanah 0 - 7cm (m3/m3)        |
| `soil_moisture_7_28cm`     | float | Kelembapan tanah 7 - 28cm (m3/m3)       |
| `soil_moisture_28_100_cm`  | float | Kelembapan tanah 28 - 100cm (m3/m3)     |
| `soil_moisture_100_255_cm` | float | Kelembapan tanah 100 - 255cm (m3/m3)    |
| `surface_pressure_hpa`     | float | Tekanan udara (hPa)                     |
| `pressure_msl_hpa`         | float | Tekanan udara mean sea level (hPa)      |
| `built_surface_m2`         | float | Luas permukaan terbangun (m3)           |
| `landcover_class`          | int   | Kelas tutupan lahan (10-80)             |
| `landcover_name`           | str   | Nama kelas tutupan lahan                |
| `rmm1`                     | float | Komponen RMM1 MJO (harian)              |
| `rmm2`                     | float | Komponen RMM2 MJO (harian)              |
| `mjo_phase`                | int   | Fase MJO (1-8)                          |
| `mjo_amplitude`            | float | Amplitudo MJO                           |
| `mjo_active`               | int   | MJO aktif (1) / tidak (2)               |
| `nino_34`                  | float | Indeks ENSO Nino 3.4 (Celsius, bulanan) |

## Alur Notebook

Notebook ini disusun agar dapat dijalankan dari atas ke bawah:

1. **Setup dan konfigurasi global**
2. **EDA** pada data mentah
3. **Preprocessing terpadu**: cleaning, spike handling, data lingkungan, feature engineering, dan fitur spasial
4. **Validasi time-series**
5. **Modeling eksperimen 1–5**
6. **Quantile forecasting dan deteksi anomali prediksi per pos**
7. **Evaluasi dan submission**

> Deskripsi dataset asli dipertahankan tanpa penghapusan.


## 1. Setup, Library, dan Konfigurasi 📑


In [ ]:
# Install package
!pip install geopandas pyogrio shapely rtree networkx scikit-learn torch torch-geometric catboost neuralforecast psutil umap-learn -q

In [ ]:
# %pip uninstall -y accelerate transformers neuralforecast
# %pip install --no-cache-dir --upgrade \
#     neuralforecast \
#     accelerate \
#     transformers

In [ ]:
# Import Library
from __future__ import annotations

import re
import warnings
from typing import Iterator, Tuple
from dataclasses import dataclass
import gc
from pathlib import Path
from collections.abc import Sequence

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib


from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import xgboost as xgb
from xgboost import XGBRegressor, plot_importance
import lightgbm as lgb
from lightgbm import LGBMRegressor


warnings.filterwarnings("ignore")


In [ ]:
TARGET = "tma_mdpl"
DATETIME_COL = "datetime"
GROUP_COL = "nama_pos"

# Resolusi observasi TMA nyatanya 3x/hari (jam 06, 12, 18) -> lag dalam satuan
# BARIS, bukan jam asli. lag_1 = 6 jam, lag_24 = 6 hari, lag_48 = 12 hari, dst.
LAG_LIST = [1, 3, 6, 12, 24, 48]
ROLLING_WINDOWS = [3, 6, 12, 24, 72]

# Curah hujan juga penting dilag -> efeknya ke TMA biasanya delayed (bukan instan)
EXTRA_LAG_COLS = ["rainfall_mm", "rainfall_openmeteo_mm"]

VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Faktor untuk deteksi spike sensor per stasiun (lihat fungsi clean_target_spikes)
SPIKE_FACTOR = 8.0
SPIKE_Z_FLOOR = 4.0

# Margin toleransi untuk clipping prediksi rekursif (persentase dari rentang
# min-max historis per stasiun). Mencegah error menggelinding tak terkendali.
CLIP_MARGIN_PCT = 0.20

OBSERVATIONS_PER_DAY = 3
RANDOM_SEED = 42
PIPELINE_VERSION = "2026-07-hydrologic-ensemble-v3"


# Conservative settings intended to fit comfortably in a 15 GB GPU.
# CatBoost stays on CPU by default, leaving GPU memory for the neural model.
DEFAULT_CONFIG = {
    "device": "auto",                 # "auto", "gpu", or "cpu"
    "gpu_index": 0,
    "precision": "32-true",           # More stable than AMP for this dataset.
    "neural_model": "nhits",          # "nhits" (recommended) or "tide"
    "input_days": 120,
    "tide_hidden_size": 64,
    "tide_batch_size": 8,
    "tide_windows_batch_size": 32,
    "tide_inference_batch_size": 32,
    "tide_max_steps": 300,
    "catboost_task_type": "CPU",     # Change to "GPU" if desired.
    "catboost_iterations": 800,
    "catboost_depth": 7,
    "catboost_gpu_ram_part": 0.25,
    "add_hydrology_features": True,
    "report_worst_stations": 10,
    "seasonal_window_days": 15,
    "seasonal_anchor_half_life_days": 3,
    "weight_step": 0.05,
    "ensemble_objective": "balanced",  # "balanced", "rmse", or "mae"
    "validation_mode": "same_season",  # "same_season" or "last_block"
    "output_file": "submission_ens.csv",
}


In [ ]:
# Mengambil dataset dari google drive (jika tidak perlu, comment or delete this cell)
from google.colab import drive
drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/sebelas-maret-statistics-data-science-2026')

In [ ]:
HYDRO = ROOT / 'data_pendukung/HydroRIVERS_v10_au_shp/HydroRIVERS_v10_au.shp'
train_df = pd.read_csv(ROOT / 'train.csv')
test_df = pd.read_csv(ROOT / 'test.csv')
datling_df = pd.read_csv(ROOT / 'data_pendukung/data_lingkungan.csv')
koor_df = pd.read_csv(ROOT / 'data_pendukung/koordinat_pos.csv')
df_submission = pd.read_csv(ROOT / 'sample_submission.csv')

train_df[DATETIME_COL] = pd.to_datetime(train_df[DATETIME_COL])
datling_df[DATETIME_COL] = pd.to_datetime(datling_df[DATETIME_COL])

print(f"Shape train_df    : {train_df.shape}")
print(f"Shape test_df     : {test_df.shape}")
print(f"Shape koor_df     : {koor_df.shape}")
print(f"Shape datling_df  : {datling_df.shape}")
# print(f"Shape koorv2_df   : {koorv2_df.shape}")


## 2. Exploratory Data Analysis (EDA) 📊


Missing Value & Duplikat Data

In [ ]:
# @title
# Fungsi untuk mengecek missing value
def check_missing_values(df):
    total_missing = df.isnull().sum()
    if total_missing.sum() == 0:
        print("Tidak ada missing value pada df ini")
    else:
        percent_missing = (df.isnull().sum() / len(df)) * 100
        missing_data = pd.DataFrame({
            'Total Missing': total_missing,
            'Percentage (%)': percent_missing.round(5)
        })
        # Hanya tampilkan kolom yang memiliki missing value > 0
        display(missing_data[missing_data['Total Missing'] > 0].sort_values(by='Total Missing', ascending=False))

# Contoh penggunaan untuk semua dataframe
print("--- Train DataFrame ---")
check_missing_values(train_df)

print("\n--- Test DataFrame ---")
check_missing_values(test_df)

print("\n--- Data Lingkungan DataFrame ---")
check_missing_values(datling_df)

print("\n--- Koordinat Pos DataFrame ---")
check_missing_values(koor_df)

Visualisasi Data per-pos

In [ ]:
# @title
# Memastikan kolom datetime dalam format yang benar
if 'datetime' not in train_df.columns or not pd.api.types.is_datetime64_any_dtype(train_df['datetime']):
    train_df['datetime'] = pd.to_datetime(train_df['datetime'])

all_pos = train_df['nama_pos'].unique()
plot_df = train_df[train_df['nama_pos'].isin(all_pos)].copy()
plot_df = plot_df.sort_values(['nama_pos', 'datetime'])
plot_df['n_day'] = plot_df.groupby('nama_pos').cumcount()

def calc_iqr_bounds(series, k=1.5):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return q1, q3, iqr, q1 - k * iqr, q3 + k * iqr

outlier_records = []

fig, axes = plt.subplots(len(all_pos), 1, figsize=(15, 6 * len(all_pos)))
if len(all_pos) == 1:
    axes = [axes]

for i, pos in enumerate(all_pos):
    data_pos = plot_df[plot_df['nama_pos'] == pos].copy()

    q1, q3, iqr, lower, upper = calc_iqr_bounds(data_pos['tma_mdpl'])
    data_min, data_max = data_pos['tma_mdpl'].min(), data_pos['tma_mdpl'].max()
    median_val = data_pos['tma_mdpl'].median()

    data_pos['is_outlier'] = (data_pos['tma_mdpl'] < lower) | (data_pos['tma_mdpl'] > upper)

    # Garis utama
    sns.lineplot(data=data_pos, x='n_day', y='tma_mdpl', ax=axes[i],
                 sort=False, lw=1.5, label='TMA', color='steelblue', zorder=2)

    # Range (min/max)
    axes[i].axhline(data_min, color='gray', linestyle='--', lw=0.8, alpha=0.6, label=f'Min ({data_min:.2f})')
    axes[i].axhline(data_max, color='gray', linestyle='--', lw=0.8, alpha=0.6, label=f'Max ({data_max:.2f})')

    # IQR: Q1, Q3, median
    axes[i].axhline(q1, color='orange', lw=1, alpha=0.8, label=f'Q1 ({q1:.2f})')
    axes[i].axhline(q3, color='orange', lw=1, alpha=0.8, label=f'Q3 ({q3:.2f})')
    axes[i].axhline(median_val, color='green', lw=1, alpha=0.8, label=f'Median ({median_val:.2f})')

    # Fence outlier (Q1-1.5*IQR, Q3+1.5*IQR)
    axes[i].axhline(lower, color='darkred', linestyle=':', lw=1.2, alpha=0.9, label=f'Lower Fence ({lower:.2f})')
    axes[i].axhline(upper, color='darkred', linestyle=':', lw=1.2, alpha=0.9, label=f'Upper Fence ({upper:.2f})')

    # Tandai titik outlier
    outliers = data_pos[data_pos['is_outlier']]
    if not outliers.empty:
        axes[i].scatter(outliers['n_day'], outliers['tma_mdpl'], color='red', s=30,
                         zorder=5, label=f'Outlier ({len(outliers)})')

    for _, row in outliers.iterrows():
        outlier_records.append({
            'nama_pos': pos,
            'datetime': row['datetime'],
            'n_day': row['n_day'],
            'tma_mdpl': row['tma_mdpl'],
            'lower_fence': lower,
            'upper_fence': upper,
            'tipe': 'Rendah (< Q1-1.5*IQR)' if row['tma_mdpl'] < lower else 'Tinggi (> Q3+1.5*IQR)'
        })

    # Garis batas tahun
    for year in [2024, 2025]:
        boundary = data_pos[data_pos['datetime'].dt.year == year]['n_day']
        if not boundary.empty:
            x_val = boundary.iloc[0]
            axes[i].axvline(x=x_val, color='black', linestyle='--', alpha=0.5)
            axes[i].text(x_val, data_max, f' Awal {year}', color='black', rotation=90, va='top', ha='right', fontsize=8)

    axes[i].set_title(f'Timeline TMA_MDPL + Range/IQR di Pos: {pos}  |  Outlier: {len(outliers)}', fontsize=13)
    axes[i].set_xlabel('Hari ke-n (Cumulative Days)')
    axes[i].set_ylabel('TMA (mdpl)')
    axes[i].grid(True, linestyle=':', alpha=0.5)
    axes[i].legend(loc='upper right', fontsize=8, ncol=2)

plt.tight_layout()
plt.show()

# ==== Tabel ringkasan rentang timeline outlier per pos ====
outlier_df = pd.DataFrame(outlier_records)

if not outlier_df.empty:
    summary_rows = []
    for pos, grp in outlier_df.groupby('nama_pos'):
        grp = grp.sort_values('datetime')
        summary_rows.append({
            'nama_pos': pos,
            'jumlah_outlier': len(grp),
            'tanggal_outlier_pertama': grp['datetime'].min().date(),
            'tanggal_outlier_terakhir': grp['datetime'].max().date(),
            'n_day_range': f"{int(grp['n_day'].min())}–{int(grp['n_day'].max())}",
            'nilai_min_outlier': round(grp['tma_mdpl'].min(), 3),
            'nilai_max_outlier': round(grp['tma_mdpl'].max(), 3),
        })
    outlier_summary_df = pd.DataFrame(summary_rows).sort_values('jumlah_outlier', ascending=False)

    print("=== Ringkasan Outlier per Pos ===")
    display(outlier_summary_df)

    print("\n=== Detail Semua Titik Outlier ===")
    display(outlier_df.sort_values(['nama_pos', 'datetime']).reset_index(drop=True))
else:
    print("Tidak ditemukan outlier berdasarkan metode IQR (k=1.5) di seluruh pos.")

## 3. Preprocessing Data


### 3.1 Cleaning Data

In [ ]:
# @title
def clean_environment(environment):
    environment = environment.copy()

    # -999 is a missing-value sentinel, not real solar radiation.
    if "solar_radiation_mj_m2" in environment.columns:
        bad_solar = environment["solar_radiation_mj_m2"].eq(-999)
        environment["solar_radiation_missing"] = bad_solar.astype(int)
        environment.loc[bad_solar, "solar_radiation_mj_m2"] = np.nan

    environment = environment.drop_duplicates(["nama_pos", "datetime"])
    environment = environment.sort_values(["nama_pos", "datetime"])

    # The neural model requires complete covariates. Keep an indicator so it
    # can distinguish measured values from filled values.
    numeric_columns = environment.select_dtypes(include="number").columns.tolist()

    for column in numeric_columns:
        environment[column] = environment[column].replace([np.inf, -np.inf], np.nan)

        if environment[column].isna().any():
            environment[column + "_missing"] = environment[column].isna().astype(int)
            environment[column] = environment.groupby("nama_pos")[column].transform(
                lambda values: values.interpolate(limit_direction="both")
            )
            environment[column] = environment[column].fillna(environment[column].median())

    return environment
clean_environment(datling_df)
datling_df.dropna(inplace=True)
print("datling_df setelah cleaning:")
print(f"Shape: {datling_df.shape}")
print("Missing values setelah cleaning:")
# Use the previously defined function to check for missing values
check_missing_values(datling_df)

### 3.2 Pembersihan Spike Sensor per Stasiun 🧹

Sebelum feature engineering apa pun, kita bersihkan `tma_mdpl` dulu. Ciri khas
error sensor: **satu baris meledak jauh dari nilai normal, lalu baris
berikutnya langsung balik ke nilai yang mirip dengan sebelum error terjadi**.
Ini beda dari banjir asli yang naik-turun bertahap lewat beberapa jam/hari.

Deteksi dilakukan **per stasiun** (bukan global), karena tiap stasiun punya
skala/elevasi dasar yang beda jauh (ada yang ~2 m, ada yang ~130 m) — deteksi
global akan salah kaprah membandingkan skala yang tidak sepadan.

In [ ]:
# @title
def _runs(mask):
    groups = mask.ne(mask.shift()).cumsum()
    return mask.groupby(groups).transform("size")


def clean_tma_spikes(
    df,
    station_col="nama_pos",
    time_col="datetime",
    value_col="tma_mdpl",
    window=7,
    z_limit=8,
    jump_limit=6,
    return_limit=3,
    max_run=2,
    min_points=15,
    keep_diagnostics=False,   # <-- BARU: simpan _robust_score & _is_spike untuk EDA lanjutan
):
    """
    Remove isolated gauge impulses per station without deleting real peaks.
    (Logika deteksi spike TIDAK diubah — sudah MAD-based & per-pos.)
    """
    needed = {station_col, time_col, value_col}
    missing = needed.difference(df.columns)
    if missing:
        raise ValueError("Missing columns: " + ", ".join(sorted(missing)))

    data = df.copy()
    data[time_col] = pd.to_datetime(data[time_col])
    data["_row_order"] = np.arange(len(data))
    data["_original_tma"] = data[value_col]
    data["_is_spike"] = False
    data["_robust_score"] = np.nan

    output = []

    for station, part in data.groupby(station_col, sort=False, dropna=False):
        part = part.sort_values(time_col).copy()
        x = pd.to_numeric(part[value_col], errors="coerce")

        if x.notna().sum() < min_points:
            output.append(part)
            continue

        median = x.rolling(window, center=True, min_periods=3).median()
        deviation = (x - median).abs()
        mad = deviation.rolling(window, center=True, min_periods=3).median()

        station_scale = np.nanmedian(np.abs(x - np.nanmedian(x))) * 1.4826
        diff_scale = np.nanmedian(np.abs(x.diff() - np.nanmedian(x.diff()))) * 1.4826
        scale_floor = max(station_scale * 0.02, diff_scale, 1e-6)
        local_scale = (mad * 1.4826).clip(lower=scale_floor)
        score = deviation / local_scale

        prev = x.shift(1)
        nxt = x.shift(-1)
        prev2 = x.shift(2)
        nxt2 = x.shift(-2)
        jump_in = (x - prev).abs() / local_scale
        jump_out = (x - nxt).abs() / local_scale
        neighbors_match = (prev - nxt).abs() / local_scale <= return_limit
        stable_before = (prev - prev2).abs() / local_scale <= return_limit
        stable_after = (nxt - nxt2).abs() / local_scale <= return_limit

        candidate = (
            (score >= z_limit)
            & (jump_in >= jump_limit)
            & (jump_out >= jump_limit)
            & neighbors_match
            & stable_before
            & stable_after
            & prev.notna()
            & nxt.notna()
            & prev2.notna()
            & nxt2.notna()
        )

        short_run = _runs(candidate) <= max_run
        spike = candidate & short_run

        broad_candidate = score >= z_limit
        broad_group = broad_candidate.ne(broad_candidate.shift()).cumsum()

        for group_id, indices in part[broad_candidate].groupby(broad_group).groups.items():
            positions = part.index.get_indexer(indices)
            if len(positions) > max_run or len(positions) == 0:
                continue
            left_pos = positions.min() - 1
            right_pos = positions.max() + 1
            outer_left_pos = left_pos - 1
            outer_right_pos = right_pos + 1
            if outer_left_pos < 0 or outer_right_pos >= len(part):
                continue
            left = x.iloc[left_pos]
            right = x.iloc[right_pos]
            outer_left = x.iloc[outer_left_pos]
            outer_right = x.iloc[outer_right_pos]
            local = local_scale.iloc[positions].max()
            anchors_exist = all(pd.notna(v) for v in [left, right, outer_left, outer_right])
            anchors_match = abs(left - right) <= return_limit * local
            anchors_stable = (
                abs(left - outer_left) <= return_limit * local
                and abs(right - outer_right) <= return_limit * local
            )
            if anchors_exist and anchors_match and anchors_stable:
                jumps = min(abs(x.iloc[positions[0]] - left), abs(x.iloc[positions[-1]] - right))
                if jumps >= jump_limit * local:
                    spike.iloc[positions] = True

        cleaned = x.mask(spike)
        cleaned = cleaned.interpolate(method="linear", limit=max_run, limit_area="inside")

        part[value_col] = cleaned
        part["_is_spike"] = spike.to_numpy()
        part["_robust_score"] = score.to_numpy()
        output.append(part)

    result = pd.concat(output).sort_values("_row_order")

    audit = result.loc[result["_is_spike"], [
        station_col, time_col, "_original_tma", value_col, "_robust_score",
    ]].copy()
    audit = audit.rename(columns={
        "_original_tma": "tma_original",
        value_col: "tma_replacement",
        "_robust_score": "robust_score",
    })
    audit["reason"] = "isolated_jump_and_return"
    audit["koreksi_abs"] = (audit["tma_original"] - audit["tma_replacement"]).abs()

    drop_cols = ["_row_order", "_original_tma"]
    if not keep_diagnostics:
        drop_cols += ["_is_spike", "_robust_score"]
    result = result.drop(columns=drop_cols)

    return result, audit.reset_index(drop=True)


def spike_summary(audit, raw_df=None, station_col="nama_pos"):
    """
    Laporan statistik per pos untuk EDA:
    - jumlah & persentase data yang diganti
    - rata-rata & maksimum robust score (seberapa ekstrem spike-nya)
    - rata-rata & maksimum magnitude koreksi (seberapa besar nilainya diubah)
    - rentang tanggal terjadinya spike
    Diurutkan dari pos dengan spike terbanyak -> paling sedikit.
    """
    cols = [
        station_col, "total_data", "jumlah_spike", "persen_spike",
        "avg_robust_score", "max_robust_score",
        "avg_koreksi_abs", "max_koreksi_abs",
        "tanggal_spike_pertama", "tanggal_spike_terakhir",
    ]
    if audit.empty:
        return pd.DataFrame(columns=cols)

    summary = (
        audit.groupby(station_col, dropna=False)
        .agg(
            jumlah_spike=("robust_score", "size"),
            avg_robust_score=("robust_score", "mean"),
            max_robust_score=("robust_score", "max"),
            avg_koreksi_abs=("koreksi_abs", "mean"),
            max_koreksi_abs=("koreksi_abs", "max"),
            tanggal_spike_pertama=("datetime", "min"),
            tanggal_spike_terakhir=("datetime", "max"),
        )
        .round(3)
        .reset_index()
    )

    if raw_df is not None:
        total_points = raw_df.groupby(station_col).size().rename("total_data")
        summary = summary.merge(total_points, on=station_col, how="left")
        summary["persen_spike"] = (summary["jumlah_spike"] / summary["total_data"] * 100).round(3)
        summary = summary[cols]
    else:
        summary = summary[[c for c in cols if c not in ("total_data", "persen_spike")]]

    return summary.sort_values("jumlah_spike", ascending=False).reset_index(drop=True)


def spike_score_distribution(clean_df_with_diagnostics, station_col="nama_pos"):
    """
    EDA tambahan: distribusi robust_score SELURUH titik data (bukan hanya
    yang sudah lolos threshold jadi spike). Berguna untuk mengecek apakah
    z_limit=8 sudah pas, atau perlu diturunkan/naikkan per pos.
    Memerlukan clean_tma_spikes(..., keep_diagnostics=True).
    """
    if "_robust_score" not in clean_df_with_diagnostics.columns:
        raise ValueError("Panggil clean_tma_spikes(..., keep_diagnostics=True) dulu.")

    return (
        clean_df_with_diagnostics.groupby(station_col)["_robust_score"]
        .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
        .round(3)
        .sort_values("max", ascending=False)
    )


# ==== Contoh pemakaian ====
train_df_clean, spike_audit = clean_tma_spikes(train_df, keep_diagnostics=True)

report = spike_summary(spike_audit, raw_df=train_df)
print("=== Laporan Spike per Pos (urut terbanyak -> tersedikit) ===")
display(report)

print("\n=== Distribusi Robust Score per Pos (untuk EDA / tuning threshold) ===")
display(spike_score_distribution(train_df_clean))

print("\n=== Detail Titik yang Diganti ===")
display(spike_audit.head(30))

In [ ]:
# @title
# ==== Ambil top 10 pos dengan spike terbanyak ====
top_n = 10
spike_counts = spike_audit.groupby(GROUP_COL).size().sort_values(ascending=False)
top_pos_list = spike_counts.head(top_n).index.tolist()

# Untuk tiap pos, pusatkan window di tanggal dengan robust_score TERTINGGI
# (spike paling ekstrem di pos tsb) agar sanity-check fokus ke kejadian terparah
check_examples = []
for pos in top_pos_list:
    pos_audit = spike_audit[spike_audit[GROUP_COL] == pos]
    center_row = pos_audit.loc[pos_audit["robust_score"].idxmax()]
    check_examples.append((pos, center_row[DATETIME_COL], len(pos_audit)))

# ==== Plot before vs after untuk masing-masing pos ====
fig, axes = plt.subplots(len(check_examples), 1, figsize=(11, 4 * len(check_examples)))
if len(check_examples) == 1:
    axes = [axes]

for ax, (pos, center_date, n_spike) in zip(axes, check_examples):
    center = pd.Timestamp(center_date)
    window = (center - pd.Timedelta(days=5), center + pd.Timedelta(days=5))

    raw = train_df[(train_df[GROUP_COL] == pos) &
                    (train_df[DATETIME_COL].between(*window))]
    clean = train_df_clean[(train_df_clean[GROUP_COL] == pos) &
                            (train_df_clean[DATETIME_COL].between(*window))]

    ax.plot(raw[DATETIME_COL], raw[TARGET], "o-", color="#E24B4A", alpha=0.6, label="Sebelum cleaning")
    ax.plot(clean[DATETIME_COL], clean[TARGET], "o-", color="#1D9E75", alpha=0.8, label="Sesudah cleaning")

    # Tandai titik yang benar-benar terdeteksi sebagai spike dalam window ini
    pos_spikes_in_window = spike_audit[
        (spike_audit[GROUP_COL] == pos) &
        (spike_audit[DATETIME_COL].between(*window))
    ]
    if not pos_spikes_in_window.empty:
        ax.scatter(
            pos_spikes_in_window[DATETIME_COL],
            pos_spikes_in_window["tma_original"],
            color="black", marker="x", s=90, zorder=5,
            label="Titik terdeteksi spike"
        )

    ax.set_title(f"{pos}  |  total spike: {n_spike}  |  window sekitar spike terparah ({center.date()})")
    ax.set_xlabel("Tanggal")
    ax.set_ylabel(TARGET)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# @title
diagnostic_rows = []

for pos in top_pos_list:
    part = train_df[train_df[GROUP_COL] == pos].sort_values(DATETIME_COL).copy()
    x = pd.to_numeric(part[TARGET], errors="coerce")

    median = x.rolling(7, center=True, min_periods=3).median()
    deviation = (x - median).abs()
    mad = deviation.rolling(7, center=True, min_periods=3).median()

    station_scale = np.nanmedian(np.abs(x - np.nanmedian(x))) * 1.4826
    diff_scale = np.nanmedian(np.abs(x.diff() - np.nanmedian(x.diff()))) * 1.4826
    scale_floor = max(station_scale * 0.02, diff_scale, 1e-6)
    local_scale = (mad * 1.4826).clip(lower=scale_floor)
    score = deviation / local_scale

    prev, nxt = x.shift(1), x.shift(-1)
    jump_in = (x - prev).abs() / local_scale
    jump_out = (x - nxt).abs() / local_scale
    neighbors_match = (prev - nxt).abs() / local_scale <= 3  # return_limit default

    # Kandidat "ekstrem berdasarkan score" TANPA syarat run-length / return-to-baseline
    raw_extreme = score >= 8  # z_limit default

    # Bandingkan dengan hasil aktual dari clean_tma_spikes
    actual_flagged = spike_audit[spike_audit[GROUP_COL] == pos][DATETIME_COL]
    part_dt = part[DATETIME_COL].reset_index(drop=True)
    actual_mask = part_dt.isin(actual_flagged).values

    n_raw_extreme = raw_extreme.sum()
    n_actually_cleaned = actual_mask.sum()

    # H1: cek apakah ada titik raw_extreme yang score-nya "diredam" jadi < 8
    # karena local_scale membesar dibanding scale_floor
    inflated_scale_ratio = (local_scale / scale_floor)
    n_scale_inflated_near_extreme = ((inflated_scale_ratio > 3) & (deviation > deviation.median() * 5)).sum()

    # H2: cek titik dengan score tinggi tapi gagal jadi run pendek
    candidate_like = (score >= 8) & (jump_in >= 6) & (jump_out.notna())
    run_len = _runs(candidate_like)
    n_failed_run_length = ((candidate_like) & (run_len > 2)).sum()

    diagnostic_rows.append({
        "pos": pos,
        "n_raw_extreme_score": n_raw_extreme,
        "n_actually_cleaned": n_actually_cleaned,
        "n_gap_missed": n_raw_extreme - n_actually_cleaned,  # indikasi H1/H2 terjadi
        "n_scale_inflated_near_extreme": n_scale_inflated_near_extreme,  # indikasi H1
        "n_failed_run_length": n_failed_run_length,  # indikasi H2
    })

diagnostic_df = pd.DataFrame(diagnostic_rows).sort_values("n_gap_missed", ascending=False)
display(diagnostic_df)

In [ ]:
# @title
scale_check_rows = []

for pos in top_pos_list:
    part = train_df[train_df[GROUP_COL] == pos].sort_values(DATETIME_COL).copy()
    x = pd.to_numeric(part[TARGET], errors="coerce")

    station_scale = np.nanmedian(np.abs(x - np.nanmedian(x))) * 1.4826
    diff_scale = np.nanmedian(np.abs(x.diff() - np.nanmedian(x.diff()))) * 1.4826
    scale_floor = max(station_scale * 0.02, diff_scale, 1e-6)

    audit_here = spike_audit[spike_audit[GROUP_COL] == pos]

    scale_check_rows.append({
        "pos": pos,
        "station_scale": round(station_scale, 5),
        "diff_scale": round(diff_scale, 5),
        "scale_floor": round(scale_floor, 5),
        "pct_nilai_unik": round(x.nunique() / x.notna().sum() * 100, 2),
        "avg_koreksi_abs": round(audit_here["koreksi_abs"].mean(), 5) if len(audit_here) else np.nan,
        "median_koreksi_abs": round(audit_here["koreksi_abs"].median(), 5) if len(audit_here) else np.nan,
        "max_koreksi_abs": round(audit_here["koreksi_abs"].max(), 5) if len(audit_here) else np.nan,
    })

scale_check_df = pd.DataFrame(scale_check_rows).sort_values("scale_floor")
display(scale_check_df)

In [ ]:
# @title
from __future__ import annotations

from collections.abc import Sequence
import numpy as np
import pandas as pd


def parse_year_selector(
    years: int | str | slice | Sequence[int],
) -> list[int]:
    """
    Mengubah berbagai format pemilihan tahun menjadi list tahun.

    Contoh
    -------
    2024
        -> [2024]

    [2024, 2025]
        -> [2024, 2025]

    "2023:2025"
        -> [2023, 2024, 2025]

    slice(2023, 2025)
        -> [2023, 2024, 2025]

    Catatan
    -------
    Batas akhir dibuat inklusif agar konsisten dengan slicing tanggal pandas.
    """
    if isinstance(years, (int, np.integer)):
        return [int(years)]

    if isinstance(years, str):
        value = years.strip()

        if ":" in value:
            parts = value.split(":")

            if len(parts) != 2:
                raise ValueError(
                    "Format rentang tahun harus seperti '2023:2025'."
                )

            start_year = int(parts[0])
            end_year = int(parts[1])

            if start_year > end_year:
                raise ValueError(
                    "Tahun awal tidak boleh lebih besar dari tahun akhir."
                )

            return list(range(start_year, end_year + 1))

        return [int(value)]

    if isinstance(years, slice):
        if years.start is None or years.stop is None:
            raise ValueError(
                "slice tahun harus memiliki start dan stop, "
                "misalnya slice(2023, 2025)."
            )

        start_year = int(years.start)
        end_year = int(years.stop)
        step = 1 if years.step is None else int(years.step)

        if step <= 0:
            raise ValueError("Step tahun harus lebih besar dari nol.")

        if start_year > end_year:
            raise ValueError(
                "Tahun awal tidak boleh lebih besar dari tahun akhir."
            )

        return list(range(start_year, end_year + 1, step))

    if isinstance(years, Sequence):
        selected_years = sorted({
            int(year)
            for year in years
        })

        if not selected_years:
            raise ValueError("Daftar tahun tidak boleh kosong.")

        return selected_years

    raise TypeError(
        "years harus berupa int, list, tuple, string rentang, atau slice."
    )

In [ ]:
# @title
# Pastikan datetime bertipe datetime
train_df_clean["datetime"] = pd.to_datetime(train_df["datetime"])

# Semua outlier yang ingin diperbaiki
outliers = [
    ("Bojonegoro - Kali Kethek", "2023-11-23 12:00:00"),
    ("Bojonegoro - Kali Kethek", "2023-11-28 06:00:00"),
    ('Bojonegoro - Kali Kethek',	'2023-09-14 06:00:00'),

    ('Arjowinangun - Pacitan','2024-03-09 18:00:00'),

    ('Babat','2025-09-08 12:00:00'),

    ('Colo Weir', '2024-05-08 06:00:00'),
    ('Colo Weir', '2024-07-12 06:00:00'),

    ("Floodway Bridge C", "2025-06-15 18:00:00"),

    ("Jarum", "2023-02-22 18:00:00"),
    ("Jarum", "2023-03-10 18:00:00"),
    ("Jarum", "2023-03-11 12:00:00"),
    ("Jarum", "2023-05-11 18:00:00"),
    ("Jarum", "2025-06-25 12:00:00"),

    ("Jurug", "2025-02-28 06:00:00"),

    ("Kajangan", "2023-08-02 12:00:00"),

    ("Kali Anyar - Kreteg Abang", "2023-06-13 18:00:00"),

    ("Kali Pepe - PTPN", "2023-05-24 06:00:00"),
    ("Kali Pepe - PTPN", "2023-11-04 18:00:00"),
    ("Kali Pepe - PTPN", "2023-12-18 12:00:00"),

    ("Karangnongko", "2023-05-30 12:00:00"),
    ("Karangnongko", "2023-06-09 18:00:00"),
    ("Karangnongko", "2023-06-29 18:00:00"),

    ("Kedungupit", "2023-10-25 18:00:00"),

    ("Ketonggo", "2023-10-11 18:00:00"),

    ("Napel", "2023-04-10 06:00:00"),

    ("Peren", "2023-11-09 18:00:00"),
    ("Peren", "2025-04-29 06:00:00"),
]

# Ubah ke datetime
outliers = [(s, pd.Timestamp(t)) for s, t in outliers]

# Tandai sebagai NaN
mask = pd.Series(False, index=train_df.index)

for station, dt in outliers:
    mask |= (
        (train_df["nama_pos"] == station) &
        (train_df["datetime"] == dt)
    )

print(f"Total outlier diperbaiki: {mask.sum()}")

train_df.loc[mask, "tma_mdpl"] = np.nan

# Interpolasi PER STASIUN
train_df_clean = train_df.sort_values(["nama_pos", "datetime"])

train_df_clean["tma_mdpl"] = (
    train_df_clean
    .groupby("nama_pos")["tma_mdpl"]
    .transform(lambda s: s.interpolate(method="linear"))
)


In [ ]:
# @title
# Memastikan kolom datetime dalam format yang benar
if 'datetime' not in train_df.columns or not pd.api.types.is_datetime64_any_dtype(train_df_clean['datetime']):
    train_df_clean['datetime'] = pd.to_datetime(train_df_clean['datetime'])

all_pos = train_df_clean['nama_pos'].unique()
plot_df = train_df_clean[train_df_clean['nama_pos'].isin(all_pos)].copy()
plot_df = plot_df.sort_values(['nama_pos', 'datetime'])
plot_df['n_day'] = plot_df.groupby('nama_pos').cumcount()

def calc_iqr_bounds(series, k=1.5):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return q1, q3, iqr, q1 - k * iqr, q3 + k * iqr

outlier_records = []

fig, axes = plt.subplots(len(all_pos), 1, figsize=(15, 6 * len(all_pos)))
if len(all_pos) == 1:
    axes = [axes]

for i, pos in enumerate(all_pos):
    data_pos = plot_df[plot_df['nama_pos'] == pos].copy()

    q1, q3, iqr, lower, upper = calc_iqr_bounds(data_pos['tma_mdpl'])
    data_min, data_max = data_pos['tma_mdpl'].min(), data_pos['tma_mdpl'].max()
    median_val = data_pos['tma_mdpl'].median()

    data_pos['is_outlier'] = (data_pos['tma_mdpl'] < lower) | (data_pos['tma_mdpl'] > upper)

    # Garis utama
    sns.lineplot(data=data_pos, x='n_day', y='tma_mdpl', ax=axes[i],
                 sort=False, lw=1.5, label='TMA', color='steelblue', zorder=2)

    # Range (min/max)
    axes[i].axhline(data_min, color='gray', linestyle='--', lw=0.8, alpha=0.6, label=f'Min ({data_min:.2f})')
    axes[i].axhline(data_max, color='gray', linestyle='--', lw=0.8, alpha=0.6, label=f'Max ({data_max:.2f})')

    # IQR: Q1, Q3, median
    axes[i].axhline(q1, color='orange', lw=1, alpha=0.8, label=f'Q1 ({q1:.2f})')
    axes[i].axhline(q3, color='orange', lw=1, alpha=0.8, label=f'Q3 ({q3:.2f})')
    axes[i].axhline(median_val, color='green', lw=1, alpha=0.8, label=f'Median ({median_val:.2f})')

    # Fence outlier (Q1-1.5*IQR, Q3+1.5*IQR)
    axes[i].axhline(lower, color='darkred', linestyle=':', lw=1.2, alpha=0.9, label=f'Lower Fence ({lower:.2f})')
    axes[i].axhline(upper, color='darkred', linestyle=':', lw=1.2, alpha=0.9, label=f'Upper Fence ({upper:.2f})')

    # Tandai titik outlier
    outliers = data_pos[data_pos['is_outlier']]
    if not outliers.empty:
        axes[i].scatter(outliers['n_day'], outliers['tma_mdpl'], color='red', s=30,
                         zorder=5, label=f'Outlier ({len(outliers)})')

    for _, row in outliers.iterrows():
        outlier_records.append({
            'nama_pos': pos,
            'datetime': row['datetime'],
            'n_day': row['n_day'],
            'tma_mdpl': row['tma_mdpl'],
            'lower_fence': lower,
            'upper_fence': upper,
            'tipe': 'Rendah (< Q1-1.5*IQR)' if row['tma_mdpl'] < lower else 'Tinggi (> Q3+1.5*IQR)'
        })

    # Garis batas tahun
    for year in [2024, 2025]:
        boundary = data_pos[data_pos['datetime'].dt.year == year]['n_day']
        if not boundary.empty:
            x_val = boundary.iloc[0]
            axes[i].axvline(x=x_val, color='black', linestyle='--', alpha=0.5)
            axes[i].text(x_val, data_max, f' Awal {year}', color='black', rotation=90, va='top', ha='right', fontsize=8)

    axes[i].set_title(f'Timeline TMA_MDPL + Range/IQR di Pos: {pos}  |  Outlier: {len(outliers)}', fontsize=13)
    axes[i].set_xlabel('Hari ke-n (Cumulative Days)')
    axes[i].set_ylabel('TMA (mdpl)')
    axes[i].grid(True, linestyle=':', alpha=0.5)
    axes[i].legend(loc='upper right', fontsize=8, ncol=2)

plt.tight_layout()
plt.show()

# ==== Tabel ringkasan rentang timeline outlier per pos ====
outlier_df = pd.DataFrame(outlier_records)

if not outlier_df.empty:
    summary_rows = []
    for pos, grp in outlier_df.groupby('nama_pos'):
        grp = grp.sort_values('datetime')
        summary_rows.append({
            'nama_pos': pos,
            'jumlah_outlier': len(grp),
            'tanggal_outlier_pertama': grp['datetime'].min().date(),
            'tanggal_outlier_terakhir': grp['datetime'].max().date(),
            'n_day_range': f"{int(grp['n_day'].min())}–{int(grp['n_day'].max())}",
            'nilai_min_outlier': round(grp['tma_mdpl'].min(), 3),
            'nilai_max_outlier': round(grp['tma_mdpl'].max(), 3),
        })
    outlier_summary_df = pd.DataFrame(summary_rows).sort_values('jumlah_outlier', ascending=False)

    print("=== Ringkasan Outlier per Pos ===")
    display(outlier_summary_df)

    print("\n=== Detail Semua Titik Outlier ===")
    display(outlier_df.sort_values(['nama_pos', 'datetime']).reset_index(drop=True))
else:
    print("Tidak ditemukan outlier berdasarkan metode IQR (k=1.5) di seluruh pos.")

### 3.3 Cleaning Spike Manual

In [ ]:
# @title
from dataclasses import dataclass


@dataclass
class SpikeDetectionResult:
    """Output pemeriksaan spike."""

    data: pd.DataFrame
    spikes: pd.DataFrame
    summary: pd.DataFrame


def detect_target_spikes(
    df: pd.DataFrame,
    station_name: str | None = None,
    years: int | str | slice | Sequence[int] | None = None,
    datetime_col: str = "datetime",
    station_col: str = "nama_pos",
    target_col: str = "tma_mdpl",
    rolling_window: int = 9,
    hampel_threshold: float = 4.0,
    jump_threshold: float = 4.0,
    reversal_ratio: float = 0.35,
    min_absolute_jump: float | None = None,
    require_reversal: bool = True,
) -> SpikeDetectionResult:
    """
    Mendeteksi kandidat spike pada target tanpa menghapus baris.

    Spike dideteksi menggunakan:
    1. deviasi terhadap median lokal;
    2. perubahan tajam terhadap observasi sebelumnya;
    3. pembalikan menuju level sebelumnya pada observasi berikutnya.

    Parameters
    ----------
    df:
        DataFrame sumber.

    station_name:
        Nama pos yang ingin diperiksa. Jika None, seluruh pos diperiksa.

    years:
        Pilihan tahun:
        - 2024
        - [2024, 2025]
        - "2023:2025"
        - slice(2023, 2025)
        - None untuk seluruh periode.

    rolling_window:
        Jumlah observasi untuk median dan MAD lokal.
        Sebaiknya bilangan ganjil.

    hampel_threshold:
        Ambang robust z-score terhadap median lokal.

    jump_threshold:
        Ambang robust z-score untuk perubahan nilai.

    reversal_ratio:
        Setelah spike, jarak titik berikutnya terhadap level sebelum spike
        harus kurang dari rasio ini dikali besar lonjakan.

    min_absolute_jump:
        Batas minimum perubahan target dalam satuan asli.
        Jika None, hanya threshold robust yang digunakan.

    require_reversal:
        Jika True, kandidat spike harus segera kembali mendekati level awal.
        Jika False, semua lonjakan ekstrem ikut ditandai.

    Returns
    -------
    SpikeDetectionResult:
        data:
            Data terpilih dengan kolom diagnostik.

        spikes:
            Hanya kandidat spike.

        summary:
            Jumlah spike per pos dan tahun.
    """
    required_columns = {
        datetime_col,
        station_col,
        target_col,
    }

    missing = required_columns - set(df.columns)
    if missing:
        raise KeyError(
            f"Kolom berikut tidak ditemukan: {sorted(missing)}"
        )

    if rolling_window < 5:
        raise ValueError(
            "rolling_window sebaiknya minimal 5 observasi."
        )

    if rolling_window % 2 == 0:
        raise ValueError(
            "rolling_window harus ganjil, misalnya 7, 9, atau 15."
        )

    result = df.copy()

    # Mempertahankan index asli untuk proses pemeriksaan manual.
    result["_original_index"] = result.index

    result[datetime_col] = pd.to_datetime(
        result[datetime_col],
        errors="coerce",
    )

    result[target_col] = pd.to_numeric(
        result[target_col],
        errors="coerce",
    )

    invalid_datetime = result[datetime_col].isna()
    if invalid_datetime.any():
        raise ValueError(
            f"Terdapat {int(invalid_datetime.sum())} datetime tidak valid."
        )

    if station_name is not None:
        available_stations = set(
            result[station_col].dropna().astype(str)
        )

        if station_name not in available_stations:
            similar = [
                value
                for value in sorted(available_stations)
                if station_name.lower() in value.lower()
                or value.lower() in station_name.lower()
            ][:10]

            raise ValueError(
                f"Pos '{station_name}' tidak ditemukan. "
                f"Kandidat serupa: {similar}"
            )

        result = result[
            result[station_col].astype(str).eq(station_name)
        ].copy()

    selected_years = parse_year_selector(years)

    if selected_years is not None:
        result = result[
            result[datetime_col].dt.year.isin(selected_years)
        ].copy()

    if result.empty:
        raise ValueError(
            "Data kosong setelah pemilihan pos dan tahun."
        )

    result = result.sort_values(
        [station_col, datetime_col]
    ).reset_index(drop=True)

    group = result.groupby(
        station_col,
        observed=True,
        sort=False,
    )

    # Nilai sebelum dan sesudah.
    result["target_previous"] = group[target_col].shift(1)
    result["target_next"] = group[target_col].shift(-1)

    # Perubahan satu langkah.
    result["delta_previous"] = (
        result[target_col] - result["target_previous"]
    )

    result["delta_next"] = (
        result["target_next"] - result[target_col]
    )

    result["absolute_jump"] = result["delta_previous"].abs()

    # Rolling median yang centered untuk melihat pola lokal.
    result["local_median"] = group[target_col].transform(
        lambda values: values.rolling(
            window=rolling_window,
            center=True,
            min_periods=max(3, rolling_window // 2),
        ).median()
    )

    absolute_deviation = (
        result[target_col] - result["local_median"]
    ).abs()

    # Median Absolute Deviation lokal.
    result["local_mad"] = group[target_col].transform(
        lambda values: values.rolling(
            window=rolling_window,
            center=True,
            min_periods=max(3, rolling_window // 2),
        ).apply(
            lambda window_values: np.nanmedian(
                np.abs(
                    window_values
                    - np.nanmedian(window_values)
                )
            ),
            raw=True,
        )
    )

    # Hindari pembagian nol pada seri yang sangat stabil.
    station_mad = group[target_col].transform(
        lambda values: np.nanmedian(
            np.abs(values - np.nanmedian(values))
        )
    )

    global_mad = float(
        np.nanmedian(
            np.abs(
                result[target_col]
                - np.nanmedian(result[target_col])
            )
        )
    )

    fallback_scale = (
        station_mad
        .replace(0, np.nan)
        .fillna(global_mad)
        .replace(0, 1e-6)
    )

    result["local_mad"] = (
        result["local_mad"]
        .replace(0, np.nan)
        .fillna(fallback_scale)
        .clip(lower=1e-6)
    )

    # Faktor 1.4826 membuat MAD kira-kira setara standar deviasi
    # pada distribusi normal.
    result["hampel_score"] = (
        absolute_deviation
        / (1.4826 * result["local_mad"])
    )

    # Skala robust untuk perubahan antar-observasi.
    delta_absolute = result["delta_previous"].abs()

    delta_scale = delta_absolute.groupby(
        result[station_col],
        observed=True,
    ).transform(
        lambda values: np.nanmedian(
            np.abs(values - np.nanmedian(values))
        )
    )

    delta_scale = (
        delta_scale
        .replace(0, np.nan)
        .fillna(
            delta_absolute.groupby(
                result[station_col],
                observed=True,
            ).transform("median")
        )
        .replace(0, np.nan)
        .fillna(1e-6)
        .clip(lower=1e-6)
    )

    result["jump_score"] = (
        result["absolute_jump"]
        / (1.4826 * delta_scale)
    )

    # Reversal: setelah lonjakan, data kembali mendekati nilai sebelumnya.
    distance_next_to_previous = (
        result["target_next"]
        - result["target_previous"]
    ).abs()

    result["reversal_ratio"] = (
        distance_next_to_previous
        / result["absolute_jump"].clip(lower=1e-6)
    )

    result["has_fast_reversal"] = (
        result["reversal_ratio"] <= reversal_ratio
    )

    # Lonjakan naik lalu turun, atau turun lalu naik.
    result["direction_reversed"] = (
        np.sign(result["delta_previous"])
        != np.sign(result["delta_next"])
    )

    result["is_local_extreme"] = (
        (
            result[target_col] > result["target_previous"]
        )
        & (
            result[target_col] > result["target_next"]
        )
    ) | (
        (
            result[target_col] < result["target_previous"]
        )
        & (
            result[target_col] < result["target_next"]
        )
    )

    hampel_condition = (
        result["hampel_score"] >= hampel_threshold
    )

    jump_condition = (
        result["jump_score"] >= jump_threshold
    )

    if min_absolute_jump is not None:
        jump_condition &= (
            result["absolute_jump"] >= min_absolute_jump
        )

    reversal_condition = (
        result["has_fast_reversal"]
        & result["direction_reversed"]
        & result["is_local_extreme"]
    )

    if require_reversal:
        result["is_spike_candidate"] = (
            hampel_condition
            & jump_condition
            & reversal_condition
        )
    else:
        result["is_spike_candidate"] = (
            hampel_condition
            & jump_condition
        )

    result["spike_direction"] = np.select(
        [
            result["is_spike_candidate"]
            & (result["delta_previous"] > 0),
            result["is_spike_candidate"]
            & (result["delta_previous"] < 0),
        ],
        [
            "upward_spike",
            "downward_spike",
        ],
        default="normal",
    )

    # Gabungan score hanya untuk ranking pemeriksaan.
    result["spike_score"] = (
        result["hampel_score"].fillna(0)
        + result["jump_score"].fillna(0)
        + np.where(
            result["has_fast_reversal"],
            2.0,
            0.0,
        )
    )

    spikes = (
        result[result["is_spike_candidate"]]
        .sort_values(
            "spike_score",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    if spikes.empty:
        summary = pd.DataFrame(
            columns=[
                station_col,
                "year",
                "n_spikes",
                "maximum_spike_score",
            ]
        )
    else:
        spikes["year"] = spikes[datetime_col].dt.year

        summary = (
            spikes.groupby(
                [station_col, "year"],
                observed=True,
                as_index=False,
            )
            .agg(
                n_spikes=(
                    "is_spike_candidate",
                    "size",
                ),
                maximum_spike_score=(
                    "spike_score",
                    "max",
                ),
                median_absolute_jump=(
                    "absolute_jump",
                    "median",
                ),
            )
            .sort_values(
                "n_spikes",
                ascending=False,
            )
        )

    return SpikeDetectionResult(
        data=result,
        spikes=spikes,
        summary=summary,
    )

import matplotlib.pyplot as plt


def plot_detected_spikes(
    spike_result: SpikeDetectionResult,
    datetime_col: str = "datetime",
    station_col: str = "nama_pos",
    target_col: str = "tma_mdpl",
    station_name: str | None = None,
):
    """
    Menampilkan target dan kandidat spike.
    """
    data = spike_result.data.copy()

    if station_name is not None:
        data = data[
            data[station_col].astype(str).eq(station_name)
        ].copy()

    available_stations = data[station_col].dropna().unique()

    if len(available_stations) != 1:
        raise ValueError(
            "Visualisasi harus menggunakan tepat satu pos. "
            "Isi parameter station_name."
        )

    spikes = data[data["is_spike_candidate"]]

    fig, ax = plt.subplots(figsize=(15, 6))

    ax.plot(
        data[datetime_col],
        data[target_col],
        linewidth=1,
        label="TMA aktual",
    )

    ax.plot(
        data[datetime_col],
        data["local_median"],
        linewidth=1.5,
        label="Median lokal",
    )

    ax.scatter(
        spikes[datetime_col],
        spikes[target_col],
        s=60,
        marker="x",
        label="Kandidat spike",
    )

    ax.set_title(
        f"Deteksi spike — {available_stations[0]}"
    )
    ax.set_xlabel("Waktu")
    ax.set_ylabel(target_col)
    ax.legend()
    ax.grid(alpha=0.2)

    fig.tight_layout()

    return fig, ax

In [ ]:
spike_result = detect_target_spikes(
    df=train_df_clean,
    station_name=None,
    years=None,
    datetime_col="datetime",
    station_col="nama_pos",
    target_col="tma_mdpl",
    rolling_window=9,
    hampel_threshold=4.0,
    jump_threshold=4.0,
    reversal_ratio=0.35,
    min_absolute_jump=0.5,
    require_reversal=True,
)

display(spike_result.summary)

In [ ]:
# Visualisasi Spike pada suatu pos

plot_detected_spikes(
    spike_result,
    station_name="Wonogiri Dam",
)

In [ ]:
# @title
def apply_manual_spike_removal(
    df: pd.DataFrame,
    reviewed_spikes: pd.DataFrame,
    decision_col: str = "manual_decision",
    index_col: str = "_original_index",
    remove_value: str = "remove",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Menghapus hanya baris yang secara manual diberi keputusan 'remove'.

    Returns
    -------
    cleaned_df:
        Data setelah penghapusan manual.

    removed_rows:
        Baris yang benar-benar dihapus.
    """
    required = {decision_col, index_col}

    missing = required - set(reviewed_spikes.columns)
    if missing:
        raise KeyError(
            f"Kolom review tidak ditemukan: {sorted(missing)}"
        )

    rows_to_remove = (
        reviewed_spikes.loc[
            reviewed_spikes[decision_col]
            .astype(str)
            .str.lower()
            .eq(remove_value.lower()),
            index_col,
        ]
        .dropna()
        .astype(int)
        .unique()
    )

    invalid_indices = set(rows_to_remove) - set(df.index)

    if invalid_indices:
        raise ValueError(
            "Sebagian index review tidak ditemukan pada DataFrame asli: "
            f"{sorted(invalid_indices)[:20]}"
        )

    removed_rows = df.loc[rows_to_remove].copy()

    cleaned_df = (
        df.drop(index=rows_to_remove)
        .copy()
    )

    print(f"Baris awal       : {len(df):,}")
    print(f"Baris dihapus    : {len(removed_rows):,}")
    print(f"Baris tersisa    : {len(cleaned_df):,}")

    return cleaned_df, removed_rows

In [ ]:
display(
    spike_result.spikes[
        [
            "_original_index",
            "datetime",
            "nama_pos",
            "target_previous",
            "tma_mdpl",
            "target_next",
            "absolute_jump",
            "hampel_score",
            "jump_score",
            "reversal_ratio",
            "spike_score",
        ]
    ]
)

In [ ]:
# @title
from dataclasses import dataclass

import numpy as np
import pandas as pd


@dataclass
class SpikeRemovalResult:
    """Hasil penghapusan kandidat spike."""

    cleaned_data: pd.DataFrame
    removed_spikes: pd.DataFrame
    audit: pd.DataFrame


def remove_detected_spikes(
    original_df: pd.DataFrame,
    spike_result,
    index_col: str = "_original_index",
    datetime_col: str = "datetime",
    station_col: str = "nama_pos",
    target_col: str = "tma_mdpl",
    reset_index: bool = True,
    require_all_indices_found: bool = True,
) -> SpikeRemovalResult:
    """
    Menghapus semua kandidat spike yang terdapat pada spike_result.spikes.

    Fungsi ini tidak mengubah original_df secara langsung. Hasil penghapusan
    dikembalikan sebagai DataFrame baru.

    Parameters
    ----------
    original_df:
        DataFrame asli yang sebelumnya digunakan pada detect_target_spikes().

    spike_result:
        Hasil dari fungsi detect_target_spikes().

    index_col:
        Kolom yang menyimpan index asli. Default: "_original_index".

    reset_index:
        Jika True, index cleaned_data dibuat ulang setelah penghapusan.

    require_all_indices_found:
        Jika True, fungsi menghasilkan error apabila ada index kandidat yang
        tidak ditemukan dalam original_df.

    Returns
    -------
    SpikeRemovalResult
        cleaned_data:
            DataFrame setelah seluruh kandidat spike dihapus.

        removed_spikes:
            Baris asli yang telah dihapus.

        audit:
            Informasi kandidat, score, dan alasan penghapusan.
    """
    if not isinstance(original_df, pd.DataFrame):
        raise TypeError("original_df harus berupa pandas DataFrame.")

    if not hasattr(spike_result, "spikes"):
        raise TypeError(
            "spike_result harus merupakan hasil detect_target_spikes() "
            "dan memiliki atribut .spikes."
        )

    spikes = spike_result.spikes.copy()

    if spikes.empty:
        print("Tidak ada kandidat spike. Tidak ada baris yang dihapus.")

        return SpikeRemovalResult(
            cleaned_data=original_df.copy(),
            removed_spikes=original_df.iloc[0:0].copy(),
            audit=pd.DataFrame(),
        )

    if index_col not in spikes.columns:
        raise KeyError(
            f"Kolom '{index_col}' tidak ditemukan pada spike_result.spikes."
        )

    candidate_indices = (
        pd.to_numeric(
            spikes[index_col],
            errors="coerce",
        )
        .dropna()
        .astype(int)
        .drop_duplicates()
        .tolist()
    )

    if not candidate_indices:
        raise ValueError(
            "Tidak ada index kandidat spike yang valid untuk dihapus."
        )

    original_indices = set(original_df.index)
    missing_indices = sorted(
        set(candidate_indices) - original_indices
    )

    if missing_indices and require_all_indices_found:
        raise ValueError(
            f"{len(missing_indices)} index kandidat tidak ditemukan dalam "
            f"original_df. Contoh: {missing_indices[:20]}\n"
            "Pastikan original_df belum di-reset index atau diubah setelah "
            "proses deteksi spike."
        )

    valid_indices = [
        index
        for index in candidate_indices
        if index in original_indices
    ]

    removed_spikes = (
        original_df.loc[valid_indices]
        .copy()
    )

    removed_spikes.insert(
        0,
        "_removed_original_index",
        removed_spikes.index,
    )

    cleaned_data = (
        original_df.drop(index=valid_indices)
        .copy()
    )

    audit_columns = [
        index_col,
        datetime_col,
        station_col,
        target_col,
        "target_previous",
        "target_next",
        "local_median",
        "local_mad",
        "absolute_jump",
        "hampel_score",
        "jump_score",
        "reversal_ratio",
        "spike_score",
        "spike_direction",
    ]

    available_audit_columns = [
        column
        for column in audit_columns
        if column in spikes.columns
    ]

    audit = (
        spikes[available_audit_columns]
        .copy()
    )

    audit["removal_action"] = "removed"
    audit["removal_reason"] = (
        "Detected spike satisfying configured statistical thresholds"
    )

    if reset_index:
        cleaned_data = cleaned_data.reset_index(drop=True)

    print("=" * 65)
    print("PENGHAPUSAN SPIKE SELESAI")
    print("=" * 65)
    print(f"Baris awal             : {len(original_df):,}")
    print(f"Kandidat spike         : {len(spikes):,}")
    print(f"Index unik dihapus      : {len(valid_indices):,}")
    print(f"Index tidak ditemukan   : {len(missing_indices):,}")
    print(f"Baris setelah cleaning  : {len(cleaned_data):,}")

    if station_col in audit.columns:
        print("\nJumlah spike yang dihapus per pos:")

        station_summary = (
            audit.groupby(
                station_col,
                observed=True,
            )
            .size()
            .sort_values(ascending=False)
        )

        print(station_summary.to_string())

    return SpikeRemovalResult(
        cleaned_data=cleaned_data,
        removed_spikes=removed_spikes,
        audit=audit,
    )

In [ ]:
removal_result = remove_detected_spikes(
    original_df=train_df_clean,
    spike_result=spike_result,
)

df_train_clean = removal_result.cleaned_data
display(removal_result.removed_spikes)

display(removal_result.audit)

Visualisasi Sebelum spike dihapus dan sesudah.

In [ ]:
# @title
import matplotlib.pyplot as plt


def plot_before_after_spike_removal(
    original_df: pd.DataFrame,
    cleaned_df: pd.DataFrame,
    removed_spikes: pd.DataFrame,
    station_name: str,
    years=None,
    datetime_col: str = "datetime",
    station_col: str = "nama_pos",
    target_col: str = "tma_mdpl",
    removed_index_col: str = "_removed_original_index",
):
    """
    Membandingkan time series sebelum dan sesudah kandidat spike dihapus.
    """
    original = original_df.copy()
    cleaned = cleaned_df.copy()
    removed = removed_spikes.copy()

    original[datetime_col] = pd.to_datetime(
        original[datetime_col],
        errors="coerce",
    )

    cleaned[datetime_col] = pd.to_datetime(
        cleaned[datetime_col],
        errors="coerce",
    )

    if datetime_col in removed.columns:
        removed[datetime_col] = pd.to_datetime(
            removed[datetime_col],
            errors="coerce",
        )

    original = original[
        original[station_col].astype(str).eq(str(station_name))
    ].copy()

    cleaned = cleaned[
        cleaned[station_col].astype(str).eq(str(station_name))
    ].copy()

    removed = removed[
        removed[station_col].astype(str).eq(str(station_name))
    ].copy()

    selected_years = parse_year_selector(years)

    if selected_years is not None:
        original = original[
            original[datetime_col].dt.year.isin(selected_years)
        ]

        cleaned = cleaned[
            cleaned[datetime_col].dt.year.isin(selected_years)
        ]

        removed = removed[
            removed[datetime_col].dt.year.isin(selected_years)
        ]

    if original.empty:
        raise ValueError(
            f"Tidak ada data untuk pos '{station_name}' "
            "pada periode yang dipilih."
        )

    fig, ax = plt.subplots(figsize=(16, 6))

    ax.plot(
        original[datetime_col],
        original[target_col],
        linewidth=1,
        alpha=0.55,
        label="Data asli",
    )

    ax.plot(
        cleaned[datetime_col],
        cleaned[target_col],
        linewidth=1.4,
        label="Data setelah spike dihapus",
    )

    if not removed.empty:
        ax.scatter(
            removed[datetime_col],
            removed[target_col],
            marker="x",
            s=110,
            linewidths=2.5,
            label="Spike yang dihapus",
            zorder=6,
        )

    ax.set_title(
        f"Sebelum dan Sesudah Penghapusan Spike — {station_name}"
    )
    ax.set_xlabel("Waktu")
    ax.set_ylabel(target_col)
    ax.legend()
    ax.grid(alpha=0.25)

    fig.tight_layout()

    return fig, ax

In [ ]:
plot_before_after_spike_removal(
    original_df=train_df_clean,
    cleaned_df=df_train_clean,
    removed_spikes=removal_result.removed_spikes,
    station_name = "Bojonegoro - Kali Kethek",
    years=[2023,2024,2025]
)

### 3.4 Pendeteksi Spike Tma Catboots (Opsional)

In [ ]:
# @title
# ======================================================================
# CATBOOST-BASED TARGET SPIKE DETECTION
# SATU CELL GOOGLE COLAB
# ======================================================================

# Instalasi package. Baris ini aman dijalankan di Google Colab.
!pip install -q catboost pandas numpy matplotlib scikit-learn

from __future__ import annotations

import gc
import math
import re
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Sequence, Union

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


# ======================================================================
# 1. KONFIGURASI GLOBAL
# ======================================================================

RANDOM_SEED = 42

YearSelector = Union[
    int,
    str,
    slice,
    Sequence[int],
    None,
]


@dataclass
class CatBoostSpikeResult:
    """
    Menyimpan seluruh hasil deteksi spike.

    Attributes
    ----------
    data:
        Data yang sudah memiliki prediksi CatBoost, residual,
        spike score, dan penanda spike.

    spikes:
        Hanya baris yang dideteksi sebagai kandidat spike.

    summary:
        Ringkasan jumlah spike per nama_pos dan tahun.

    fold_metrics:
        Evaluasi prediksi CatBoost pada setiap fold waktu.

    models:
        Daftar model CatBoost yang dilatih untuk setiap fold.

    feature_columns:
        Daftar fitur yang digunakan model.
    """

    data: pd.DataFrame
    spikes: pd.DataFrame
    summary: pd.DataFrame
    fold_metrics: pd.DataFrame
    models: list
    feature_columns: list[str]


# ======================================================================
# 2. PEMILIHAN TAHUN
# ======================================================================

def parse_year_selector(
    years: YearSelector,
) -> Optional[list[int]]:
    """
    Mengubah pilihan tahun menjadi list tahun.

    Format yang didukung
    --------------------
    None
        Seluruh tahun.

    2024
        Hanya tahun 2024.

    [2024, 2025]
        Tahun 2024 dan 2025.

    "2023:2025"
        Tahun 2023, 2024, dan 2025.

    slice(2023, 2025)
        Tahun 2023, 2024, dan 2025.

    Catatan
    -------
    Rentang akhir bersifat inklusif.
    """

    if years is None:
        return None

    if isinstance(years, (int, np.integer)):
        return [int(years)]

    if isinstance(years, str):
        value = years.strip()

        if not value:
            raise ValueError("Parameter years tidak boleh berupa string kosong.")

        if ":" not in value:
            return [int(value)]

        parts = value.split(":")

        if len(parts) != 2:
            raise ValueError(
                "Format rentang tahun harus seperti '2023:2025'."
            )

        start_year = int(parts[0].strip())
        end_year = int(parts[1].strip())

        if start_year > end_year:
            raise ValueError(
                "Tahun awal tidak boleh lebih besar daripada tahun akhir."
            )

        return list(range(start_year, end_year + 1))

    if isinstance(years, slice):
        if years.start is None or years.stop is None:
            raise ValueError(
                "Slice tahun harus mempunyai start dan stop, "
                "misalnya slice(2023, 2025)."
            )

        start_year = int(years.start)
        end_year = int(years.stop)
        step = 1 if years.step is None else int(years.step)

        if step <= 0:
            raise ValueError("Step tahun harus lebih besar dari nol.")

        if start_year > end_year:
            raise ValueError(
                "Tahun awal tidak boleh lebih besar daripada tahun akhir."
            )

        return list(range(start_year, end_year + 1, step))

    if isinstance(years, Sequence):
        selected_years = sorted({
            int(year)
            for year in years
        })

        if not selected_years:
            raise ValueError("Daftar tahun tidak boleh kosong.")

        return selected_years

    raise TypeError(
        "years harus berupa int, list, tuple, string rentang, slice, atau None."
    )


# ======================================================================
# 3. VALIDASI DAN STANDARDISASI DATA
# ======================================================================

def validate_spike_dataframe(
    df: pd.DataFrame,
    datetime_col: str,
    station_col: str,
    target_col: str,
) -> pd.DataFrame:
    """
    Memvalidasi kolom utama dan menstandarkan tipe data.
    """

    if not isinstance(df, pd.DataFrame):
        raise TypeError("df harus berupa pandas DataFrame.")

    required = {
        datetime_col,
        station_col,
        target_col,
    }

    missing = required - set(df.columns)

    if missing:
        raise KeyError(
            f"Kolom wajib tidak ditemukan: {sorted(missing)}"
        )

    data = df.copy()

    # Simpan index asli agar kandidat bisa ditemukan kembali
    # pada DataFrame awal.
    data["_original_index"] = data.index

    data[datetime_col] = pd.to_datetime(
        data[datetime_col],
        errors="coerce",
    )

    invalid_datetime = data[datetime_col].isna()

    if invalid_datetime.any():
        examples = data.loc[
            invalid_datetime,
            [datetime_col, station_col],
        ].head(10)

        raise ValueError(
            f"Terdapat {int(invalid_datetime.sum())} datetime tidak valid.\n"
            f"Contoh:\n{examples.to_string(index=False)}"
        )

    data[station_col] = (
        data[station_col]
        .astype("string")
        .str.strip()
    )

    data[target_col] = pd.to_numeric(
        data[target_col],
        errors="coerce",
    )

    if data[station_col].isna().any():
        raise ValueError(
            f"Terdapat {int(data[station_col].isna().sum())} nama pos kosong."
        )

    if data[target_col].notna().sum() == 0:
        raise ValueError(
            f"Kolom target '{target_col}' tidak memiliki nilai numerik."
        )

    duplicate_keys = data.duplicated(
        [station_col, datetime_col],
        keep=False,
    )

    if duplicate_keys.any():
        warnings.warn(
            f"Terdapat {int(duplicate_keys.sum())} baris dengan kombinasi "
            f"{station_col} dan {datetime_col} yang duplikat. "
            "Periksa apakah data perlu diagregasi terlebih dahulu."
        )

    data = (
        data.sort_values(
            [station_col, datetime_col]
        )
        .reset_index(drop=True)
    )

    return data


# ======================================================================
# 4. PEMBUATAN FITUR CAUSAL
# ======================================================================

def _grouped_rolling_feature(
    shifted_series: pd.Series,
    group_key: pd.Series,
    window: int,
    operation: str,
) -> pd.Series:
    """
    Helper rolling per pos.
    """

    minimum_periods = max(2, window // 3)

    rolling_object = (
        shifted_series
        .groupby(group_key, observed=True)
        .rolling(
            window=window,
            min_periods=minimum_periods,
        )
    )

    if operation == "median":
        values = rolling_object.median()
    elif operation == "mean":
        values = rolling_object.mean()
    elif operation == "std":
        values = rolling_object.std()
    elif operation == "min":
        values = rolling_object.min()
    elif operation == "max":
        values = rolling_object.max()
    else:
        raise ValueError(
            f"Rolling operation tidak dikenal: {operation}"
        )

    return (
        values
        .reset_index(level=0, drop=True)
        .sort_index()
    )


def build_catboost_spike_features(
    df: pd.DataFrame,
    datetime_col: str = "datetime",
    station_col: str = "nama_pos",
    target_col: str = "tma_mdpl",
    lag_steps: Sequence[int] = (1, 2, 3, 6, 9),
    rolling_windows: Sequence[int] = (3, 9, 21),
    include_existing_numeric_features: bool = True,
) -> tuple[pd.DataFrame, list[str]]:
    """
    Membuat fitur CatBoost yang aman untuk time series.

    Semua fitur yang berasal dari target menggunakan shift(1),
    sehingga target pada waktu sekarang tidak masuk ke fitur prediksinya.

    Fitur yang dibuat
    -----------------
    - Kalender.
    - Lag target.
    - Rolling median, mean, std, min, dan max.
    - Perubahan target masa lalu.
    - Semua fitur numerik lain yang sudah ada pada df, misalnya:
      rainfall, soil moisture, river order, atau embedding GAT.
    """

    data = validate_spike_dataframe(
        df=df,
        datetime_col=datetime_col,
        station_col=station_col,
        target_col=target_col,
    )

    # --------------------------------------------------------------
    # Fitur kalender
    # --------------------------------------------------------------

    data["calendar_year"] = data[datetime_col].dt.year
    data["month"] = data[datetime_col].dt.month
    data["day_of_year"] = data[datetime_col].dt.dayofyear
    data["day_of_month"] = data[datetime_col].dt.day
    data["day_of_week"] = data[datetime_col].dt.dayofweek
    data["hour"] = data[datetime_col].dt.hour

    data["day_sin"] = np.sin(
        2 * np.pi * data["day_of_year"] / 365.25
    ).astype("float32")

    data["day_cos"] = np.cos(
        2 * np.pi * data["day_of_year"] / 365.25
    ).astype("float32")

    data["hour_sin"] = np.sin(
        2 * np.pi * data["hour"] / 24
    ).astype("float32")

    data["hour_cos"] = np.cos(
        2 * np.pi * data["hour"] / 24
    ).astype("float32")

    station_group = data.groupby(
        station_col,
        observed=True,
        sort=False,
    )

    generated_target_features = []

    # --------------------------------------------------------------
    # Lag target
    # --------------------------------------------------------------

    for lag in lag_steps:
        if lag <= 0:
            raise ValueError("Semua lag harus lebih besar dari nol.")

        column = f"{target_col}_lag_{lag}"

        data[column] = station_group[target_col].shift(lag)

        generated_target_features.append(column)

    # --------------------------------------------------------------
    # Perubahan historis
    # --------------------------------------------------------------

    data[f"{target_col}_previous_change"] = (
        data[f"{target_col}_lag_1"]
        - data[f"{target_col}_lag_2"]
    )

    generated_target_features.append(
        f"{target_col}_previous_change"
    )

    # shift(1) wajib agar current target tidak bocor.
    shifted_target = station_group[target_col].shift(1)

    # --------------------------------------------------------------
    # Rolling historis
    # --------------------------------------------------------------

    for window in rolling_windows:
        if window < 2:
            raise ValueError(
                "Rolling window harus minimal 2 observasi."
            )

        for operation in [
            "median",
            "mean",
            "std",
            "min",
            "max",
        ]:
            column = (
                f"{target_col}_rolling_{operation}_{window}"
            )

            data[column] = _grouped_rolling_feature(
                shifted_series=shifted_target,
                group_key=data[station_col],
                window=window,
                operation=operation,
            )

            generated_target_features.append(column)

    # --------------------------------------------------------------
    # Pilih fitur
    # --------------------------------------------------------------

    ignored_columns = {
        datetime_col,
        target_col,
        "_original_index",
    }

    generated_calendar_features = [
        "calendar_year",
        "month",
        "day_of_year",
        "day_of_month",
        "day_of_week",
        "hour",
        "day_sin",
        "day_cos",
        "hour_sin",
        "hour_cos",
    ]

    if include_existing_numeric_features:
        numeric_features = [
            column
            for column in data.select_dtypes(
                include="number"
            ).columns
            if column not in ignored_columns
        ]
    else:
        numeric_features = (
            generated_calendar_features
            + generated_target_features
        )

    numeric_features = list(dict.fromkeys(
        numeric_features
    ))

    feature_columns = [
        station_col
    ] + numeric_features

    return data, feature_columns


# ======================================================================
# 5. TIME-SERIES FOLDS
# ======================================================================

def make_expanding_time_folds(
    df: pd.DataFrame,
    datetime_col: str = "datetime",
    n_folds: int = 5,
    min_train_fraction: float = 0.50,
) -> list[tuple[pd.Timestamp, pd.Timestamp, pd.Timestamp]]:
    """
    Membuat expanding-window time-series validation.

    Contoh
    -------
    Fold 1:
        Train periode awal sampai cutoff 1.
        Validation setelah cutoff 1.

    Fold 2:
        Train diperpanjang sampai cutoff 2.
        Validation setelah cutoff 2.
    """

    if n_folds < 1:
        raise ValueError("n_folds minimal 1.")

    if not 0.20 <= min_train_fraction < 1.0:
        raise ValueError(
            "min_train_fraction harus berada pada rentang 0.20 sampai < 1."
        )

    unique_times = np.array(
        sorted(
            pd.to_datetime(
                df[datetime_col],
                errors="coerce",
            )
            .dropna()
            .unique()
        )
    )

    if len(unique_times) < 30:
        raise ValueError(
            "Timestamp terlalu sedikit untuk expanding-window validation. "
            "Minimal disarankan 30 timestamp."
        )

    first_validation_index = int(
        len(unique_times) * min_train_fraction
    )

    remaining_count = (
        len(unique_times) - first_validation_index
    )

    fold_size = max(
        1,
        math.ceil(remaining_count / n_folds),
    )

    folds = []

    for fold_number in range(n_folds):
        validation_start_index = (
            first_validation_index
            + fold_number * fold_size
        )

        if validation_start_index >= len(unique_times):
            break

        validation_end_index = min(
            validation_start_index
            + fold_size
            - 1,
            len(unique_times) - 1,
        )

        train_end_index = (
            validation_start_index - 1
        )

        if train_end_index < 0:
            continue

        folds.append(
            (
                pd.Timestamp(
                    unique_times[train_end_index]
                ),
                pd.Timestamp(
                    unique_times[validation_start_index]
                ),
                pd.Timestamp(
                    unique_times[validation_end_index]
                ),
            )
        )

    if not folds:
        raise ValueError(
            "Tidak ada fold yang berhasil dibuat."
        )

    return folds


# ======================================================================
# 6. IMPUTASI FITUR TANPA MEMAKAI VALIDATION MEDIAN
# ======================================================================

def prepare_fold_features(
    train_data: pd.DataFrame,
    validation_data: pd.DataFrame,
    feature_columns: list[str],
    categorical_columns: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Mengisi nilai fitur kosong menggunakan statistik training fold saja.

    Ini mencegah leakage dari validation ke training.
    """

    train_x = train_data[feature_columns].copy()
    validation_x = validation_data[feature_columns].copy()

    numeric_columns = [
        column
        for column in feature_columns
        if column not in categorical_columns
    ]

    for column in categorical_columns:
        train_x[column] = (
            train_x[column]
            .astype("string")
            .fillna("UNKNOWN")
        )

        validation_x[column] = (
            validation_x[column]
            .astype("string")
            .fillna("UNKNOWN")
        )

    for column in numeric_columns:
        train_x[column] = pd.to_numeric(
            train_x[column],
            errors="coerce",
        ).replace(
            [np.inf, -np.inf],
            np.nan,
        )

        validation_x[column] = pd.to_numeric(
            validation_x[column],
            errors="coerce",
        ).replace(
            [np.inf, -np.inf],
            np.nan,
        )

        training_median = train_x[column].median()

        if pd.isna(training_median):
            training_median = 0.0

        train_x[column] = train_x[column].fillna(
            training_median
        )

        validation_x[column] = (
            validation_x[column]
            .fillna(training_median)
        )

    return train_x, validation_x


# ======================================================================
# 7. PREDIKSI OUT-OF-FOLD CATBOOST
# ======================================================================

def generate_catboost_oof_predictions(
    feature_df: pd.DataFrame,
    feature_columns: list[str],
    datetime_col: str = "datetime",
    station_col: str = "nama_pos",
    target_col: str = "tma_mdpl",
    n_folds: int = 5,
    min_train_fraction: float = 0.50,
    iterations: int = 700,
    depth: int = 7,
    learning_rate: float = 0.04,
    early_stopping_rounds: int = 80,
    task_type: str = "CPU",
    verbose: Union[int, bool] = 100,
    random_seed: int = RANDOM_SEED,
) -> tuple[pd.DataFrame, list, pd.DataFrame]:
    """
    Melatih CatBoost pada expanding time folds dan menghasilkan
    prediksi out-of-fold.

    Loss MAE digunakan agar model tidak terlalu dipengaruhi oleh spike.
    RMSE tetap dihitung sebagai metrik evaluasi.
    """

    data = feature_df.copy()

    if target_col not in data.columns:
        raise KeyError(
            f"Target '{target_col}' tidak ditemukan."
        )

    categorical_columns = [station_col]

    data["catboost_expected_tma"] = np.nan
    data["oof_fold"] = pd.NA

    folds = make_expanding_time_folds(
        df=data,
        datetime_col=datetime_col,
        n_folds=n_folds,
        min_train_fraction=min_train_fraction,
    )

    models = []
    metric_rows = []

    task_type = str(task_type).upper()

    if task_type not in {"CPU", "GPU"}:
        raise ValueError(
            "task_type harus 'CPU' atau 'GPU'."
        )

    for fold_number, (
        train_end,
        validation_start,
        validation_end,
    ) in enumerate(folds, start=1):

        train_mask = (
            data[datetime_col].le(train_end)
            & data[target_col].notna()
        )

        validation_mask = (
            data[datetime_col].between(
                validation_start,
                validation_end,
                inclusive="both",
            )
            & data[target_col].notna()
        )

        train_data = data.loc[train_mask].copy()
        validation_data = data.loc[
            validation_mask
        ].copy()

        if train_data.empty:
            warnings.warn(
                f"Fold {fold_number} dilewati karena training kosong."
            )
            continue

        if validation_data.empty:
            warnings.warn(
                f"Fold {fold_number} dilewati karena validation kosong."
            )
            continue

        train_x, validation_x = prepare_fold_features(
            train_data=train_data,
            validation_data=validation_data,
            feature_columns=feature_columns,
            categorical_columns=categorical_columns,
        )

        train_y = train_data[target_col]
        validation_y = validation_data[target_col]

        print("=" * 80)
        print(f"FOLD {fold_number}")
        print(
            f"Training      : <= {train_end}"
        )
        print(
            f"Validation    : {validation_start} sampai {validation_end}"
        )
        print(
            f"Jumlah train  : {len(train_data):,}"
        )
        print(
            f"Jumlah valid  : {len(validation_data):,}"
        )

        parameters = {
            "iterations": iterations,
            "depth": depth,
            "learning_rate": learning_rate,
            "loss_function": "MAE",
            "eval_metric": "RMSE",
            "l2_leaf_reg": 7,
            "random_strength": 0.5,
            "random_seed": random_seed + fold_number,
            "allow_writing_files": False,
            "verbose": verbose,
            "task_type": task_type,
        }

        model = CatBoostRegressor(
            **parameters
        )

        model.fit(
            train_x,
            train_y,
            cat_features=categorical_columns,
            eval_set=(
                validation_x,
                validation_y,
            ),
            use_best_model=True,
            early_stopping_rounds=early_stopping_rounds,
        )

        predictions = model.predict(
            validation_x
        )

        data.loc[
            validation_mask,
            "catboost_expected_tma",
        ] = predictions

        data.loc[
            validation_mask,
            "oof_fold",
        ] = fold_number

        rmse = float(
            np.sqrt(
                mean_squared_error(
                    validation_y,
                    predictions,
                )
            )
        )

        mae = float(
            mean_absolute_error(
                validation_y,
                predictions,
            )
        )

        bias = float(
            np.mean(
                predictions
                - validation_y.to_numpy()
            )
        )

        metric_rows.append({
            "fold": fold_number,
            "train_end": train_end,
            "validation_start": validation_start,
            "validation_end": validation_end,
            "train_rows": len(train_data),
            "validation_rows": len(validation_data),
            "RMSE": rmse,
            "MAE": mae,
            "bias": bias,
            "best_iteration": model.get_best_iteration(),
        })

        models.append(model)

        print(
            f"Fold RMSE     : {rmse:.6f}"
        )
        print(
            f"Fold MAE      : {mae:.6f}"
        )
        print(
            f"Fold bias     : {bias:.6f}"
        )

        gc.collect()

    fold_metrics = pd.DataFrame(
        metric_rows
    )

    if not fold_metrics.empty:
        print("=" * 80)
        print("RINGKASAN VALIDASI CATBOOST")
        print(
            fold_metrics[
                [
                    "fold",
                    "RMSE",
                    "MAE",
                    "bias",
                    "validation_rows",
                ]
            ].to_string(
                index=False,
                float_format=lambda value: f"{value:.6f}",
            )
        )

        print(
            "\nRata-rata RMSE:",
            f"{fold_metrics['RMSE'].mean():.6f}",
        )

        print(
            "Standar deviasi RMSE:",
            f"{fold_metrics['RMSE'].std(ddof=0):.6f}",
        )

    return data, models, fold_metrics


# ======================================================================
# 8. ROBUST RESIDUAL SCORE DAN DETEKSI SPIKE
# ======================================================================

def score_catboost_spikes(
    oof_df: pd.DataFrame,
    datetime_col: str = "datetime",
    station_col: str = "nama_pos",
    target_col: str = "tma_mdpl",
    prediction_col: str = "catboost_expected_tma",
    residual_threshold: float = 5.0,
    min_absolute_residual: float = 0.40,
    reversal_ratio_threshold: float = 0.35,
    require_fast_reversal: bool = True,
    minimum_mad: float = 1e-4,
) -> pd.DataFrame:
    """
    Menandai kandidat spike menggunakan residual CatBoost.

    Kondisi utama
    --------------
    1. Aktual jauh dari prediksi CatBoost.
    2. Residual ekstrem dibanding residual normal pos tersebut.
    3. Perubahan mempunyai nilai fisik minimum.
    4. Opsional: nilai setelah spike cepat kembali ke kondisi sebelumnya.
    """

    required = {
        datetime_col,
        station_col,
        target_col,
        prediction_col,
    }

    missing = required - set(oof_df.columns)

    if missing:
        raise KeyError(
            f"Kolom untuk scoring tidak lengkap: {sorted(missing)}"
        )

    result = (
        oof_df.sort_values(
            [station_col, datetime_col]
        )
        .copy()
    )

    # --------------------------------------------------------------
    # Residual model
    # --------------------------------------------------------------

    result["residual"] = (
        result[target_col]
        - result[prediction_col]
    )

    result["absolute_residual"] = (
        result["residual"].abs()
    )

    station_group = result.groupby(
        station_col,
        observed=True,
        sort=False,
    )

    # Median residual per pos.
    result["station_residual_median"] = (
        station_group["residual"]
        .transform("median")
    )

    # Median Absolute Deviation residual.
    result["station_residual_mad"] = (
        station_group["residual"]
        .transform(
            lambda values: np.nanmedian(
                np.abs(
                    values
                    - np.nanmedian(values)
                )
            )
        )
    )

    fallback_residual_scale = (
        station_group["absolute_residual"]
        .transform("median")
    )

    global_residual_mad = float(
        np.nanmedian(
            np.abs(
                result["residual"]
                - np.nanmedian(result["residual"])
            )
        )
    )

    if not np.isfinite(global_residual_mad):
        global_residual_mad = minimum_mad

    result["station_residual_mad"] = (
        result["station_residual_mad"]
        .replace(0, np.nan)
        .fillna(fallback_residual_scale)
        .replace(0, np.nan)
        .fillna(global_residual_mad)
        .replace(0, minimum_mad)
        .clip(lower=minimum_mad)
    )

    # Robust z-score residual.
    result["residual_robust_z"] = (
        np.abs(
            result["residual"]
            - result["station_residual_median"]
        )
        / (
            1.4826
            * result["station_residual_mad"]
        )
    )

    # --------------------------------------------------------------
    # Konteks sebelum-sesudah
    # --------------------------------------------------------------

    result["target_previous"] = (
        station_group[target_col]
        .shift(1)
    )

    result["target_next"] = (
        station_group[target_col]
        .shift(-1)
    )

    result["delta_previous"] = (
        result[target_col]
        - result["target_previous"]
    )

    result["delta_next"] = (
        result["target_next"]
        - result[target_col]
    )

    result["absolute_jump"] = (
        result["delta_previous"].abs()
    )

    result["distance_next_to_previous"] = (
        result["target_next"]
        - result["target_previous"]
    ).abs()

    # Seberapa dekat titik setelah spike kembali
    # ke kondisi sebelum spike.
    result["reversal_ratio"] = (
        result["distance_next_to_previous"]
        / result["absolute_jump"].clip(lower=1e-6)
    )

    result["has_fast_reversal"] = (
        result["reversal_ratio"]
        <= reversal_ratio_threshold
    )

    # Arah sebelum dan setelah titik harus berlawanan.
    result["direction_reversed"] = (
        np.sign(result["delta_previous"])
        != np.sign(result["delta_next"])
    )

    # Titik harus menjadi puncak atau lembah lokal.
    result["is_local_extreme"] = (
        (
            result[target_col]
            > result["target_previous"]
        )
        & (
            result[target_col]
            > result["target_next"]
        )
    ) | (
        (
            result[target_col]
            < result["target_previous"]
        )
        & (
            result[target_col]
            < result["target_next"]
        )
    )

    residual_condition = (
        result["residual_robust_z"]
        >= residual_threshold
    )

    physical_condition = (
        result["absolute_residual"]
        >= min_absolute_residual
    )

    oof_available_condition = (
        result[prediction_col].notna()
    )

    reversal_condition = (
        result["has_fast_reversal"]
        & result["direction_reversed"]
        & result["is_local_extreme"]
    )

    if require_fast_reversal:
        result["is_spike_candidate"] = (
            residual_condition
            & physical_condition
            & oof_available_condition
            & reversal_condition
        )
    else:
        result["is_spike_candidate"] = (
            residual_condition
            & physical_condition
            & oof_available_condition
        )

    result["spike_direction"] = np.select(
        [
            result["is_spike_candidate"]
            & result["residual"].gt(0),

            result["is_spike_candidate"]
            & result["residual"].lt(0),
        ],
        [
            "upward_spike",
            "downward_spike",
        ],
        default="normal",
    )

    # Score hanya untuk mengurutkan kandidat.
    result["catboost_spike_score"] = (
        result["residual_robust_z"]
        .fillna(0)
        + np.log1p(
            result["absolute_residual"]
            .fillna(0)
        )
        + np.where(
            result["has_fast_reversal"],
            2.0,
            0.0,
        )
    )

    return result


# ======================================================================
# 9. FUNGSI UTAMA END-TO-END
# ======================================================================

def detect_spikes_with_catboost(
    df: pd.DataFrame,
    years: YearSelector = None,
    station_names: Optional[Sequence[str]] = None,
    datetime_col: str = "datetime",
    station_col: str = "nama_pos",
    target_col: str = "tma_mdpl",
    lag_steps: Sequence[int] = (1, 2, 3, 6, 9),
    rolling_windows: Sequence[int] = (3, 9, 21),
    include_existing_numeric_features: bool = True,
    n_folds: int = 5,
    min_train_fraction: float = 0.50,
    iterations: int = 700,
    depth: int = 7,
    learning_rate: float = 0.04,
    early_stopping_rounds: int = 80,
    task_type: str = "CPU",
    residual_threshold: float = 5.0,
    min_absolute_residual: float = 0.40,
    reversal_ratio_threshold: float = 0.35,
    require_fast_reversal: bool = True,
    verbose: Union[int, bool] = 100,
) -> CatBoostSpikeResult:
    """
    Menjalankan seluruh proses deteksi spike.

    Penting
    -------
    years dan station_names hanya membatasi hasil review.

    Model tetap dilatih menggunakan seluruh data historis yang tersedia
    sebelum setiap validation fold.
    """

    print("Membangun fitur causal...")

    feature_df, feature_columns = (
        build_catboost_spike_features(
            df=df,
            datetime_col=datetime_col,
            station_col=station_col,
            target_col=target_col,
            lag_steps=lag_steps,
            rolling_windows=rolling_windows,
            include_existing_numeric_features=(
                include_existing_numeric_features
            ),
        )
    )

    print(
        f"Jumlah fitur CatBoost: {len(feature_columns)}"
    )

    print(
        f"Jumlah baris: {len(feature_df):,}"
    )

    print(
        f"Jumlah pos: "
        f"{feature_df[station_col].nunique():,}"
    )

    print("\nMelatih CatBoost dengan expanding time folds...")

    oof_df, models, fold_metrics = (
        generate_catboost_oof_predictions(
            feature_df=feature_df,
            feature_columns=feature_columns,
            datetime_col=datetime_col,
            station_col=station_col,
            target_col=target_col,
            n_folds=n_folds,
            min_train_fraction=min_train_fraction,
            iterations=iterations,
            depth=depth,
            learning_rate=learning_rate,
            early_stopping_rounds=early_stopping_rounds,
            task_type=task_type,
            verbose=verbose,
        )
    )

    print("\nMenghitung residual dan kandidat spike...")

    scored = score_catboost_spikes(
        oof_df=oof_df,
        datetime_col=datetime_col,
        station_col=station_col,
        target_col=target_col,
        residual_threshold=residual_threshold,
        min_absolute_residual=min_absolute_residual,
        reversal_ratio_threshold=(
            reversal_ratio_threshold
        ),
        require_fast_reversal=(
            require_fast_reversal
        ),
    )

    # --------------------------------------------------------------
    # Filter hasil, bukan filter training
    # --------------------------------------------------------------

    selected = scored.copy()

    selected_years = parse_year_selector(
        years
    )

    if selected_years is not None:
        selected = selected[
            selected[datetime_col]
            .dt.year
            .isin(selected_years)
        ].copy()

    if station_names is not None:
        requested_stations = [
            str(value).strip()
            for value in station_names
        ]

        available_stations = set(
            selected[station_col]
            .dropna()
            .astype(str)
        )

        missing_stations = (
            set(requested_stations)
            - available_stations
        )

        if missing_stations:
            warnings.warn(
                "Pos berikut tidak ditemukan pada hasil: "
                f"{sorted(missing_stations)}"
            )

        selected = selected[
            selected[station_col]
            .astype(str)
            .isin(requested_stations)
        ].copy()

    spikes = (
        selected[
            selected["is_spike_candidate"]
        ]
        .sort_values(
            "catboost_spike_score",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    if spikes.empty:
        summary = pd.DataFrame(
            columns=[
                station_col,
                "year",
                "n_spikes",
                "maximum_spike_score",
                "median_absolute_residual",
                "median_absolute_jump",
                "minimum_reversal_ratio",
            ]
        )

    else:
        spikes["year"] = (
            spikes[datetime_col].dt.year
        )

        summary = (
            spikes.groupby(
                [station_col, "year"],
                observed=True,
                as_index=False,
            )
            .agg(
                n_spikes=(
                    "is_spike_candidate",
                    "size",
                ),
                maximum_spike_score=(
                    "catboost_spike_score",
                    "max",
                ),
                median_absolute_residual=(
                    "absolute_residual",
                    "median",
                ),
                median_absolute_jump=(
                    "absolute_jump",
                    "median",
                ),
                minimum_reversal_ratio=(
                    "reversal_ratio",
                    "min",
                ),
            )
            .sort_values(
                [
                    "n_spikes",
                    "maximum_spike_score",
                ],
                ascending=[
                    False,
                    False,
                ],
            )
            .reset_index(drop=True)
        )

    print("=" * 80)
    print("HASIL DETEKSI")
    print(
        f"Jumlah kandidat spike: {len(spikes):,}"
    )

    if not summary.empty:
        print(
            summary.head(30).to_string(
                index=False,
                float_format=lambda value: f"{value:.6f}",
            )
        )
    else:
        print(
            "Tidak ditemukan kandidat spike "
            "dengan konfigurasi saat ini."
        )

    return CatBoostSpikeResult(
        data=selected,
        spikes=spikes,
        summary=summary,
        fold_metrics=fold_metrics,
        models=models,
        feature_columns=feature_columns,
    )


# ======================================================================
# 10. VISUALISASI SATU POS
# ======================================================================

def plot_catboost_spikes_for_station(
    result: CatBoostSpikeResult,
    station_name: str,
    datetime_col: str = "datetime",
    station_col: str = "nama_pos",
    target_col: str = "tma_mdpl",
    prediction_col: str = "catboost_expected_tma",
    figsize: tuple[int, int] = (17, 6),
):
    """
    Menampilkan seluruh periode satu pos.

    Garis utama:
    - Aktual TMA.
    - Expected TMA dari CatBoost.
    - Kandidat spike.
    """

    station_data = (
        result.data[
            result.data[station_col]
            .astype(str)
            .eq(str(station_name))
        ]
        .sort_values(datetime_col)
        .copy()
    )

    if station_data.empty:
        available = sorted(
            result.data[station_col]
            .dropna()
            .astype(str)
            .unique()
        )[:20]

        raise ValueError(
            f"Pos '{station_name}' tidak ditemukan. "
            f"Contoh pos tersedia: {available}"
        )

    spikes = station_data[
        station_data["is_spike_candidate"]
    ]

    fig, ax = plt.subplots(
        figsize=figsize
    )

    ax.plot(
        station_data[datetime_col],
        station_data[target_col],
        linewidth=1.0,
        label="TMA aktual",
    )

    ax.plot(
        station_data[datetime_col],
        station_data[prediction_col],
        linewidth=1.5,
        alpha=0.85,
        label="Expected TMA CatBoost",
    )

    ax.scatter(
        spikes[datetime_col],
        spikes[target_col],
        marker="x",
        s=100,
        linewidths=2.5,
        label="Kandidat spike",
        zorder=6,
    )

    # Garis vertikal kecil antara expected dan actual.
    for _, row in spikes.iterrows():
        if pd.notna(row[prediction_col]):
            ax.plot(
                [
                    row[datetime_col],
                    row[datetime_col],
                ],
                [
                    row[prediction_col],
                    row[target_col],
                ],
                linestyle="--",
                linewidth=1,
                alpha=0.55,
            )

    ax.set_title(
        f"Deteksi Spike CatBoost — {station_name}\n"
        f"Jumlah kandidat: {len(spikes)}"
    )

    ax.set_xlabel("Waktu")
    ax.set_ylabel(target_col)
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()

    return fig, ax


# ======================================================================
# 11. VISUALISASI KONTEKS SATU SPIKE
# ======================================================================

def plot_catboost_spike_context(
    result: CatBoostSpikeResult,
    spike_number: int = 0,
    context_points: int = 15,
    datetime_col: str = "datetime",
    station_col: str = "nama_pos",
    target_col: str = "tma_mdpl",
    prediction_col: str = "catboost_expected_tma",
    figsize: tuple[int, int] = (15, 6),
):
    """
    Menampilkan beberapa titik sebelum dan sesudah satu kandidat spike.

    spike_number menggunakan index hasil result.spikes:
    0 untuk spike pertama,
    1 untuk spike kedua,
    dan seterusnya.
    """

    if result.spikes.empty:
        raise ValueError(
            "Tidak ada kandidat spike untuk divisualisasikan."
        )

    if not 0 <= spike_number < len(result.spikes):
        raise IndexError(
            f"spike_number harus berada di antara "
            f"0 dan {len(result.spikes) - 1}."
        )

    if context_points < 1:
        raise ValueError(
            "context_points minimal 1."
        )

    selected_spike = (
        result.spikes.iloc[spike_number]
    )

    station_name = (
        selected_spike[station_col]
    )

    original_index = (
        selected_spike["_original_index"]
    )

    station_data = (
        result.data[
            result.data[station_col]
            .astype(str)
            .eq(str(station_name))
        ]
        .sort_values(datetime_col)
        .reset_index(drop=True)
    )

    matched_positions = station_data.index[
        station_data["_original_index"]
        .eq(original_index)
    ]

    if len(matched_positions) == 0:
        raise ValueError(
            "Spike tidak ditemukan kembali pada data hasil."
        )

    position = int(
        matched_positions[0]
    )

    start = max(
        0,
        position - context_points,
    )

    end = min(
        len(station_data),
        position + context_points + 1,
    )

    context = (
        station_data.iloc[start:end]
        .copy()
    )

    spike_row = (
        station_data.iloc[position]
    )

    fig, ax = plt.subplots(
        figsize=figsize
    )

    ax.plot(
        context[datetime_col],
        context[target_col],
        marker="o",
        markersize=4,
        linewidth=1.1,
        label="TMA aktual",
    )

    ax.plot(
        context[datetime_col],
        context[prediction_col],
        marker=".",
        markersize=5,
        linewidth=1.5,
        label="Expected TMA CatBoost",
    )

    ax.scatter(
        [spike_row[datetime_col]],
        [spike_row[target_col]],
        marker="x",
        s=150,
        linewidths=3,
        label="Kandidat spike",
        zorder=7,
    )

    ax.axvline(
        spike_row[datetime_col],
        linestyle="--",
        alpha=0.6,
    )

    ax.set_title(
        f"{station_name} — {spike_row[datetime_col]}\n"
        f"Aktual={spike_row[target_col]:.4f} | "
        f"Expected={spike_row[prediction_col]:.4f} | "
        f"Residual={spike_row['residual']:.4f} | "
        f"Robust Z={spike_row['residual_robust_z']:.2f} | "
        f"Reversal={spike_row['reversal_ratio']:.3f}"
    )

    ax.set_xlabel("Waktu")
    ax.set_ylabel(target_col)
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()

    return context, fig, ax


# ======================================================================
# 12. VISUALISASI SCORE PER SPIKE
# ======================================================================

def plot_spike_score_ranking(
    result: CatBoostSpikeResult,
    top_n: int = 30,
    station_col: str = "nama_pos",
    datetime_col: str = "datetime",
    figsize: tuple[int, int] = (12, 7),
):
    """
    Menampilkan kandidat dengan spike score terbesar.
    """

    if result.spikes.empty:
        raise ValueError(
            "Tidak ada kandidat spike."
        )

    top_spikes = (
        result.spikes
        .nlargest(
            top_n,
            "catboost_spike_score",
        )
        .copy()
    )

    top_spikes["label"] = (
        top_spikes[station_col].astype(str)
        + " | "
        + top_spikes[datetime_col]
        .dt.strftime("%Y-%m-%d %H:%M")
    )

    top_spikes = top_spikes.sort_values(
        "catboost_spike_score",
        ascending=True,
    )

    fig, ax = plt.subplots(
        figsize=figsize
    )

    ax.barh(
        top_spikes["label"],
        top_spikes["catboost_spike_score"],
    )

    ax.set_title(
        f"{min(top_n, len(top_spikes))} Kandidat Spike dengan Score Terbesar"
    )

    ax.set_xlabel(
        "CatBoost spike score"
    )

    ax.set_ylabel(
        "Pos dan waktu"
    )

    ax.grid(
        axis="x",
        alpha=0.25,
    )

    fig.tight_layout()

    return fig, ax


# ======================================================================
# 13. MENYIMPAN SEMUA GRAFIK POS
# ======================================================================

def save_all_station_spike_plots(
    result: CatBoostSpikeResult,
    output_dir: str = "catboost_spike_plots",
    datetime_col: str = "datetime",
    station_col: str = "nama_pos",
    target_col: str = "tma_mdpl",
    dpi: int = 160,
) -> list[str]:
    """
    Menyimpan satu grafik untuk setiap pos yang mempunyai kandidat spike.
    """

    output_path = Path(
        output_dir
    )

    output_path.mkdir(
        parents=True,
        exist_ok=True,
    )

    stations = sorted(
        result.spikes[station_col]
        .dropna()
        .astype(str)
        .unique()
    )

    saved_paths = []

    for station_name in stations:
        fig, _ = plot_catboost_spikes_for_station(
            result=result,
            station_name=station_name,
            datetime_col=datetime_col,
            station_col=station_col,
            target_col=target_col,
        )

        safe_name = re.sub(
            r"[^A-Za-z0-9_-]+",
            "_",
            station_name,
        ).strip("_")

        file_path = (
            output_path
            / f"{safe_name}_catboost_spikes.png"
        )

        fig.savefig(
            file_path,
            dpi=dpi,
            bbox_inches="tight",
        )

        plt.close(fig)

        saved_paths.append(
            str(file_path)
        )

    print(
        f"{len(saved_paths)} grafik tersimpan di: "
        f"{output_path.resolve()}"
    )

    return saved_paths


# ======================================================================
# 14. EXPORT FILE REVIEW MANUAL
# ======================================================================

def export_spike_review(
    result: CatBoostSpikeResult,
    output_file: str = "catboost_spike_manual_review.csv",
    datetime_col: str = "datetime",
    station_col: str = "nama_pos",
    target_col: str = "tma_mdpl",
) -> pd.DataFrame:
    """
    Menyimpan kandidat spike ke CSV untuk diperiksa manusia.

    Kolom manual_decision dapat diisi:
    - keep
    - replace
    - remove
    - review
    """

    review_columns = [
        "_original_index",
        datetime_col,
        station_col,
        "oof_fold",
        "target_previous",
        target_col,
        "catboost_expected_tma",
        "target_next",
        "residual",
        "absolute_residual",
        "station_residual_median",
        "station_residual_mad",
        "residual_robust_z",
        "delta_previous",
        "delta_next",
        "absolute_jump",
        "distance_next_to_previous",
        "reversal_ratio",
        "has_fast_reversal",
        "is_local_extreme",
        "catboost_spike_score",
        "spike_direction",
    ]

    available_columns = [
        column
        for column in review_columns
        if column in result.spikes.columns
    ]

    review = (
        result.spikes[
            available_columns
        ]
        .copy()
    )

    review["manual_decision"] = "review"
    review["manual_note"] = ""

    output_path = Path(
        output_file
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    review.to_csv(
        output_path,
        index=False,
    )

    print(
        f"File review tersimpan di: "
        f"{output_path.resolve()}"
    )

    return review


# ======================================================================
# 15. MENERAPKAN KEPUTUSAN MANUAL
# ======================================================================

def apply_manual_spike_decisions(
    original_df: pd.DataFrame,
    reviewed_spikes: pd.DataFrame,
    target_col: str = "tma_mdpl",
    original_index_col: str = "_original_index",
    decision_col: str = "manual_decision",
    replacement_col: str = "catboost_expected_tma",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Menerapkan keputusan manual.

    Keputusan
    ---------
    keep
        Nilai tetap seperti data asli.

    replace
        Target diganti dengan catboost_expected_tma.

    remove
        Baris dihapus.

    review
        Belum dilakukan perubahan.

    Returns
    -------
    cleaned_df:
        DataFrame setelah keputusan manual.

    audit:
        Catatan baris yang diganti atau dihapus.
    """

    required = {
        original_index_col,
        decision_col,
    }

    missing = required - set(
        reviewed_spikes.columns
    )

    if missing:
        raise KeyError(
            f"Kolom review tidak lengkap: {sorted(missing)}"
        )

    cleaned = original_df.copy()

    review = reviewed_spikes.copy()

    review[decision_col] = (
        review[decision_col]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    valid_decisions = {
        "keep",
        "replace",
        "remove",
        "review",
    }

    invalid_decisions = set(
        review[decision_col].unique()
    ) - valid_decisions

    if invalid_decisions:
        raise ValueError(
            "Keputusan manual tidak valid: "
            f"{sorted(invalid_decisions)}"
        )

    replace_rows = review[
        review[decision_col].eq("replace")
    ].copy()

    remove_rows = review[
        review[decision_col].eq("remove")
    ].copy()

    audit_rows = []

    # --------------------------------------------------------------
    # Replace
    # --------------------------------------------------------------

    if not replace_rows.empty:
        if replacement_col not in replace_rows.columns:
            raise KeyError(
                f"Kolom replacement '{replacement_col}' "
                "tidak ditemukan."
            )

        for _, row in replace_rows.iterrows():
            original_index = int(
                row[original_index_col]
            )

            if original_index not in cleaned.index:
                warnings.warn(
                    f"Index {original_index} tidak ditemukan; "
                    "replacement dilewati."
                )
                continue

            old_value = cleaned.at[
                original_index,
                target_col,
            ]

            new_value = row[
                replacement_col
            ]

            if pd.isna(new_value):
                warnings.warn(
                    f"Expected value kosong pada index "
                    f"{original_index}; replacement dilewati."
                )
                continue

            cleaned.at[
                original_index,
                target_col,
            ] = new_value

            audit_rows.append({
                "original_index": original_index,
                "decision": "replace",
                "old_value": old_value,
                "new_value": new_value,
            })

    # --------------------------------------------------------------
    # Remove
    # --------------------------------------------------------------

    indices_to_remove = []

    if not remove_rows.empty:
        for _, row in remove_rows.iterrows():
            original_index = int(
                row[original_index_col]
            )

            if original_index not in cleaned.index:
                warnings.warn(
                    f"Index {original_index} tidak ditemukan; "
                    "penghapusan dilewati."
                )
                continue

            old_value = cleaned.at[
                original_index,
                target_col,
            ]

            indices_to_remove.append(
                original_index
            )

            audit_rows.append({
                "original_index": original_index,
                "decision": "remove",
                "old_value": old_value,
                "new_value": np.nan,
            })

    if indices_to_remove:
        cleaned = cleaned.drop(
            index=indices_to_remove
        )

    audit = pd.DataFrame(
        audit_rows
    )

    print("=" * 60)
    print(f"Baris awal      : {len(original_df):,}")
    print(f"Baris diganti   : {len(replace_rows):,}")
    print(f"Baris dihapus   : {len(indices_to_remove):,}")
    print(f"Baris akhir     : {len(cleaned):,}")

    return cleaned, audit


# ======================================================================
# 16. CONTOH PENGGUNAAN
# ======================================================================

print("""
==============================================================================
FUNGSI BERHASIL DIDEFINISIKAN
==============================================================================

Contoh penggunaan dasar:

result_spike = detect_spikes_with_catboost(
    df=df_model,
    years="2024:2025",
    station_names=None,
    datetime_col="datetime",
    station_col="nama_pos",
    target_col="tma_mdpl",
    n_folds=5,
    min_train_fraction=0.50,
    iterations=700,
    depth=7,
    learning_rate=0.04,
    task_type="CPU",
    residual_threshold=5.0,
    min_absolute_residual=0.40,
    reversal_ratio_threshold=0.35,
    require_fast_reversal=True,
)

Melihat ringkasan:
display(result_spike.summary)

Melihat kandidat:
display(result_spike.spikes)

Visualisasi Pos Babat:
plot_catboost_spikes_for_station(
    result_spike,
    station_name="Babat",
)

Visualisasi kandidat spike pertama:
context, fig, ax = plot_catboost_spike_context(
    result_spike,
    spike_number=0,
    context_points=15,
)

Ranking spike:
plot_spike_score_ranking(
    result_spike,
    top_n=30,
)

Menyimpan seluruh gambar:
saved_plots = save_all_station_spike_plots(
    result_spike,
    output_dir="/content/catboost_spike_plots",
)

Mengekspor tabel review:
review_df = export_spike_review(
    result_spike,
    output_file="/content/catboost_spike_manual_review.csv",
)

Setelah manual_decision diedit dan file diunggah kembali:

reviewed_df = pd.read_csv(
    "/content/catboost_spike_manual_review_edited.csv"
)

df_model_clean, audit = apply_manual_spike_decisions(
    original_df=df_model,
    reviewed_spikes=reviewed_df,
    target_col="tma_mdpl",
)

==============================================================================
""")

In [ ]:
df_model = df_train_clean.copy()

df_model["datetime"] = pd.to_datetime(
    df_model["datetime"]
)

display(df_model.head())
print(df_model.shape)
print(df_model.columns.tolist())

In [ ]:
df_lingkungan = datling_df.copy()

df_lingkungan["datetime"] = pd.to_datetime(
    df_lingkungan["datetime"]
)

df_model = df_model.merge(
    df_lingkungan,
    on=[
        "nama_pos",
        "datetime",
    ],
    how="left",
    validate="many_to_one",
)

print(df_model.shape)

In [ ]:
result_spike = detect_spikes_with_catboost(
    df=df_model,
    years=None,
    station_names="Bojonegoro - Kali Kethek",
)

display(result_spike.summary)

In [ ]:
plot_catboost_spikes_for_station(
    result_spike,
    station_name=None  ,
)

### 3.5 Hydrology Feature Engineering


In [ ]:
def add_hydrology_features(environment):
    """Add causal rainfall-memory, soil-state, and weather-tendency features.

    The environment is hourly, so every rolling feature below uses only the
    current and earlier rows. This is important: a centered or forward window
    would leak future weather into historical validation rows.
    """
    environment = environment.copy()
    environment = environment.sort_values(["nama_pos", "datetime"])
    station = environment["nama_pos"]

    # The documentation shows these two rainfall columns are duplicates. Drop
    # one only when that is true in the supplied frame.
    if {"rainfall_mm", "rainfall_openmeteo_mm"}.issubset(environment.columns):
        left = environment["rainfall_mm"].to_numpy()
        right = environment["rainfall_openmeteo_mm"].to_numpy()

        if np.allclose(left, right, equal_nan=True):
            environment = environment.drop(columns="rainfall_openmeteo_mm")

    if "rainfall_mm" in environment.columns:
        rain = environment["rainfall_mm"].clip(lower=0).fillna(0)

        # Antecedent rainfall at several physical response times.
        for days in [1, 3, 7, 14, 30]:
            hours = 24 * days
            environment[f"rain_sum_{days}d"] = rain.groupby(station).transform(
                lambda values: values.rolling(hours, min_periods=1).sum()
            )

        for days in [1, 3, 7]:
            hours = 24 * days
            environment[f"rain_max_{days}d"] = rain.groupby(station).transform(
                lambda values: values.rolling(hours, min_periods=1).max()
            )

        wet = rain.gt(0.1).astype(float)

        for days in [3, 7]:
            hours = 24 * days
            environment[f"wet_fraction_{days}d"] = wet.groupby(station).transform(
                lambda values: values.rolling(hours, min_periods=1).mean()
            )

        # API is a decaying rainfall memory. Convert the daily retention to an
        # hourly retention because df_lingkungan has one row per hour.
        for retention in [0.70, 0.85, 0.93]:
            hourly_retention = retention ** (1 / 24)
            alpha = 1 - hourly_retention
            name = f"api_{int(retention * 100):03d}"
            environment[name] = rain.groupby(station).transform(
                lambda values: values.ewm(alpha=alpha, adjust=False).mean()
            )

        # Number of hourly observations since meaningful rain.
        rain_event = rain.gt(0.1)
        event_number = rain_event.groupby(station).cumsum()
        environment["hours_since_rain"] = (~rain_event).groupby(
            [station, event_number]
        ).cumsum().clip(upper=24 * 60)

    soil_columns = [
        column
        for column in [
            "soil_moisture_0_7cm",
            "soil_moisture_7_28cm",
            "soil_moisture_28_100cm",
            "soil_moisture_100_255cm",
        ]
        if column in environment.columns
    ]

    if soil_columns:
        shallow = soil_columns[:2]
        deep = soil_columns[2:]
        environment["soil_shallow_mean"] = environment[shallow].mean(axis=1)
        environment["soil_deep_mean"] = (
            environment[deep].mean(axis=1)
            if deep
            else environment[soil_columns].mean(axis=1)
        )

        # Rain on saturated ground behaves differently from rain on dry soil.
        interaction_soils = ["soil_shallow_mean", "soil_deep_mean"]

        for rain_feature in ["api_085", "rain_sum_3d", "rain_sum_7d"]:
            if rain_feature in environment.columns:
                for soil_feature in interaction_soils:
                    name = rain_feature + "_x_" + soil_feature
                    environment[name] = (
                        environment[rain_feature] * environment[soil_feature]
                    )

    if {"temperature_c", "dew_point_c"}.issubset(environment.columns):
        environment["dewpoint_depression"] = (
            environment["temperature_c"] - environment["dew_point_c"]
        )

    if {"cloud_cover_pct", "humidity_pct"}.issubset(environment.columns):
        environment["cloud_x_humidity"] = (
            environment["cloud_cover_pct"] * environment["humidity_pct"] / 100
        )

    # One- and three-day tendencies capture changing atmospheric and soil state.
    tendency_columns = [
        column
        for column in soil_columns
        + [
            "humidity_pct",
            "temperature_c",
            "surface_pressure_hpa",
            "pressure_msl_hpa",
            "cloud_cover_pct",
            "dew_point_c",
        ]
        if column in environment.columns
    ]

    for column in tendency_columns:
        for days in [1, 3]:
            lag = 24 * days
            environment[f"{column}_change_{days}d"] = (
                environment[column]
                - environment.groupby("nama_pos")[column].shift(lag)
            ).fillna(0)

    return environment

datlingv2_df = add_hydrology_features(datling_df)

### 3.6 Penggabungan Train dengan Fitur Pendukung 📲


In [ ]:
def build_panel(train, test, environment):
    # Pastikan test memiliki kolom datetime dan nama_pos untuk proses grid
    test = test.copy()
    if "datetime" not in test.columns and "id" in test.columns:
        test[["datetime", "nama_pos"]] = test["id"].str.split(" - ", n=1, expand=True)
        test["datetime"] = pd.to_datetime(test["datetime"])

    first_time = train["datetime"].min()
    train_end = train["datetime"].max()

    # Recreate the expected training timetable
    target_hours = sorted(test["datetime"].dt.hour.unique())
    training_days = pd.date_range(
        first_time.normalize(),
        train_end.normalize(),
        freq="D",
    )

    training_times = []
    for day in training_days:
        for hour in target_hours:
            timestamp = day + pd.Timedelta(hours=int(hour))
            if first_time <= timestamp <= train_end:
                training_times.append(timestamp)

    all_times = sorted(set(training_times) | set(test["datetime"]))
    stations = sorted(test["nama_pos"].unique())

    grid = pd.MultiIndex.from_product(
        [stations, all_times],
        names=["nama_pos", "datetime"],
    ).to_frame(index=False)

    panel = grid.merge(
        environment,
        on=["nama_pos", "datetime"],
        how="left",
        indicator="_environment_merge",
    )
    panel["environment_row_missing"] = panel["_environment_merge"].eq(
        "left_only"
    ).astype(int)
    panel = panel.drop(columns="_environment_merge")

    environment_numeric = [
        column
        for column in environment.select_dtypes(include="number").columns
        if column in panel.columns
    ]

    for column in environment_numeric:
        if panel[column].isna().any():
            panel[column] = panel.groupby("nama_pos")[column].transform(
                lambda values: values.interpolate(limit_direction="both")
            )
            panel[column] = panel[column].fillna(panel[column].median())

    panel = panel.merge(
        train[["datetime", "nama_pos", "tma_mdpl"]],
        on=["datetime", "nama_pos"],
        how="left",
    )

    panel = panel.sort_values(["nama_pos", "datetime"]).reset_index(drop=True)

    all_times_series = pd.Series(panel["datetime"].sort_values().unique())
    time_to_step = pd.Series(np.arange(len(all_times_series)), index=all_times_series)
    panel["step"] = panel["datetime"].map(time_to_step)

    return panel

# Eksekusi fungsi dengan data yang sudah dibersihkan
merged_df = build_panel(train_df_clean, test_df, datling_df)
print(f"Shape merged_df: {merged_df.shape}")
merged_df.head()

In [ ]:
def check_missing_values(df):
    total_missing = df.isnull().sum()
    if total_missing.sum() == 0:
        print("Tidak ada missing value pada df ini")
        return
    percent_missing = (df.isnull().sum() / len(df)) * 100
    missing_data = pd.DataFrame({
        "Total Missing": total_missing,
        "Percentage (%)": percent_missing.round(5)
    })
    display(missing_data[missing_data["Total Missing"] > 0])


check_missing_values(merged_df)


### 3.7 Feature Engineering


#### 1. Fitur Waktu (Cyclical Encoding)

Supaya model tahu jam 23 dekat dengan jam 0, dan Desember dekat dengan Januari.

In [ ]:
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    dt = df[DATETIME_COL]

    df["hour"] = dt.dt.hour
    df["dayofweek"] = dt.dt.dayofweek
    df["day"] = dt.dt.day
    df["month"] = dt.dt.month
    df["dayofyear"] = dt.dt.dayofyear
    df["year"] = dt.dt.year

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)
    df["doy_sin"] = np.sin(2 * np.pi * df["dayofyear"] / 365.25)
    df["doy_cos"] = np.cos(2 * np.pi * df["dayofyear"] / 365.25)
    return df


merged_df = add_time_features(merged_df)


#### 2. Handle Missing Value pada Fitur Eksogen

Kolom lingkungan (`rainfall`, `soil_moisture`, `mjo`, `nino_34`, dst) kadang NaN
karena beda resolusi/sumber data. Interpolasi waktu **per pos** jauh lebih
masuk akal daripada isi rata-rata global.

In [ ]:
def fill_exogenous_missing(df: pd.DataFrame, exclude_cols) -> pd.DataFrame:
    df = df.copy()
    num_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                if c not in exclude_cols]

    df[num_cols] = (
        df.groupby(GROUP_COL)[num_cols]
        .apply(lambda g: g.interpolate(method="linear").ffill().bfill())
        .reset_index(level=0, drop=True)
    )
    return df


exclude_from_fill = [TARGET, "latitude", "longitude"]
merged_df = fill_exogenous_missing(merged_df, exclude_cols=exclude_from_fill)


#### 3. Lag & Rolling Features (per `nama_pos`)

Histori TMA & curah hujan beberapa langkah sebelumnya biasanya jadi fitur
paling berpengaruh. `shift(1)` diterapkan dulu sebelum rolling supaya rolling
tidak "mengintip" nilai saat ini (mencegah data leakage).

In [ ]:
def add_lag_and_rolling(df: pd.DataFrame, lag_cols) -> pd.DataFrame:
    df = df.copy()

    for col in lag_cols:
        grouped = df.groupby(GROUP_COL)[col]

        for lag in LAG_LIST:
            df[f"{col}_lag_{lag}"] = grouped.shift(lag)

        shifted = grouped.shift(1)
        for window in ROLLING_WINDOWS:
            roll = shifted.groupby(df[GROUP_COL]).rolling(window, min_periods=1)
            df[f"{col}_roll_mean_{window}"] = roll.mean().reset_index(level=0, drop=True)
            df[f"{col}_roll_std_{window}"] = roll.std().reset_index(level=0, drop=True)

        df[f"{col}_diff_1"] = grouped.diff(1)

    return df


lag_cols = [TARGET] + [c for c in EXTRA_LAG_COLS if c in merged_df.columns]
merged_df = add_lag_and_rolling(merged_df, lag_cols)
merged_df.head()


#### 4. Encoding Kategorikal (`nama_pos`, `landcover_name`)

In [ ]:
def encode_categoricals(df: pd.DataFrame):
    df = df.copy()
    encoders = {}

    for col in [GROUP_COL, "landcover_name"]:
        if col in df.columns:
            le = LabelEncoder()
            df[f"{col}_encoded"] = le.fit_transform(df[col].astype(str))
            encoders[col] = le

    return df, encoders


merged_df, encoders = encode_categoricals(merged_df)
merged_df.head()

#### 5. Feature Engineering lainnya... *ikut pipeline teman:P

In [ ]:
def fit_climatology(panel, cutoff):
    # Re-calculate month and hour just in case they are missing from build_panel
    panel = panel.copy()
    panel['month'] = panel['datetime'].dt.month
    panel['hour'] = panel['datetime'].dt.hour

    history = panel[
        panel["datetime"].le(cutoff) & panel["tma_mdpl"].notna()
    ].copy()

    # Monthly station-hour medians are stable and easy to estimate
    climatology = (
        history.groupby(["nama_pos", "month", "hour"], as_index=False)["tma_mdpl"]
        .median()
        .rename(columns={"tma_mdpl": "climatology"})
    )

    station_median = history.groupby("nama_pos")["tma_mdpl"].median()
    return climatology, station_median


def apply_climatology(frame, climatology, station_median):
    frame = frame.copy()
    # Ensure month/hour exist
    if 'month' not in frame.columns: frame['month'] = frame['datetime'].dt.month
    if 'hour' not in frame.columns: frame['hour'] = frame['datetime'].dt.hour

    frame = frame.drop(columns=["climatology", "anomaly"], errors="ignore")
    frame = frame.merge(climatology, on=["nama_pos", "month", "hour"], how="left")

    # Fallback to station median then 0 if specific month/hour combo is missing
    frame["climatology"] = frame["climatology"].fillna(
        frame["nama_pos"].map(station_median)
    ).fillna(0)

    frame["anomaly"] = frame["tma_mdpl"] - frame["climatology"]
    return frame

# --- Eksekusi pada Pipeline ---
# Gunakan data training terakhir sebagai batas perhitungan statistik
if 'merged_df' in globals() and 'train_df_clean' in globals():
    val_cutoff = train_df_clean['datetime'].max()
    print(f"Fitting climatology statistics up to: {val_cutoff}")

    climatology_df, station_median_series = fit_climatology(merged_df, val_cutoff)
    merged_df = apply_climatology(merged_df, climatology_df, station_median_series)

    print("Kolom 'climatology' dan 'anomaly' berhasil ditambahkan ke merged_df.")
    display(merged_df[['datetime', 'nama_pos', 'tma_mdpl', 'climatology', 'anomaly']].head())
else:
    print("Pastikan 'merged_df' dan 'train_df_clean' sudah tersedia di memory.")

### Training Untuk Anomaly Target

In [ ]:
def create_anomaly_target(df):
    """
    Daripada ngeprediksi tma_mdpl mentah mentah gitu aja, kita prediksi anomaly: selisih dari tiap tiap rata rata (mean) stasiun

    Physics:
      tma_mdpl = station_baseline(lat, lon, elevation_proxy) + anomaly(weather, climate)

    Jadi nanti model hanya liat dan mempelajari anomalinya aja, untuk mengeneralisir terhadap semua statisun
    """
    df = df.copy()

    # Statistik Per-station
    station_stats = df.groupby('nama_pos')['tma_mdpl'].agg(
        tma_mean='mean',
        tma_std='std',
        tma_median='median',
        tma_q25=lambda x: x.quantile(0.25),
        tma_q75=lambda x: x.quantile(0.75)
    ).reset_index()

    # Ambil info spasial tiap stasiun
    if 'latitude' in df.columns and 'longitude' in df.columns:
        station_spatial = df.groupby('nama_pos').agg(
            latitude=('latitude', 'first'),
            longitude=('longitude', 'first')
        ).reset_index()
        station_stats = station_stats.merge(station_spatial, on='nama_pos')

    # Gabungin
    df = df.merge(station_stats[['nama_pos', 'tma_mean', 'tma_std', 'tma_median']],
                  on='nama_pos', how='left')

    # Buat target anomali (zero-centered per station)
    df['tma_anomaly'] = df['tma_mdpl'] - df['tma_mean']

    # anomali yang udah dinormalized (z-score per station) — biar nanti pas belajar cross-station gampaing dibandingkan
    df['tma_anomaly_normalized'] = df['tma_anomaly'] / (df['tma_std'] + 1e-6)

    return df, station_stats

def get_model(model_name, random_state=42):
    """
    Factory model — objective disamakan (huber-like) biar perbandingan adil.
    """
    if model_name == 'lgbm':
        return lgb.LGBMRegressor(
            objective='huber', metric='mae', boosting_type='gbdt',
            n_estimators=3000, learning_rate=0.02, num_leaves=63,
            max_depth=7, min_child_samples=30, subsample=0.75,
            colsample_bytree=0.75, reg_alpha=0.5, reg_lambda=2.0,
            random_state=random_state, verbose=-1
        )
    elif model_name == 'xgboost':
        return xgb.XGBRegressor(
            objective='reg:pseudohubererror', eval_metric='mae',
            n_estimators=3000, learning_rate=0.02, max_depth=7,
            min_child_weight=30, subsample=0.75, colsample_bytree=0.75,
            reg_alpha=0.5, reg_lambda=2.0, random_state=random_state,
            early_stopping_rounds=100, verbosity=0
        )
    else:
        raise ValueError(f"model_name tidak dikenal: {model_name}")


def fit_model(model_name, model, X_train, y_train, X_test, y_test, cat_features):
    """
    Setiap library punya API early stopping & categorical yang beda,
    jadi logic fit dipisah di sini.
    """
    if model_name == 'lgbm':
        model.fit(
            X_train, y_train,
            eval_set=[(X_test, y_test)],
            callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
            categorical_feature=cat_features
        )
    elif model_name == 'xgboost':
        # xgboost butuh category dtype juga, tapi enable_categorical di set X saja
        model.set_params(enable_categorical=True)
        model.fit(
            X_train, y_train,
            eval_set=[(X_test, y_test)],
            verbose=False
        )
    return model


def evaluate_temporal_hybrid_multi(df, models=('lgbm', 'xgboost'),
                                    include_station_options=(False, True),
                                    test_frac=0.2, random_state=42):
    """
    Sama seperti evaluate_temporal_hybrid, tapi loop untuk beberapa model
    sekaligus. Mengembalikan DataFrame ringkasan biar gampang dibandingkan.

    PERHATIAN (leakage rekursif):
    lag/rolling features (tma_anomaly_lag*, dst) di sini dihitung dari nilai
    AKTUAL sepanjang timeline (df digabung dulu baru displit). Ini cocok untuk
    "one-step-ahead eval" tapi TIDAK merepresentasikan skenario deployment
    rekursifmu di Kaggle, di mana lag h+2 dst berasal dari prediksi model
    sendiri, bukan nilai aktual. Kalau mau angka yang comparable dengan skor
    submission Kaggle, evaluasi ini perlu diganti jadi walk-forward rekursif
    (prediksi 1 langkah -> dipakai sebagai lag utk langkah berikutnya).
    """
    df_processed, station_stats = create_anomaly_target(df)
    df_processed = add_time_features(df_processed)
    df_processed = df_processed.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)

    target = 'tma_anomaly'
    drop_cols = [
        'datetime', 'tma_mdpl', 'tma_anomaly', 'tma_anomaly_normalized',
        'tma_mean', 'tma_std', 'tma_median', 'tma_scaled', 'landcover_name',
    ]

    results = []

    for include_station in include_station_options:
        label_station = "WITH nama_pos" if include_station else "WITHOUT nama_pos"

        _drop = drop_cols.copy()
        if not include_station:
            _drop.append('nama_pos')
        _drop = [c for c in _drop if c in df_processed.columns]

        valid_mask = df_processed[target].notna()
        df_valid = df_processed[valid_mask].reset_index(drop=True)

        X_all = df_valid.drop(columns=_drop, errors='ignore')
        y_all = df_valid[target]

        cat_features = ['mjo_phase', 'landcover_class', 'month']
        if include_station:
            cat_features.append('nama_pos')
        cat_features = [c for c in cat_features if c in X_all.columns]

        # Fix: Convert 'mjo_phase' to int before converting to category for XGBoost compatibility
        if 'mjo_phase' in cat_features and (X_all['mjo_phase'].dtype == float or X_all['mjo_phase'].dtype == np.float64):
            X_all['mjo_phase'] = X_all['mjo_phase'].astype(int)

        for c in cat_features:
            X_all[c] = X_all[c].astype('category')

        # Split temporal per stasiun
        train_idx, test_idx = [], []
        for station in df_valid['nama_pos'].unique():
            s_idx = df_valid.index[df_valid['nama_pos'] == station].tolist()
            split = int(len(s_idx) * (1 - test_frac))
            train_idx.extend(s_idx[:split])
            test_idx.extend(s_idx[split:])

        X_train, X_test = X_all.iloc[train_idx], X_all.iloc[test_idx]
        y_train, y_test = y_all.iloc[train_idx], y_all.iloc[test_idx]

        for model_name in models:
            model = get_model(model_name, random_state=random_state)
            model = fit_model(model_name, model, X_train, y_train, X_test, y_test, cat_features)

            pred_anomaly = model.predict(X_test)

            test_means = df_valid.iloc[test_idx]['tma_mean'].values
            pred_raw = test_means + pred_anomaly
            actual_raw = df_valid.iloc[test_idx]['tma_mdpl'].values

            mae = mean_absolute_error(actual_raw, pred_raw)
            rmse = np.sqrt(mean_squared_error(actual_raw, pred_raw))
            r2 = r2_score(actual_raw, pred_raw)

            anomaly_mae = mean_absolute_error(y_test, pred_anomaly)
            anomaly_r2 = r2_score(y_test, pred_anomaly)

            print(f"\n[{model_name}] Validasi Temporal ({label_station}):")
            print(f"  Mentah  → MAE: {mae:.4f}, RMSE: {rmse:.4f}, R²: {r2:.4f}")
            print(f"  Anomali → MAE: {anomaly_mae:.4f}, R²: {anomaly_r2:.4f}")

            results.append({
                'model': model_name,
                'station_feature': label_station,
                'mae_raw': mae, 'rmse_raw': rmse, 'r2_raw': r2,
                'mae_anomaly': anomaly_mae, 'r2_anomaly': anomaly_r2,
            })

    results_df = pd.DataFrame(results).sort_values('rmse_raw').reset_index(drop=True)
    return results_df


# Contoh pemakaian:
results_df = evaluate_temporal_hybrid_multi(train_df, models=('lgbm', 'xgboost'), test_frac=0.2)
results_df

### 3.9 Konfigurasi Preprocessing dan Ensemble Terpadu untuk Eksperimen 3–5

Seluruh eksperimen lanjutan menggunakan konfigurasi dasar yang sama. Perbedaan eksperimen hanya pada encoder spasial dan bobot ensemble. Bobot dapat diedit langsung pada `EXPERIMENT_WEIGHT_CONFIG`.

- **1–7 hari:** lebih berat ke model neural.
- **8–30 hari:** lebih berat ke model seasonal.
- **31+ hari:** lebih berat ke CatBoost.


In [ ]:
from copy import deepcopy

GLOBAL_MODEL_CONFIG = {
    "random_seed": 42,
    "device": "auto",
    "catboost_task_type": "CPU",
    "catboost_iterations": 800,
    "catboost_depth": 7,
    "neural_model": "nhits",
    "input_days": 120,
    "weight_mode": "manual",  # ganti menjadi "optimize" untuk grid search
    "weight_step": 0.05,
    "ensemble_objective": "balanced",
    "validation_mode": "same_season",
    "quantiles": [0.10, 0.50, 0.90],
    "anomaly_lower_quantile": 0.10,
    "anomaly_upper_quantile": 0.90,
}

EXPERIMENT_WEIGHT_CONFIG = {
    "experiment_3": {
        "1-7 days":   {"catboost": 0.20, "neural": 0.60, "seasonal": 0.20},
        "8-30 days":  {"catboost": 0.20, "neural": 0.25, "seasonal": 0.55},
        "31-90 days": {"catboost": 0.60, "neural": 0.20, "seasonal": 0.20},
        "91+ days":   {"catboost": 0.65, "neural": 0.15, "seasonal": 0.20},
    },
    "experiment_4": {
        "1-7 days":   {"catboost": 0.20, "neural": 0.55, "seasonal": 0.25},
        "8-30 days":  {"catboost": 0.20, "neural": 0.20, "seasonal": 0.60},
        "31-90 days": {"catboost": 0.55, "neural": 0.25, "seasonal": 0.20},
        "91+ days":   {"catboost": 0.65, "neural": 0.15, "seasonal": 0.20},
    },
    "experiment_5": {
        "1-7 days":   {"catboost": 0.20, "neural": 0.60, "seasonal": 0.20},
        "8-30 days":  {"catboost": 0.25, "neural": 0.20, "seasonal": 0.55},
        "31-90 days": {"catboost": 0.65, "neural": 0.20, "seasonal": 0.15},
        "91+ days":   {"catboost": 0.70, "neural": 0.15, "seasonal": 0.15},
    },
}

def validate_horizon_weights(weight_config):
    required_bands = {"1-7 days", "8-30 days", "31-90 days", "91+ days"}
    required_models = {"catboost", "neural", "seasonal"}
    missing_bands = required_bands.difference(weight_config)
    if missing_bands:
        raise ValueError(f"Horizon band belum lengkap: {sorted(missing_bands)}")
    for band, weights in weight_config.items():
        if set(weights) != required_models:
            raise ValueError(f"{band}: model harus {sorted(required_models)}")
        if any(float(v) < 0 for v in weights.values()):
            raise ValueError(f"{band}: bobot tidak boleh negatif.")
        total = sum(float(v) for v in weights.values())
        if not np.isclose(total, 1.0, atol=1e-8):
            raise ValueError(f"{band}: total bobot harus 1.0, saat ini {total:.6f}")
    return True

def build_experiment_config(experiment_name, **overrides):
    if experiment_name not in EXPERIMENT_WEIGHT_CONFIG:
        raise KeyError(f"Eksperimen tidak dikenal: {experiment_name}")
    weights = deepcopy(EXPERIMENT_WEIGHT_CONFIG[experiment_name])
    validate_horizon_weights(weights)
    config = deepcopy(GLOBAL_MODEL_CONFIG)
    config.update({
        "experiment_name": experiment_name,
        "manual_horizon_weights": weights,
    })
    config.update(overrides)
    return config

for _name, _weights in EXPERIMENT_WEIGHT_CONFIG.items():
    validate_horizon_weights(_weights)

print("Konfigurasi global dan horizon weights eksperimen 3–5 siap.")


## 4. Validasi Time Series dan Split Data 🔁


**Kenapa diganti dari `expanding_window_split`:** versi sebelumnya train selalu
membesar (expanding) dan val selalu fix 30 hari. Versi ini pakai **rolling
window** murni (ukuran train TETAP, digeser maju tiap fold) supaya CV lebih
mirip kondisi nyata (model selalu dilatih dari histori dengan panjang yang
sama, bukan histori yang terus menumpuk).

**Catatan satuan penting:** resolusi observasi asli tetap 3x/hari (06/12/18 →
1 baris = 6 jam, lihat catatan di `LAG_LIST`). Jadi:
- **Train digeser maju 1 baris per fold** — inilah granularitas paling halus
  yang tersedia di data ini, dirujuk sebagai "1 jam" pada permintaan, meski
  literal 1 baris ≈ 6 jam asli.
- **Val horizon bertambah bertahap dari 1 hingga 6 baris ke depan** antar
  fold — fold awal cuma diuji 1 langkah ke depan (paling gampang), fold
  terakhir diuji sampai 6 langkah ke depan (mendekati kondisi submission
  yang recursive). Ini yang dimaksud "rentang 6 jam" — bukan literal 6 jam
  kalender, tapi 6 baris/step observasi.

Val dievaluasi dengan **simulasi recursive forecast sungguhan** (bukan
one-step predict biasa) — lag & rolling di val dihitung dari prediksi model
sendiri, sama persis seperti nanti dinilai Kaggle. Fungsi recursive-nya
(`build_static_features`, `recursive_forecast_eval`) dipindah lebih awal
supaya bisa dipakai langsung di dalam CV ini.


#### Fungsi Pendukung: Fitur Statis & Simulasi Recursive Forecast

In [ ]:
def sanitize_feature_names(df: pd.DataFrame) -> pd.DataFrame:
    """LightGBM menolak nama kolom dengan karakter non alfanumerik."""
    df = df.copy()
    df.columns = [re.sub(r"[^0-9a-zA-Z_]+", "_", str(c)) for c in df.columns]
    return df


def build_static_features(df_raw, koor_df, datling_df, encoders):
    """Fitur yang TIDAK butuh rekursi: waktu, spasial, dan eksogen."""
    df = merge_all(df_raw, koor_df, datling_df, env_how="left")
    df = add_time_features(df)

    df = fill_exogenous_missing(df, exclude_cols=[TARGET, "latitude", "longitude"])

    if "landcover_name" in df.columns:
        df = df.sort_values([GROUP_COL, DATETIME_COL])
        df["landcover_name"] = df.groupby(GROUP_COL)["landcover_name"].transform(
            lambda s: s.ffill().bfill()
        )

    df = add_lag_and_rolling(df, [c for c in EXTRA_LAG_COLS if c in df.columns])
    for col, le in encoders.items():
        if col in df.columns:
            df[f"{col}_encoded"] = le.transform(df[col].astype(str))
    return df


def recursive_forecast_eval(history_df, target_df, koor_df, datling_df, model,
                             feature_cols, encoders, pos_bounds, is_lgb=False):
    """
    Simulasi recursive forecast dengan pengaman clipping per stasiun.
    history_df : data yang datetime-nya SEBELUM target_df (histori asli/diketahui).
    target_df  : data yang ingin diprediksi & dievaluasi (punya kolom TARGET asli).
    pos_bounds : DataFrame index nama_pos dengan kolom [\'min\',\'max\'] (rentang historis).
    """
    target_df = target_df.sort_values([GROUP_COL, DATETIME_COL]).reset_index(drop=True)
    max_hist = max(LAG_LIST + ROLLING_WINDOWS) + 5

    tail_per_pos = history_df.sort_values(DATETIME_COL).groupby(GROUP_COL).tail(max_hist)

    combined_raw = pd.concat([
        tail_per_pos[[DATETIME_COL, GROUP_COL, TARGET]],
        target_df[[DATETIME_COL, GROUP_COL]].assign(**{TARGET: np.nan})
    ], ignore_index=True)

    static_df = build_static_features(combined_raw, koor_df, datling_df, encoders)
    static_df = static_df.sort_values([GROUP_COL, DATETIME_COL]).reset_index(drop=True)
    if is_lgb:
        static_df = sanitize_feature_names(static_df)

    # Kolom kategorikal LightGBM diturunkan dari `encoders`, bukan variabel
    # global -- supaya fungsi ini bisa dipanggil berkali-kali di dalam CV
    # tanpa bergantung urutan eksekusi cell.
    cat_cols_here = [f"{c}_encoded" for c in encoders.keys()]
    if is_lgb:
        cat_cols_here = [re.sub(r"[^0-9a-zA-Z_]+", "_", c) for c in cat_cols_here]
    cat_cols_here = [c for c in cat_cols_here if c in static_df.columns]

    y_true_map = target_df.set_index([GROUP_COL, DATETIME_COL])[TARGET]

    predictions = []
    for pos, group in static_df.groupby(GROUP_COL):
        group = group.sort_values(DATETIME_COL).reset_index(drop=True)
        target_dates_pos = set(target_df.loc[target_df[GROUP_COL] == pos, DATETIME_COL])

        history = list(
            tail_per_pos.loc[tail_per_pos[GROUP_COL] == pos].sort_values(DATETIME_COL)[TARGET]
        )

        if pos in pos_bounds.index:
            lo, hi = pos_bounds.loc[pos, "min"], pos_bounds.loc[pos, "max"]
            margin = (hi - lo) * CLIP_MARGIN_PCT
            clip_lo, clip_hi = lo - margin, hi + margin
        else:
            clip_lo, clip_hi = -np.inf, np.inf

        step = 0
        for idx in range(len(group)):
            row = group.iloc[idx]
            if row[DATETIME_COL] not in target_dates_pos:
                continue
            step += 1

            row = row.copy()
            hist_arr = np.array(history[-max_hist:])

            for lag in LAG_LIST:
                row[f"{TARGET}_lag_{lag}"] = hist_arr[-lag] if len(hist_arr) >= lag else np.nan
            for window in ROLLING_WINDOWS:
                w = hist_arr[-window:]
                row[f"{TARGET}_roll_mean_{window}"] = w.mean() if len(w) > 0 else np.nan
                row[f"{TARGET}_roll_std_{window}"] = w.std() if len(w) > 1 else np.nan
            row[f"{TARGET}_diff_1"] = (hist_arr[-1] - hist_arr[-2]) if len(hist_arr) >= 2 else np.nan

            X_row = pd.DataFrame([row])
            if is_lgb:
                for c in cat_cols_here:
                    if c in X_row.columns:
                        X_row[c] = X_row[c].astype("category")

            pred = model.predict(X_row[feature_cols])[0]
            pred = float(np.clip(pred, clip_lo, clip_hi))  # <-- pengaman utama

            history.append(pred)
            predictions.append({
                GROUP_COL: pos, DATETIME_COL: row[DATETIME_COL],
                "step_ahead": step, "y_pred": pred,
                "y_true": y_true_map.get((pos, row[DATETIME_COL]), np.nan),
            })

    return pd.DataFrame(predictions)


In [ ]:
def merge_all(df_main, df_koor, df_env, env_how='left'):
    """
    Utility function to merge main TMA data with coordinates and environmental features.
    """
    # 1. Merge with coordinates
    df = df_main.merge(df_koor, on=GROUP_COL, how='left')

    # 2. Merge with environmental data (datling)
    # Ensure datetime and station columns match
    df = df.merge(
        df_env,
        on=[GROUP_COL, DATETIME_COL],
        how=env_how
    )

    return df

In [ ]:
def rolling_window_split(
    df: pd.DataFrame,
    datetime_col: str,
    group_col: str,
    train_window_rows: int,
    n_splits: int = 5,
    train_step: int = 1,        # geser train tiap 1 baris ("1 jam" pada istilah request)
    val_horizon_start: int = 1,
    val_horizon_end: int = 1,   # horizon val bertambah bertahap s.d. 6 baris ("6 jam")
    verbose: bool = True,
) -> Iterator[Tuple[pd.DataFrame, pd.DataFrame, int]]:
    """
    Rolling (sliding) window split -- BUKAN expanding window.

    Beda dengan `expanding_window_split` versi sebelumnya: ukuran window
    train di sini TETAP (`train_window_rows`) dan digeser maju `train_step`
    baris tiap fold, bukan terus membesar. Window val TIDAK statis:
    horizon-nya bertambah bertahap tiap fold dari `val_horizon_start` sampai
    `val_horizon_end` baris ke depan.

    "Baris" di sini = satu timestamp observasi unik lintas seluruh pos
    (konsisten dengan cara `expanding_window_split` lama membagi berdasarkan
    tanggal/waktu, bukan per baris per stasiun) -- supaya semua pos tetap
    berbagi batas fold yang sama.

    Yields
    ------
    train_df, valid_df, horizon
    """
    df = df.sort_values([datetime_col, group_col]).reset_index(drop=True)
    unique_ts = np.array(sorted(df[datetime_col].unique()))
    n_ts = len(unique_ts)

    if n_ts < train_window_rows + n_splits * train_step + val_horizon_end:
        raise ValueError(
            "Jumlah observasi unik tidak cukup untuk kombinasi "
            "train_window_rows / n_splits / train_step / val_horizon_end ini."
        )

    for fold in range(n_splits):
        # Horizon val naik bertahap linear dari val_horizon_start -> val_horizon_end
        if n_splits > 1:
            horizon = val_horizon_start + round(
                (val_horizon_end - val_horizon_start) * fold / (n_splits - 1)
            )
        else:
            horizon = val_horizon_end
        horizon = int(max(val_horizon_start, min(val_horizon_end, horizon)))

        train_start = fold * train_step
        train_end = train_start + train_window_rows
        val_start = train_end
        val_end = val_start + horizon

        train_ts = unique_ts[train_start:train_end]
        val_ts = unique_ts[val_start:val_end]

        train_fold = df[df[datetime_col].isin(train_ts)].copy()
        val_fold = df[df[datetime_col].isin(val_ts)].copy()

        if verbose:
            print("=" * 70)
            print(f"Fold {fold + 1} | horizon val = {horizon} baris (~{horizon*6} jam nyata)")
            print(f"Train : {train_fold[datetime_col].min()} → {train_fold[datetime_col].max()}  ({len(train_fold):,} baris)")
            print(f"Valid : {val_fold[datetime_col].min()} → {val_fold[datetime_col].max()}  ({len(val_fold):,} baris)")

        yield train_fold, val_fold, horizon


In [ ]:
def run_rolling_cv(train_df_raw, koor_df, datling_df, model_factory, is_lgb,
                    train_window_rows, n_splits=5, train_step=1,
                    val_horizon_start=1, val_horizon_end=6, verbose=True):
    """
    Jalankan rolling-window CV dengan perbaikan fitur lengkap (Climatology, Anomaly, Step).
    """
    splitter = rolling_window_split(
        train_df_raw, datetime_col=DATETIME_COL, group_col=GROUP_COL,
        train_window_rows=train_window_rows, n_splits=n_splits,
        train_step=train_step, val_horizon_start=val_horizon_start,
        val_horizon_end=val_horizon_end, verbose=verbose,
    )

    fold_scores = []
    all_recur_preds = []

    for fold, (train_fold_raw, val_fold_raw, horizon) in enumerate(splitter, start=1):
        # 1. Base Merging & Cleaning
        merged_fold = build_panel(train_fold_raw, val_fold_raw, datling_df)

        # 2. Time Features
        merged_fold = add_time_features(merged_fold)

        # 3. Filling & Hydrology/Lag Features
        merged_fold = fill_exogenous_missing(merged_fold, exclude_cols=[TARGET, "latitude", "longitude"])
        lag_cols_fold = [TARGET] + [c for c in EXTRA_LAG_COLS if c in merged_fold.columns]
        merged_fold = add_lag_and_rolling(merged_fold, lag_cols_fold)

        # 4. Encoding
        for col, le in encoders.items():
            if col in merged_fold.columns:
                merged_fold[f"{col}_encoded"] = le.transform(merged_fold[col].astype(str))

        # 5. Climatology & Anomaly (Penyebab utama KeyError)
        fold_cutoff = train_fold_raw[DATETIME_COL].max()
        clim_df, stat_med = fit_climatology(merged_fold, fold_cutoff)
        merged_fold = apply_climatology(merged_fold, clim_df, stat_med)

        # 6. Filter X dan y (hanya untuk data training di fold ini)
        train_mask = merged_fold[DATETIME_COL].le(fold_cutoff) & merged_fold[TARGET].notna()
        X_fold = merged_fold.loc[train_mask, feature_cols]
        y_fold = merged_fold.loc[train_mask, TARGET]

        if is_lgb:
            X_fold = sanitize_feature_names(X_fold)
            cat_cols_fold = [c for c in cat_cols_lgb if c in X_fold.columns]
            for c in cat_cols_fold:
                X_fold[c] = X_fold[c].astype("category")

        model = model_factory()
        if is_lgb:
            model.fit(
                X_fold, y_fold, eval_set=[(X_fold, y_fold)], eval_metric="rmse",
                categorical_feature=cat_cols_fold,
                callbacks=[lgb.early_stopping(100, verbose=False)],
            )
        else:
            # Untuk XGBoost, early stopping dihandle via parameter di model_factory atau fit
            model.fit(X_fold, y_fold, eval_set=[(X_fold, y_fold)], verbose=False)

        # --- Evaluasi recursive forecast ---
        pos_bounds_fold = train_fold_raw.groupby(GROUP_COL)[TARGET].agg(["min", "max"])
        recur_pred = recursive_forecast_eval(
            train_fold_raw, val_fold_raw, koor_df, datling_df, model,
            feature_cols, encoders, pos_bounds_fold, is_lgb=is_lgb,
        )

        recur_pred = recur_pred.dropna(subset=["y_true"])
        recur_pred["fold"] = fold
        recur_pred["horizon"] = horizon
        all_recur_preds.append(recur_pred)

        score = np.sqrt(mean_squared_error(recur_pred["y_true"], recur_pred["y_pred"]))
        fold_scores.append(score)
        if verbose:
            print(f"Fold {fold} RMSE (recursive, horizon={horizon}) = {score:.4f}\n")

    oof = pd.concat(all_recur_preds, ignore_index=True)
    print("========================")
    print(f"Mean RMSE : {np.mean(fold_scores):.4f}")
    print(f"Std RMSE  : {np.std(fold_scores):.4f}")
    return {"scores": fold_scores, "mean": np.mean(fold_scores), "std": np.std(fold_scores), "oof": oof}

### 4.3 Algoritma Model dan Fitur


In [ ]:
# Algoritma Model yang akan digunakan

def create_xgb():
    return XGBRegressor(
        n_estimators=2000,
        learning_rate=0.02,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        early_stopping_rounds=100,
        eval_metric="rmse",
        tree_method="hist", # Enable GPU for XGBoost
        device= 'cuda', # Specify GPU device ID (0 for the first GPU)
    )

def create_lgb():
    return LGBMRegressor(
        n_estimators=2000,
        learning_rate=0.02,
        max_depth=7,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        min_child_samples=10,  # diturunkan dari default (20) -> beberapa pos histori pendek
        random_state=42,
        device='gpu', # Enable GPU for LightGBM
        gpu_platform_id=0, # Specify GPU platform ID
        gpu_device_id=0, # Specify GPU device ID
    )

# OPSIONAL ALGORITMA

In [ ]:
drop_cols = [DATETIME_COL, TARGET, GROUP_COL, "landcover_name"]
feature_cols = [c for c in merged_df.columns if c not in drop_cols]

cat_cols_lgb = [c for c in ["nama_pos_encoded", "landcover_name_encoded"] if c in feature_cols]


#### Jalankan Rolling-Window CV — XGBoost vs LightGBM

`train_window_rows` dipilih cukup besar (mayoritas data) supaya tiap fold
tetap punya histori yang representatif; `n_splits=5`, `train_step=1`
(geser 1 baris tiap fold — granularitas terhalus / "1 jam"), horizon val
bertahap `1 → 6` baris ("6 jam").

In [ ]:
ROLL_CV_KWARGS = dict(
    train_window_rows=int(train_df[DATETIME_COL].nunique() * 0.7),
    n_splits=5,
    train_step=1,
    val_horizon_start=1,
    val_horizon_end=6,
)

xgb_cv_result = run_rolling_cv(
    train_df, koor_df, datling_df, model_factory=create_xgb, is_lgb=False,
    verbose=True, **ROLL_CV_KWARGS,
)

In [ ]:
xgb_cv_result = run_rolling_cv(
    train_df, koor_df, datling_df, model_factory=create_xgb, is_lgb=False,
    verbose=True, **ROLL_CV_KWARGS,
)

In [ ]:
lgb_cv_result = run_rolling_cv(
    train_df, koor_df, datling_df, model_factory=create_lgb, is_lgb=True,
    verbose=True, **ROLL_CV_KWARGS,
)


### 4.5 Split Final Train–Validation–Test


Rolling CV di atas dipakai untuk *memilih & memvalidasi metodologi*. Untuk
model final yang disubmit, tetap dipakai satu split kronologis
train → val → test (val untuk memantau training, test untuk evaluasi akhir
yang sama sekali tidak dilihat model).


In [ ]:
# Menentukan feature yang digunakan

X_train = merged_df[feature_cols]
y_train = merged_df[TARGET]

# Drop rows with NaN in the target variable before training the final models
nan_mask_final = y_train.isna()
X_train = X_train[~nan_mask_final]
y_train = y_train[~nan_mask_final]

X_train

## 5. Modeling dan Eksperimen 🤖


### 5.1 Eksperimen 1 — Baseline XGBoost


In [ ]:
model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.02,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    early_stopping_rounds=100,
    eval_metric="rmse",
    tree_method="hist",
    tree_method="gpu_hist", # Enable GPU for XGBoost
    gpu_id=0, # Specify GPU device ID (0 for the first GPU)
)

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train)],
    verbose=100
)


### 5.2 Eksperimen 2 — LightGBM


In [ ]:
def sanitize_feature_names(df: pd.DataFrame) -> pd.DataFrame:
    """LightGBM menolak nama kolom dengan karakter non alfanumerik."""
    df = df.copy()
    df.columns = [re.sub(r"[^0-9a-zA-Z_]+", "_", str(c)) for c in df.columns]
    return df


X_train_lgb = sanitize_feature_names(X_train)
cat_cols_lgb = [c for c in ["nama_pos_encoded", "landcover_name_encoded"] if c in X_train_lgb.columns]
for c in cat_cols_lgb:
    X_train_lgb[c] = X_train_lgb[c].astype("category")


In [ ]:
evals_result_dict = {}

lgb_model = LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.02,
    max_depth=7,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    min_child_samples=10,  # diturunkan dari default (20) -> beberapa pos histori pendek
    random_state=42,
    device='gpu', # Enable GPU for LightGBM
    gpu_platform_id=0, # Specify GPU platform ID
    gpu_device_id=0, # Specify GPU device ID
)

lgb_model.fit(
    X_train_lgb, y_train,
    eval_set=[(X_train_lgb, y_train)],
    eval_metric="rmse",
    categorical_feature=cat_cols_lgb,
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100), lgb.record_evaluation(evals_result_dict)],
)


### 5.3 Eksperimen 3 — CatBoost + N-HiTS + Seasonal


Versi pipeline lama yang berpotensi membuat Colab kehabisan memori digantikan oleh versi **hardened** di bawah. Tahap preprocessing menggunakan hasil Section 3 dan konfigurasi model menggunakan `build_experiment_config("experiment_3")`.


Yang membuat google colab saya crash<br>
|<br>
|<br>
v

In [ ]:
# @title
"""
CatBoost + neural + seasonal ensemble for forecasting tma_mdpl.
HARDENED VERSION — optimized to avoid Google Colab session crashes (OOM).

Changes vs. the original pipeline (see inline "# PATCH" comments):
  1. float32 downcasting everywhere numeric data is created.
  2. `nama_pos` stored as pandas `category` instead of `object`.
  3. Rolling/EWM features use native pandas groupby().rolling()/.ewm()
     instead of groupby().transform(lambda ...) — faster and lighter.
  4. Explicit `del` + `gc.collect()` between validation and final fit,
     and inside run_models(), so CatBoost/Neural objects don't pile up.
  5. A size guard on the panel cross-join (stations x timestamps) that
     raises early instead of letting Colab silently OOM.
  6. Optional cap on the number of exogenous features fed to the neural
     model, ranked by CatBoost feature importance.
  7. Lightweight RAM logging (via psutil) at key checkpoints.
  8. Two entry points, `run_validation_only` and `run_final_only`, so the
     two heaviest phases (validation fit, final fit) can be run in
     separate Colab runtime sessions if RAM is tight. `run_ensemble`
     still exists and calls both in sequence for convenience.

Install dependencies:
    pip install pandas numpy catboost neuralforecast torch psutil

Use from a notebook (all-in-one, same interface as before):
    submission, metrics, weights, validation = run_ensemble(
        df_train,
        df_test,
        df_lingkungan,
        df_submission,
        config={"device": "auto"},
    )

Or, to protect against Colab OOM, split into two runtime sessions:
    # --- Session 1 ---
    weights, metrics, validation = run_validation_only(
        df_train, df_test, df_lingkungan, config={"device": "auto"}
    )
    # weights/metrics are saved to disk automatically.
    # Runtime > Restart runtime here to fully release RAM.

    # --- Session 2 (after restart, re-load df_train/df_test/df_lingkungan) ---
    submission = run_final_only(
        df_train, df_test, df_lingkungan, df_submission,
        weights_path="weights.csv", config={"device": "auto"},
    )
"""

import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from catboost import CatBoostRegressor
from neuralforecast import NeuralForecast
from neuralforecast.losses.pytorch import MAE
from neuralforecast.models import NHITS, TiDE

try:
    import psutil
    _HAS_PSUTIL = True
except ImportError:
    _HAS_PSUTIL = False


OBSERVATIONS_PER_DAY = 3
RANDOM_SEED = 42
PIPELINE_VERSION = "2026-07-hydrologic-ensemble-v4-hardened"


DEFAULT_CONFIG = {
    "device": "auto",                 # "auto", "gpu", or "cpu"
    "gpu_index": 0,
    "precision": "32-true",
    "neural_model": "nhits",          # "nhits" (recommended) or "tide"
    "input_days": 120,
    "tide_hidden_size": 64,
    "tide_batch_size": 8,
    "tide_windows_batch_size": 32,
    "tide_inference_batch_size": 32,
    "tide_max_steps": 300,
    "catboost_task_type": "CPU",
    "catboost_iterations": 800,
    "catboost_depth": 7,
    "catboost_gpu_ram_part": 0.25,
    "add_hydrology_features": True,
    "report_worst_stations": 10,
    "seasonal_window_days": 15,
    "seasonal_anchor_half_life_days": 3,
    "weight_step": 0.05,
    "ensemble_objective": "balanced",
    "validation_mode": "same_season",
    "output_file": "submission_ens.csv",

    # --- PATCH: new memory-safety knobs ---
    "max_panel_rows": 5_000_000,      # PATCH: guard against runaway cross-join
    "neural_max_exog_features": 40,   # PATCH: cap neural exogenous feature count
    "log_memory": True,               # PATCH: print RAM usage at checkpoints
    "weights_output_file": "weights.csv",
    "metrics_output_file": "metrics.csv",
}


# ----------------------------------------------------------------------
# PATCH: memory helpers
# ----------------------------------------------------------------------

def log_memory(tag, config=None):
    """Print current host RAM usage. Cheap, but very useful for diagnosis."""
    if config is not None and not config.get("log_memory", True):
        return

    if not _HAS_PSUTIL:
        return

    mem = psutil.virtual_memory()
    print(
        f"[mem] {tag}: {mem.percent:.1f}% used "
        f"({mem.used / 1e9:.2f} GB / {mem.total / 1e9:.2f} GB)"
    )


def downcast_numeric(frame):
    """Downcast float64 -> float32 and shrink integer dtypes where safe.

    This alone typically halves the memory footprint of the environment /
    panel DataFrames, which are the largest objects in this pipeline.
    """
    frame = frame.copy()

    float_columns = frame.select_dtypes(include="float64").columns
    if len(float_columns):
        frame[float_columns] = frame[float_columns].astype("float32")

    int_columns = frame.select_dtypes(include="int64").columns
    for column in int_columns:
        frame[column] = pd.to_numeric(frame[column], downcast="integer")

    return frame


def free(*objects):
    """Delete references and force garbage collection.

    Note: this reliably frees pure-Python objects (DataFrames, arrays).
    It does NOT guarantee that native allocators used by CatBoost/PyTorch
    return memory to the OS immediately — for a hard guarantee, run the
    validation and final phases in separate Colab sessions
    (see run_validation_only / run_final_only).
    """
    del objects
    gc.collect()


def safe_fillna(series, fill_value):
    """fillna() that never triggers pandas' LossySetitemError.

    PATCH (bugfix): when `series` has been downcast to float32/int* and the
    replacement value(s) are float64 (e.g. a not-yet-downcast lookup table),
    pandas' Copy-on-Write internals can raise AssertionError instead of
    silently upcasting (pandas GH#52927-style failure). Explicitly casting
    the fill value to the series' own dtype first avoids the mixed-dtype
    write entirely.
    """
    if isinstance(fill_value, (pd.Series, np.ndarray)):
        fill_value = pd.Series(fill_value).astype(series.dtype)
    else:
        fill_value = series.dtype.type(fill_value)
    return series.fillna(fill_value)


# ----------------------------------------------------------------------
# Config handling
# ----------------------------------------------------------------------

def prepare_config(config=None):
    """Merge user settings and select GPU or CPU safely."""
    settings = DEFAULT_CONFIG.copy()

    if config is not None:
        settings.update(config)

    requested_device = str(settings["device"]).lower()

    if requested_device not in {"auto", "gpu", "cpu"}:
        raise ValueError("config['device'] must be 'auto', 'gpu', or 'cpu'.")

    gpu_available = torch.cuda.is_available()

    if requested_device == "gpu" and not gpu_available:
        print("CUDA is unavailable. The neural model will use CPU instead.")

    use_gpu = gpu_available and requested_device in {"auto", "gpu"}
    settings["accelerator"] = "gpu" if use_gpu else "cpu"
    settings["precision"] = settings["precision"] if use_gpu else "32-true"
    settings["devices"] = [settings["gpu_index"]] if use_gpu else 1

    catboost_device = str(settings["catboost_task_type"]).upper()

    if catboost_device not in {"CPU", "GPU"}:
        raise ValueError("config['catboost_task_type'] must be 'CPU' or 'GPU'.")

    if catboost_device == "GPU" and not gpu_available:
        print("CUDA is unavailable. CatBoost will use CPU instead.")
        catboost_device = "CPU"

    settings["catboost_task_type"] = catboost_device

    if settings["ensemble_objective"] not in {"balanced", "rmse", "mae"}:
        raise ValueError(
            "config['ensemble_objective'] must be 'balanced', 'rmse', or 'mae'."
        )

    if settings["validation_mode"] not in {"same_season", "last_block"}:
        raise ValueError(
            "config['validation_mode'] must be 'same_season' or 'last_block'."
        )

    settings["neural_model"] = str(settings["neural_model"]).lower()

    if settings["neural_model"] not in {"nhits", "tide"}:
        raise ValueError("config['neural_model'] must be 'nhits' or 'tide'.")

    return settings


def release_gpu_memory():
    """Release cached GPU memory before fitting the next neural model."""
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ----------------------------------------------------------------------
# Data preparation
# ----------------------------------------------------------------------

def prepare_data(train_df_clean, test_df, datling_df):
    """Copy and standardize the DataFrames without changing the originals."""
    train = train_df_clean.copy()
    test = test_df.copy()
    environment = datling_df.copy()

    train["datetime"] = pd.to_datetime(train["datetime"])
    environment["datetime"] = pd.to_datetime(environment["datetime"])

    if "datetime" not in test.columns or "nama_pos" not in test.columns:
        parsed = test["id"].str.extract(
            r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) - (.*)$"
        )

        if parsed.isna().any().any():
            raise ValueError("Some test IDs do not match 'datetime - nama_pos'.")

        test["datetime"] = pd.to_datetime(parsed[0])
        test["nama_pos"] = parsed[1]
    else:
        test["datetime"] = pd.to_datetime(test["datetime"])

    if "id" not in test.columns:
        test["id"] = test["datetime"].astype(str) + " - " + test["nama_pos"]

    test["original_order"] = np.arange(len(test))

    # PATCH: category dtype for station identity saves memory everywhere
    # downstream (panel, environment, predictions all carry nama_pos).
    train["nama_pos"] = train["nama_pos"].astype("string")
    test["nama_pos"] = test["nama_pos"].astype("string")
    environment["nama_pos"] = environment["nama_pos"].astype("string")

    return train, test, environment


def clean_environment(environment):
    environment = environment.copy()

    if "solar_radiation_mj_m2" in environment.columns:
        bad_solar = environment["solar_radiation_mj_m2"].eq(-999)
        environment["solar_radiation_missing"] = bad_solar.astype(int)
        environment.loc[bad_solar, "solar_radiation_mj_m2"] = np.nan

    environment = environment.drop_duplicates(["nama_pos", "datetime"])
    environment = environment.sort_values(["nama_pos", "datetime"])

    numeric_columns = environment.select_dtypes(include="number").columns.tolist()

    for column in numeric_columns:
        environment[column] = environment[column].replace([np.inf, -np.inf], np.nan)

        if environment[column].isna().any():
            environment[column + "_missing"] = environment[column].isna().astype(int)
            environment[column] = environment.groupby("nama_pos", observed=True)[column].transform(
                lambda values: values.interpolate(limit_direction="both")
            )
            environment[column] = environment[column].fillna(environment[column].median())

    # NOTE (bugfix): do NOT downcast here. If we downcast to float32 now,
    # the interpolate()/fillna() calls that still run later in
    # add_hydrology_features() and build_panel() can try to write float64
    # results back into an already-float32 column, which pandas' CoW
    # internals reject with AssertionError/LossySetitemError. Keep
    # everything float64 until all transforms are done — downcast only
    # once, at the very end of build_panel().

    return environment


def _grouped_rolling(series, group_key, window, min_periods=1, agg="sum"):
    """PATCH: native pandas groupby().rolling() instead of
    groupby().transform(lambda values: values.rolling(...)).

    This is both faster and lighter on memory than the lambda/transform
    pattern, because it avoids building one temporary Series object per
    group under the hood.
    """
    rolled = series.groupby(group_key, observed=True).rolling(
        window, min_periods=min_periods
    )
    result = getattr(rolled, agg)()
    return result.reset_index(level=0, drop=True).sort_index()


def _grouped_ewm(series, group_key, alpha):
    """PATCH: native pandas groupby().ewm() instead of transform(lambda)."""
    ewm = series.groupby(group_key, observed=True).ewm(alpha=alpha, adjust=False)
    result = ewm.mean()
    return result.reset_index(level=0, drop=True).sort_index()


def add_hydrology_features(environment):
    """Add causal rainfall-memory, soil-state, and weather-tendency features.

    The environment is hourly, so every rolling feature below uses only the
    current and earlier rows (no centered/forward windows for causal
    features) — this avoids leaking future weather into validation rows.
    """
    environment = environment.copy()
    environment = environment.sort_values(["nama_pos", "datetime"])
    station = environment["nama_pos"]

    if {"rainfall_mm", "rainfall_openmeteo_mm"}.issubset(environment.columns):
        left = environment["rainfall_mm"].to_numpy()
        right = environment["rainfall_openmeteo_mm"].to_numpy()

        if np.allclose(left, right, equal_nan=True):
            environment = environment.drop(columns="rainfall_openmeteo_mm")

    if "rainfall_mm" in environment.columns:
        rain = environment["rainfall_mm"].clip(lower=0).fillna(0)

        # PATCH: native groupby().rolling() instead of transform(lambda)
        for days in [1, 3, 7, 14, 30]:
            hours = 24 * days
            environment[f"rain_sum_{days}d"] = _grouped_rolling(
                rain, station, hours, min_periods=1, agg="sum"
            )

        for days in [1, 3, 7]:
            hours = 24 * days
            environment[f"rain_max_{days}d"] = _grouped_rolling(
                rain, station, hours, min_periods=1, agg="max"
            )

        wet = rain.gt(0.1).astype("float32")

        for days in [3, 7]:
            hours = 24 * days
            environment[f"wet_fraction_{days}d"] = _grouped_rolling(
                wet, station, hours, min_periods=1, agg="mean"
            )

        # API is a decaying rainfall memory (hourly retention from daily).
        for retention in [0.70, 0.85, 0.93]:
            hourly_retention = retention ** (1 / 24)
            alpha = 1 - hourly_retention
            name = f"api_{int(retention * 100):03d}"
            environment[name] = _grouped_ewm(rain, station, alpha)

        rain_event = rain.gt(0.1)
        event_number = rain_event.groupby(station, observed=True).cumsum()
        environment["hours_since_rain"] = (~rain_event).groupby(
            [station, event_number], observed=True
        ).cumsum().clip(upper=24 * 60)

    soil_columns = [
        column
        for column in [
            "soil_moisture_0_7cm",
            "soil_moisture_7_28cm",
            "soil_moisture_28_100cm",
            "soil_moisture_100_255cm",
        ]
        if column in environment.columns
    ]

    if soil_columns:
        shallow = soil_columns[:2]
        deep = soil_columns[2:]
        environment["soil_shallow_mean"] = environment[shallow].mean(axis=1)
        environment["soil_deep_mean"] = (
            environment[deep].mean(axis=1)
            if deep
            else environment[soil_columns].mean(axis=1)
        )

        interaction_soils = ["soil_shallow_mean", "soil_deep_mean"]

        for rain_feature in ["api_085", "rain_sum_3d", "rain_sum_7d"]:
            if rain_feature in environment.columns:
                for soil_feature in interaction_soils:
                    name = rain_feature + "_x_" + soil_feature
                    environment[name] = (
                        environment[rain_feature] * environment[soil_feature]
                    )

    if {"temperature_c", "dew_point_c"}.issubset(environment.columns):
        environment["dewpoint_depression"] = (
            environment["temperature_c"] - environment["dew_point_c"]
        )

    if {"cloud_cover_pct", "humidity_pct"}.issubset(environment.columns):
        environment["cloud_x_humidity"] = (
            environment["cloud_cover_pct"] * environment["humidity_pct"] / 100
        )

    tendency_columns = [
        column
        for column in soil_columns
        + [
            "humidity_pct",
            "temperature_c",
            "surface_pressure_hpa",
            "pressure_msl_hpa",
            "cloud_cover_pct",
            "dew_point_c",
        ]
        if column in environment.columns
    ]

    for column in tendency_columns:
        for days in [1, 3]:
            lag = 24 * days
            environment[f"{column}_change_{days}d"] = (
                environment[column]
                - environment.groupby("nama_pos", observed=True)[column].shift(lag)
            ).fillna(0)

    # NOTE (bugfix): still do not downcast here — build_panel() runs one
    # more interpolate()/fillna() pass on these columns after merging into
    # the grid. Downcasting is deferred to the single call at the end of
    # build_panel(), after every transform is finished.

    return environment


def add_calendar_features(frame):
    frame = frame.copy()
    day = frame["datetime"].dt.dayofyear
    hour = frame["datetime"].dt.hour

    frame["month"] = frame["datetime"].dt.month
    frame["day_of_year"] = day
    frame["hour"] = hour
    frame["day_sin"] = np.sin(2 * np.pi * day / 365.25).astype("float32")
    frame["day_cos"] = np.cos(2 * np.pi * day / 365.25).astype("float32")
    frame["hour_sin"] = np.sin(2 * np.pi * hour / 24).astype("float32")
    frame["hour_cos"] = np.cos(2 * np.pi * hour / 24).astype("float32")

    if "wind_direction_deg" in frame.columns:
        angle = np.deg2rad(frame["wind_direction_deg"])
        frame["wind_sin"] = np.sin(angle).astype("float32")
        frame["wind_cos"] = np.cos(angle).astype("float32")

    if "mjo_phase" in frame.columns:
        angle = 2 * np.pi * (frame["mjo_phase"] - 1) / 8
        frame["mjo_phase_sin"] = np.sin(angle).astype("float32")
        frame["mjo_phase_cos"] = np.cos(angle).astype("float32")

    return frame


def build_panel(train, test, environment, config):
    first_time = train["datetime"].min()
    train_end = train["datetime"].max()

    target_hours = sorted(test["datetime"].dt.hour.unique())
    training_days = pd.date_range(
        first_time.normalize(),
        train_end.normalize(),
        freq="D",
    )

    training_times = []

    for day in training_days:
        for hour in target_hours:
            timestamp = day + pd.Timedelta(hours=int(hour))

            if first_time <= timestamp <= train_end:
                training_times.append(timestamp)

    all_times = sorted(set(training_times) | set(test["datetime"]))
    stations = sorted(test["nama_pos"].unique())

    # PATCH: guard against a runaway cross-join before it ever gets built.
    # Better to fail fast with a clear message than to silently OOM.
    estimated_rows = len(stations) * len(all_times)
    max_rows = config.get("max_panel_rows", 5_000_000)

    if estimated_rows > max_rows:
        raise MemoryError(
            f"Panel grid would have ~{estimated_rows:,} rows "
            f"({len(stations)} stations x {len(all_times)} timestamps), "
            f"which exceeds config['max_panel_rows']={max_rows:,}. "
            "Consider shortening the training window, processing stations "
            "in batches, or raising max_panel_rows if you are sure your "
            "instance has enough RAM."
        )

    log_memory("before panel cross-join", config)

    grid = pd.MultiIndex.from_product(
        [stations, all_times],
        names=["nama_pos", "datetime"],
    ).to_frame(index=False)
    grid["nama_pos"] = grid["nama_pos"].astype("category")

    log_memory("after panel cross-join", config)

    panel = grid.merge(
        environment,
        on=["nama_pos", "datetime"],
        how="left",
        indicator="_environment_merge",
    )
    panel["environment_row_missing"] = panel["_environment_merge"].eq(
        "left_only"
    ).astype("int8")
    panel = panel.drop(columns="_environment_merge")

    free(grid)  # PATCH: grid is no longer needed once merged

    environment_numeric = [
        column
        for column in environment.select_dtypes(include="number").columns
        if column in panel.columns
    ]

    for column in environment_numeric:
        if panel[column].isna().any():
            panel[column] = panel.groupby("nama_pos", observed=True)[column].transform(
                lambda values: values.interpolate(limit_direction="both")
            )
            # PATCH (bugfix): safe_fillna — at this point panel[column] is
            # still float64 (downcast happens once, later, at the end of
            # build_panel), so this is mostly a defensive no-op here, but
            # keeps the pattern consistent everywhere fillna touches a
            # column that might already be downcast in other call paths.
            panel[column] = safe_fillna(panel[column], panel[column].median())

    panel = panel.merge(
        train[["datetime", "nama_pos", "tma_mdpl"]],
        on=["datetime", "nama_pos"],
        how="left",
    )

    panel = add_calendar_features(panel)
    panel = panel.sort_values(["nama_pos", "datetime"]).reset_index(drop=True)

    all_times_series = pd.Series(panel["datetime"].sort_values().unique())
    time_to_step = pd.Series(np.arange(len(all_times_series)), index=all_times_series)
    panel["step"] = panel["datetime"].map(time_to_step).astype("int32")

    # PATCH: final downcast pass on the largest object in the pipeline.
    panel = downcast_numeric(panel)

    log_memory("after build_panel", config)

    return panel


# ----------------------------------------------------------------------
# Climatology
# ----------------------------------------------------------------------

def fit_climatology(panel, cutoff):
    history = panel[
        panel["datetime"].le(cutoff) & panel["tma_mdpl"].notna()
    ].copy()

    climatology = (
        history.groupby(["nama_pos", "month", "hour"], as_index=False, observed=True)["tma_mdpl"]
        .median()
        .rename(columns={"tma_mdpl": "climatology"})
    )

    station_median = history.groupby("nama_pos", observed=True)["tma_mdpl"].median()

    free(history)

    return climatology, station_median


def apply_climatology(frame, climatology, station_median):
    frame = frame.drop(columns=["climatology", "anomaly"], errors="ignore").copy()
    frame = frame.merge(climatology, on=["nama_pos", "month", "hour"], how="left")
    # PATCH (bugfix): use safe_fillna — frame["climatology"] may be float32
    # (from an already-downcast panel) while the station_median lookup is
    # float64; a raw .fillna() with mismatched dtypes can raise
    # AssertionError/LossySetitemError under pandas Copy-on-Write.
    frame["climatology"] = safe_fillna(
        frame["climatology"], frame["nama_pos"].map(station_median)
    )
    frame["climatology"] = safe_fillna(frame["climatology"], 0)
    frame["anomaly"] = frame["tma_mdpl"] - frame["climatology"]
    return frame


def get_model_features(panel):
    ignored = {
        "datetime",
        "tma_mdpl",
        "climatology",
        "anomaly",
        "step",
        "landcover_name",
    }

    numeric_features = [
        column
        for column in panel.select_dtypes(include="number").columns
        if column not in ignored
    ]

    catboost_features = ["nama_pos"] + numeric_features
    neural_features = numeric_features

    return catboost_features, neural_features


# ----------------------------------------------------------------------
# PATCH: neural feature selection, ranked by CatBoost importance
# ----------------------------------------------------------------------

def select_neural_features(neural_features, config, catboost_model=None, catboost_features=None):
    """Cap the number of exogenous features fed to the neural model.

    Fewer exogenous features means a smaller input tensor per window
    (input_days * OBSERVATIONS_PER_DAY * n_features), which directly
    reduces both GPU and host memory pressure during neural training.
    Ranking uses CatBoost feature importance when available, so the most
    predictive features are kept.
    """
    limit = config.get("neural_max_exog_features")

    if not limit or len(neural_features) <= limit:
        return neural_features

    if catboost_model is not None and catboost_features is not None:
        importances = pd.Series(
            catboost_model.get_feature_importance(),
            index=catboost_features,
        )
        ranked = [
            feature
            for feature in importances.sort_values(ascending=False).index
            if feature in neural_features
        ]
        selected = ranked[:limit]
    else:
        selected = neural_features[:limit]

    print(
        f"Neural exogenous features capped: {len(neural_features)} -> "
        f"{len(selected)} (config['neural_max_exog_features']={limit})"
    )

    return selected


# ----------------------------------------------------------------------
# CatBoost
# ----------------------------------------------------------------------

def train_catboost(panel, cutoff, features, config):
    training = panel[
        panel["datetime"].le(cutoff) & panel["anomaly"].notna()
    ]

    parameters = {
        "iterations": config["catboost_iterations"],
        "depth": config["catboost_depth"],
        "learning_rate": 0.05,
        "loss_function": "RMSE",
        "l2_leaf_reg": 5,
        "random_strength": 0.5,
        "random_seed": RANDOM_SEED,
        "verbose": 100,
        "allow_writing_files": False,
        "task_type": config["catboost_task_type"],
    }

    if str(config["catboost_task_type"]).upper() == "GPU":
        parameters["devices"] = str(config["gpu_index"])
        parameters["gpu_ram_part"] = config["catboost_gpu_ram_part"]
    else:
        parameters["rsm"] = 0.85

    model = CatBoostRegressor(**parameters)

    model.fit(
        training[features],
        training["anomaly"],
        cat_features=["nama_pos"],
    )

    return model


def predict_catboost(model, future, features):
    future = future.copy()
    future["catboost"] = model.predict(future[features]) + future["climatology"]
    return future[["nama_pos", "datetime", "catboost"]]


# ----------------------------------------------------------------------
# Seasonal model
# ----------------------------------------------------------------------

def predict_anchored_seasonal(panel, cutoff, horizon, config):
    """Forecast from the smoothed value one year ago plus a decaying offset."""
    history = panel[panel["datetime"].le(cutoff)].copy()
    future = panel[panel["datetime"].gt(cutoff)].copy()
    future = future.groupby("nama_pos", group_keys=False, observed=True).head(horizon)

    history = history.sort_values(["nama_pos", "hour", "datetime"])
    window = config["seasonal_window_days"]

    # PATCH: native rolling instead of transform(lambda). Centered windows
    # are fine here because this uses only *past* years' data as reference,
    # not future data relative to the forecast cutoff.
    history["seasonal_reference"] = (
        history.groupby(["nama_pos", "hour"], observed=True)["tma_mdpl"]
        .rolling(window=window, center=True, min_periods=max(3, window // 3))
        .median()
        .reset_index(level=[0, 1], drop=True)
        .sort_index()
    )

    reference = history[
        ["nama_pos", "datetime", "seasonal_reference"]
    ].rename(columns={"datetime": "reference_time"})

    future["reference_time"] = future["datetime"].map(
        lambda timestamp: timestamp - pd.DateOffset(years=1)
    )
    future = future.merge(
        reference,
        on=["nama_pos", "reference_time"],
        how="left",
    )

    last_observed = (
        history.dropna(subset=["tma_mdpl"])
        .sort_values(["nama_pos", "datetime"])
        .groupby("nama_pos", as_index=False, observed=True)
        .tail(1)
        [["nama_pos", "datetime", "tma_mdpl"]]
        .rename(
            columns={
                "datetime": "origin_time",
                "tma_mdpl": "origin_tma",
            }
        )
    )
    last_observed["reference_time"] = last_observed["origin_time"].map(
        lambda timestamp: timestamp - pd.DateOffset(years=1)
    )
    last_observed = last_observed.merge(
        reference,
        on=["nama_pos", "reference_time"],
        how="left",
    )
    last_observed["anchor_offset"] = (
        last_observed["origin_tma"] - last_observed["seasonal_reference"]
    ).fillna(0)

    future = future.merge(
        last_observed[["nama_pos", "anchor_offset"]],
        on="nama_pos",
        how="left",
    )
    future["anchor_offset"] = future["anchor_offset"].fillna(0)
    future["horizon_step"] = future.groupby("nama_pos", observed=True).cumcount() + 1
    future["horizon_day"] = np.ceil(
        future["horizon_step"] / OBSERVATIONS_PER_DAY
    )

    half_life = config["seasonal_anchor_half_life_days"]
    decay = 0.5 ** (future["horizon_day"] / half_life)
    # PATCH (bugfix): safe_fillna avoids AssertionError/LossySetitemError
    # when seasonal_reference (derived from a downcast float32 tma_mdpl)
    # and climatology (float64, from a separate lookup table) don't share
    # a dtype.
    seasonal_base = safe_fillna(future["seasonal_reference"], future["climatology"])
    future["seasonal"] = seasonal_base + decay * future["anchor_offset"]

    free(history, reference, last_observed)

    return future[["nama_pos", "datetime", "seasonal"]]


# ----------------------------------------------------------------------
# Neural model
# ----------------------------------------------------------------------

def prepare_neural_frames(panel, cutoff, horizon, neural_features):
    history = panel[panel["datetime"].le(cutoff)].copy()
    future = panel[panel["datetime"].gt(cutoff)].copy()
    future = future.groupby("nama_pos", group_keys=False, observed=True).head(horizon)

    counts = future.groupby("nama_pos", observed=True).size()
    if counts.empty or not counts.eq(horizon).all():
        raise ValueError("The environmental file does not cover the full horizon.")

    history["y"] = history["anomaly"]
    history["y"] = history.groupby("nama_pos", observed=True)["y"].transform(
        lambda values: values.interpolate(limit=6, limit_direction="both")
    )
    history["y"] = history["y"].fillna(0)

    neural_scale = history.groupby("nama_pos", observed=True)["y"].std()
    neural_scale = neural_scale.replace([np.inf, -np.inf, 0], np.nan).fillna(1)
    neural_scale = neural_scale.clip(lower=0.1)

    history["neural_scale"] = history["nama_pos"].map(neural_scale)
    future["neural_scale"] = future["nama_pos"].map(neural_scale).fillna(1)
    history["y"] = history["y"] / history["neural_scale"]

    history = history.rename(columns={"nama_pos": "unique_id", "step": "ds"})
    future = future.rename(columns={"nama_pos": "unique_id", "step": "ds"})

    history["unique_id"] = history["unique_id"].astype(str)
    future["unique_id"] = future["unique_id"].astype(str)

    train_columns = ["unique_id", "ds", "y"] + neural_features
    future_columns = ["unique_id", "ds"] + neural_features

    return history[train_columns], future[future_columns], future


def train_and_predict_neural(panel, cutoff, horizon, neural_features, config):
    history, future_covariates, future_map = prepare_neural_frames(
        panel, cutoff, horizon, neural_features
    )

    common = {
        "h": horizon,
        "input_size": config["input_days"] * OBSERVATIONS_PER_DAY,
        "futr_exog_list": neural_features,
        "loss": MAE(),
        "scaler_type": "identity",
        "max_steps": config["tide_max_steps"],
        "batch_size": config["tide_batch_size"],
        "windows_batch_size": config["tide_windows_batch_size"],
        "inference_windows_batch_size": config["tide_inference_batch_size"],
        "start_padding_enabled": True,
        "random_seed": RANDOM_SEED,
        "accelerator": config["accelerator"],
        "devices": config["devices"],
        "precision": config["precision"],
        "alias": "neural",
    }

    if config["neural_model"] == "nhits":
        model = NHITS(
            n_blocks=[1, 1, 1],
            mlp_units=[[128, 128], [128, 128], [128, 128]],
            n_pool_kernel_size=[4, 2, 1],
            n_freq_downsample=[12, 4, 1],
            pooling_mode="AvgPool1d",
            dropout_prob_theta=0.1,
            **common,
        )
    else:
        model = TiDE(
            hidden_size=config["tide_hidden_size"],
            decoder_output_dim=16,
            temporal_decoder_dim=32,
            dropout=0.2,
            **common,
        )

    forecast = NeuralForecast(models=[model], freq=1)

    try:
        log_memory("before neural fit", config)
        forecast.fit(df=history)
        prediction = forecast.predict(futr_df=future_covariates).reset_index()
        log_memory("after neural fit", config)
    except (torch.cuda.OutOfMemoryError, RuntimeError) as error:
        is_gpu_oom = (
            config["accelerator"] == "gpu"
            and "out of memory" in str(error).lower()
        )

        if not is_gpu_oom:
            raise

        print("GPU memory was insufficient. Retrying the neural model on CPU...")
        free(forecast, model)
        release_gpu_memory()

        cpu_config = config.copy()
        cpu_config["accelerator"] = "cpu"
        cpu_config["devices"] = 1
        cpu_config["precision"] = "32-true"

        return train_and_predict_neural(
            panel,
            cutoff,
            horizon,
            neural_features,
            cpu_config,
        )

    if "neural" in prediction.columns:
        prediction_column = "neural"
    else:
        prediction_columns = [
            column
            for column in prediction.columns
            if column not in {"index", "unique_id", "ds", "cutoff"}
        ]

        if len(prediction_columns) != 1:
            raise ValueError(
                "Could not identify the neural prediction column. "
                f"Available columns: {prediction.columns.tolist()}"
            )

        prediction_column = prediction_columns[0]

    prediction = prediction.rename(
        columns={prediction_column: "neural_scaled_anomaly"}
    )

    non_finite = ~np.isfinite(prediction["neural_scaled_anomaly"])

    if non_finite.any():
        print(
            f"Warning: {non_finite.sum()} neural values were non-finite; "
            "using zero anomaly (climatology) for those rows."
        )
        prediction.loc[non_finite, "neural_scaled_anomaly"] = 0

    prediction["unique_id"] = prediction["unique_id"].astype(str)
    future_map["unique_id"] = future_map["unique_id"].astype(str)

    prediction = prediction.sort_values(["unique_id", "ds"]).copy()
    prediction["forecast_order"] = prediction.groupby("unique_id").cumcount()

    future_lookup = future_map.sort_values(["unique_id", "datetime"]).copy()
    future_lookup["forecast_order"] = future_lookup.groupby("unique_id").cumcount()

    prediction_counts = prediction.groupby("unique_id").size()
    future_counts = future_lookup.groupby("unique_id").size()

    if not prediction_counts.equals(future_counts):
        raise ValueError(
            "The neural model returned a different number of rows than requested.\n"
            f"Predicted counts:\n{prediction_counts}\n"
            f"Expected counts:\n{future_counts}"
        )

    prediction = prediction.merge(
        future_lookup[
            [
                "unique_id",
                "forecast_order",
                "datetime",
                "climatology",
                "neural_scale",
            ]
        ],
        on=["unique_id", "forecast_order"],
        how="left",
        validate="one_to_one",
    )
    prediction["neural_anomaly"] = (
        prediction["neural_scaled_anomaly"] * prediction["neural_scale"]
    )
    prediction["neural"] = (
        prediction["neural_anomaly"] + prediction["climatology"]
    )
    prediction = prediction.rename(columns={"unique_id": "nama_pos"})
    prediction["nama_pos"] = prediction["nama_pos"].astype("category")

    # PATCH: release intermediate neural objects immediately.
    free(forecast, model, history, future_covariates, future_map)
    release_gpu_memory()

    return prediction[["nama_pos", "datetime", "neural"]]


# ----------------------------------------------------------------------
# Horizon banding
# ----------------------------------------------------------------------

def add_horizon_band(frame):
    frame = frame.sort_values(["nama_pos", "datetime"]).copy()
    frame["horizon_step"] = frame.groupby("nama_pos", observed=True).cumcount() + 1
    frame["horizon_day"] = np.ceil(
        frame["horizon_step"] / OBSERVATIONS_PER_DAY
    ).astype(int)

    frame["horizon_band"] = pd.cut(
        frame["horizon_day"],
        bins=[0, 7, 30, 90, np.inf],
        labels=["1-7 days", "8-30 days", "31-90 days", "91+ days"],
    )

    return frame


# ----------------------------------------------------------------------
# Ensemble weighting
# ----------------------------------------------------------------------

def learn_ensemble_weights(validation, config):
    """Gunakan bobot manual per horizon atau optimasi grid sebagai fallback."""
    manual_weights = config.get("manual_horizon_weights")
    weight_mode = str(config.get("weight_mode", "optimize")).lower()

    if weight_mode == "manual" and manual_weights:
        rows = []
        for band, weights in manual_weights.items():
            rows.append({
                "horizon_band": str(band),
                "score": np.nan,
                "catboost_weight": float(weights["catboost"]),
                "neural_weight": float(weights["neural"]),
                "seasonal_weight": float(weights["seasonal"]),
            })
        return pd.DataFrame(rows, columns=[
            "horizon_band", "score", "catboost_weight",
            "neural_weight", "seasonal_weight"
        ])

    return _optimize_ensemble_weights(validation, config)


def _optimize_ensemble_weights(validation, config):
    rows = []
    models = ["catboost", "neural", "seasonal"]
    usable = validation.dropna(subset=["tma_mdpl"] + models)
    step = config["weight_step"]
    candidate_weights = np.arange(0, 1 + step / 2, step)

    for band, group in usable.groupby("horizon_band", observed=True):
        actual = group["tma_mdpl"].to_numpy()

        individual_rmse = []
        individual_mae = []

        for model in models:
            error = group[model].to_numpy() - actual
            individual_rmse.append(np.sqrt(np.mean(error ** 2)))
            individual_mae.append(np.mean(np.abs(error)))

        rmse_scale = max(min(individual_rmse), 1e-8)
        mae_scale = max(min(individual_mae), 1e-8)
        best = None

        for catboost_weight in candidate_weights:
            for neural_weight in candidate_weights:
                seasonal_weight = 1 - catboost_weight - neural_weight

                if seasonal_weight < -1e-9:
                    continue

                seasonal_weight = max(0, seasonal_weight)
                blended = (
                    catboost_weight * group["catboost"].to_numpy()
                    + neural_weight * group["neural"].to_numpy()
                    + seasonal_weight * group["seasonal"].to_numpy()
                )
                error = blended - actual
                rmse = np.sqrt(np.mean(error ** 2))
                mae = np.mean(np.abs(error))

                if config["ensemble_objective"] == "rmse":
                    score = rmse
                elif config["ensemble_objective"] == "mae":
                    score = mae
                else:
                    score = 0.5 * rmse / rmse_scale + 0.5 * mae / mae_scale

                if best is None or score < best["score"]:
                    best = {
                        "score": score,
                        "catboost_weight": float(catboost_weight),
                        "neural_weight": float(neural_weight),
                        "seasonal_weight": float(seasonal_weight),
                    }

        rows.append({"horizon_band": str(band), **best})

    return pd.DataFrame(
        rows,
        columns=[
            "horizon_band",
            "score",
            "catboost_weight",
            "neural_weight",
            "seasonal_weight",
        ],
    )


def apply_ensemble_weights(predictions, weights):
    predictions = predictions.copy()
    weight_table = weights.set_index("horizon_band")
    band = predictions["horizon_band"].astype(str)

    for model in ["catboost", "neural", "seasonal"]:
        weight_column = model + "_weight"
        predictions[weight_column] = band.map(weight_table[weight_column]).fillna(1 / 3)

    numerator = np.zeros(len(predictions))
    denominator = np.zeros(len(predictions))

    for model in ["catboost", "neural", "seasonal"]:
        weight = predictions[model + "_weight"].to_numpy()
        value = predictions[model].to_numpy()
        available = np.isfinite(value)
        numerator += np.where(available, weight * value, 0)
        denominator += np.where(available, weight, 0)

    predictions["tma_mdpl"] = np.divide(
        numerator,
        denominator,
        out=np.full(len(predictions), np.nan),
        where=denominator > 0,
    )

    return predictions


def calculate_validation_metrics(validation, weights, config):
    """Return RMSE, MSE, and MAE for every model and horizon band."""
    actual = validation["tma_mdpl"].copy()
    scored = apply_ensemble_weights(validation.drop(columns="tma_mdpl"), weights)
    scored = scored.rename(columns={"tma_mdpl": "ensemble"})
    scored["actual"] = actual.to_numpy()

    groups = [("Overall", scored)]
    groups += [
        (str(band), group)
        for band, group in scored.groupby("horizon_band", observed=True)
    ]

    rows = []

    for horizon_band, group in groups:
        for model in ["catboost", "neural", "seasonal", "ensemble"]:
            model_group = group.dropna(subset=["actual", model])
            error = model_group[model] - model_group["actual"]
            mse = np.mean(error ** 2)
            rows.append(
                {
                    "horizon_band": horizon_band,
                    "model": model,
                    "RMSE": np.sqrt(mse),
                    "MSE": mse,
                    "MAE": np.mean(np.abs(error)),
                    "rows": len(model_group),
                }
            )

    metrics = pd.DataFrame(rows)

    station_rows = []

    for station, group in scored.groupby("nama_pos", observed=True):
        for model in ["catboost", "neural", "seasonal", "ensemble"]:
            model_group = group.dropna(subset=["actual", model])
            error = model_group[model] - model_group["actual"]
            mse = np.mean(error ** 2)
            station_rows.append(
                {
                    "nama_pos": station,
                    "model": model,
                    "RMSE": np.sqrt(mse),
                    "MAE": np.mean(np.abs(error)),
                    "bias": np.mean(error),
                    "rows": len(model_group),
                }
            )

    station_metrics = pd.DataFrame(station_rows)
    metrics.attrs["station_metrics"] = station_metrics
    scored.attrs["station_metrics"] = station_metrics

    report_count = int(config["report_worst_stations"])

    if report_count > 0:
        worst = (
            station_metrics[station_metrics["model"].eq("ensemble")]
            .sort_values("RMSE", ascending=False)
            .head(report_count)
        )
        print("\nWorst validation stations (ensemble):")
        print(worst.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

    return metrics, scored


# ----------------------------------------------------------------------
# Model orchestration
# ----------------------------------------------------------------------

def run_models(panel, cutoff, horizon, catboost_features, neural_features, config):
    climatology, station_median = fit_climatology(panel, cutoff)
    data = apply_climatology(panel, climatology, station_median)

    future = data[data["datetime"].gt(cutoff)].copy()
    future = future.groupby("nama_pos", group_keys=False, observed=True).head(horizon)

    log_memory("before CatBoost fit", config)
    catboost_model = train_catboost(data, cutoff, catboost_features, config)
    catboost_prediction = predict_catboost(catboost_model, future, catboost_features)
    log_memory("after CatBoost fit", config)

    # PATCH: cap + rank neural exogenous features using CatBoost importance
    # before the (much heavier) neural fit.
    selected_neural_features = select_neural_features(
        neural_features, config, catboost_model, catboost_features
    )

    # PATCH: CatBoost model is no longer needed after this point in this call.
    free(catboost_model)

    neural_prediction = train_and_predict_neural(
        data,
        cutoff,
        horizon,
        selected_neural_features,
        config,
    )
    seasonal_prediction = predict_anchored_seasonal(
        data,
        cutoff,
        horizon,
        config,
    )

    prediction = catboost_prediction.merge(
        neural_prediction,
        on=["nama_pos", "datetime"],
        how="left",
        validate="one_to_one",
    )
    prediction = prediction.merge(
        seasonal_prediction,
        on=["nama_pos", "datetime"],
        how="left",
        validate="one_to_one",
    )

    missing_neural = prediction["neural"].isna().sum()

    if missing_neural:
        print(
            f"Warning: {missing_neural} neural rows are missing; "
            "using the other models as fallback."
        )

    prediction = add_horizon_band(prediction)

    # PATCH: drop large intermediates from this scope before returning.
    free(data, future, catboost_prediction, neural_prediction, seasonal_prediction)
    log_memory("end of run_models", config)

    return prediction


def choose_validation_cutoff(panel, train, test, horizon, config):
    """Choose a chronological cutoff, preferably matching the test season."""
    train_end = train["datetime"].max()
    training_times = sorted(
        panel.loc[panel["datetime"].le(train_end), "datetime"].unique()
    )

    if len(training_times) <= horizon:
        raise ValueError("Training history is shorter than the test horizon.")

    if config["validation_mode"] == "same_season":
        validation_start = test["datetime"].min() - pd.DateOffset(years=1)
        eligible_cutoffs = [time for time in training_times if time < validation_start]

        if eligible_cutoffs:
            cutoff = eligible_cutoffs[-1]
            future_times = [time for time in training_times if time > cutoff][:horizon]

            if len(future_times) == horizon:
                return cutoff, future_times[0], future_times[-1]

        print("Same-season validation was unavailable; using the last block.")

    cutoff = training_times[-horizon - 1]
    future_times = [time for time in training_times if time > cutoff][:horizon]
    return cutoff, future_times[0], future_times[-1]


# ----------------------------------------------------------------------
# Shared setup used by both entry points
# ----------------------------------------------------------------------

def _build_common_state(df_train, df_test, df_lingkungan, config):
    train, test, environment = prepare_data(df_train, df_test, df_lingkungan)
    environment = clean_environment(environment)

    if config["add_hydrology_features"]:
        print("Adding causal hydrology features...")
        environment = add_hydrology_features(environment)

    panel = build_panel(train, test, environment, config)

    # PATCH: environment no longer needed once merged into panel.
    free(environment)

    catboost_features, neural_features = get_model_features(panel)

    horizon_counts = test.groupby("nama_pos", observed=True).size()

    if not horizon_counts.eq(horizon_counts.iloc[0]).all():
        raise ValueError("Every station in df_test must have the same horizon.")

    horizon = int(horizon_counts.iloc[0])

    return train, test, panel, catboost_features, neural_features, horizon


# ----------------------------------------------------------------------
# PATCH: entry point 1 of 2 — validation phase only
# ----------------------------------------------------------------------

def run_validation_only(df_train, df_test, df_lingkungan, config=None):
    """Run only the (cheaper) validation phase and persist weights/metrics.

    Intended usage: run this in one Colab session, then restart the
    runtime before calling run_final_only() in a fresh session. This
    guarantees RAM used by validation is fully released before the
    heavier final fit begins.
    """
    config = prepare_config(config)

    print("Pipeline version:", PIPELINE_VERSION, "(validation phase)")
    log_memory("start of run_validation_only", config)

    train, test, panel, catboost_features, neural_features, horizon = (
        _build_common_state(df_train, df_test, df_lingkungan, config)
    )

    validation_cutoff, validation_start, validation_end = choose_validation_cutoff(
        panel, train, test, horizon, config
    )

    print(
        "Validation period:",
        pd.Timestamp(validation_start),
        "to",
        pd.Timestamp(validation_end),
    )

    print("\nFitting validation models...")
    validation_prediction = run_models(
        panel, validation_cutoff, horizon, catboost_features, neural_features, config
    )

    validation_actual = train[["nama_pos", "datetime", "tma_mdpl"]]
    validation_prediction = validation_prediction.merge(
        validation_actual, on=["nama_pos", "datetime"], how="left"
    )

    weights = learn_ensemble_weights(validation_prediction, config)
    metrics, validation_prediction = calculate_validation_metrics(
        validation_prediction, weights, config
    )

    print("\nLearned horizon weights:")
    print(weights.to_string(index=False))
    print("\nValidation metrics (lower is better):")
    print(metrics.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

    weights_path = Path(config["weights_output_file"]).expanduser()
    metrics_path = Path(config["metrics_output_file"]).expanduser()
    weights.to_csv(weights_path, index=False)
    metrics.to_csv(metrics_path, index=False)
    print(f"\nSaved weights to: {weights_path.resolve()}")
    print(f"Saved metrics to: {metrics_path.resolve()}")

    free(train, test, panel)
    release_gpu_memory()
    log_memory("end of run_validation_only", config)

    print(
        "\nIf you are on a tight-RAM Colab instance, this is a good point "
        "to Runtime > Restart runtime before calling run_final_only()."
    )

    return weights, metrics, validation_prediction


# ----------------------------------------------------------------------
# PATCH: entry point 2 of 2 — final fit + submission
# ----------------------------------------------------------------------

def run_final_only(
    df_train,
    df_test,
    df_lingkungan,
    df_submission=None,
    weights=None,
    weights_path=None,
    config=None,
):
    """Run only the final fit + submission phase.

    Provide either `weights` (a DataFrame from run_validation_only) or
    `weights_path` (a CSV saved by run_validation_only).
    """
    config = prepare_config(config)

    if weights is None:
        if weights_path is None:
            weights_path = config["weights_output_file"]
        weights = pd.read_csv(weights_path)

    print("Pipeline version:", PIPELINE_VERSION, "(final phase)")
    log_memory("start of run_final_only", config)

    train, test, panel, catboost_features, neural_features, horizon = (
        _build_common_state(df_train, df_test, df_lingkungan, config)
    )

    train_end = train["datetime"].max()

    print("\nFitting final models on all training data...")
    test_prediction = run_models(
        panel, train_end, horizon, catboost_features, neural_features, config
    )
    test_prediction = apply_ensemble_weights(test_prediction, weights)

    submission = test.merge(
        test_prediction[
            ["nama_pos", "datetime", "catboost", "neural", "seasonal", "tma_mdpl"]
        ],
        on=["nama_pos", "datetime"],
        how="left",
        indicator="_prediction_merge",
    )

    missing_prediction = submission[
        submission["tma_mdpl"].isna() | submission["_prediction_merge"].ne("both")
    ]

    if not missing_prediction.empty:
        examples = missing_prediction[
            [
                "nama_pos",
                "datetime",
                "_prediction_merge",
                "catboost",
                "neural",
                "seasonal",
                "tma_mdpl",
            ]
        ].head(5)
        raise ValueError(
            f"{len(missing_prediction)} test rows did not receive a prediction.\n"
            f"First unmatched rows:\n{examples.to_string(index=False)}"
        )

    submission = submission.drop(columns="_prediction_merge")
    prediction_by_id = submission.set_index("id")["tma_mdpl"]

    if df_submission is None:
        final_submission = test.sort_values("original_order")[["id"]].copy()
    else:
        final_submission = df_submission.copy()

    final_submission["tma_mdpl"] = final_submission["id"].map(prediction_by_id)

    if final_submission["tma_mdpl"].isna().any():
        raise ValueError("df_submission contains IDs that were not found in df_test.")

    output_path = Path(config["output_file"]).expanduser()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    final_submission.to_csv(output_path, index=False)

    if not output_path.exists() or output_path.stat().st_size == 0:
        raise OSError(f"Submission file was not written correctly: {output_path}")

    final_submission.attrs["output_file"] = str(output_path.resolve())
    print("\nSaved submission:", output_path.resolve())
    print("Submission rows:", len(final_submission))

    free(train, test, panel, test_prediction, submission)
    release_gpu_memory()
    log_memory("end of run_final_only", config)

    return final_submission


# ----------------------------------------------------------------------
# Convenience wrapper — same interface as the original pipeline
# ----------------------------------------------------------------------

def run_ensemble(df_train, df_test, df_lingkungan, df_submission=None, config=None):
    """
    Train, validate, and forecast with CatBoost + neural + seasonal models.

    This wrapper simply calls run_validation_only() then run_final_only()
    in the same process, for interface compatibility with the original
    pipeline. If you are hitting Colab RAM limits, prefer calling
    run_validation_only() and run_final_only() in two separate runtime
    sessions instead (see module docstring).

    Returns:
        submission             final test predictions
        metrics                RMSE, MSE, and MAE by model and horizon
        weights                learned ensemble weight per horizon band
        validation_prediction  validation predictions for further analysis
    """
    config = prepare_config(config)

    weights, metrics, validation_prediction = run_validation_only(
        df_train, df_test, df_lingkungan, config
    )

    # PATCH: explicit free + gc between phases, in addition to the GPU
    # cache clear. This does not guarantee OS-level memory release for
    # native allocators (CatBoost/PyTorch), but it does ensure Python-side
    # objects (large DataFrames) are eligible for collection immediately.
    gc.collect()
    release_gpu_memory()

    submission = run_final_only(
        df_train,
        df_test,
        df_lingkungan,
        df_submission,
        weights=weights,
        config=config,
    )

    return submission, metrics, weights, validation_prediction

In [ ]:
exp3_config = build_experiment_config(
    "experiment_3",
    device="gpu",
    catboost_task_type="GPU",
    output_file="/content/submission_exp3.csv",
)

submission, metrics, weights, validation = run_ensemble(
    df_train_clean,
    test_df,
    datling_df,
    df_submission,
    config=exp3_config,
)

display(metrics)
display(weights)
display(submission.head())


### 5.4 Eksperimen 4 — ST-GNN + CatBoost + N-HiTS + Seasonal


#### St-GNN

In [ ]:
# @title
"""
ST-GNN SPATIAL ENCODER — pipeline mandiri untuk mengubah hubungan antar pos
(koordinat_pos_v2.csv) menjadi fitur embedding baru yang lalu digabung ke
`panel` dan dipakai sebagai fitur tambahan oleh model utama (CatBoost /
NHITS / TiDE).

Modul ini mengimplementasikan langkah 6-13 dari diagram pipeline:

    6. SPATIAL GRAPH BUILDER      -> build_spatial_graph()
    7. NODE FEATURE BUILDER       -> build_node_static_features()
                                      + _pivot_dynamic_features()
    8. EDGE FEATURE BUILDER       -> build_edge_attributes()
    9. GRAPH ATTENTION ST-GNN     -> GraphAttentionLayer, STGNNEncoder,
       ENCODER                      train_stgnn_embeddings()
   10. SPATIAL EMBEDDING          -> generate_stgnn_features()
   11. EMBEDDING SELECTION        -> select_stgnn_embeddings()
   12. MERGE TO PANEL             -> merge_stgnn_to_panel()
   13. FEATURE LIST UPDATE        -> update_feature_lists()

Beda dengan versi GCN+GRU yang lebih sederhana di script ensemble utama:
  - Layer graf sekarang benar-benar GRAPH ATTENTION (bukan spectral GCN):
    bobot antar node dipelajari lewat attention, bukan cuma normalisasi
    derajat statis.
  - Attention memakai EDGE FEATURES (jarak sungai, beda orde aliran,
    kemiripan luas DAS, beda debit, arah aliran, travel time, dst) sebagai
    bagian dari skor attention -> "Edge Tensor (E x D)" di diagram.
  - Ada langkah EMBEDDING SELECTION eksplisit (variance filter ->
    correlation filter -> feature importance) sebelum embedding
    digabungkan ke panel, sesuai diagram butir 11.

Desain anti-leakage (sama seperti versi sebelumnya):
  - Graph (topologi + edge attribute) 100% statis dari atribut fisik pos
    di koordinat_pos_v2.csv -> TIDAK PERNAH memakai tma_mdpl.
  - Input encoder per timestep = fitur cuaca/kalender (exogenous) yang
    sudah tersedia di seluruh horizon (train + test), sama seperti
    futr_exog_list neural model.
  - Target (anomaly) hanya dipakai sebagai sinyal loss self-supervised,
    dan hanya dari baris datetime <= cutoff.

Cara pakai (drop-in, dipanggil dari dalam run_models()/add_stgnn_features()
di script ensemble utama, atau berdiri sendiri):

    graph = build_spatial_graph(df_koordinat, method="hydrology", k=6)

    embeddings_df, feature_list, encoder_state = fit_stgnn_pipeline(
        panel=panel,                 # sudah punya kolom anomaly, exogenous
        koordinat_df=df_koordinat,
        neural_features=neural_features,
        cutoff=cutoff,
        horizon=horizon,
        config=config,
    )

    panel, embedding_columns = merge_stgnn_to_panel(panel, embeddings_df)
    catboost_features, neural_features = update_feature_lists(
        catboost_features, neural_features, embedding_columns
    )
"""

import gc

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.neighbors import NearestNeighbors

RANDOM_SEED = 42
OBSERVATIONS_PER_DAY = 3  # samakan dengan script ensemble utama

# ----------------------------------------------------------------------
# Konfigurasi default khusus modul ini (bisa dioverride lewat `config`)
# ----------------------------------------------------------------------

STGNN_DEFAULTS = {
    "stgnn_graph_method": "hydrology",     # "hydrology" atau "knn_latlon"
    "stgnn_k_neighbors": 6,
    "stgnn_max_input_features": 24,        # cap fitur exogenous jadi input GAT
    "stgnn_gat_hidden_dim": 32,
    "stgnn_gat_heads": 2,
    "stgnn_edge_hidden_dim": 8,
    "stgnn_gru_hidden_dim": 32,
    "stgnn_embedding_dim": 16,
    "stgnn_input_days": 14,
    "stgnn_epochs": 60,
    "stgnn_batch_size": 32,
    "stgnn_learning_rate": 1e-3,
    "stgnn_weight_decay": 1e-5,
    "stgnn_dropout": 0.1,
    "stgnn_log_every": 20,
    # embedding selection
    "stgnn_variance_threshold": 1e-4,
    "stgnn_corr_threshold": 0.95,
    "stgnn_top_k_embeddings": 12,
}


def _cfg(config, key):
    if config is not None and key in config:
        return config[key]
    return STGNN_DEFAULTS[key]


def free(*objects):
    del objects
    gc.collect()


# ========================================================================
# 6. SPATIAL GRAPH BUILDER
# ========================================================================

def build_spatial_graph(koordinat_df, method="hydrology", k=6, self_loops=True):
    """Bangun graf antar pos dari koordinat_pos_v2.csv.

    Graf statis, dibangun dari kNN di ruang fitur fisik (bukan dari
    tma_mdpl), jadi aman dipakai di seluruh horizon (train + test) tanpa
    risiko kebocoran data.

    Kolom yang dipakai (jika tersedia di koordinat_pos_v2.csv):
        latitude, longitude, DIST_DN_KM, DIST_UP_KM, CATCH_SKM,
        UPLAND_SKM, DIS_AV_CMS, ORD_STRA, ORD_FLOW

    Returns dict:
        node_order          : list[str], urutan node -> indeks node
        adjacency_mask       : (N, N) bool tensor, True jika edge ada
        adjacency_norm        : (N, N) float tensor, adjacency ternormalisasi
                                (dipakai kalau butuh residual/GCN-style mix)
        edge_weight_raw      : (N, N) numpy, bobot kNN mentah (1/(1+dist))
        feature_columns_used : kolom yang dipakai untuk kNN
        koordinat            : DataFrame koordinat yang sudah dibersihkan
                                (index sejajar dengan node_order)
    """
    df = koordinat_df.drop_duplicates("nama_pos").reset_index(drop=True)
    node_order = df["nama_pos"].astype(str).tolist()
    n = len(node_order)

    if method == "hydrology":
        candidate_columns = [
            "latitude", "longitude", "DIST_DN_KM", "DIST_UP_KM",
            "CATCH_SKM", "UPLAND_SKM", "DIS_AV_CMS", "ORD_STRA", "ORD_FLOW",
        ]
    elif method == "knn_latlon":
        candidate_columns = ["latitude", "longitude"]
    else:
        raise ValueError("method harus 'hydrology' atau 'knn_latlon'.")

    feature_columns = [c for c in candidate_columns if c in df.columns]
    if not feature_columns:
        raise ValueError(f"Tidak ada kolom {candidate_columns} di koordinat_df.")

    features = df[feature_columns].astype("float64").to_numpy()
    mean = features.mean(axis=0)
    std = features.std(axis=0)
    std[std == 0] = 1.0
    features_scaled = (features - mean) / std

    k_eff = min(k + 1, n)
    nn_model = NearestNeighbors(n_neighbors=k_eff)
    nn_model.fit(features_scaled)
    distances, indices = nn_model.kneighbors(features_scaled)

    adjacency_raw = np.zeros((n, n), dtype="float64")
    for i in range(n):
        for rank, j in enumerate(indices[i]):
            if i == j:
                continue
            weight = 1.0 / (1.0 + distances[i, rank])
            adjacency_raw[i, j] = max(adjacency_raw[i, j], weight)

    # graf tidak berarah -> simetriskan
    adjacency_raw = np.maximum(adjacency_raw, adjacency_raw.T)

    adjacency_mask = adjacency_raw > 0

    if self_loops:
        np.fill_diagonal(adjacency_raw, np.maximum(adjacency_raw.diagonal(), 1.0))
        np.fill_diagonal(adjacency_mask, True)

    degree = adjacency_raw.sum(axis=1)
    degree[degree == 0] = 1.0
    d_inv_sqrt = np.diag(1.0 / np.sqrt(degree))
    adjacency_norm = d_inv_sqrt @ adjacency_raw @ d_inv_sqrt

    return {
        "node_order": node_order,
        "adjacency_mask": torch.tensor(adjacency_mask, dtype=torch.bool),
        "adjacency_norm": torch.tensor(adjacency_norm, dtype=torch.float32),
        "edge_weight_raw": adjacency_raw,
        "feature_columns_used": feature_columns,
        "koordinat": df,
    }


# ========================================================================
# 7. NODE FEATURE BUILDER — bagian statis
# ========================================================================

def build_node_static_features(graph, config=None):
    """Fitur statis per node: posisi + atribut hidrologi pos.

    Diagram menyebut "Elevation" sebagai fitur statis, tapi
    koordinat_pos_v2.csv TIDAK punya kolom elevasi -> kolom itu otomatis
    dilewati (tidak dipaksa diisi angka palsu). Kolom lain dipakai kalau
    tersedia.
    """
    df = graph["koordinat"]
    node_order = graph["node_order"]
    df = df.set_index("nama_pos").loc[node_order].reset_index()

    candidate_columns = [
        "latitude", "longitude", "CATCH_SKM", "UPLAND_SKM",
        "ORD_STRA", "ORD_FLOW", "DIS_AV_CMS", "DIST_UP_KM", "DIST_DN_KM",
    ]
    static_columns = [c for c in candidate_columns if c in df.columns]

    features = df[static_columns].astype("float64").to_numpy()
    # standardisasi per kolom supaya skala sebanding dengan fitur dinamis
    mean = features.mean(axis=0)
    std = features.std(axis=0)
    std[std == 0] = 1.0
    features_scaled = ((features - mean) / std).astype("float32")

    return torch.tensor(features_scaled, dtype=torch.float32), static_columns


# ========================================================================
# 7b. NODE FEATURE BUILDER — bagian dinamis (dari panel exogenous)
# ========================================================================

def select_stgnn_input_features(neural_features, config=None):
    """Cap kolom exogenous yang jadi input dinamis encoder (bukan target)."""
    limit = _cfg(config, "stgnn_max_input_features")
    override = None if config is None else config.get("stgnn_feature_columns")

    if override:
        return [c for c in override if c in neural_features]
    if not limit or len(neural_features) <= limit:
        return list(neural_features)
    return list(neural_features[:limit])


def _pivot_dynamic_features(panel, node_order, feature_columns):
    """Susun tensor dinamis (T, N, F) dari kolom exogenous. Causal-safe:
    fitur ini diketahui di seluruh horizon (train+test), persis seperti
    futr_exog_list yang dipakai neural model.
    """
    pivots = []
    for column in feature_columns:
        pivot = panel.pivot_table(
            index="datetime", columns="nama_pos", values=column, aggfunc="mean"
        )
        pivot = pivot.reindex(columns=node_order).sort_index()
        pivot = pivot.interpolate(limit_direction="both").fillna(0.0)
        pivots.append(pivot.to_numpy())

    all_datetimes = pivot.index
    stacked = np.stack(pivots, axis=-1)  # (T, N, F)
    return all_datetimes, torch.tensor(stacked, dtype=torch.float32)


def _pivot_target(panel, node_order, all_datetimes, target_column="anomaly"):
    """Target (T, N), NaN dibiarkan (jadi mask di loss self-supervised)."""
    pivot = panel.pivot_table(
        index="datetime", columns="nama_pos", values=target_column, aggfunc="mean"
    )
    pivot = pivot.reindex(index=all_datetimes, columns=node_order)
    return torch.tensor(pivot.to_numpy(), dtype=torch.float32)


# ========================================================================
# 8. EDGE FEATURE BUILDER
# ========================================================================

def build_edge_attributes(graph, config=None):
    """Hitung atribut edge (N, N, D) untuk setiap pasangan node yang
    terhubung di adjacency_mask. Semua atribut berasal dari koordinat
    fisik pos -> statis, tidak pernah menyentuh tma_mdpl.

    Atribut yang dihitung (tergantung kolom yang tersedia):
        - river_distance        : |DIST_DN_i - DIST_DN_j| (proxy jarak
                                   sepanjang aliran; makin kecil makin
                                   dekat secara hidrologi)
        - flow_direction        : DIST_DN_j - DIST_DN_i (tanda menunjukkan
                                   arah hilir/hulu antar pos)
        - river_order_diff      : |ORD_STRA_i - ORD_STRA_j|
        - catchment_similarity  : 1 / (1 + |log(CATCH_i) - log(CATCH_j)|)
        - mean_flow_diff        : |DIS_AV_CMS_i - DIS_AV_CMS_j|
        - connectivity_weight   : bobot kNN mentah (edge_weight_raw)
        - travel_time_proxy     : LENGTH_KM_i / max(DIS_AV_CMS_i, eps)
                                   (perkiraan kasar waktu tempuh air di pos
                                   sumber; bukan travel time antar-pos
                                   sesungguhnya karena tidak ada data rute)
        - great_circle_distance : jarak lat/lon (km, haversine) sebagai
                                   fallback kalau kolom sungai tidak ada

    NOTE: koordinat_pos_v2.csv tidak punya kolom elevasi, jadi
    "Elevation Difference" dari diagram TIDAK dihitung di sini.

    Returns:
        edge_attr : (N, N, D) float tensor, nol di pasangan yang tidak
                    terhubung (sudah otomatis di-mask oleh adjacency_mask
                    saat dipakai di attention layer)
        columns   : nama kolom edge_attr
    """
    df = graph["koordinat"].set_index("nama_pos").loc[graph["node_order"]]
    n = len(graph["node_order"])
    mask = graph["adjacency_mask"].numpy()
    eps = 1e-6

    attrs = {}

    if "DIST_DN_KM" in df.columns:
        dist_dn = df["DIST_DN_KM"].to_numpy()
        attrs["river_distance"] = np.abs(dist_dn[:, None] - dist_dn[None, :])
        attrs["flow_direction"] = dist_dn[None, :] - dist_dn[:, None]

    if "ORD_STRA" in df.columns:
        order = df["ORD_STRA"].to_numpy()
        attrs["river_order_diff"] = np.abs(order[:, None] - order[None, :])

    if "CATCH_SKM" in df.columns:
        log_catch = np.log(df["CATCH_SKM"].clip(lower=eps).to_numpy())
        attrs["catchment_similarity"] = 1.0 / (
            1.0 + np.abs(log_catch[:, None] - log_catch[None, :])
        )

    if "DIS_AV_CMS" in df.columns:
        flow = df["DIS_AV_CMS"].to_numpy()
        attrs["mean_flow_diff"] = np.abs(flow[:, None] - flow[None, :])

    if "LENGTH_KM" in df.columns and "DIS_AV_CMS" in df.columns:
        length = df["LENGTH_KM"].to_numpy()
        flow = df["DIS_AV_CMS"].clip(lower=eps).to_numpy()
        travel = length / flow
        attrs["travel_time_proxy"] = np.broadcast_to(travel[:, None], (n, n)).copy()

    if {"latitude", "longitude"}.issubset(df.columns):
        lat = np.radians(df["latitude"].to_numpy())
        lon = np.radians(df["longitude"].to_numpy())
        dlat = lat[:, None] - lat[None, :]
        dlon = lon[:, None] - lon[None, :]
        a = (
            np.sin(dlat / 2) ** 2
            + np.cos(lat[:, None]) * np.cos(lat[None, :]) * np.sin(dlon / 2) ** 2
        )
        attrs["great_circle_km"] = 2 * 6371.0 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

    attrs["connectivity_weight"] = graph["edge_weight_raw"].copy()

    columns = list(attrs.keys())
    stacked = np.stack([attrs[c] for c in columns], axis=-1)  # (N, N, D)

    # standardisasi tiap kolom edge (di antara edge yang benar-benar ada)
    for d in range(stacked.shape[-1]):
        values = stacked[:, :, d][mask]
        if values.size == 0:
            continue
        std = values.std()
        mean = values.mean()
        if std < eps:
            std = 1.0
        stacked[:, :, d] = (stacked[:, :, d] - mean) / std

    stacked[~mask] = 0.0

    return torch.tensor(stacked, dtype=torch.float32), columns


# ========================================================================
# 9. GRAPH ATTENTION ST-GNN ENCODER
# ========================================================================

class GraphAttentionLayer(nn.Module):
    """GAT dengan edge features, versi dense (cocok untuk graf kecil,
    puluhan node seperti jumlah pos di koordinat_pos_v2.csv).

    Skor attention e_ij dihitung dari [W h_i || W h_j || W_e edge_ij],
    lalu di-softmax hanya di antara tetangga (adjacency_mask). Ini
    mengimplementasikan "Graph Attention Layer" di diagram, dengan
    "Edge Tensor (E x D)" masuk sebagai bagian dari skor attention.
    """

    def __init__(self, in_dim, out_dim, edge_dim, heads=2, dropout=0.1):
        super().__init__()
        self.heads = heads
        self.out_dim = out_dim

        self.node_proj = nn.Linear(in_dim, out_dim * heads, bias=False)
        self.edge_proj = nn.Linear(edge_dim, out_dim * heads, bias=False)
        self.attn = nn.Parameter(torch.empty(heads, 3 * out_dim))
        nn.init.xavier_uniform_(self.attn)

        self.leaky_relu = nn.LeakyReLU(0.2)
        self.dropout = nn.Dropout(dropout)
        self.out_proj = nn.Linear(out_dim * heads, out_dim)

    def forward(self, node_features, adjacency_mask, edge_attr):
        # node_features: (batch, N, F_in)
        # adjacency_mask: (N, N) bool
        # edge_attr: (N, N, edge_dim)
        batch, n, _ = node_features.shape
        heads, out_dim = self.heads, self.out_dim

        wh = self.node_proj(node_features).view(batch, n, heads, out_dim)
        we = self.edge_proj(edge_attr).view(n, n, heads, out_dim)

        wh_i = wh.unsqueeze(2).expand(batch, n, n, heads, out_dim)   # target i
        wh_j = wh.unsqueeze(1).expand(batch, n, n, heads, out_dim)   # source j
        we_ij = we.unsqueeze(0).expand(batch, n, n, heads, out_dim)

        concat = torch.cat([wh_i, wh_j, we_ij], dim=-1)  # (batch,N,N,heads,3*out)
        scores = self.leaky_relu(
            torch.einsum("bijhd,hd->bijh", concat, self.attn)
        )  # (batch, N, N, heads)

        mask = adjacency_mask.view(1, n, n, 1)
        scores = scores.masked_fill(~mask, float("-inf"))
        alpha = torch.softmax(scores, dim=2)  # normalisasi atas j (sumber)
        alpha = torch.nan_to_num(alpha, nan=0.0)
        alpha = self.dropout(alpha)

        # PENTING: agregasi pakai `wh` (proyeksi asli, 4D: batch,N,heads,out_dim),
        # BUKAN `wh_j` yang sudah di-expand jadi 5D (batch,N,N,heads,out_dim) untuk
        # keperluan `concat` di atas. `wh` berperan sebagai fitur sumber (indeks j)
        # di sini karena einsum mengontraksi lewat label "j" pada `alpha`.
        out = torch.einsum("bijh,bjhd->bihd", alpha, wh)  # (batch,N,heads,out)
        out = out.reshape(batch, n, heads * out_dim)
        return self.out_proj(out)


class STGNNEncoder(nn.Module):
    """Graph Attention Layer -> Temporal GRU -> Graph Attention Layer ->
    Embedding Layer, persis urutan di diagram butir 9.

    Fitur node di setiap timestep = fitur dinamis (cuaca/kalender) +
    fitur statis pos (broadcast di semua waktu). Edge attribute statis
    dipakai di kedua GAT layer.
    """

    def __init__(self, dynamic_dim, static_dim, edge_dim, gat_hidden_dim,
                 gat_heads, gru_hidden_dim, embedding_dim, dropout=0.1):
        super().__init__()
        input_dim = dynamic_dim + static_dim

        self.gat_1 = GraphAttentionLayer(
            input_dim, gat_hidden_dim, edge_dim, heads=gat_heads, dropout=dropout
        )
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

        self.gru = nn.GRU(
            input_size=gat_hidden_dim, hidden_size=gru_hidden_dim, batch_first=True
        )

        self.gat_2 = GraphAttentionLayer(
            gru_hidden_dim, gat_hidden_dim, edge_dim, heads=gat_heads, dropout=dropout
        )

        self.embedding_head = nn.Linear(gat_hidden_dim, embedding_dim)
        self.prediction_head = nn.Linear(embedding_dim, 1)

    def forward(self, window, static_features, adjacency_mask, edge_attr):
        # window: (batch, L, N, F_dyn)
        batch, seq_len, n, _ = window.shape
        static_expanded = static_features.view(1, 1, n, -1).expand(
            batch, seq_len, n, -1
        )
        node_input = torch.cat([window, static_expanded], dim=-1)

        gat1_out = []
        for t in range(seq_len):
            step_out = self.activation(
                self.gat_1(node_input[:, t], adjacency_mask, edge_attr)
            )
            gat1_out.append(step_out)
        gat1_out = torch.stack(gat1_out, dim=1)  # (batch, L, N, H)

        gru_input = gat1_out.permute(0, 2, 1, 3).reshape(batch * n, seq_len, -1)
        gru_input = self.dropout(gru_input)
        _, hidden = self.gru(gru_input)
        hidden = hidden.squeeze(0).view(batch, n, -1)

        refined = self.activation(self.gat_2(hidden, adjacency_mask, edge_attr))
        embedding = self.embedding_head(refined)
        prediction = self.prediction_head(embedding).squeeze(-1)
        return embedding, prediction


def _masked_mse(prediction, target):
    mask = torch.isfinite(target)
    if mask.sum() == 0:
        return prediction.sum() * 0.0
    safe_target = torch.where(mask, target, torch.zeros_like(target))
    error = (prediction - safe_target) * mask.float()
    return (error ** 2).sum() / mask.float().sum()


class _CausalWindowDataset(torch.utils.data.Dataset):
    """Sample = window fitur [idx-L : idx] -> target di idx.

    Hanya index dengan target valid dan datetime <= cutoff_step yang
    dipakai, supaya training causal terhadap cutoff yang sama dipakai
    CatBoost/neural model lain di ensemble.
    """

    def __init__(self, features, targets, input_len, max_step_inclusive):
        self.features = features
        self.targets = targets
        self.input_len = input_len
        self.valid_indices = [
            idx
            for idx in range(input_len, min(targets.shape[0], max_step_inclusive + 1))
            if torch.isfinite(targets[idx]).any()
        ]

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, position):
        idx = self.valid_indices[position]
        window = self.features[idx - self.input_len: idx]
        target = self.targets[idx]
        return window, target


def train_stgnn_embeddings(panel, graph, edge_attr, static_features,
                            feature_columns, cutoff, horizon,
                            target_column="anomaly", config=None):
    """Latih ST-GNN encoder secara causal terhadap `cutoff`.

    Input dibangun dari seluruh panel (train+test datetimes) karena fitur
    exogenous diketahui di semua waktu — hanya TARGET yang dibatasi
    <= cutoff, dan hanya baris itu yang dipakai untuk update bobot.
    """
    torch.manual_seed(RANDOM_SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if config is not None and "torch_device" in config:
        device = config["torch_device"]

    node_order = graph["node_order"]
    adjacency_mask = graph["adjacency_mask"].to(device)
    edge_attr = edge_attr.to(device)
    static_features = static_features.to(device)

    all_datetimes, feature_tensor = _pivot_dynamic_features(
        panel, node_order, feature_columns
    )
    target_tensor = _pivot_target(panel, node_order, all_datetimes, target_column)

    input_len = _cfg(config, "stgnn_input_days") * OBSERVATIONS_PER_DAY
    cutoff_step = int(
        np.searchsorted(all_datetimes.values, np.datetime64(cutoff), side="right") - 1
    )

    if cutoff_step <= input_len:
        raise ValueError(
            f"Histori sampai cutoff terlalu pendek ({cutoff_step} step) untuk "
            f"stgnn_input_days={_cfg(config, 'stgnn_input_days')}. "
            "Perkecil config['stgnn_input_days']."
        )

    dataset = _CausalWindowDataset(feature_tensor, target_tensor, input_len, cutoff_step)
    if len(dataset) == 0:
        raise ValueError("Tidak ada sample training ST-GNN yang valid sampai cutoff ini.")

    loader = torch.utils.data.DataLoader(
        dataset, batch_size=_cfg(config, "stgnn_batch_size"), shuffle=True, drop_last=False
    )

    model = STGNNEncoder(
        dynamic_dim=len(feature_columns),
        static_dim=static_features.shape[-1],
        edge_dim=edge_attr.shape[-1],
        gat_hidden_dim=_cfg(config, "stgnn_gat_hidden_dim"),
        gat_heads=_cfg(config, "stgnn_gat_heads"),
        gru_hidden_dim=_cfg(config, "stgnn_gru_hidden_dim"),
        embedding_dim=_cfg(config, "stgnn_embedding_dim"),
        dropout=_cfg(config, "stgnn_dropout"),
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=_cfg(config, "stgnn_learning_rate"),
        weight_decay=_cfg(config, "stgnn_weight_decay"),
    )

    epochs = _cfg(config, "stgnn_epochs")
    log_every = _cfg(config, "stgnn_log_every")

    model.train()
    for epoch in range(epochs):
        epoch_loss, n_batches = 0.0, 0
        for window, target in loader:
            window = window.to(device)
            target = target.to(device)

            optimizer.zero_grad()
            _, prediction = model(window, static_features, adjacency_mask, edge_attr)
            loss = _masked_mse(prediction, target)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        if (epoch + 1) % log_every == 0 or epoch == 0:
            print(f"[ST-GNN/GAT] epoch {epoch + 1}/{epochs} "
                  f"loss={epoch_loss / max(n_batches, 1):.5f}")

    model.eval()

    state = {
        "device": device,
        "node_order": node_order,
        "feature_columns": feature_columns,
        "input_len": input_len,
        "embedding_dim": _cfg(config, "stgnn_embedding_dim"),
        "all_datetimes": all_datetimes,
        "feature_tensor": feature_tensor,
        "static_features": static_features,
        "edge_attr": edge_attr,
        "adjacency_mask": adjacency_mask,
    }
    return model, state


# ========================================================================
# 10. SPATIAL EMBEDDING (generate)
# ========================================================================

@torch.no_grad()
def generate_stgnn_features(model, state):
    """Hasilkan embedding causal untuk SETIAP (nama_pos, datetime) di
    panel, termasuk baris future/test — forward pass hanya memakai fitur
    exogenous yang memang sudah diketahui di semua waktu.

    Returns DataFrame: nama_pos, datetime, stgnn_emb_0 .. stgnn_emb_{D-1}
    """
    device = state["device"]
    node_order = state["node_order"]
    input_len = state["input_len"]
    embedding_dim = state["embedding_dim"]
    all_datetimes = state["all_datetimes"]
    feature_tensor = state["feature_tensor"].to(device)
    static_features = state["static_features"]
    adjacency_mask = state["adjacency_mask"]
    edge_attr = state["edge_attr"]

    n_steps = feature_tensor.shape[0]
    embeddings = np.full((n_steps, len(node_order), embedding_dim), np.nan, dtype="float32")

    model.eval()
    for t in range(input_len, n_steps):
        window = feature_tensor[t - input_len:t].unsqueeze(0)
        embedding, _ = model(window, static_features, adjacency_mask, edge_attr)
        embeddings[t] = embedding.squeeze(0).cpu().numpy()

    embedding_columns = [f"stgnn_emb_{i}" for i in range(embedding_dim)]
    rows = []
    for t_idx, timestamp in enumerate(all_datetimes):
        if np.isnan(embeddings[t_idx]).all():
            continue
        for n_idx, station in enumerate(node_order):
            rows.append([station, timestamp] + embeddings[t_idx, n_idx].tolist())

    result = pd.DataFrame(rows, columns=["nama_pos", "datetime"] + embedding_columns)
    result["nama_pos"] = result["nama_pos"].astype("string")
    result[embedding_columns] = result[embedding_columns].astype("float32")
    return result, embedding_columns


# ========================================================================
# 11. EMBEDDING SELECTION
# ========================================================================

def select_stgnn_embeddings(embeddings_df, embedding_columns, panel=None,
                             target_column="anomaly", config=None):
    """Variance filter -> Correlation filter -> Feature importance ->
    Top-K, sesuai diagram butir 11.

    - Variance filter  : buang kolom embedding yang nyaris konstan.
    - Correlation filter: dari sepasang kolom yang korelasinya > threshold,
                           buang salah satu (redundan).
    - Feature importance: kalau `panel` + target_column tersedia, ranking
                           sisa kolom pakai |korelasi Pearson| ke target
                           (memakai baris dengan datetime <= cutoff yang
                           sudah tersimpan di panel['anomaly'] non-null,
                           jadi tetap causal — tidak memakai baris test).
                           Kalau tidak tersedia, urutan asli dipertahankan.
    - Top-K             : ambil K kolom teratas.

    Returns: list nama kolom embedding terpilih.
    """
    variance_threshold = _cfg(config, "stgnn_variance_threshold")
    corr_threshold = _cfg(config, "stgnn_corr_threshold")
    top_k = _cfg(config, "stgnn_top_k_embeddings")

    # --- variance filter ---
    variances = embeddings_df[embedding_columns].var()
    kept = variances[variances > variance_threshold].index.tolist()
    if not kept:
        kept = list(embedding_columns)

    # --- correlation filter ---
    corr_matrix = embeddings_df[kept].corr().abs()
    to_drop = set()
    for i, col_i in enumerate(kept):
        if col_i in to_drop:
            continue
        for col_j in kept[i + 1:]:
            if col_j in to_drop:
                continue
            if corr_matrix.loc[col_i, col_j] > corr_threshold:
                to_drop.add(col_j)
    kept = [c for c in kept if c not in to_drop]

    # --- feature importance (korelasi ke target, causal-safe) ---
    if panel is not None and target_column in panel.columns:
        merged = panel[["nama_pos", "datetime", target_column]].merge(
            embeddings_df[["nama_pos", "datetime"] + kept],
            on=["nama_pos", "datetime"], how="inner",
        ).dropna(subset=[target_column])

        if len(merged) > 10:
            importance = merged[kept].apply(
                lambda col: abs(col.corr(merged[target_column]))
            ).fillna(0)
            kept = importance.sort_values(ascending=False).index.tolist()

    # --- top-K ---
    selected = kept[:top_k] if top_k else kept

    print(
        f"[ST-GNN] embedding selection: {len(embedding_columns)} -> "
        f"{len(kept)} (setelah variance+correlation filter) -> "
        f"{len(selected)} (top-{top_k})"
    )
    return selected


# ========================================================================
# 12. MERGE TO PANEL
# ========================================================================

def merge_stgnn_to_panel(panel, embeddings_df, selected_columns):
    """Gabungkan embedding terpilih ke panel ON (nama_pos, datetime)."""
    to_merge = embeddings_df[["nama_pos", "datetime"] + selected_columns]
    panel = panel.merge(to_merge, on=["nama_pos", "datetime"], how="left")
    panel[selected_columns] = panel[selected_columns].fillna(0.0)
    return panel


# ========================================================================
# 13. FEATURE LIST UPDATE
# ========================================================================

def update_feature_lists(catboost_features, neural_features, selected_columns):
    """Tambahkan kolom embedding terpilih ke daftar fitur CatBoost & neural."""
    catboost_features = list(catboost_features) + list(selected_columns)
    neural_features = list(neural_features) + list(selected_columns)
    return catboost_features, neural_features


# ========================================================================
# ORKESTRASI — satu pemanggilan untuk seluruh pipeline encoder spasial
# ========================================================================

def fit_stgnn_pipeline(panel, koordinat_df, neural_features, cutoff, horizon,
                        target_column="anomaly", config=None):
    """Jalankan langkah 6-11 sekaligus: graph -> node/edge feature ->
    latih GAT-ST-GNN -> generate embedding -> seleksi embedding.

    Tidak melakukan merge ke panel (langkah 12) supaya caller bisa
    memutuskan sendiri kapan merge terjadi (biasanya dalam run_models()
    di script ensemble utama, seperti add_stgnn_features() sebelumnya).

    Returns:
        embeddings_df     : DataFrame nama_pos, datetime, stgnn_emb_*
        selected_columns   : kolom embedding yang lolos seleksi
        graph              : dict graph (bisa dipakai ulang lintas cutoff
                              karena topologinya statis)
    """
    graph = build_spatial_graph(
        koordinat_df,
        method=_cfg(config, "stgnn_graph_method"),
        k=_cfg(config, "stgnn_k_neighbors"),
    )

    missing_nodes = set(panel["nama_pos"].unique()) - set(graph["node_order"])
    if missing_nodes:
        raise ValueError(
            f"Pos berikut ada di panel tapi tidak ada di koordinat_df: {missing_nodes}"
        )

    static_features, static_columns = build_node_static_features(graph, config)
    edge_attr, edge_columns = build_edge_attributes(graph, config)
    feature_columns = select_stgnn_input_features(neural_features, config)

    print(f"[ST-GNN] node statis: {static_columns}")
    print(f"[ST-GNN] edge attr : {edge_columns}")
    print(f"[ST-GNN] input dinamis ({len(feature_columns)}): {feature_columns}")

    model, state = train_stgnn_embeddings(
        panel, graph, edge_attr, static_features, feature_columns,
        cutoff, horizon, target_column=target_column, config=config,
    )
    embeddings_df, embedding_columns = generate_stgnn_features(model, state)

    selected_columns = select_stgnn_embeddings(
        embeddings_df, embedding_columns, panel=panel,
        target_column=target_column, config=config,
    )

    free(model, state)

    return embeddings_df, selected_columns, graph


# ========================================================================
# Contoh integrasi ke run_models() milik script ensemble utama
# (referensi saja — tidak dieksekusi otomatis oleh modul ini)
# ========================================================================
"""
def add_stgnn_features(panel, koordinat_df, cutoff, horizon, neural_features, config):
    embeddings_df, selected_columns, graph = fit_stgnn_pipeline(
        panel, koordinat_df, neural_features, cutoff, horizon,
        target_column="anomaly", config=config,
    )
    panel = merge_stgnn_to_panel(panel, embeddings_df, selected_columns)
    return panel, selected_columns


# di dalam run_models():
if config.get("use_stgnn"):
    data, embedding_columns = add_stgnn_features(
        data, df_koordinat, cutoff, horizon, neural_features, config
    )
    catboost_features, neural_features = update_feature_lists(
        catboost_features, neural_features, embedding_columns
    )
"""

In [ ]:
# @title
"""
CatBoost + ST-GNN + neural + seasonal ensemble for forecasting tma_mdpl.
HARDENED VERSION — memory-safe + spatial encoder (ST-GNN) terintegrasi.

Pipeline:
    Weather -> Feature engineering -> Spatial feature -> Graph construction
    -> ST-GNN (spatial encoder) -> Feature fusion -> CatBoost + Neural + Seasonal
    ensemble -> Submission

ST-GNN di sini HANYA menghasilkan fitur embedding tambahan per
(nama_pos, datetime) yang ditambahkan ke `panel` sebelum CatBoost/neural
dilatih — dia bukan model prediksi akhir, hanya spatial encoder. Karena
itu tidak ada perubahan pada logika ensemble weighting/horizon banding
yang sudah ada.

Poin penting soal desain ST-GNN (supaya tidak bocor data):
  - Graph (topologi + bobot edge) statis, dibangun dari atribut fisik pos
    (koordinat_pos: jarak sungai, luas DAS, debit rata-rata, orde aliran)
    — tidak pernah memakai tma_mdpl.
  - Input ST-GNN per timestep adalah fitur CUACA/kalender (exogenous),
    BUKAN anomaly/tma_mdpl. Fitur-fitur ini sudah tersedia untuk seluruh
    horizon (termasuk periode test), persis seperti futr_exog_list yang
    dipakai NHITS/TiDE di bagian neural model. Karena itu embedding aman
    dihasilkan untuk baris future/test tanpa menyentuh target.
  - Target (anomaly) hanya dipakai sebagai sinyal loss self-supervised
    saat training, dan hanya dari baris dengan datetime <= cutoff — sama
    persis pola cutoff yang sudah dipakai train_catboost/train_and_predict_neural.

Install dependencies:
    pip install pandas numpy catboost neuralforecast torch psutil scikit-learn

Pemakaian (sama seperti sebelumnya, tambah satu argumen df_koordinat):
    submission, metrics, weights, validation = run_ensemble(
        df_train, df_test, df_lingkungan, df_koordinat, df_submission,
        config={"device": "auto"},
    )

Atau split dua sesi Colab seperti sebelumnya:
    weights, metrics, validation = run_validation_only(
        df_train, df_test, df_lingkungan, df_koordinat, config={"device": "auto"}
    )
    # Runtime > Restart runtime di sini kalau perlu.
    submission = run_final_only(
        df_train, df_test, df_lingkungan, df_koordinat, df_submission,
        weights_path="weights.csv", config={"device": "auto"},
    )
"""

import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from catboost import CatBoostRegressor
from neuralforecast import NeuralForecast
from neuralforecast.losses.pytorch import MAE
from neuralforecast.models import NHITS, TiDE
from sklearn.neighbors import NearestNeighbors
from stgnn_spatial_encoder import fit_stgnn_pipeline, merge_stgnn_to_panel, update_feature_lists

try:
    import psutil
    _HAS_PSUTIL = True
except ImportError:
    _HAS_PSUTIL = False


OBSERVATIONS_PER_DAY = 3
RANDOM_SEED = 42
PIPELINE_VERSION = "2026-07-hydrologic-ensemble-v5-stgnn"


DEFAULT_CONFIG = {
    "device": "auto",                 # "auto", "gpu", or "cpu"
    "gpu_index": 0,
    "precision": "32-true",
    "neural_model": "nhits",          # "nhits" (recommended) or "tide"
    "input_days": 120,
    "tide_hidden_size": 64,
    "tide_batch_size": 8,
    "tide_windows_batch_size": 32,
    "tide_inference_batch_size": 32,
    "tide_max_steps": 300,
    "catboost_task_type": "CPU",
    "catboost_iterations": 800,
    "catboost_depth": 7,
    "catboost_gpu_ram_part": 0.25,
    "add_hydrology_features": True,
    "report_worst_stations": 10,
    "seasonal_window_days": 15,
    "seasonal_anchor_half_life_days": 3,
    "weight_step": 0.05,
    "ensemble_objective": "balanced",
    "validation_mode": "same_season",
    "output_file": "submission_ens.csv",

    "max_panel_rows": 5_000_000,      # guard against runaway cross-join
    "neural_max_exog_features": 40,   # cap neural exogenous feature count
    "log_memory": True,               # print RAM usage at checkpoints
    "weights_output_file": "weights.csv",
    "metrics_output_file": "metrics.csv",

    # --- ST-GNN spatial encoder knobs ---
    "use_stgnn": True,
    "stgnn_graph_method": "hydrology",     # "hydrology" atau "knn_latlon"
    "stgnn_k_neighbors": 6,
    "stgnn_max_input_features": 24,        # cap kolom exogenous yang jadi input GCN
    "stgnn_embedding_dim": 16,
    "stgnn_gcn_hidden_dim": 32,
    "stgnn_gru_hidden_dim": 32,
    "stgnn_input_days": 14,
    "stgnn_epochs": 60,
    "stgnn_learning_rate": 1e-3,
    "stgnn_weight_decay": 1e-5,
    "stgnn_dropout": 0.1,
    "stgnn_log_every": 20,
}


# ----------------------------------------------------------------------
# Memory helpers
# ----------------------------------------------------------------------

def log_memory(tag, config=None):
    """Print current host RAM usage. Cheap, but very useful for diagnosis."""
    if config is not None and not config.get("log_memory", True):
        return

    if not _HAS_PSUTIL:
        return

    mem = psutil.virtual_memory()
    print(
        f"[mem] {tag}: {mem.percent:.1f}% used "
        f"({mem.used / 1e9:.2f} GB / {mem.total / 1e9:.2f} GB)"
    )


def downcast_numeric(frame):
    """Downcast float64 -> float32 and shrink integer dtypes where safe."""
    frame = frame.copy()

    float_columns = frame.select_dtypes(include="float64").columns
    if len(float_columns):
        frame[float_columns] = frame[float_columns].astype("float32")

    int_columns = frame.select_dtypes(include="int64").columns
    for column in int_columns:
        frame[column] = pd.to_numeric(frame[column], downcast="integer")

    return frame


def free(*objects):
    """Delete references and force garbage collection."""
    del objects
    gc.collect()


def safe_fillna(series, fill_value):
    """fillna() that never triggers pandas' LossySetitemError."""
    if isinstance(fill_value, (pd.Series, np.ndarray)):
        fill_value = pd.Series(fill_value).astype(series.dtype)
    else:
        fill_value = series.dtype.type(fill_value)
    return series.fillna(fill_value)


# ----------------------------------------------------------------------
# Config handling
# ----------------------------------------------------------------------

def prepare_config(config=None):
    """Merge user settings and select GPU or CPU safely."""
    settings = DEFAULT_CONFIG.copy()

    if config is not None:
        settings.update(config)

    requested_device = str(settings["device"]).lower()

    if requested_device not in {"auto", "gpu", "cpu"}:
        raise ValueError("config['device'] must be 'auto', 'gpu', or 'cpu'.")

    gpu_available = torch.cuda.is_available()

    if requested_device == "gpu" and not gpu_available:
        print("CUDA is unavailable. The neural model will use CPU instead.")

    use_gpu = gpu_available and requested_device in {"auto", "gpu"}
    settings["accelerator"] = "gpu" if use_gpu else "cpu"
    settings["precision"] = settings["precision"] if use_gpu else "32-true"
    settings["devices"] = [settings["gpu_index"]] if use_gpu else 1
    settings["torch_device"] = torch.device("cuda" if use_gpu else "cpu")

    catboost_device = str(settings["catboost_task_type"]).upper()

    if catboost_device not in {"CPU", "GPU"}:
        raise ValueError("config['catboost_task_type'] must be 'CPU' or 'GPU'.")

    if catboost_device == "GPU" and not gpu_available:
        print("CUDA is unavailable. CatBoost will use CPU instead.")
        catboost_device = "CPU"

    settings["catboost_task_type"] = catboost_device

    if settings["ensemble_objective"] not in {"balanced", "rmse", "mae"}:
        raise ValueError(
            "config['ensemble_objective'] must be 'balanced', 'rmse', or 'mae'."
        )

    if settings["validation_mode"] not in {"same_season", "last_block"}:
        raise ValueError(
            "config['validation_mode'] must be 'same_season' or 'last_block'."
        )

    settings["neural_model"] = str(settings["neural_model"]).lower()

    if settings["neural_model"] not in {"nhits", "tide"}:
        raise ValueError("config['neural_model'] must be 'nhits' or 'tide'.")

    if settings["stgnn_graph_method"] not in {"hydrology", "knn_latlon"}:
        raise ValueError(
            "config['stgnn_graph_method'] must be 'hydrology' or 'knn_latlon'."
        )

    return settings


def release_gpu_memory():
    """Release cached GPU memory before fitting the next neural model."""
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ----------------------------------------------------------------------
# Data preparation
# ----------------------------------------------------------------------

def prepare_data(train_df_clean, test_df, datling_df):
    """Copy and standardize the DataFrames without changing the originals."""
    train = train_df_clean.copy()
    test = test_df.copy()
    environment = datling_df.copy()

    train["datetime"] = pd.to_datetime(train["datetime"])
    environment["datetime"] = pd.to_datetime(environment["datetime"])

    if "datetime" not in test.columns or "nama_pos" not in test.columns:
        parsed = test["id"].str.extract(
            r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) - (.*)$"
        )

        if parsed.isna().any().any():
            raise ValueError("Some test IDs do not match 'datetime - nama_pos'.")

        test["datetime"] = pd.to_datetime(parsed[0])
        test["nama_pos"] = parsed[1]
    else:
        test["datetime"] = pd.to_datetime(test["datetime"])

    if "id" not in test.columns:
        test["id"] = test["datetime"].astype(str) + " - " + test["nama_pos"]

    test["original_order"] = np.arange(len(test))

    train["nama_pos"] = train["nama_pos"].astype("string")
    test["nama_pos"] = test["nama_pos"].astype("string")
    environment["nama_pos"] = environment["nama_pos"].astype("string")

    return train, test, environment


def prepare_koordinat(koordinat_df):
    """Standardize the spatial/hydrology attribute table used for the graph."""
    koordinat = koordinat_df.copy()
    koordinat["nama_pos"] = koordinat["nama_pos"].astype("string")
    koordinat = koordinat.drop_duplicates("nama_pos").reset_index(drop=True)
    return koordinat


def clean_environment(environment):
    environment = environment.copy()

    if "solar_radiation_mj_m2" in environment.columns:
        bad_solar = environment["solar_radiation_mj_m2"].eq(-999)
        environment["solar_radiation_missing"] = bad_solar.astype(int)
        environment.loc[bad_solar, "solar_radiation_mj_m2"] = np.nan

    environment = environment.drop_duplicates(["nama_pos", "datetime"])
    environment = environment.sort_values(["nama_pos", "datetime"])

    numeric_columns = environment.select_dtypes(include="number").columns.tolist()

    for column in numeric_columns:
        environment[column] = environment[column].replace([np.inf, -np.inf], np.nan)

        if environment[column].isna().any():
            environment[column + "_missing"] = environment[column].isna().astype(int)
            environment[column] = environment.groupby("nama_pos", observed=True)[column].transform(
                lambda values: values.interpolate(limit_direction="both")
            )
            environment[column] = environment[column].fillna(environment[column].median())

    # NOTE (bugfix): do NOT downcast here — see build_panel() for details.
    return environment


def _grouped_rolling(series, group_key, window, min_periods=1, agg="sum"):
    rolled = series.groupby(group_key, observed=True).rolling(
        window, min_periods=min_periods
    )
    result = getattr(rolled, agg)()
    return result.reset_index(level=0, drop=True).sort_index()


def _grouped_ewm(series, group_key, alpha):
    ewm = series.groupby(group_key, observed=True).ewm(alpha=alpha, adjust=False)
    result = ewm.mean()
    return result.reset_index(level=0, drop=True).sort_index()


def add_hydrology_features(environment):
    """Add causal rainfall-memory, soil-state, and weather-tendency features."""
    environment = environment.copy()
    environment = environment.sort_values(["nama_pos", "datetime"])
    station = environment["nama_pos"]

    if {"rainfall_mm", "rainfall_openmeteo_mm"}.issubset(environment.columns):
        left = environment["rainfall_mm"].to_numpy()
        right = environment["rainfall_openmeteo_mm"].to_numpy()

        if np.allclose(left, right, equal_nan=True):
            environment = environment.drop(columns="rainfall_openmeteo_mm")

    if "rainfall_mm" in environment.columns:
        rain = environment["rainfall_mm"].clip(lower=0).fillna(0)

        for days in [1, 3, 7, 14, 30]:
            hours = 24 * days
            environment[f"rain_sum_{days}d"] = _grouped_rolling(
                rain, station, hours, min_periods=1, agg="sum"
            )

        for days in [1, 3, 7]:
            hours = 24 * days
            environment[f"rain_max_{days}d"] = _grouped_rolling(
                rain, station, hours, min_periods=1, agg="max"
            )

        wet = rain.gt(0.1).astype("float32")

        for days in [3, 7]:
            hours = 24 * days
            environment[f"wet_fraction_{days}d"] = _grouped_rolling(
                wet, station, hours, min_periods=1, agg="mean"
            )

        for retention in [0.70, 0.85, 0.93]:
            hourly_retention = retention ** (1 / 24)
            alpha = 1 - hourly_retention
            name = f"api_{int(retention * 100):03d}"
            environment[name] = _grouped_ewm(rain, station, alpha)

        rain_event = rain.gt(0.1)
        event_number = rain_event.groupby(station, observed=True).cumsum()
        environment["hours_since_rain"] = (~rain_event).groupby(
            [station, event_number], observed=True
        ).cumsum().clip(upper=24 * 60)

    soil_columns = [
        column
        for column in [
            "soil_moisture_0_7cm",
            "soil_moisture_7_28cm",
            "soil_moisture_28_100cm",
            "soil_moisture_100_255cm",
        ]
        if column in environment.columns
    ]

    if soil_columns:
        shallow = soil_columns[:2]
        deep = soil_columns[2:]
        environment["soil_shallow_mean"] = environment[shallow].mean(axis=1)
        environment["soil_deep_mean"] = (
            environment[deep].mean(axis=1)
            if deep
            else environment[soil_columns].mean(axis=1)
        )

        interaction_soils = ["soil_shallow_mean", "soil_deep_mean"]

        for rain_feature in ["api_085", "rain_sum_3d", "rain_sum_7d"]:
            if rain_feature in environment.columns:
                for soil_feature in interaction_soils:
                    name = rain_feature + "_x_" + soil_feature
                    environment[name] = (
                        environment[rain_feature] * environment[soil_feature]
                    )

    if {"temperature_c", "dew_point_c"}.issubset(environment.columns):
        environment["dewpoint_depression"] = (
            environment["temperature_c"] - environment["dew_point_c"]
        )

    if {"cloud_cover_pct", "humidity_pct"}.issubset(environment.columns):
        environment["cloud_x_humidity"] = (
            environment["cloud_cover_pct"] * environment["humidity_pct"] / 100
        )

    tendency_columns = [
        column
        for column in soil_columns
        + [
            "humidity_pct",
            "temperature_c",
            "surface_pressure_hpa",
            "pressure_msl_hpa",
            "cloud_cover_pct",
            "dew_point_c",
        ]
        if column in environment.columns
    ]

    for column in tendency_columns:
        for days in [1, 3]:
            lag = 24 * days
            environment[f"{column}_change_{days}d"] = (
                environment[column]
                - environment.groupby("nama_pos", observed=True)[column].shift(lag)
            ).fillna(0)

    return environment


def add_calendar_features(frame):
    frame = frame.copy()
    day = frame["datetime"].dt.dayofyear
    hour = frame["datetime"].dt.hour

    frame["month"] = frame["datetime"].dt.month
    frame["day_of_year"] = day
    frame["hour"] = hour
    frame["day_sin"] = np.sin(2 * np.pi * day / 365.25).astype("float32")
    frame["day_cos"] = np.cos(2 * np.pi * day / 365.25).astype("float32")
    frame["hour_sin"] = np.sin(2 * np.pi * hour / 24).astype("float32")
    frame["hour_cos"] = np.cos(2 * np.pi * hour / 24).astype("float32")

    if "wind_direction_deg" in frame.columns:
        angle = np.deg2rad(frame["wind_direction_deg"])
        frame["wind_sin"] = np.sin(angle).astype("float32")
        frame["wind_cos"] = np.cos(angle).astype("float32")

    if "mjo_phase" in frame.columns:
        angle = 2 * np.pi * (frame["mjo_phase"] - 1) / 8
        frame["mjo_phase_sin"] = np.sin(angle).astype("float32")
        frame["mjo_phase_cos"] = np.cos(angle).astype("float32")

    return frame


def build_panel(train, test, environment, config):
    first_time = train["datetime"].min()
    train_end = train["datetime"].max()

    target_hours = sorted(test["datetime"].dt.hour.unique())
    training_days = pd.date_range(
        first_time.normalize(),
        train_end.normalize(),
        freq="D",
    )

    training_times = []

    for day in training_days:
        for hour in target_hours:
            timestamp = day + pd.Timedelta(hours=int(hour))

            if first_time <= timestamp <= train_end:
                training_times.append(timestamp)

    all_times = sorted(set(training_times) | set(test["datetime"]))
    stations = sorted(test["nama_pos"].unique())

    estimated_rows = len(stations) * len(all_times)
    max_rows = config.get("max_panel_rows", 5_000_000)

    if estimated_rows > max_rows:
        raise MemoryError(
            f"Panel grid would have ~{estimated_rows:,} rows "
            f"({len(stations)} stations x {len(all_times)} timestamps), "
            f"which exceeds config['max_panel_rows']={max_rows:,}. "
            "Consider shortening the training window, processing stations "
            "in batches, or raising max_panel_rows if you are sure your "
            "instance has enough RAM."
        )

    log_memory("before panel cross-join", config)

    grid = pd.MultiIndex.from_product(
        [stations, all_times],
        names=["nama_pos", "datetime"],
    ).to_frame(index=False)
    grid["nama_pos"] = grid["nama_pos"].astype("category")

    log_memory("after panel cross-join", config)

    panel = grid.merge(
        environment,
        on=["nama_pos", "datetime"],
        how="left",
        indicator="_environment_merge",
    )
    panel["environment_row_missing"] = panel["_environment_merge"].eq(
        "left_only"
    ).astype("int8")
    panel = panel.drop(columns="_environment_merge")

    free(grid)

    environment_numeric = [
        column
        for column in environment.select_dtypes(include="number").columns
        if column in panel.columns
    ]

    for column in environment_numeric:
        if panel[column].isna().any():
            panel[column] = panel.groupby("nama_pos", observed=True)[column].transform(
                lambda values: values.interpolate(limit_direction="both")
            )
            panel[column] = safe_fillna(panel[column], panel[column].median())

    panel = panel.merge(
        train[["datetime", "nama_pos", "tma_mdpl"]],
        on=["datetime", "nama_pos"],
        how="left",
    )

    panel = add_calendar_features(panel)
    panel = panel.sort_values(["nama_pos", "datetime"]).reset_index(drop=True)

    all_times_series = pd.Series(panel["datetime"].sort_values().unique())
    time_to_step = pd.Series(np.arange(len(all_times_series)), index=all_times_series)
    panel["step"] = panel["datetime"].map(time_to_step).astype("int32")

    panel = downcast_numeric(panel)

    log_memory("after build_panel", config)

    return panel


# ----------------------------------------------------------------------
# Climatology
# ----------------------------------------------------------------------

def fit_climatology(panel, cutoff):
    history = panel[
        panel["datetime"].le(cutoff) & panel["tma_mdpl"].notna()
    ].copy()

    climatology = (
        history.groupby(["nama_pos", "month", "hour"], as_index=False, observed=True)["tma_mdpl"]
        .median()
        .rename(columns={"tma_mdpl": "climatology"})
    )

    station_median = history.groupby("nama_pos", observed=True)["tma_mdpl"].median()

    free(history)

    return climatology, station_median


def apply_climatology(frame, climatology, station_median):
    frame = frame.drop(columns=["climatology", "anomaly"], errors="ignore").copy()
    frame = frame.merge(climatology, on=["nama_pos", "month", "hour"], how="left")
    frame["climatology"] = safe_fillna(
        frame["climatology"], frame["nama_pos"].map(station_median)
    )
    frame["climatology"] = safe_fillna(frame["climatology"], 0)
    frame["anomaly"] = frame["tma_mdpl"] - frame["climatology"]
    return frame


def get_model_features(panel):
    ignored = {
        "datetime",
        "tma_mdpl",
        "climatology",
        "anomaly",
        "step",
        "landcover_name",
    }

    numeric_features = [
        column
        for column in panel.select_dtypes(include="number").columns
        if column not in ignored
    ]

    catboost_features = ["nama_pos"] + numeric_features
    neural_features = numeric_features

    return catboost_features, neural_features


def select_neural_features(neural_features, config, catboost_model=None, catboost_features=None):
    """Cap the number of exogenous features fed to the neural model."""
    limit = config.get("neural_max_exog_features")

    if not limit or len(neural_features) <= limit:
        return neural_features

    if catboost_model is not None and catboost_features is not None:
        importances = pd.Series(
            catboost_model.get_feature_importance(),
            index=catboost_features,
        )
        ranked = [
            feature
            for feature in importances.sort_values(ascending=False).index
            if feature in neural_features
        ]
        selected = ranked[:limit]
    else:
        selected = neural_features[:limit]

    print(
        f"Neural exogenous features capped: {len(neural_features)} -> "
        f"{len(selected)} (config['neural_max_exog_features']={limit})"
    )

    return selected


# ----------------------------------------------------------------------
# ST-GNN spatial encoder
# ----------------------------------------------------------------------

def add_stgnn_features(panel, df_koordinat, cutoff, horizon, neural_features, config):
    embeddings_df, selected_columns, graph = fit_stgnn_pipeline(
        panel, df_koordinat, neural_features, cutoff, horizon,
        target_column="anomaly", config=config,
    )
    panel = merge_stgnn_to_panel(panel, embeddings_df, selected_columns)
    return panel, selected_columns

# ----------------------------------------------------------------------
# CatBoost
# ----------------------------------------------------------------------

def train_catboost(panel, cutoff, features, config):
    training = panel[
        panel["datetime"].le(cutoff) & panel["anomaly"].notna()
    ]

    parameters = {
        "iterations": config["catboost_iterations"],
        "depth": config["catboost_depth"],
        "learning_rate": 0.05,
        "loss_function": "RMSE",
        "l2_leaf_reg": 5,
        "random_strength": 0.5,
        "random_seed": RANDOM_SEED,
        "verbose": 100,
        "allow_writing_files": False,
        "task_type": config["catboost_task_type"],
    }

    if str(config["catboost_task_type"]).upper() == "GPU":
        parameters["devices"] = str(config["gpu_index"])
        parameters["gpu_ram_part"] = config["catboost_gpu_ram_part"]
    else:
        parameters["rsm"] = 0.85

    model = CatBoostRegressor(**parameters)

    model.fit(
        training[features],
        training["anomaly"],
        cat_features=["nama_pos"],
    )

    return model


def predict_catboost(model, future, features):
    future = future.copy()
    future["catboost"] = model.predict(future[features]) + future["climatology"]
    return future[["nama_pos", "datetime", "catboost"]]


# ----------------------------------------------------------------------
# Seasonal model
# ----------------------------------------------------------------------

def predict_anchored_seasonal(panel, cutoff, horizon, config):
    """Forecast from the smoothed value one year ago plus a decaying offset."""
    history = panel[panel["datetime"].le(cutoff)].copy()
    future = panel[panel["datetime"].gt(cutoff)].copy()
    future = future.groupby("nama_pos", group_keys=False, observed=True).head(horizon)

    history = history.sort_values(["nama_pos", "hour", "datetime"])
    window = config["seasonal_window_days"]

    history["seasonal_reference"] = (
        history.groupby(["nama_pos", "hour"], observed=True)["tma_mdpl"]
        .rolling(window=window, center=True, min_periods=max(3, window // 3))
        .median()
        .reset_index(level=[0, 1], drop=True)
        .sort_index()
    )

    reference = history[
        ["nama_pos", "datetime", "seasonal_reference"]
    ].rename(columns={"datetime": "reference_time"})

    future["reference_time"] = future["datetime"].map(
        lambda timestamp: timestamp - pd.DateOffset(years=1)
    )
    future = future.merge(
        reference,
        on=["nama_pos", "reference_time"],
        how="left",
    )

    last_observed = (
        history.dropna(subset=["tma_mdpl"])
        .sort_values(["nama_pos", "datetime"])
        .groupby("nama_pos", as_index=False, observed=True)
        .tail(1)
        [["nama_pos", "datetime", "tma_mdpl"]]
        .rename(
            columns={
                "datetime": "origin_time",
                "tma_mdpl": "origin_tma",
            }
        )
    )
    last_observed["reference_time"] = last_observed["origin_time"].map(
        lambda timestamp: timestamp - pd.DateOffset(years=1)
    )
    last_observed = last_observed.merge(
        reference,
        on=["nama_pos", "reference_time"],
        how="left",
    )
    last_observed["anchor_offset"] = (
        last_observed["origin_tma"] - last_observed["seasonal_reference"]
    ).fillna(0)

    future = future.merge(
        last_observed[["nama_pos", "anchor_offset"]],
        on="nama_pos",
        how="left",
    )
    future["anchor_offset"] = future["anchor_offset"].fillna(0)
    future["horizon_step"] = future.groupby("nama_pos", observed=True).cumcount() + 1
    future["horizon_day"] = np.ceil(
        future["horizon_step"] / OBSERVATIONS_PER_DAY
    )

    half_life = config["seasonal_anchor_half_life_days"]
    decay = 0.5 ** (future["horizon_day"] / half_life)
    seasonal_base = safe_fillna(future["seasonal_reference"], future["climatology"])
    future["seasonal"] = seasonal_base + decay * future["anchor_offset"]

    free(history, reference, last_observed)

    return future[["nama_pos", "datetime", "seasonal"]]


# ----------------------------------------------------------------------
# Neural model
# ----------------------------------------------------------------------

def prepare_neural_frames(panel, cutoff, horizon, neural_features):
    history = panel[panel["datetime"].le(cutoff)].copy()
    future = panel[panel["datetime"].gt(cutoff)].copy()
    future = future.groupby("nama_pos", group_keys=False, observed=True).head(horizon)

    counts = future.groupby("nama_pos", observed=True).size()
    if counts.empty or not counts.eq(horizon).all():
        raise ValueError("The environmental file does not cover the full horizon.")

    history["y"] = history["anomaly"]
    history["y"] = history.groupby("nama_pos", observed=True)["y"].transform(
        lambda values: values.interpolate(limit=6, limit_direction="both")
    )
    history["y"] = history["y"].fillna(0)

    neural_scale = history.groupby("nama_pos", observed=True)["y"].std()
    neural_scale = neural_scale.replace([np.inf, -np.inf, 0], np.nan).fillna(1)
    neural_scale = neural_scale.clip(lower=0.1)

    history["neural_scale"] = history["nama_pos"].map(neural_scale)
    future["neural_scale"] = future["nama_pos"].map(neural_scale).fillna(1)
    history["y"] = history["y"] / history["neural_scale"]

    history = history.rename(columns={"nama_pos": "unique_id", "step": "ds"})
    future = future.rename(columns={"nama_pos": "unique_id", "step": "ds"})

    history["unique_id"] = history["unique_id"].astype(str)
    future["unique_id"] = future["unique_id"].astype(str)

    train_columns = ["unique_id", "ds", "y"] + neural_features
    future_columns = ["unique_id", "ds"] + neural_features

    return history[train_columns], future[future_columns], future


def train_and_predict_neural(panel, cutoff, horizon, neural_features, config):
    history, future_covariates, future_map = prepare_neural_frames(
        panel, cutoff, horizon, neural_features
    )

    common = {
        "h": horizon,
        "input_size": config["input_days"] * OBSERVATIONS_PER_DAY,
        "futr_exog_list": neural_features,
        "loss": MAE(),
        "scaler_type": "identity",
        "max_steps": config["tide_max_steps"],
        "batch_size": config["tide_batch_size"],
        "windows_batch_size": config["tide_windows_batch_size"],
        "inference_windows_batch_size": config["tide_inference_batch_size"],
        "start_padding_enabled": True,
        "random_seed": RANDOM_SEED,
        "accelerator": config["accelerator"],
        "devices": config["devices"],
        "precision": config["precision"],
        "alias": "neural",
    }

    if config["neural_model"] == "nhits":
        model = NHITS(
            n_blocks=[1, 1, 1],
            mlp_units=[[128, 128], [128, 128], [128, 128]],
            n_pool_kernel_size=[4, 2, 1],
            n_freq_downsample=[12, 4, 1],
            pooling_mode="AvgPool1d",
            dropout_prob_theta=0.1,
            **common,
        )
    else:
        model = TiDE(
            hidden_size=config["tide_hidden_size"],
            decoder_output_dim=16,
            temporal_decoder_dim=32,
            dropout=0.2,
            **common,
        )

    forecast = NeuralForecast(models=[model], freq=1)

    try:
        log_memory("before neural fit", config)
        forecast.fit(df=history)
        prediction = forecast.predict(futr_df=future_covariates).reset_index()
        log_memory("after neural fit", config)
    except (torch.cuda.OutOfMemoryError, RuntimeError) as error:
        is_gpu_oom = (
            config["accelerator"] == "gpu"
            and "out of memory" in str(error).lower()
        )

        if not is_gpu_oom:
            raise

        print("GPU memory was insufficient. Retrying the neural model on CPU...")
        free(forecast, model)
        release_gpu_memory()

        cpu_config = config.copy()
        cpu_config["accelerator"] = "cpu"
        cpu_config["devices"] = 1
        cpu_config["precision"] = "32-true"

        return train_and_predict_neural(
            panel,
            cutoff,
            horizon,
            neural_features,
            cpu_config,
        )

    if "neural" in prediction.columns:
        prediction_column = "neural"
    else:
        prediction_columns = [
            column
            for column in prediction.columns
            if column not in {"index", "unique_id", "ds", "cutoff"}
        ]

        if len(prediction_columns) != 1:
            raise ValueError(
                "Could not identify the neural prediction column. "
                f"Available columns: {prediction.columns.tolist()}"
            )

        prediction_column = prediction_columns[0]

    prediction = prediction.rename(
        columns={prediction_column: "neural_scaled_anomaly"}
    )

    non_finite = ~np.isfinite(prediction["neural_scaled_anomaly"])

    if non_finite.any():
        print(
            f"Warning: {non_finite.sum()} neural values were non-finite; "
            "using zero anomaly (climatology) for those rows."
        )
        prediction.loc[non_finite, "neural_scaled_anomaly"] = 0

    prediction["unique_id"] = prediction["unique_id"].astype(str)
    future_map["unique_id"] = future_map["unique_id"].astype(str)

    prediction = prediction.sort_values(["unique_id", "ds"]).copy()
    prediction["forecast_order"] = prediction.groupby("unique_id").cumcount()

    future_lookup = future_map.sort_values(["unique_id", "datetime"]).copy()
    future_lookup["forecast_order"] = future_lookup.groupby("unique_id").cumcount()

    prediction_counts = prediction.groupby("unique_id").size()
    future_counts = future_lookup.groupby("unique_id").size()

    if not prediction_counts.equals(future_counts):
        raise ValueError(
            "The neural model returned a different number of rows than requested.\n"
            f"Predicted counts:\n{prediction_counts}\n"
            f"Expected counts:\n{future_counts}"
        )

    prediction = prediction.merge(
        future_lookup[
            [
                "unique_id",
                "forecast_order",
                "datetime",
                "climatology",
                "neural_scale",
            ]
        ],
        on=["unique_id", "forecast_order"],
        how="left",
        validate="one_to_one",
    )
    prediction["neural_anomaly"] = (
        prediction["neural_scaled_anomaly"] * prediction["neural_scale"]
    )
    prediction["neural"] = (
        prediction["neural_anomaly"] + prediction["climatology"]
    )
    prediction = prediction.rename(columns={"unique_id": "nama_pos"})
    prediction["nama_pos"] = prediction["nama_pos"].astype("category")

    free(forecast, model, history, future_covariates, future_map)
    release_gpu_memory()

    return prediction[["nama_pos", "datetime", "neural"]]


# ----------------------------------------------------------------------
# Horizon banding
# ----------------------------------------------------------------------

def add_horizon_band(frame):
    frame = frame.sort_values(["nama_pos", "datetime"]).copy()
    frame["horizon_step"] = frame.groupby("nama_pos", observed=True).cumcount() + 1
    frame["horizon_day"] = np.ceil(
        frame["horizon_step"] / OBSERVATIONS_PER_DAY
    ).astype(int)

    frame["horizon_band"] = pd.cut(
        frame["horizon_day"],
        bins=[0, 7, 30, 90, np.inf],
        labels=["1-7 days", "8-30 days", "31-90 days", "91+ days"],
    )

    return frame


# ----------------------------------------------------------------------
# Ensemble weighting
# ----------------------------------------------------------------------

def learn_ensemble_weights(validation, config):
    """Gunakan bobot manual per horizon atau optimasi grid sebagai fallback."""
    manual_weights = config.get("manual_horizon_weights")
    weight_mode = str(config.get("weight_mode", "optimize")).lower()

    if weight_mode == "manual" and manual_weights:
        rows = []
        for band, weights in manual_weights.items():
            rows.append({
                "horizon_band": str(band),
                "score": np.nan,
                "catboost_weight": float(weights["catboost"]),
                "neural_weight": float(weights["neural"]),
                "seasonal_weight": float(weights["seasonal"]),
            })
        return pd.DataFrame(rows, columns=[
            "horizon_band", "score", "catboost_weight",
            "neural_weight", "seasonal_weight"
        ])

    return _optimize_ensemble_weights(validation, config)


def _optimize_ensemble_weights(validation, config):
    rows = []
    models = ["catboost", "neural", "seasonal"]
    usable = validation.dropna(subset=["tma_mdpl"] + models)
    step = config["weight_step"]
    candidate_weights = np.arange(0, 1 + step / 2, step)

    for band, group in usable.groupby("horizon_band", observed=True):
        actual = group["tma_mdpl"].to_numpy()

        individual_rmse = []
        individual_mae = []

        for model in models:
            error = group[model].to_numpy() - actual
            individual_rmse.append(np.sqrt(np.mean(error ** 2)))
            individual_mae.append(np.mean(np.abs(error)))

        rmse_scale = max(min(individual_rmse), 1e-8)
        mae_scale = max(min(individual_mae), 1e-8)
        best = None

        for catboost_weight in candidate_weights:
            for neural_weight in candidate_weights:
                seasonal_weight = 1 - catboost_weight - neural_weight

                if seasonal_weight < -1e-9:
                    continue

                seasonal_weight = max(0, seasonal_weight)
                blended = (
                    catboost_weight * group["catboost"].to_numpy()
                    + neural_weight * group["neural"].to_numpy()
                    + seasonal_weight * group["seasonal"].to_numpy()
                )
                error = blended - actual
                rmse = np.sqrt(np.mean(error ** 2))
                mae = np.mean(np.abs(error))

                if config["ensemble_objective"] == "rmse":
                    score = rmse
                elif config["ensemble_objective"] == "mae":
                    score = mae
                else:
                    score = 0.5 * rmse / rmse_scale + 0.5 * mae / mae_scale

                if best is None or score < best["score"]:
                    best = {
                        "score": score,
                        "catboost_weight": float(catboost_weight),
                        "neural_weight": float(neural_weight),
                        "seasonal_weight": float(seasonal_weight),
                    }

        rows.append({"horizon_band": str(band), **best})

    return pd.DataFrame(
        rows,
        columns=[
            "horizon_band",
            "score",
            "catboost_weight",
            "neural_weight",
            "seasonal_weight",
        ],
    )


def apply_ensemble_weights(predictions, weights):
    predictions = predictions.copy()
    weight_table = weights.set_index("horizon_band")
    band = predictions["horizon_band"].astype(str)

    for model in ["catboost", "neural", "seasonal"]:
        weight_column = model + "_weight"
        predictions[weight_column] = band.map(weight_table[weight_column]).fillna(1 / 3)

    numerator = np.zeros(len(predictions))
    denominator = np.zeros(len(predictions))

    for model in ["catboost", "neural", "seasonal"]:
        weight = predictions[model + "_weight"].to_numpy()
        value = predictions[model].to_numpy()
        available = np.isfinite(value)
        numerator += np.where(available, weight * value, 0)
        denominator += np.where(available, weight, 0)

    predictions["tma_mdpl"] = np.divide(
        numerator,
        denominator,
        out=np.full(len(predictions), np.nan),
        where=denominator > 0,
    )

    return predictions


def calculate_validation_metrics(validation, weights, config):
    """Return RMSE, MSE, and MAE for every model and horizon band."""
    actual = validation["tma_mdpl"].copy()
    scored = apply_ensemble_weights(validation.drop(columns="tma_mdpl"), weights)
    scored = scored.rename(columns={"tma_mdpl": "ensemble"})
    scored["actual"] = actual.to_numpy()

    groups = [("Overall", scored)]
    groups += [
        (str(band), group)
        for band, group in scored.groupby("horizon_band", observed=True)
    ]

    rows = []

    for horizon_band, group in groups:
        for model in ["catboost", "neural", "seasonal", "ensemble"]:
            model_group = group.dropna(subset=["actual", model])
            error = model_group[model] - model_group["actual"]
            mse = np.mean(error ** 2)
            rows.append(
                {
                    "horizon_band": horizon_band,
                    "model": model,
                    "RMSE": np.sqrt(mse),
                    "MSE": mse,
                    "MAE": np.mean(np.abs(error)),
                    "rows": len(model_group),
                }
            )

    metrics = pd.DataFrame(rows)

    station_rows = []

    for station, group in scored.groupby("nama_pos", observed=True):
        for model in ["catboost", "neural", "seasonal", "ensemble"]:
            model_group = group.dropna(subset=["actual", model])
            error = model_group[model] - model_group["actual"]
            mse = np.mean(error ** 2)
            station_rows.append(
                {
                    "nama_pos": station,
                    "model": model,
                    "RMSE": np.sqrt(mse),
                    "MAE": np.mean(np.abs(error)),
                    "bias": np.mean(error),
                    "rows": len(model_group),
                }
            )

    station_metrics = pd.DataFrame(station_rows)
    metrics.attrs["station_metrics"] = station_metrics
    scored.attrs["station_metrics"] = station_metrics

    report_count = int(config["report_worst_stations"])

    if report_count > 0:
        worst = (
            station_metrics[station_metrics["model"].eq("ensemble")]
            .sort_values("RMSE", ascending=False)
            .head(report_count)
        )
        print("\nWorst validation stations (ensemble):")
        print(worst.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

    return metrics, scored


# ----------------------------------------------------------------------
# Model orchestration
# ----------------------------------------------------------------------

def run_models(panel, cutoff, horizon, catboost_features, neural_features, config, df_koordinat=None):
    climatology, station_median = fit_climatology(panel, cutoff)
    data = apply_climatology(panel, climatology, station_median)

    catboost_features = list(catboost_features)
    neural_features = list(neural_features)

    if config.get("use_stgnn") and df_koordinat is not None:
        data, embedding_columns = add_stgnn_features(
            data, df_koordinat, cutoff, horizon, neural_features, config
        )
        catboost_features = catboost_features + embedding_columns
        neural_features = neural_features + embedding_columns

    future = data[data["datetime"].gt(cutoff)].copy()
    future = future.groupby("nama_pos", group_keys=False, observed=True).head(horizon)

    log_memory("before CatBoost fit", config)
    catboost_model = train_catboost(data, cutoff, catboost_features, config)
    catboost_prediction = predict_catboost(catboost_model, future, catboost_features)
    log_memory("after CatBoost fit", config)

    selected_neural_features = select_neural_features(
        neural_features, config, catboost_model, catboost_features
    )

    free(catboost_model)

    neural_prediction = train_and_predict_neural(
        data,
        cutoff,
        horizon,
        selected_neural_features,
        config,
    )
    seasonal_prediction = predict_anchored_seasonal(
        data,
        cutoff,
        horizon,
        config,
    )

    prediction = catboost_prediction.merge(
        neural_prediction,
        on=["nama_pos", "datetime"],
        how="left",
        validate="one_to_one",
    )
    prediction = prediction.merge(
        seasonal_prediction,
        on=["nama_pos", "datetime"],
        how="left",
        validate="one_to_one",
    )

    missing_neural = prediction["neural"].isna().sum()

    if missing_neural:
        print(
            f"Warning: {missing_neural} neural rows are missing; "
            "using the other models as fallback."
        )

    prediction = add_horizon_band(prediction)

    free(data, future, catboost_prediction, neural_prediction, seasonal_prediction)
    log_memory("end of run_models", config)

    return prediction


def choose_validation_cutoff(panel, train, test, horizon, config):
    """Choose a chronological cutoff, preferably matching the test season."""
    train_end = train["datetime"].max()
    training_times = sorted(
        panel.loc[panel["datetime"].le(train_end), "datetime"].unique()
    )

    if len(training_times) <= horizon:
        raise ValueError("Training history is shorter than the test horizon.")

    if config["validation_mode"] == "same_season":
        validation_start = test["datetime"].min() - pd.DateOffset(years=1)
        eligible_cutoffs = [time for time in training_times if time < validation_start]

        if eligible_cutoffs:
            cutoff = eligible_cutoffs[-1]
            future_times = [time for time in training_times if time > cutoff][:horizon]

            if len(future_times) == horizon:
                return cutoff, future_times[0], future_times[-1]

        print("Same-season validation was unavailable; using the last block.")

    cutoff = training_times[-horizon - 1]
    future_times = [time for time in training_times if time > cutoff][:horizon]
    return cutoff, future_times[0], future_times[-1]


# ----------------------------------------------------------------------
# Shared setup used by both entry points
# ----------------------------------------------------------------------

def _build_common_state(df_train, df_test, df_lingkungan, node_embedding, config):
    train, test, environment, node = prepare_data(df_train, df_test, df_lingkungan, node_embedding)
    environment = clean_environment(environment)

    if config["add_hydrology_features"]:
        print("Adding causal hydrology features...")
        environment = add_hydrology_features(environment)

    panel = build_panel(train, test, environment, config)

    free(environment)

    if config.get("use_stgnn"):
        missing_nodes = set(test["nama_pos"].unique()) - set(node["nama_pos"].unique())
        if missing_nodes:
            raise ValueError(
                f"Pos berikut ada di df_test tapi tidak ada di node: {missing_nodes}"
            )

    catboost_features, neural_features = get_model_features(panel)

    horizon_counts = test.groupby("nama_pos", observed=True).size()

    if not horizon_counts.eq(horizon_counts.iloc[0]).all():
        raise ValueError("Every station in df_test must have the same horizon.")

    horizon = int(horizon_counts.iloc[0])

    return train, test, panel, node, catboost_features, neural_features, horizon


# ----------------------------------------------------------------------
# Entry point 1 of 2 — validation phase only
# ----------------------------------------------------------------------

def run_validation_only(df_train, df_test, df_lingkungan, node_embedding, config=None):
    """Run only the (cheaper) validation phase and persist weights/metrics."""
    config = prepare_config(config)

    print("Pipeline version:", PIPELINE_VERSION, "(validation phase)")
    log_memory("start of run_validation_only", config)

    train, test, panel, graph, catboost_features, neural_features, horizon = (
        _build_common_state(df_train, df_test, df_lingkungan, node_embedding, config)
    )

    validation_cutoff, validation_start, validation_end = choose_validation_cutoff(
        panel, train, test, horizon, config
    )

    print(
        "Validation period:",
        pd.Timestamp(validation_start),
        "to",
        pd.Timestamp(validation_end),
    )

    print("\nFitting validation models...")
    validation_prediction = run_models(
        panel, validation_cutoff, horizon, catboost_features, neural_features, config, graph
    )

    validation_actual = train[["nama_pos", "datetime", "tma_mdpl"]]
    validation_prediction = validation_prediction.merge(
        validation_actual, on=["nama_pos", "datetime"], how="left"
    )

    weights = learn_ensemble_weights(validation_prediction, config)
    metrics, validation_prediction = calculate_validation_metrics(
        validation_prediction, weights, config
    )

    print("\nLearned horizon weights:")
    print(weights.to_string(index=False))
    print("\nValidation metrics (lower is better):")
    print(metrics.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

    weights_path = Path(config["weights_output_file"]).expanduser()
    metrics_path = Path(config["metrics_output_file"]).expanduser()
    weights.to_csv(weights_path, index=False)
    metrics.to_csv(metrics_path, index=False)
    print(f"\nSaved weights to: {weights_path.resolve()}")
    print(f"Saved metrics to: {metrics_path.resolve()}")

    free(train, test, panel)
    release_gpu_memory()
    log_memory("end of run_validation_only", config)

    print(
        "\nIf you are on a tight-RAM Colab instance, this is a good point "
        "to Runtime > Restart runtime before calling run_final_only()."
    )

    return weights, metrics, validation_prediction


# ----------------------------------------------------------------------
# Entry point 2 of 2 — final fit + submission
# ----------------------------------------------------------------------

def run_final_only(
    df_train,
    df_test,
    df_lingkungan,
    node_embedding,
    df_submission=None,
    weights=None,
    weights_path=None,
    config=None,
):
    """Run only the final fit + submission phase."""
    config = prepare_config(config)

    if weights is None:
        if weights_path is None:
            weights_path = config["weights_output_file"]
        weights = pd.read_csv(weights_path)

    print("Pipeline version:", PIPELINE_VERSION, "(final phase)")
    log_memory("start of run_final_only", config)

    train, test, panel, graph, catboost_features, neural_features, horizon = (
        _build_common_state(df_train, df_test, df_lingkungan, df_koordinat, config)
    )

    train_end = train["datetime"].max()

    print("\nFitting final models on all training data...")
    test_prediction = run_models(
        panel, train_end, horizon, catboost_features, neural_features, config, graph
    )
    test_prediction = apply_ensemble_weights(test_prediction, weights)

    submission = test.merge(
        test_prediction[
            ["nama_pos", "datetime", "catboost", "neural", "seasonal", "tma_mdpl"]
        ],
        on=["nama_pos", "datetime"],
        how="left",
        indicator="_prediction_merge",
    )

    missing_prediction = submission[
        submission["tma_mdpl"].isna() | submission["_prediction_merge"].ne("both")
    ]

    if not missing_prediction.empty:
        examples = missing_prediction[
            [
                "nama_pos",
                "datetime",
                "_prediction_merge",
                "catboost",
                "neural",
                "seasonal",
                "tma_mdpl",
            ]
        ].head(5)
        raise ValueError(
            f"{len(missing_prediction)} test rows did not receive a prediction.\n"
            f"First unmatched rows:\n{examples.to_string(index=False)}"
        )

    submission = submission.drop(columns="_prediction_merge")
    prediction_by_id = submission.set_index("id")["tma_mdpl"]

    if df_submission is None:
        final_submission = test.sort_values("original_order")[["id"]].copy()
    else:
        final_submission = df_submission.copy()

    final_submission["tma_mdpl"] = final_submission["id"].map(prediction_by_id)

    if final_submission["tma_mdpl"].isna().any():
        raise ValueError("df_submission contains IDs that were not found in df_test.")

    output_path = Path(config["output_file"]).expanduser()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    final_submission.to_csv(output_path, index=False)

    if not output_path.exists() or output_path.stat().st_size == 0:
        raise OSError(f"Submission file was not written correctly: {output_path}")

    final_submission.attrs["output_file"] = str(output_path.resolve())
    print("\nSaved submission:", output_path.resolve())
    print("Submission rows:", len(final_submission))

    free(train, test, panel, test_prediction, submission)
    release_gpu_memory()
    log_memory("end of run_final_only", config)

    return final_submission


# ----------------------------------------------------------------------
# Convenience wrapper — same interface as before, plus df_koordinat
# ----------------------------------------------------------------------

def run_ensemble(df_train, df_test, df_lingkungan, df_koordinat, df_submission=None, config=None):
    """
    Train, validate, and forecast with CatBoost + ST-GNN + neural + seasonal.

    Returns:
        submission             final test predictions
        metrics                RMSE, MSE, and MAE by model and horizon
        weights                learned ensemble weight per horizon band
        validation_prediction  validation predictions for further analysis
    """
    config = prepare_config(config)

    weights, metrics, validation_prediction = run_validation_only(
        df_train, df_test, df_lingkungan, df_koordinat, config
    )

    gc.collect()
    release_gpu_memory()

    submission = run_final_only(
        df_train,
        df_test,
        df_lingkungan,
        df_koordinat,
        df_submission,
        weights=weights,
        config=config,
    )

    return submission, metrics, weights, validation_prediction

In [ ]:
exp4_config = build_experiment_config(
    "experiment_4",
    device="auto",
    output_file="/content/submission_exp4.csv",
)

submission, metrics, weights, validation = run_ensemble(
    df_train_clean,
    test_df,
    datling_df,
    koor_df,
    df_submission,
    config=exp4_config,
)

display(metrics)
display(weights)
display(submission.head())


### 5.5 Eksperimen 5 — River GAT + CatBoost + N-HiTS + Seasonal


In [ ]:
# @title
# -*- coding: utf-8 -*-
"""
FULL RIVER-SPATIAL FORECASTING PIPELINE
HydroRIVERS -> directed topology -> self-supervised GATv2 -> node_embedding
-> CatBoost + N-HiTS/TiDE + seasonal ensemble.

Designed for the SeMa Statistics Data Science 2026 dataset and Google Colab.
The downstream ensemble code is based on the user-provided hardened pipeline.
"""

# ======================================================================
# 1. Imports and dependency installation
# ======================================================================

# Google Colab installation cell (run manually if packages are missing):
# !pip install -q geopandas pyogrio shapely rtree networkx scikit-learn \
#     torch torch-geometric catboost neuralforecast psutil umap-learn

from __future__ import annotations

import gc
import json
import logging
import math
import os
import random
import time
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Set, Tuple

import numpy as np
import pandas as pd

try:
    import geopandas as gpd
    from shapely.geometry import Point
except ImportError as exc:
    raise ImportError("Install geopandas and shapely before running this module.") from exc

import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import RobustScaler, StandardScaler

try:
    from torch_geometric.data import Data
    from torch_geometric.nn import GATv2Conv
except ImportError as exc:
    raise ImportError("Install torch-geometric before running the GAT section.") from exc

LOGGER = logging.getLogger("river_gat")
if not LOGGER.handlers:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
    )


# ======================================================================
# 2. Configuration
# ======================================================================

@dataclass
class RiverGATConfig:
    """Configuration for HydroRIVERS topology and self-supervised GAT."""

    target_crs: str = "EPSG:4326"
    projected_crs: Optional[str] = None
    max_snap_distance_m: float = 5_000.0
    snap_bbox_buffer_deg: float = 1.0
    river_order_priority: Tuple[str, ...] = ("ORD_STRA", "ORD_FLOW", "ORD_CLAS")

    include_all_upstream: bool = True
    include_path_to_outlet: bool = True
    upstream_hops: Optional[int] = None
    downstream_hops: Optional[int] = None
    max_graph_nodes: int = 250_000
    max_graph_edges: int = 500_000

    scaler: str = "robust"
    hidden_dim: int = 64
    embedding_dim: int = 32
    heads: int = 4
    num_layers: int = 2
    dropout: float = 0.15
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    epochs: int = 300
    patience: int = 30
    gradient_clip_val: float = 1.0
    link_loss_weight: float = 1.0
    feature_loss_weight: float = 0.5
    val_fraction: float = 0.10
    test_fraction: float = 0.10
    negative_ratio: float = 1.0
    random_seed: int = 42
    device: str = "auto"
    use_edge_features: bool = True
    add_reverse_edges_for_message_passing: bool = True

    output_dir: str = "."
    embedding_filename: str = "node_embedding_gat.csv"
    model_filename: str = "river_gat_encoder.pt"
    scaler_filename: str = "river_node_scaler.pkl"


HYDRO_ATTRIBUTE_CANDIDATES: Dict[str, Tuple[str, ...]] = {
    "segment_id": ("HYRIV_ID", "hyriv_id", "segment_id"),
    "downstream_id": ("NEXT_DOWN", "next_down", "downstream_id"),
    "river_order": ("ORD_STRA", "ORD_FLOW", "ORD_CLAS", "river_order"),
    "length": ("LENGTH_KM", "length_km", "length"),
    "upstream_area": ("UPLAND_SKM", "upstream_area", "upland_skm"),
    "catchment_area": ("CATCH_SKM", "catchment_area", "catch_skm"),
    "distance_to_outlet": ("DIST_DN_KM", "distance_to_outlet", "dist_dn_km"),
    "distance_upstream": ("DIST_UP_KM", "distance_upstream", "dist_up_km"),
    "discharge": ("DIS_AV_CMS", "average_discharge", "discharge"),
    "main_river_id": ("MAIN_RIV", "main_river_id"),
    "basin_id": ("HYBAS_L12", "basin_id"),
    "endorheic": ("ENDORHEIC", "endorheic"),
}

CONTINUOUS_LOG_FEATURES = {
    "length_km",
    "catchment_area",
    "upstream_area",
    "average_discharge",
    "number_of_upstream_nodes",
    "upstream_total_length",
    "upstream_total_area",
}


# ======================================================================
# 3. Seed and device handling
# ======================================================================

def set_global_seed(seed: int = 42) -> None:
    """Set deterministic seeds for Python, NumPy, and PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass


def resolve_device(requested: str = "auto") -> torch.device:
    """Resolve auto/cpu/gpu into a valid torch.device."""
    requested = str(requested).lower()
    if requested not in {"auto", "cpu", "gpu", "cuda"}:
        raise ValueError("device must be one of: auto, cpu, gpu, cuda")
    if requested in {"gpu", "cuda"} and not torch.cuda.is_available():
        LOGGER.warning("CUDA unavailable; falling back to CPU.")
        return torch.device("cpu")
    if requested == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device("cuda" if requested in {"gpu", "cuda"} else "cpu")


# ======================================================================
# 4. Load HydroRIVERS
# ======================================================================

def _find_column(columns: Iterable[str], candidates: Sequence[str], required: bool = False) -> Optional[str]:
    lookup = {str(c).lower(): str(c) for c in columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    if required:
        raise KeyError(f"None of these columns were found: {list(candidates)}")
    return None


def resolve_hydroriver_columns(columns: Iterable[str]) -> Dict[str, Optional[str]]:
    """Map flexible HydroRIVERS column names to canonical semantic names."""
    mapping = {
        key: _find_column(columns, candidates, required=key in {"segment_id", "downstream_id"})
        for key, candidates in HYDRO_ATTRIBUTE_CANDIDATES.items()
    }
    if mapping["river_order"] is None:
        warnings.warn("No river-order field found; river order will be filled as zero.")
    elif mapping["river_order"] != "ORD_STRA":
        warnings.warn(f"ORD_STRA unavailable; using {mapping['river_order']} as river order.")
    return mapping


def _repair_geometries(gdf: gpd.GeoDataFrame) -> Tuple[gpd.GeoDataFrame, int]:
    invalid_mask = ~gdf.geometry.is_valid & gdf.geometry.notna()
    n_invalid = int(invalid_mask.sum())
    if n_invalid:
        LOGGER.warning("Repairing %s invalid geometries with make_valid/buffer(0).", n_invalid)
        try:
            gdf.loc[invalid_mask, "geometry"] = gdf.loc[invalid_mask, "geometry"].make_valid()
        except Exception:
            gdf.loc[invalid_mask, "geometry"] = gdf.loc[invalid_mask, "geometry"].buffer(0)
    return gdf, n_invalid


def load_hydrorivers(
    hydroriver_path: str | Path,
    target_crs: str = "EPSG:4326",
    bbox: Optional[Tuple[float, float, float, float]] = None,
) -> Tuple[gpd.GeoDataFrame, Dict[str, Any], Dict[str, Optional[str]]]:
    """Read, validate, clean, and report HydroRIVERS geometries.

    Parameters
    ----------
    hydroriver_path:
        Path to the HydroRIVERS shapefile.
    target_crs:
        Output geographic CRS.
    bbox:
        Optional `(minx, miny, maxx, maxy)` spatial filter. This is strongly
        recommended for station snapping on continental HydroRIVERS files.
    """
    path = Path(hydroriver_path)
    if not path.exists():
        raise FileNotFoundError(path)

    read_kwargs: Dict[str, Any] = {}
    if bbox is not None:
        read_kwargs["bbox"] = bbox
    gdf = gpd.read_file(path, **read_kwargs)
    if gdf.empty:
        raise ValueError("HydroRIVERS read returned zero segments.")

    mapping = resolve_hydroriver_columns(gdf.columns)
    segment_col = mapping["segment_id"]
    downstream_col = mapping["downstream_id"]
    order_col = mapping["river_order"]

    original_crs = gdf.crs
    if original_crs is None:
        warnings.warn(f"HydroRIVERS CRS is missing; assuming {target_crs}.")
        gdf = gdf.set_crs(target_crs)
    elif str(original_crs) != target_crs:
        gdf = gdf.to_crs(target_crs)

    n_empty_geometry = int(gdf.geometry.isna().sum() + gdf.geometry.is_empty.sum())
    gdf = gdf.loc[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    gdf, n_invalid_geometry = _repair_geometries(gdf)

    n_missing_hyriv_id = int(gdf[segment_col].isna().sum())
    n_duplicate_hyriv_id = int(gdf[segment_col].duplicated(keep=False).sum())
    n_missing_next_down = int(gdf[downstream_col].isna().sum())

    gdf = gdf.drop_duplicates().copy()
    gdf[segment_col] = pd.to_numeric(gdf[segment_col], errors="coerce").astype("Int64")
    gdf[downstream_col] = pd.to_numeric(gdf[downstream_col], errors="coerce").fillna(0).astype("Int64")

    if n_missing_hyriv_id:
        warnings.warn(f"Dropping {n_missing_hyriv_id} rows with missing segment ID.")
        gdf = gdf[gdf[segment_col].notna()].copy()
    if gdf[segment_col].duplicated().any():
        dup = gdf.loc[gdf[segment_col].duplicated(keep=False), segment_col].head(20).tolist()
        raise ValueError(f"Duplicate segment IDs remain after exact deduplication: {dup}")

    distribution = {}
    if order_col:
        distribution = (
            pd.to_numeric(gdf[order_col], errors="coerce")
            .value_counts(dropna=False)
            .sort_index()
            .to_dict()
        )

    report = {
        "n_segments": int(len(gdf)),
        "n_invalid_geometry": n_invalid_geometry,
        "n_empty_geometry": n_empty_geometry,
        "n_missing_hyriv_id": n_missing_hyriv_id,
        "n_duplicate_hyriv_id": n_duplicate_hyriv_id,
        "n_missing_next_down": n_missing_next_down,
        "river_order_distribution": distribution,
        "source_crs": str(original_crs),
        "output_crs": str(gdf.crs),
        "bbox_filter": bbox,
    }
    LOGGER.info("Loaded %,d HydroRIVERS segments.", len(gdf))
    return gdf, report, mapping


def load_hydroriver_attributes(
    hydroriver_path: str | Path,
) -> Tuple[pd.DataFrame, Dict[str, Optional[str]]]:
    """Read the complete HydroRIVERS DBF attributes without geometry.

    This keeps topology relationships available while avoiding the memory cost
    of loading every continental geometry.
    """
    path = Path(hydroriver_path)
    try:
        import pyogrio
        frame = pyogrio.read_dataframe(path, read_geometry=False)
    except Exception:
        geometry = gpd.read_file(path)
        frame = pd.DataFrame(geometry.drop(columns="geometry"))
    mapping = resolve_hydroriver_columns(frame.columns)
    sid, did = mapping["segment_id"], mapping["downstream_id"]
    frame[sid] = pd.to_numeric(frame[sid], errors="coerce").astype("Int64")
    frame[did] = pd.to_numeric(frame[did], errors="coerce").fillna(0).astype("Int64")
    frame = frame.dropna(subset=[sid]).drop_duplicates(subset=[sid], keep="first")
    LOGGER.info("Loaded %,d HydroRIVERS attribute rows without geometry.", len(frame))
    return frame, mapping


# ======================================================================
# 5. Validate station coordinates
# ======================================================================

def validate_station_coordinates(df_koordinat: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Standardize station name/latitude/longitude and report duplicates."""
    if not isinstance(df_koordinat, pd.DataFrame):
        raise TypeError("df_koordinat must be a pandas DataFrame")

    station_col = _find_column(df_koordinat.columns, ("nama_pos", "station", "station_name"), True)
    lat_col = _find_column(
        df_koordinat.columns,
        ("latitude", "lat", "station_latitude", "y"),
        True,
    )
    lon_col = _find_column(
        df_koordinat.columns,
        ("longitude", "lon", "lng", "long", "station_longitude", "x"),
        True,
    )
    result = df_koordinat.rename(
        columns={station_col: "nama_pos", lat_col: "latitude", lon_col: "longitude"}
    )[["nama_pos", "latitude", "longitude"]].copy()

    result["nama_pos"] = result["nama_pos"].astype("string").str.strip()
    result["latitude"] = pd.to_numeric(result["latitude"], errors="coerce")
    result["longitude"] = pd.to_numeric(result["longitude"], errors="coerce")

    missing_coordinate = result[["latitude", "longitude"]].isna().any(axis=1)
    invalid_lat = ~result["latitude"].between(-90, 90, inclusive="both")
    invalid_lon = ~result["longitude"].between(-180, 180, inclusive="both")
    duplicate_station = result["nama_pos"].duplicated(keep=False)
    duplicate_coordinate = result.duplicated(["latitude", "longitude"], keep=False)

    report = {
        "n_rows": int(len(result)),
        "n_missing_coordinates": int(missing_coordinate.sum()),
        "n_invalid_latitude": int(invalid_lat.sum()),
        "n_invalid_longitude": int(invalid_lon.sum()),
        "duplicate_station_names": result.loc[duplicate_station, "nama_pos"].tolist(),
        "duplicate_coordinates": result.loc[
            duplicate_coordinate, ["nama_pos", "latitude", "longitude"]
        ].to_dict("records"),
    }
    if missing_coordinate.any() or invalid_lat.any() or invalid_lon.any():
        bad = result.loc[missing_coordinate | invalid_lat | invalid_lon]
        raise ValueError(f"Invalid station coordinates:\n{bad.to_string(index=False)}")
    if duplicate_station.any():
        raise ValueError(f"nama_pos must be unique: {report['duplicate_station_names']}")
    if duplicate_coordinate.any():
        warnings.warn("Multiple stations share identical coordinates; see validation report.")
    return result.reset_index(drop=True), report


def station_bbox(df_koordinat: pd.DataFrame, buffer_deg: float = 1.0) -> Tuple[float, float, float, float]:
    """Build a geographic bounding box around all stations."""
    return (
        float(df_koordinat["longitude"].min() - buffer_deg),
        float(df_koordinat["latitude"].min() - buffer_deg),
        float(df_koordinat["longitude"].max() + buffer_deg),
        float(df_koordinat["latitude"].max() + buffer_deg),
    )


def estimate_utm_crs(latitude: float, longitude: float) -> str:
    """Return a WGS84 UTM EPSG code for a centroid."""
    zone = int((longitude + 180) // 6) + 1
    epsg = 32600 + zone if latitude >= 0 else 32700 + zone
    return f"EPSG:{epsg}"


# ======================================================================
# 6. Match stations to rivers
# ======================================================================

def _snap_quality(distance: float, max_distance: float) -> str:
    if not np.isfinite(distance) or distance > max_distance:
        return "unmatched"
    if distance <= 100:
        return "excellent"
    if distance <= 500:
        return "good"
    if distance <= 2_000:
        return "acceptable"
    return "poor"


def match_stations_to_rivers(
    df_koordinat: pd.DataFrame,
    rivers_gdf: gpd.GeoDataFrame,
    max_snap_distance_m: float = 5_000,
    projected_crs: Optional[str] = None,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Snap stations to nearest HydroRIVERS segments in a metric CRS."""
    stations, _ = validate_station_coordinates(df_koordinat)
    mapping = resolve_hydroriver_columns(rivers_gdf.columns)
    sid = mapping["segment_id"]
    order_col = mapping["river_order"]

    station_gdf = gpd.GeoDataFrame(
        stations,
        geometry=gpd.points_from_xy(stations["longitude"], stations["latitude"]),
        crs="EPSG:4326",
    )
    if rivers_gdf.crs is None:
        raise ValueError("rivers_gdf must have a CRS")
    station_gdf = station_gdf.to_crs(rivers_gdf.crs)

    if projected_crs is None:
        centroid = station_gdf.to_crs("EPSG:4326").geometry.union_all().centroid
        projected_crs = estimate_utm_crs(centroid.y, centroid.x)
    stations_m = station_gdf.to_crs(projected_crs)
    rivers_m = rivers_gdf.to_crs(projected_crs)

    keep_cols = [sid, "geometry"]
    for semantic in [
        "river_order", "length", "upstream_area", "catchment_area",
        "distance_to_outlet", "discharge", "main_river_id", "basin_id"
    ]:
        col = mapping.get(semantic)
        if col and col not in keep_cols:
            keep_cols.append(col)

    nearest = gpd.sjoin_nearest(
        stations_m,
        rivers_m[keep_cols],
        how="left",
        distance_col="snap_distance_m",
    ).reset_index(drop=True)
    nearest = nearest.sort_values(["nama_pos", "snap_distance_m"]).drop_duplicates("nama_pos")
    nearest["HYRIV_ID"] = pd.to_numeric(nearest[sid], errors="coerce").astype("Int64")
    nearest["river_order"] = (
        pd.to_numeric(nearest[order_col], errors="coerce") if order_col else 0.0
    )
    nearest["snap_quality"] = nearest["snap_distance_m"].map(
        lambda value: _snap_quality(float(value), max_snap_distance_m)
    )
    nearest.loc[nearest["snap_quality"].eq("unmatched"), "HYRIV_ID"] = pd.NA

    output_cols = [
        "nama_pos", "latitude", "longitude", "HYRIV_ID", "river_order",
        "snap_distance_m", "snap_quality"
    ]
    for semantic, out_name in [
        ("length", "length_km"),
        ("upstream_area", "upstream_area"),
        ("catchment_area", "catchment_area"),
        ("distance_to_outlet", "distance_to_outlet"),
        ("discharge", "average_discharge"),
        ("main_river_id", "main_river_id"),
        ("basin_id", "basin_id"),
    ]:
        col = mapping.get(semantic)
        if col and col in nearest.columns:
            nearest[out_name] = nearest[col]
            output_cols.append(out_name)

    output = pd.DataFrame(nearest[output_cols]).sort_values("nama_pos").reset_index(drop=True)
    matched = output["HYRIV_ID"].notna()
    duplicates = output.loc[matched, "HYRIV_ID"].value_counts()
    report = {
        "projected_crs": projected_crs,
        "n_stations": int(len(output)),
        "n_matched": int(matched.sum()),
        "n_unmatched": int((~matched).sum()),
        "median_snap_distance_m": float(output.loc[matched, "snap_distance_m"].median()),
        "maximum_snap_distance_m": float(output.loc[matched, "snap_distance_m"].max()),
        "worst_stations": output.nlargest(10, "snap_distance_m")[
            ["nama_pos", "snap_distance_m", "snap_quality"]
        ].to_dict("records"),
        "n_segments_with_multiple_stations": int((duplicates > 1).sum()),
        "shared_segments": duplicates[duplicates > 1].to_dict(),
    }
    LOGGER.info("Matched %d/%d stations to rivers.", report["n_matched"], report["n_stations"])
    return output, report


# ======================================================================
# 7. Build directed river topology
# ======================================================================

def build_river_topology(
    rivers: pd.DataFrame,
    segment_id_col: str = "HYRIV_ID",
    downstream_id_col: str = "NEXT_DOWN",
) -> Tuple[nx.DiGraph, Dict[str, Any]]:
    """Build upstream->downstream segment graph and audit topology issues."""
    if segment_id_col not in rivers or downstream_id_col not in rivers:
        raise KeyError(f"Missing {segment_id_col} or {downstream_id_col}")
    frame = rivers.copy()
    frame[segment_id_col] = pd.to_numeric(frame[segment_id_col], errors="coerce").astype("Int64")
    frame[downstream_id_col] = pd.to_numeric(frame[downstream_id_col], errors="coerce").fillna(0).astype("Int64")
    if frame[segment_id_col].duplicated().any():
        raise ValueError("Duplicate segment IDs are not allowed in the topology table.")

    graph = nx.DiGraph()
    records = frame.set_index(segment_id_col).to_dict("index")
    for node_id, attrs in records.items():
        graph.add_node(int(node_id), **attrs)

    valid_ids = set(graph.nodes)
    invalid_downstream: List[Tuple[int, int]] = []
    self_loops: List[Tuple[int, int]] = []
    for source, downstream in frame[[segment_id_col, downstream_id_col]].itertuples(index=False):
        source, downstream = int(source), int(downstream)
        if downstream <= 0:
            continue
        if source == downstream:
            self_loops.append((source, downstream))
            continue
        if downstream not in valid_ids:
            invalid_downstream.append((source, downstream))
            continue
        graph.add_edge(source, downstream)

    cycle_edges = list(nx.simple_cycles(graph))
    weak_components = list(nx.weakly_connected_components(graph))
    report = {
        "n_nodes": graph.number_of_nodes(),
        "n_edges": graph.number_of_edges(),
        "n_components": len(weak_components),
        "n_headwaters": sum(graph.in_degree(n) == 0 for n in graph),
        "n_outlets": sum(graph.out_degree(n) == 0 for n in graph),
        "n_junctions": sum(graph.in_degree(n) > 1 for n in graph),
        "n_cycles": len(cycle_edges),
        "cycles": cycle_edges[:50],
        "n_self_loops": len(self_loops),
        "self_loops": self_loops[:50],
        "n_invalid_downstream_ids": len(invalid_downstream),
        "invalid_downstream_ids": invalid_downstream[:50],
        "n_isolated_nodes": sum(graph.degree(n) == 0 for n in graph),
    }
    LOGGER.info("Topology: %,d nodes, %,d edges, %d components.", graph.number_of_nodes(), graph.number_of_edges(), len(weak_components))
    return graph, report


def break_cycles(
    graph: nx.DiGraph,
    strategy: str = "remove_lowest_discharge",
    discharge_attr: str = "DIS_AV_CMS",
) -> Tuple[nx.DiGraph, pd.DataFrame]:
    """Optionally break cycles and return an explicit removed-edge audit table."""
    result = graph.copy()
    audit: List[Dict[str, Any]] = []
    for cycle in list(nx.simple_cycles(result)):
        edges = list(zip(cycle, cycle[1:] + cycle[:1]))
        if strategy == "remove_lowest_discharge":
            edge = min(edges, key=lambda e: float(result.nodes[e[0]].get(discharge_attr, np.inf) or np.inf))
        else:
            edge = edges[-1]
        if result.has_edge(*edge):
            result.remove_edge(*edge)
            audit.append({"source": edge[0], "target": edge[1], "strategy": strategy, "cycle": cycle})
    return result, pd.DataFrame(audit)


# ======================================================================
# 8. Extract relevant station subgraph
# ======================================================================

def _limited_ancestors(graph: nx.DiGraph, node: int, hops: int) -> Set[int]:
    visited = {node}
    frontier = {node}
    for _ in range(hops):
        frontier = {p for current in frontier for p in graph.predecessors(current)} - visited
        if not frontier:
            break
        visited.update(frontier)
    return visited


def _limited_descendants(graph: nx.DiGraph, node: int, hops: int) -> Set[int]:
    visited = {node}
    frontier = {node}
    for _ in range(hops):
        frontier = {p for current in frontier for p in graph.successors(current)} - visited
        if not frontier:
            break
        visited.update(frontier)
    return visited


def extract_station_river_subgraph(
    graph: nx.DiGraph,
    station_hyriv_ids: Sequence[int],
    upstream_hops: Optional[int] = None,
    downstream_hops: Optional[int] = None,
    include_all_upstream: bool = True,
    include_path_to_outlet: bool = True,
) -> nx.DiGraph:
    """Extract station segments, upstream catchments, and outlet paths."""
    station_nodes = {int(x) for x in station_hyriv_ids if pd.notna(x)}
    missing = station_nodes - set(graph.nodes)
    if missing:
        warnings.warn(f"{len(missing)} station segment IDs are absent from topology: {sorted(missing)[:20]}")
    station_nodes &= set(graph.nodes)
    selected: Set[int] = set(station_nodes)

    for node in station_nodes:
        if include_all_upstream:
            selected.update(nx.ancestors(graph, node))
        elif upstream_hops is not None:
            selected.update(_limited_ancestors(graph, node, upstream_hops))

        if include_path_to_outlet:
            current = node
            visited = {current}
            while graph.out_degree(current) > 0:
                successors = list(graph.successors(current))
                current = successors[0]
                if current in visited:
                    break
                selected.add(current)
                visited.add(current)
        elif downstream_hops is not None:
            selected.update(_limited_descendants(graph, node, downstream_hops))

    subgraph = graph.subgraph(selected).copy()
    LOGGER.info("Relevant subgraph: %,d nodes, %,d edges.", subgraph.number_of_nodes(), subgraph.number_of_edges())
    return subgraph


def estimate_graph_memory(n_nodes: int, n_edges: int, n_features: int, edge_features: int = 0) -> Dict[str, float]:
    """Approximate sparse tensor memory in MiB, excluding optimizer activations."""
    node_mb = n_nodes * n_features * 4 / 2**20
    edge_index_mb = 2 * n_edges * 8 / 2**20
    edge_attr_mb = n_edges * edge_features * 4 / 2**20
    return {
        "node_features_mb": node_mb,
        "edge_index_mb": edge_index_mb,
        "edge_attributes_mb": edge_attr_mb,
        "minimum_tensor_mb": node_mb + edge_index_mb + edge_attr_mb,
    }


# ======================================================================
# 9. Build node features
# ======================================================================

def _safe_numeric(value: Any, default: float = 0.0) -> float:
    try:
        value = float(value)
        return value if np.isfinite(value) else default
    except Exception:
        return default


def build_node_features(
    graph: nx.DiGraph,
    column_mapping: Mapping[str, Optional[str]],
    scaler_type: str = "robust",
) -> Tuple[pd.DataFrame, pd.DataFrame, Any]:
    """Create raw and scaled hydrological/topological node features."""
    if graph.number_of_nodes() == 0:
        raise ValueError("Cannot build features for an empty graph")
    is_dag = nx.is_directed_acyclic_graph(graph)
    if not is_dag:
        raise ValueError("Graph contains cycles. Use break_cycles() explicitly before feature generation.")

    topo_order = list(nx.topological_sort(graph))
    reverse_topo = topo_order[::-1]
    component_size: Dict[int, int] = {}
    for component in nx.weakly_connected_components(graph):
        size = len(component)
        for node in component:
            component_size[node] = size

    upstream_count: Dict[int, int] = {}
    upstream_total_length: Dict[int, float] = {}
    upstream_total_area: Dict[int, float] = {}
    upstream_max_order: Dict[int, float] = {}
    upstream_mean_order: Dict[int, float] = {}
    distance_from_headwater: Dict[int, int] = {}
    accumulated_orders: Dict[int, Tuple[float, int]] = {}

    order_attr = column_mapping.get("river_order")
    length_attr = column_mapping.get("length")
    area_attr = column_mapping.get("catchment_area")

    for node in topo_order:
        predecessors = list(graph.predecessors(node))
        own_order = _safe_numeric(graph.nodes[node].get(order_attr), 0.0) if order_attr else 0.0
        own_length = _safe_numeric(graph.nodes[node].get(length_attr), 0.0) if length_attr else 0.0
        own_area = _safe_numeric(graph.nodes[node].get(area_attr), 0.0) if area_attr else 0.0
        if not predecessors:
            upstream_count[node] = 0
            upstream_total_length[node] = own_length
            upstream_total_area[node] = own_area
            upstream_max_order[node] = own_order
            accumulated_orders[node] = (own_order, 1)
            upstream_mean_order[node] = own_order
            distance_from_headwater[node] = 0
        else:
            ancestor_set = set()
            for p in predecessors:
                ancestor_set.add(p)
                ancestor_set.update(nx.ancestors(graph, p))
            upstream_count[node] = len(ancestor_set)
            upstream_total_length[node] = own_length + sum(_safe_numeric(graph.nodes[a].get(length_attr), 0.0) if length_attr else 0.0 for a in ancestor_set)
            upstream_total_area[node] = own_area + sum(_safe_numeric(graph.nodes[a].get(area_attr), 0.0) if area_attr else 0.0 for a in ancestor_set)
            all_orders = [own_order] + [
                _safe_numeric(graph.nodes[a].get(order_attr), 0.0) if order_attr else 0.0
                for a in ancestor_set
            ]
            upstream_max_order[node] = max(all_orders)
            upstream_mean_order[node] = float(np.mean(all_orders))
            distance_from_headwater[node] = max(distance_from_headwater[p] + 1 for p in predecessors)

    downstream_count: Dict[int, int] = {}
    distance_to_outlet_graph: Dict[int, int] = {}
    for node in reverse_topo:
        successors = list(graph.successors(node))
        if not successors:
            downstream_count[node] = 0
            distance_to_outlet_graph[node] = 0
        else:
            descendants = set()
            for s in successors:
                descendants.add(s)
                descendants.update(nx.descendants(graph, s))
            downstream_count[node] = len(descendants)
            distance_to_outlet_graph[node] = min(distance_to_outlet_graph[s] + 1 for s in successors)

    rows: List[Dict[str, Any]] = []
    for node in graph.nodes:
        attrs = graph.nodes[node]
        order = _safe_numeric(attrs.get(order_attr), 0.0) if order_attr else 0.0
        successor = next(iter(graph.successors(node)), None)
        downstream_order = (
            _safe_numeric(graph.nodes[successor].get(order_attr), order)
            if successor is not None and order_attr else order
        )
        predecessors = list(graph.predecessors(node))
        parent_orders = [
            _safe_numeric(graph.nodes[p].get(order_attr), order) if order_attr else order
            for p in predecessors
        ]
        row = {
            "HYRIV_ID": int(node),
            "river_order_raw": order,
            "river_order_normalized": order,
            "river_order_log": np.log1p(max(order, 0)),
            "river_order_parent_difference": order - (float(np.mean(parent_orders)) if parent_orders else order),
            "upstream_max_river_order": upstream_max_order[node],
            "upstream_mean_river_order": upstream_mean_order[node],
            "downstream_order_difference": downstream_order - order,
            "is_order_transition": float(downstream_order != order),
            "length_km": _safe_numeric(attrs.get(column_mapping.get("length")), 0.0) if column_mapping.get("length") else 0.0,
            "catchment_area": _safe_numeric(attrs.get(column_mapping.get("catchment_area")), 0.0) if column_mapping.get("catchment_area") else 0.0,
            "upstream_area": _safe_numeric(attrs.get(column_mapping.get("upstream_area")), 0.0) if column_mapping.get("upstream_area") else 0.0,
            "distance_downstream": _safe_numeric(attrs.get(column_mapping.get("distance_to_outlet")), 0.0) if column_mapping.get("distance_to_outlet") else 0.0,
            "distance_upstream": _safe_numeric(attrs.get(column_mapping.get("distance_upstream")), 0.0) if column_mapping.get("distance_upstream") else 0.0,
            "average_discharge": _safe_numeric(attrs.get(column_mapping.get("discharge")), 0.0) if column_mapping.get("discharge") else 0.0,
            "endorheic": _safe_numeric(attrs.get(column_mapping.get("endorheic")), 0.0) if column_mapping.get("endorheic") else 0.0,
            "in_degree": float(graph.in_degree(node)),
            "out_degree": float(graph.out_degree(node)),
            "is_headwater": float(graph.in_degree(node) == 0),
            "is_outlet": float(graph.out_degree(node) == 0),
            "is_junction": float(graph.in_degree(node) > 1),
            "component_size": float(component_size[node]),
            "number_of_upstream_nodes": float(upstream_count[node]),
            "number_of_downstream_nodes": float(downstream_count[node]),
            "distance_to_outlet_graph": float(distance_to_outlet_graph[node]),
            "distance_from_headwater_graph": float(distance_from_headwater[node]),
            "upstream_total_length": float(upstream_total_length[node]),
            "upstream_total_area": float(upstream_total_area[node]),
        }
        rows.append(row)

    raw = pd.DataFrame(rows).set_index("HYRIV_ID").sort_index()
    max_order = max(float(raw["river_order_raw"].max()), 1.0)
    raw["river_order_normalized"] = raw["river_order_raw"] / max_order

    transformed = raw.copy()
    for feature in CONTINUOUS_LOG_FEATURES.intersection(transformed.columns):
        transformed[feature] = np.log1p(transformed[feature].clip(lower=0))
    transformed = transformed.replace([np.inf, -np.inf], np.nan)
    transformed = transformed.fillna(transformed.median(numeric_only=True)).fillna(0.0)

    if scaler_type.lower() == "robust":
        scaler = RobustScaler()
    elif scaler_type.lower() == "standard":
        scaler = StandardScaler()
    else:
        raise ValueError("scaler_type must be robust or standard")
    scaled = pd.DataFrame(
        scaler.fit_transform(transformed),
        index=transformed.index,
        columns=transformed.columns,
        dtype="float32",
    )
    return raw, scaled, scaler


# ======================================================================
# 10. Build edge features
# ======================================================================

def build_edge_features(
    graph: nx.DiGraph,
    node_feature_df: pd.DataFrame,
    column_mapping: Mapping[str, Optional[str]],
) -> pd.DataFrame:
    """Build one numeric edge-feature row for every directed river edge."""
    main_col = column_mapping.get("main_river_id")
    basin_col = column_mapping.get("basin_id")
    rows = []
    for source, target in graph.edges:
        src = node_feature_df.loc[source]
        dst = node_feature_df.loc[target]
        source_attrs = graph.nodes[source]
        target_attrs = graph.nodes[target]
        rows.append({
            "source": source,
            "target": target,
            "source_river_order": src["river_order_raw"],
            "target_river_order": dst["river_order_raw"],
            "river_order_difference": dst["river_order_raw"] - src["river_order_raw"],
            "source_length_km": src["length_km"],
            "target_length_km": dst["length_km"],
            "same_main_river": float(main_col is not None and source_attrs.get(main_col) == target_attrs.get(main_col)),
            "same_basin": float(basin_col is not None and source_attrs.get(basin_col) == target_attrs.get(basin_col)),
            "distance_transition": dst["distance_downstream"] - src["distance_downstream"],
            "estimated_travel_distance": max(src["length_km"], 0.0),
        })
    frame = pd.DataFrame(rows)
    if frame.empty:
        raise ValueError("Subgraph has no edges")
    numeric = [c for c in frame.columns if c not in {"source", "target"}]
    frame[numeric] = frame[numeric].replace([np.inf, -np.inf], np.nan).fillna(0).astype("float32")
    return frame


# ======================================================================
# 11. Convert graph to PyTorch Geometric
# ======================================================================

def convert_graph_to_pyg(
    graph: nx.DiGraph,
    node_feature_df: pd.DataFrame,
    edge_feature_df: Optional[pd.DataFrame] = None,
    station_hyriv_ids: Optional[Sequence[int]] = None,
    add_reverse_edges: bool = True,
) -> Tuple[Data, Dict[int, int], Dict[int, int]]:
    """Convert sparse NetworkX topology to a validated PyG Data object."""
    node_ids = list(node_feature_df.index.astype(int))
    if set(node_ids) != set(graph.nodes):
        raise ValueError("node_feature_df index must match graph nodes exactly")
    hyriv_to_index = {hyriv: idx for idx, hyriv in enumerate(node_ids)}
    index_to_hyriv = {idx: hyriv for hyriv, idx in hyriv_to_index.items()}

    edge_pairs = [(hyriv_to_index[s], hyriv_to_index[t]) for s, t in graph.edges]
    edge_attrs = None
    if edge_feature_df is not None:
        edge_lookup = edge_feature_df.set_index(["source", "target"])
        feature_cols = [c for c in edge_feature_df.columns if c not in {"source", "target"}]
        attrs = [edge_lookup.loc[(s, t), feature_cols].to_numpy(dtype=np.float32) for s, t in graph.edges]
        edge_attrs = np.stack(attrs)

    if add_reverse_edges:
        original_pairs = list(edge_pairs)
        edge_pairs += [(t, s) for s, t in original_pairs]
        if edge_attrs is not None:
            edge_attrs = np.concatenate([edge_attrs, edge_attrs], axis=0)

    edge_index = torch.tensor(edge_pairs, dtype=torch.long).t().contiguous()
    x = torch.tensor(node_feature_df.to_numpy(dtype=np.float32), dtype=torch.float32)
    station_set = {int(x) for x in (station_hyriv_ids or []) if pd.notna(x)}
    station_mask = torch.tensor([node in station_set for node in node_ids], dtype=torch.bool)

    data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=torch.tensor(edge_attrs, dtype=torch.float32) if edge_attrs is not None else None,
        station_mask=station_mask,
    )
    data.node_ids = torch.tensor(node_ids, dtype=torch.long)
    if torch.isnan(data.x).any():
        raise ValueError("Node feature matrix contains NaN")
    if data.edge_index.numel() and int(data.edge_index.max()) >= data.num_nodes:
        raise ValueError("edge_index contains an out-of-range node index")
    return data, hyriv_to_index, index_to_hyriv


# ======================================================================
# 12. RiverGATEncoder
# ======================================================================

class RiverGATEncoder(nn.Module):
    """Two-layer residual GATv2 encoder with optional edge attributes."""

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 64,
        embedding_dim: int = 32,
        heads: int = 4,
        dropout: float = 0.15,
        edge_dim: Optional[int] = None,
    ) -> None:
        super().__init__()
        self.dropout = dropout
        self.input_projection = nn.Linear(input_dim, hidden_dim)
        self.conv1 = GATv2Conv(
            hidden_dim,
            hidden_dim // heads,
            heads=heads,
            concat=True,
            dropout=dropout,
            edge_dim=edge_dim,
            residual=True,
        )
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.conv2 = GATv2Conv(
            hidden_dim,
            hidden_dim // heads,
            heads=heads,
            concat=True,
            dropout=dropout,
            edge_dim=edge_dim,
            residual=True,
        )
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.embedding_projection = nn.Linear(hidden_dim, embedding_dim)
        self.feature_decoder = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ELU(),
            nn.Linear(hidden_dim, input_dim),
        )

    def encode(self, x: torch.Tensor, edge_index: torch.Tensor, edge_attr: Optional[torch.Tensor] = None) -> torch.Tensor:
        h = self.input_projection(x)
        residual = h
        h = self.conv1(h, edge_index, edge_attr)
        h = self.norm1(F.elu(h))
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = self.conv2(h, edge_index, edge_attr)
        h = self.norm2(F.elu(h + residual))
        return self.embedding_projection(h)

    def decode_features(self, z: torch.Tensor) -> torch.Tensor:
        return self.feature_decoder(z)

    def decode_links(self, z: torch.Tensor, edge_pairs: torch.Tensor) -> torch.Tensor:
        source, target = edge_pairs
        return (z[source] * z[target]).sum(dim=-1)

    def forward(self, data: Data) -> torch.Tensor:
        return self.encode(data.x, data.edge_index, getattr(data, "edge_attr", None))


# ======================================================================
# 13. Self-supervised objectives
# ======================================================================

def split_positive_edges(
    graph: nx.DiGraph,
    hyriv_to_index: Mapping[int, int],
    val_fraction: float = 0.10,
    test_fraction: float = 0.10,
    random_seed: int = 42,
) -> Dict[str, torch.Tensor]:
    """Split physical river edges while grouping by weak component.

    Large components are split deterministically by a seeded permutation.
    Tiny components remain in training so they do not disappear from message passing.
    """
    rng = np.random.default_rng(random_seed)
    train_edges: List[Tuple[int, int]] = []
    val_edges: List[Tuple[int, int]] = []
    test_edges: List[Tuple[int, int]] = []
    for component in nx.weakly_connected_components(graph):
        edges = list(graph.subgraph(component).edges)
        rng.shuffle(edges)
        n = len(edges)
        if n < 10:
            train_edges.extend(edges)
            continue
        n_test = max(1, int(n * test_fraction))
        n_val = max(1, int(n * val_fraction))
        test_edges.extend(edges[:n_test])
        val_edges.extend(edges[n_test:n_test + n_val])
        train_edges.extend(edges[n_test + n_val:])

    def tensor(edges: Sequence[Tuple[int, int]]) -> torch.Tensor:
        if not edges:
            return torch.empty((2, 0), dtype=torch.long)
        return torch.tensor(
            [(hyriv_to_index[s], hyriv_to_index[t]) for s, t in edges],
            dtype=torch.long,
        ).t().contiguous()

    return {"train": tensor(train_edges), "val": tensor(val_edges), "test": tensor(test_edges)}


def sample_hydrologic_negative_edges(
    graph: nx.DiGraph,
    positive_edges: torch.Tensor,
    index_to_hyriv: Mapping[int, int],
    hyriv_to_index: Mapping[int, int],
    num_samples: int,
    random_seed: int,
    main_river_attr: Optional[str] = "MAIN_RIV",
    basin_attr: Optional[str] = "HYBAS_L12",
) -> torch.Tensor:
    """Sample negatives across different basins/main rivers when possible.

    It rejects direct edges, self-pairs, and pairs on the same known river path.
    """
    rng = np.random.default_rng(random_seed)
    node_indices = np.array(list(index_to_hyriv.keys()), dtype=int)
    positives = set(zip(positive_edges[0].tolist(), positive_edges[1].tolist()))
    negatives: Set[Tuple[int, int]] = set()
    attempts = 0
    max_attempts = max(10_000, num_samples * 100)
    while len(negatives) < num_samples and attempts < max_attempts:
        attempts += 1
        src_idx, dst_idx = rng.choice(node_indices, 2, replace=False)
        pair = (int(src_idx), int(dst_idx))
        if pair in positives or pair in negatives:
            continue
        src, dst = index_to_hyriv[pair[0]], index_to_hyriv[pair[1]]
        if graph.has_edge(src, dst) or graph.has_edge(dst, src):
            continue
        src_attrs, dst_attrs = graph.nodes[src], graph.nodes[dst]
        if basin_attr and src_attrs.get(basin_attr) == dst_attrs.get(basin_attr):
            if main_river_attr and src_attrs.get(main_river_attr) == dst_attrs.get(main_river_attr):
                continue
        negatives.add(pair)
    if len(negatives) < num_samples:
        warnings.warn(f"Only sampled {len(negatives)}/{num_samples} constrained negative edges.")
    if not negatives:
        return torch.empty((2, 0), dtype=torch.long)
    return torch.tensor(sorted(negatives), dtype=torch.long).t().contiguous()


# ======================================================================
# 14. Train GAT
# ======================================================================

def train_river_gat(
    model: RiverGATEncoder,
    pyg_data: Data,
    graph: nx.DiGraph,
    hyriv_to_index: Mapping[int, int],
    index_to_hyriv: Mapping[int, int],
    config: RiverGATConfig,
) -> Tuple[RiverGATEncoder, pd.DataFrame]:
    """Train link prediction + node-feature reconstruction with early stopping."""
    set_global_seed(config.random_seed)
    device = resolve_device(config.device)
    model = model.to(device)
    data = pyg_data.to(device)
    splits = split_positive_edges(
        graph,
        hyriv_to_index,
        val_fraction=config.val_fraction,
        test_fraction=config.test_fraction,
        random_seed=config.random_seed,
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    bce = nn.BCEWithLogitsLoss()
    mse = nn.MSELoss()
    best_state = None
    best_val = np.inf
    patience_left = config.patience
    history: List[Dict[str, float]] = []

    def edge_loss(z: torch.Tensor, positive: torch.Tensor, seed: int) -> torch.Tensor:
        positive = positive.to(device)
        n_negative = max(1, int(positive.shape[1] * config.negative_ratio))
        negative = sample_hydrologic_negative_edges(
            graph, positive.cpu(), index_to_hyriv, hyriv_to_index, n_negative, seed
        ).to(device)
        pos_logits = model.decode_links(z, positive)
        neg_logits = model.decode_links(z, negative)
        logits = torch.cat([pos_logits, neg_logits])
        labels = torch.cat([torch.ones_like(pos_logits), torch.zeros_like(neg_logits)])
        return bce(logits, labels)

    try:
        for epoch in range(1, config.epochs + 1):
            model.train()
            optimizer.zero_grad(set_to_none=True)
            z = model(data)
            link = edge_loss(z, splits["train"], config.random_seed + epoch)
            reconstruction = mse(model.decode_features(z), data.x)
            loss = config.link_loss_weight * link + config.feature_loss_weight * reconstruction
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip_val)
            optimizer.step()

            model.eval()
            with torch.no_grad():
                z_val = model(data)
                if splits["val"].shape[1]:
                    val_link = edge_loss(z_val, splits["val"], config.random_seed + 100_000 + epoch)
                else:
                    val_link = link.detach()
                val_recon = mse(model.decode_features(z_val), data.x)
                val_loss = config.link_loss_weight * val_link + config.feature_loss_weight * val_recon

            row = {
                "epoch": epoch,
                "train_loss": float(loss.item()),
                "train_link_loss": float(link.item()),
                "train_feature_loss": float(reconstruction.item()),
                "validation_loss": float(val_loss.item()),
            }
            history.append(row)
            if epoch == 1 or epoch % 20 == 0:
                LOGGER.info("GAT epoch %d | train %.5f | val %.5f", epoch, row["train_loss"], row["validation_loss"])

            if row["validation_loss"] < best_val - 1e-6:
                best_val = row["validation_loss"]
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                patience_left = config.patience
            else:
                patience_left -= 1
                if patience_left <= 0:
                    LOGGER.info("Early stopping at epoch %d.", epoch)
                    break
    except torch.cuda.OutOfMemoryError:
        if device.type != "cuda":
            raise
        LOGGER.warning("CUDA OOM. Retrying GAT training on CPU.")
        del data, model
        torch.cuda.empty_cache()
        cpu_config = RiverGATConfig(**{**config.__dict__, "device": "cpu"})
        cpu_model = RiverGATEncoder(
            input_dim=pyg_data.x.shape[1], hidden_dim=config.hidden_dim,
            embedding_dim=config.embedding_dim, heads=config.heads,
            dropout=config.dropout,
            edge_dim=pyg_data.edge_attr.shape[1] if getattr(pyg_data, "edge_attr", None) is not None else None,
        )
        return train_river_gat(cpu_model, pyg_data.cpu(), graph, hyriv_to_index, index_to_hyriv, cpu_config)

    if best_state is not None:
        model.load_state_dict(best_state)
    return model.cpu(), pd.DataFrame(history)


# ======================================================================
# 15. Generate river embeddings
# ======================================================================

def generate_node_embeddings(
    model: RiverGATEncoder,
    pyg_data: Data,
    index_to_hyriv: Mapping[int, int],
    device: str = "auto",
) -> pd.DataFrame:
    """Return one `embedding_*` vector per HydroRIVERS segment."""
    resolved = resolve_device(device)
    model = model.to(resolved).eval()
    data = pyg_data.to(resolved)
    with torch.no_grad():
        embeddings = model(data).cpu().numpy().astype("float32")
    columns = [f"embedding_{i}" for i in range(embeddings.shape[1])]
    output = pd.DataFrame(embeddings, columns=columns)
    output.insert(0, "HYRIV_ID", [index_to_hyriv[i] for i in range(len(output))])
    return output


# ======================================================================
# 16. Build station node_embedding DataFrame
# ======================================================================

def build_station_node_embedding(
    station_river_mapping: pd.DataFrame,
    river_embeddings: pd.DataFrame,
) -> pd.DataFrame:
    """Merge river embeddings onto stations with strict quality checks."""
    required = {"nama_pos", "HYRIV_ID"}
    if not required.issubset(station_river_mapping.columns):
        raise KeyError(f"station_river_mapping must include {required}")
    embedding_columns = [c for c in river_embeddings if c.startswith("embedding_")]
    if not embedding_columns:
        raise ValueError("river_embeddings has no embedding_* columns")

    output = station_river_mapping.merge(
        river_embeddings[["HYRIV_ID"] + embedding_columns],
        on="HYRIV_ID",
        how="left",
        validate="many_to_one",
    )
    if output["nama_pos"].duplicated().any():
        raise ValueError("Final node_embedding must have one row per nama_pos")
    unmatched = output["HYRIV_ID"].isna()
    missing_embedding = output[embedding_columns].isna().any(axis=1)
    if unmatched.any() or missing_embedding.any():
        bad = output.loc[unmatched | missing_embedding, ["nama_pos", "HYRIV_ID", "snap_quality"]]
        raise ValueError(
            "Some stations are unmatched or missing an embedding. They are not silently zero-filled:\n"
            + bad.to_string(index=False)
        )
    output[embedding_columns] = output[embedding_columns].astype("float32")
    return output


# ======================================================================
# 17. Save artifacts
# ======================================================================

def save_embedding_artifacts(
    node_embedding: pd.DataFrame,
    model: RiverGATEncoder,
    training_history: pd.DataFrame,
    config: RiverGATConfig,
) -> Dict[str, str]:
    """Save embedding CSV, model checkpoint, training history, and config."""
    output_dir = Path(config.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    embedding_path = output_dir / config.embedding_filename
    model_path = output_dir / config.model_filename
    history_path = output_dir / "gat_training_history.csv"
    config_path = output_dir / "river_gat_config.json"
    node_embedding.to_csv(embedding_path, index=False)
    torch.save(model.state_dict(), model_path)
    training_history.to_csv(history_path, index=False)
    config_path.write_text(json.dumps(config.__dict__, indent=2, default=str))
    return {
        "embedding": str(embedding_path.resolve()),
        "model": str(model_path.resolve()),
        "history": str(history_path.resolve()),
        "config": str(config_path.resolve()),
    }


# ======================================================================
# 18. End-to-end GAT preparation
# ======================================================================

def run_spatial_embedding_pipeline(
    hydroriver_path: str | Path,
    df_koordinat: pd.DataFrame,
    config: Optional[RiverGATConfig] = None,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Run the complete HydroRIVERS -> GAT -> station embedding pipeline."""
    config = config or RiverGATConfig()
    set_global_seed(config.random_seed)
    stations, station_report = validate_station_coordinates(df_koordinat)

    bbox = station_bbox(stations, config.snap_bbox_buffer_deg)
    river_geometry, hydro_report, geometry_mapping = load_hydrorivers(
        hydroriver_path, target_crs=config.target_crs, bbox=bbox
    )
    station_mapping, snap_report = match_stations_to_rivers(
        stations,
        river_geometry,
        max_snap_distance_m=config.max_snap_distance_m,
        projected_crs=config.projected_crs,
    )
    del river_geometry
    gc.collect()

    all_attributes, attribute_mapping = load_hydroriver_attributes(hydroriver_path)
    sid = attribute_mapping["segment_id"]
    did = attribute_mapping["downstream_id"]
    graph, topology_report = build_river_topology(all_attributes, sid, did)
    if topology_report["n_cycles"]:
        raise ValueError(
            f"HydroRIVERS topology contains {topology_report['n_cycles']} cycles. "
            "Call break_cycles() and inspect its audit table before continuing."
        )

    subgraph = extract_station_river_subgraph(
        graph,
        station_mapping["HYRIV_ID"].dropna().astype(int).tolist(),
        upstream_hops=config.upstream_hops,
        downstream_hops=config.downstream_hops,
        include_all_upstream=config.include_all_upstream,
        include_path_to_outlet=config.include_path_to_outlet,
    )
    del graph, all_attributes
    gc.collect()

    if subgraph.number_of_nodes() > config.max_graph_nodes or subgraph.number_of_edges() > config.max_graph_edges:
        estimate = estimate_graph_memory(subgraph.number_of_nodes(), subgraph.number_of_edges(), 24, 9)
        raise MemoryError(
            f"Subgraph too large: {subgraph.number_of_nodes():,} nodes / "
            f"{subgraph.number_of_edges():,} edges. Estimated minimum tensors: "
            f"{estimate['minimum_tensor_mb']:.1f} MiB. Use hop-limited mode."
        )

    raw_features, scaled_features, scaler = build_node_features(
        subgraph, attribute_mapping, scaler_type=config.scaler
    )
    edge_features = build_edge_features(subgraph, raw_features, attribute_mapping)
    pyg_data, hyriv_to_index, index_to_hyriv = convert_graph_to_pyg(
        subgraph,
        scaled_features,
        edge_features if config.use_edge_features else None,
        station_mapping["HYRIV_ID"].dropna().astype(int).tolist(),
        add_reverse_edges=config.add_reverse_edges_for_message_passing,
    )

    edge_dim = pyg_data.edge_attr.shape[1] if getattr(pyg_data, "edge_attr", None) is not None else None
    model = RiverGATEncoder(
        input_dim=pyg_data.x.shape[1],
        hidden_dim=config.hidden_dim,
        embedding_dim=config.embedding_dim,
        heads=config.heads,
        dropout=config.dropout,
        edge_dim=edge_dim,
    )
    model, training_history = train_river_gat(
        model, pyg_data, subgraph, hyriv_to_index, index_to_hyriv, config
    )
    river_embeddings = generate_node_embeddings(model, pyg_data, index_to_hyriv, config.device)
    node_embedding = build_station_node_embedding(station_mapping, river_embeddings)
    paths = save_embedding_artifacts(node_embedding, model, training_history, config)

    reports = {
        "station_validation": station_report,
        "hydrorivers": hydro_report,
        "snapping": snap_report,
        "topology": topology_report,
        "subgraph": {
            "n_nodes": subgraph.number_of_nodes(),
            "n_edges": subgraph.number_of_edges(),
            "memory_estimate": estimate_graph_memory(
                subgraph.number_of_nodes(), subgraph.number_of_edges(),
                pyg_data.x.shape[1], edge_dim or 0,
            ),
        },
        "embedding": {
            "shape": node_embedding.shape,
            "n_embedding_dimensions": config.embedding_dim,
            "paths": paths,
        },
    }
    return node_embedding, reports


# ======================================================================
# 19. Embedding quality, visualization, and ablation helpers
# ======================================================================

def embedding_quality_report(node_embedding: pd.DataFrame) -> Dict[str, Any]:
    """Report distribution, missing values, and constant dimensions."""
    columns = [c for c in node_embedding if c.startswith("embedding_")]
    stats = node_embedding[columns].describe().T
    return {
        "statistics": stats,
        "missing_values": node_embedding[columns].isna().sum().to_dict(),
        "constant_dimensions": stats.index[stats["std"].fillna(0).eq(0)].tolist(),
    }


def station_cosine_similarity(node_embedding: pd.DataFrame) -> pd.DataFrame:
    """Compute station-by-station cosine similarity matrix."""
    columns = [c for c in node_embedding if c.startswith("embedding_")]
    matrix = node_embedding[columns].to_numpy(dtype=np.float32)
    norm = np.linalg.norm(matrix, axis=1, keepdims=True)
    normalized = matrix / np.clip(norm, 1e-8, None)
    similarity = normalized @ normalized.T
    names = node_embedding["nama_pos"].astype(str)
    return pd.DataFrame(similarity, index=names, columns=names)


def project_embeddings_2d(node_embedding: pd.DataFrame, method: str = "pca") -> pd.DataFrame:
    """Project station embeddings to two dimensions using PCA or UMAP."""
    columns = [c for c in node_embedding if c.startswith("embedding_")]
    matrix = node_embedding[columns].to_numpy(dtype=np.float32)
    if method.lower() == "pca":
        projection = PCA(n_components=2, random_state=42).fit_transform(matrix)
    elif method.lower() == "umap":
        try:
            import umap
        except ImportError as exc:
            raise ImportError("Install umap-learn for UMAP projection") from exc
        projection = umap.UMAP(n_components=2, random_state=42).fit_transform(matrix)
    else:
        raise ValueError("method must be pca or umap")
    output = node_embedding[["nama_pos", "HYRIV_ID", "river_order", "snap_quality"]].copy()
    output["component_1"] = projection[:, 0]
    output["component_2"] = projection[:, 1]
    return output


def plot_embedding_projection(projected: pd.DataFrame, color_by: str = "river_order", title: str = "GAT station embedding"):
    """Create a simple matplotlib scatter plot for a 2-D projection."""
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(9, 7))
    scatter = ax.scatter(projected["component_1"], projected["component_2"], c=projected[color_by])
    for row in projected.itertuples():
        ax.annotate(str(row.nama_pos), (row.component_1, row.component_2), fontsize=7, alpha=0.75)
    ax.set_title(title)
    ax.set_xlabel("component_1")
    ax.set_ylabel("component_2")
    fig.colorbar(scatter, ax=ax, label=color_by)
    fig.tight_layout()
    return fig, ax


def summarize_ablation_result(
    experiment: str,
    model: str,
    features: str,
    actual: Sequence[float],
    predicted: Sequence[float],
    training_time: float,
    peak_memory: float = np.nan,
) -> Dict[str, Any]:
    """Create one standardized ablation result row."""
    actual_arr = np.asarray(actual, dtype=float)
    pred_arr = np.asarray(predicted, dtype=float)
    error = pred_arr - actual_arr
    return {
        "experiment": experiment,
        "model": model,
        "features": features,
        "RMSE": float(np.sqrt(np.mean(error**2))),
        "MAE": float(np.mean(np.abs(error))),
        "bias": float(np.mean(error)),
        "training_time": float(training_time),
        "peak_memory": float(peak_memory),
    }


def catboost_embedding_importance(model: Any, feature_names: Sequence[str]) -> Dict[str, Any]:
    """Summarize total and per-column CatBoost importance for embedding features."""
    importance = pd.Series(model.get_feature_importance(), index=list(feature_names)).sort_values(ascending=False)
    embedding = importance[importance.index.str.startswith("embedding_")]
    return {
        "all_importance": importance,
        "embedding_importance": embedding,
        "embedding_total_importance": float(embedding.sum()),
    }


# ======================================================================
# 20. Colab usage example
# ======================================================================

COLAB_USAGE = r"""
# 1) Extract competition archive
!unzip -q /content/sebelas-maret-statistics-data-science-2026.zip -d /content/semar

from pathlib import Path
import pandas as pd

ROOT = Path('/content/semar')
HYDRO = ROOT / 'data_pendukung/HydroRIVERS_v10_au_shp/HydroRIVERS_v10_au.shp'

df_train = pd.read_csv(ROOT / 'train.csv')
df_test = pd.read_csv(ROOT / 'test.csv')
df_lingkungan = pd.read_csv(ROOT / 'data_pendukung/data_lingkungan.csv')
df_koordinat = pd.read_csv(ROOT / 'data_pendukung/koordinat_pos.csv')
df_submission = pd.read_csv(ROOT / 'sample_submission.csv')

# Full catchment mode. If RAM is limited, set include_all_upstream=False,
# upstream_hops=20, downstream_hops=20.
gat_config = RiverGATConfig(
    output_dir='/content/artifacts',
    include_all_upstream=True,
    include_path_to_outlet=True,
    embedding_dim=32,
    device='auto',
)
node_embedding, gat_reports = run_spatial_embedding_pipeline(
    HYDRO, df_koordinat, gat_config
)

# Downstream ensemble
submission, metrics, weights, validation = run_ensemble(
    df_train,
    df_test,
    df_lingkungan,
    node_embedding,
    df_submission,
    config={
        'device': 'auto',
        'neural_model': 'nhits',
        'neural_max_exog_features': 40,
    },
)
"""



# ======================================================================
# USER-PROVIDED HARDENED DOWNSTREAM ENSEMBLE
# ======================================================================

"""
CatBoost + neural + seasonal ensemble for forecasting tma_mdpl.
HARDENED VERSION — optimized to avoid Google Colab session crashes (OOM).
NODE-EMBEDDING FIX — node_embedding (GAT emb_*) is now correctly threaded
through prepare_data -> build_panel -> get_model_features, so BOTH CatBoost
and the neural model (NHITS/TiDE) train on it as ordinary numeric features.

Bugs fixed relative to the version you pasted (see "# FIX" comments):
  1. build_panel() referenced an undefined global `node_embedding` instead
     of its own `node` parameter -> NameError at runtime.
  2. _build_common_state() called build_panel(train, test, environment, config)
     — missing the `node` argument entirely, so embeddings were never merged.
  3. run_models() doesn't need node/df_koordinat as a separate argument —
     embeddings are already merged into `panel` inside build_panel(), and
     get_model_features() picks them up automatically as numeric columns.
     The extra `node` argument being passed into run_models() in
     run_validation_only()/run_final_only() didn't match its signature and
     would raise a TypeError.
  4. run_final_only() mixed the parameter name `node_embedding` with a
     never-defined local `df_koordinat` in three places.
  5. run_ensemble() didn't forward `node_embedding` to run_validation_only()
     / run_final_only()/at all.

Because emb_* columns are just extra numeric columns on `panel` after the
fix, both models pick them up "for free":
  - CatBoost: get_model_features() includes every numeric column (except an
    ignore-list) in `catboost_features`, so emb_* are used natively as
    dense numeric features.
  - Neural (NHITS/TiDE): the same numeric columns become `neural_features`,
    passed to the model as `futr_exog_list`. Since embeddings are static
    per-station, they are trivially known for every future timestamp too,
    so they work correctly as future-known exogenous inputs. If you cap
    them with config['neural_max_exog_features'], they are ranked by
    CatBoost feature importance along with every other feature — no
    special-casing needed.

Install dependencies:
    pip install pandas numpy catboost neuralforecast torch psutil

Use from a notebook (all-in-one, same interface as before):
    submission, metrics, weights, validation = run_ensemble(
        df_train,
        df_test,
        df_lingkungan,
        node_embedding,
        df_submission,
        config={"device": "auto"},
    )

Or, to protect against Colab OOM, split into two runtime sessions:
    # --- Session 1 ---
    weights, metrics, validation = run_validation_only(
        df_train, df_test, df_lingkungan, node_embedding, config={"device": "auto"}
    )
    # weights/metrics are saved to disk automatically.
    # Runtime > Restart runtime here to fully release RAM.

    # --- Session 2 (after restart, re-load df_train/df_test/df_lingkungan/node_embedding) ---
    submission = run_final_only(
        df_train, df_test, df_lingkungan, node_embedding, df_submission,
        weights_path="weights.csv", config={"device": "auto"},
    )
"""

import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from catboost import CatBoostRegressor
from neuralforecast import NeuralForecast
from neuralforecast.losses.pytorch import MAE
from neuralforecast.models import NHITS, TiDE

try:
    import psutil
    _HAS_PSUTIL = True
except ImportError:
    _HAS_PSUTIL = False


OBSERVATIONS_PER_DAY = 3
RANDOM_SEED = 42
PIPELINE_VERSION = "2026-07-hydrologic-ensemble-v5-node-embedding-fix"


DEFAULT_CONFIG = {
    "device": "auto",                 # "auto", "gpu", or "cpu"
    "gpu_index": 0,
    "precision": "32-true",
    "neural_model": "nhits",          # "nhits" (recommended) or "tide"
    "input_days": 120,
    "tide_hidden_size": 64,
    "tide_batch_size": 8,
    "tide_windows_batch_size": 32,
    "tide_inference_batch_size": 32,
    "tide_max_steps": 300,
    "catboost_task_type": "CPU",
    "catboost_iterations": 800,
    "catboost_depth": 7,
    "catboost_gpu_ram_part": 0.25,
    "add_hydrology_features": True,
    "report_worst_stations": 10,
    "seasonal_window_days": 15,
    "seasonal_anchor_half_life_days": 3,
    "weight_step": 0.05,
    "ensemble_objective": "balanced",
    "validation_mode": "same_season",
    "output_file": "submission_ens.csv",

    "max_panel_rows": 5_000_000,      # guard against runaway cross-join
    "neural_max_exog_features": 40,   # cap neural exogenous feature count
    "neural_use_static_exog": True,    # embedding_* via stat_exog_list
    "log_memory": True,               # print RAM usage at checkpoints
    "weights_output_file": "weights.csv",
    "metrics_output_file": "metrics.csv",
}

# ----------------------------------------------------------------------
# Memory helpers
# ----------------------------------------------------------------------

def log_memory(tag, config=None):
    if config is not None and not config.get("log_memory", True):
        return
    if not _HAS_PSUTIL:
        return
    mem = psutil.virtual_memory()
    print(
        f"[mem] {tag}: {mem.percent:.1f}% used "
        f"({mem.used / 1e9:.2f} GB / {mem.total / 1e9:.2f} GB)"
    )


def downcast_numeric(frame):
    frame = frame.copy()
    float_columns = frame.select_dtypes(include="float64").columns
    if len(float_columns):
        frame[float_columns] = frame[float_columns].astype("float32")
    int_columns = frame.select_dtypes(include="int64").columns
    for column in int_columns:
        frame[column] = pd.to_numeric(frame[column], downcast="integer")
    return frame


def free(*objects):
    del objects
    gc.collect()


def safe_fillna(series, fill_value):
    if isinstance(fill_value, (pd.Series, np.ndarray)):
        fill_value = pd.Series(fill_value).astype(series.dtype)
    else:
        fill_value = series.dtype.type(fill_value)
    return series.fillna(fill_value)


# ----------------------------------------------------------------------
# Config handling
# ----------------------------------------------------------------------

def prepare_config(config=None):
    settings = DEFAULT_CONFIG.copy()
    if config is not None:
        settings.update(config)

    requested_device = str(settings["device"]).lower()
    if requested_device not in {"auto", "gpu", "cpu"}:
        raise ValueError("config['device'] must be 'auto', 'gpu', or 'cpu'.")

    gpu_available = torch.cuda.is_available()
    if requested_device == "gpu" and not gpu_available:
        print("CUDA is unavailable. The neural model will use CPU instead.")

    use_gpu = gpu_available and requested_device in {"auto", "gpu"}
    settings["accelerator"] = "gpu" if use_gpu else "cpu"
    settings["precision"] = settings["precision"] if use_gpu else "32-true"
    settings["devices"] = [settings["gpu_index"]] if use_gpu else 1

    catboost_device = str(settings["catboost_task_type"]).upper()
    if catboost_device not in {"CPU", "GPU"}:
        raise ValueError("config['catboost_task_type'] must be 'CPU' or 'GPU'.")
    if catboost_device == "GPU" and not gpu_available:
        print("CUDA is unavailable. CatBoost will use CPU instead.")
        catboost_device = "CPU"
    settings["catboost_task_type"] = catboost_device

    if settings["ensemble_objective"] not in {"balanced", "rmse", "mae"}:
        raise ValueError(
            "config['ensemble_objective'] must be 'balanced', 'rmse', or 'mae'."
        )
    if settings["validation_mode"] not in {"same_season", "last_block"}:
        raise ValueError(
            "config['validation_mode'] must be 'same_season' or 'last_block'."
        )

    settings["neural_model"] = str(settings["neural_model"]).lower()
    if settings["neural_model"] not in {"nhits", "tide"}:
        raise ValueError("config['neural_model'] must be 'nhits' or 'tide'.")

    return settings


def release_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ----------------------------------------------------------------------
# Data preparation
# ----------------------------------------------------------------------

def prepare_data(train_df_clean, test_df, datling_df, node_embedding):
    """Copy and standardize the DataFrames without changing the originals."""
    train = train_df_clean.copy()
    test = test_df.copy()
    environment = datling_df.copy()
    node = node_embedding.copy()

    train["datetime"] = pd.to_datetime(train["datetime"])
    environment["datetime"] = pd.to_datetime(environment["datetime"])

    if "datetime" not in test.columns or "nama_pos" not in test.columns:
        parsed = test["id"].str.extract(
            r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) - (.*)$"
        )
        if parsed.isna().any().any():
            raise ValueError("Some test IDs do not match 'datetime - nama_pos'.")
        test["datetime"] = pd.to_datetime(parsed[0])
        test["nama_pos"] = parsed[1]
    else:
        test["datetime"] = pd.to_datetime(test["datetime"])

    if "id" not in test.columns:
        test["id"] = test["datetime"].astype(str) + " - " + test["nama_pos"]

    test["original_order"] = np.arange(len(test))

    train["nama_pos"] = train["nama_pos"].astype("string")
    test["nama_pos"] = test["nama_pos"].astype("string")
    environment["nama_pos"] = environment["nama_pos"].astype("string")

    # FIX: node embedding table standardized here, once, and returned as-is.
    node["nama_pos"] = node["nama_pos"].astype("string")
    if "HYRIV_ID" in node.columns:
        node["HYRIV_ID"] = node["HYRIV_ID"].astype("string")

    return train, test, environment, node


def clean_environment(environment):
    environment = environment.copy()

    if "solar_radiation_mj_m2" in environment.columns:
        bad_solar = environment["solar_radiation_mj_m2"].eq(-999)
        environment["solar_radiation_missing"] = bad_solar.astype(int)
        environment.loc[bad_solar, "solar_radiation_mj_m2"] = np.nan

    environment = environment.drop_duplicates(["nama_pos", "datetime"])
    environment = environment.sort_values(["nama_pos", "datetime"])

    numeric_columns = environment.select_dtypes(include="number").columns.tolist()

    for column in numeric_columns:
        environment[column] = environment[column].replace([np.inf, -np.inf], np.nan)
        if environment[column].isna().any():
            environment[column + "_missing"] = environment[column].isna().astype(int)
            environment[column] = environment.groupby("nama_pos", observed=True)[column].transform(
                lambda values: values.interpolate(limit_direction="both")
            )
            environment[column] = environment[column].fillna(environment[column].median())

    # NOTE: do NOT downcast here — later transforms still write float64
    # results back into these columns. Downcast happens once, at the end
    # of build_panel().

    return environment


def _grouped_rolling(series, group_key, window, min_periods=1, agg="sum"):
    rolled = series.groupby(group_key, observed=True).rolling(
        window, min_periods=min_periods
    )
    result = getattr(rolled, agg)()
    return result.reset_index(level=0, drop=True).sort_index()


def _grouped_ewm(series, group_key, alpha):
    ewm = series.groupby(group_key, observed=True).ewm(alpha=alpha, adjust=False)
    result = ewm.mean()
    return result.reset_index(level=0, drop=True).sort_index()


def add_hydrology_features(environment):
    environment = environment.copy()
    environment = environment.sort_values(["nama_pos", "datetime"])
    station = environment["nama_pos"]

    if {"rainfall_mm", "rainfall_openmeteo_mm"}.issubset(environment.columns):
        left = environment["rainfall_mm"].to_numpy()
        right = environment["rainfall_openmeteo_mm"].to_numpy()
        if np.allclose(left, right, equal_nan=True):
            environment = environment.drop(columns="rainfall_openmeteo_mm")

    if "rainfall_mm" in environment.columns:
        rain = environment["rainfall_mm"].clip(lower=0).fillna(0)

        for days in [1, 3, 7, 14, 30]:
            hours = 24 * days
            environment[f"rain_sum_{days}d"] = _grouped_rolling(
                rain, station, hours, min_periods=1, agg="sum"
            )

        for days in [1, 3, 7]:
            hours = 24 * days
            environment[f"rain_max_{days}d"] = _grouped_rolling(
                rain, station, hours, min_periods=1, agg="max"
            )

        wet = rain.gt(0.1).astype("float32")
        for days in [3, 7]:
            hours = 24 * days
            environment[f"wet_fraction_{days}d"] = _grouped_rolling(
                wet, station, hours, min_periods=1, agg="mean"
            )

        for retention in [0.70, 0.85, 0.93]:
            hourly_retention = retention ** (1 / 24)
            alpha = 1 - hourly_retention
            name = f"api_{int(retention * 100):03d}"
            environment[name] = _grouped_ewm(rain, station, alpha)

        rain_event = rain.gt(0.1)
        event_number = rain_event.groupby(station, observed=True).cumsum()
        environment["hours_since_rain"] = (~rain_event).groupby(
            [station, event_number], observed=True
        ).cumsum().clip(upper=24 * 60)

    soil_columns = [
        column
        for column in [
            "soil_moisture_0_7cm",
            "soil_moisture_7_28cm",
            "soil_moisture_28_100cm",
            "soil_moisture_100_255cm",
        ]
        if column in environment.columns
    ]

    if soil_columns:
        shallow = soil_columns[:2]
        deep = soil_columns[2:]
        environment["soil_shallow_mean"] = environment[shallow].mean(axis=1)
        environment["soil_deep_mean"] = (
            environment[deep].mean(axis=1)
            if deep
            else environment[soil_columns].mean(axis=1)
        )

        interaction_soils = ["soil_shallow_mean", "soil_deep_mean"]
        for rain_feature in ["api_085", "rain_sum_3d", "rain_sum_7d"]:
            if rain_feature in environment.columns:
                for soil_feature in interaction_soils:
                    name = rain_feature + "_x_" + soil_feature
                    environment[name] = (
                        environment[rain_feature] * environment[soil_feature]
                    )

    if {"temperature_c", "dew_point_c"}.issubset(environment.columns):
        environment["dewpoint_depression"] = (
            environment["temperature_c"] - environment["dew_point_c"]
        )

    if {"cloud_cover_pct", "humidity_pct"}.issubset(environment.columns):
        environment["cloud_x_humidity"] = (
            environment["cloud_cover_pct"] * environment["humidity_pct"] / 100
        )

    tendency_columns = [
        column
        for column in soil_columns
        + [
            "humidity_pct",
            "temperature_c",
            "surface_pressure_hpa",
            "pressure_msl_hpa",
            "cloud_cover_pct",
            "dew_point_c",
        ]
        if column in environment.columns
    ]

    for column in tendency_columns:
        for days in [1, 3]:
            lag = 24 * days
            environment[f"{column}_change_{days}d"] = (
                environment[column]
                - environment.groupby("nama_pos", observed=True)[column].shift(lag)
            ).fillna(0)

    return environment


def add_calendar_features(frame):
    frame = frame.copy()
    day = frame["datetime"].dt.dayofyear
    hour = frame["datetime"].dt.hour

    frame["month"] = frame["datetime"].dt.month
    frame["day_of_year"] = day
    frame["hour"] = hour
    frame["day_sin"] = np.sin(2 * np.pi * day / 365.25).astype("float32")
    frame["day_cos"] = np.cos(2 * np.pi * day / 365.25).astype("float32")
    frame["hour_sin"] = np.sin(2 * np.pi * hour / 24).astype("float32")
    frame["hour_cos"] = np.cos(2 * np.pi * hour / 24).astype("float32")

    if "wind_direction_deg" in frame.columns:
        angle = np.deg2rad(frame["wind_direction_deg"])
        frame["wind_sin"] = np.sin(angle).astype("float32")
        frame["wind_cos"] = np.cos(angle).astype("float32")

    if "mjo_phase" in frame.columns:
        angle = 2 * np.pi * (frame["mjo_phase"] - 1) / 8
        frame["mjo_phase_sin"] = np.sin(angle).astype("float32")
        frame["mjo_phase_cos"] = np.cos(angle).astype("float32")

    return frame


def merge_node_embedding(panel, node):
    """FIX: isolated, testable merge of the GAT node embedding into panel.

    Static per-station features merge cleanly onto every row (history AND
    future) via a simple left join on nama_pos, which is exactly what both
    CatBoost (row-wise features) and NHITS/TiDE (futr_exog_list, since the
    value is known for every future timestep) need.
    """
    node = node.copy()
    node["nama_pos"] = node["nama_pos"].astype("string")

    embedding_columns = [column for column in node.columns if column.startswith("embedding_")]

    if len(embedding_columns) == 0:
        raise ValueError("Node embedding tidak memiliki kolom embedding_*.")

    if node["nama_pos"].duplicated().any():
        duplicated = node.loc[node["nama_pos"].duplicated(), "nama_pos"].unique().tolist()
        raise ValueError(f"Duplikat nama_pos pada node embedding: {duplicated}")

    panel = panel.merge(
        node[["nama_pos"] + embedding_columns],
        on="nama_pos",
        how="left",
        validate="many_to_one",
    )

    missing_nodes = panel.loc[panel[embedding_columns[0]].isna(), "nama_pos"].unique()
    if len(missing_nodes):
        print(f"⚠ {len(missing_nodes)} station tidak memiliki embedding: {list(missing_nodes)}")

    panel[embedding_columns] = panel[embedding_columns].fillna(0.0).astype("float32")

    print(f"✓ {len(embedding_columns)} kolom embedding berhasil ditambahkan ke panel.")

    return panel, embedding_columns


def build_panel(train, test, environment, node, config):
    first_time = train["datetime"].min()
    train_end = train["datetime"].max()

    target_hours = sorted(test["datetime"].dt.hour.unique())
    training_days = pd.date_range(first_time.normalize(), train_end.normalize(), freq="D")

    training_times = []
    for day in training_days:
        for hour in target_hours:
            timestamp = day + pd.Timedelta(hours=int(hour))
            if first_time <= timestamp <= train_end:
                training_times.append(timestamp)

    all_times = sorted(set(training_times) | set(test["datetime"]))
    stations = sorted(test["nama_pos"].unique())

    estimated_rows = len(stations) * len(all_times)
    max_rows = config.get("max_panel_rows", 5_000_000)
    if estimated_rows > max_rows:
        raise MemoryError(
            f"Panel grid would have ~{estimated_rows:,} rows "
            f"({len(stations)} stations x {len(all_times)} timestamps), "
            f"which exceeds config['max_panel_rows']={max_rows:,}. "
            "Consider shortening the training window, processing stations "
            "in batches, or raising max_panel_rows if you are sure your "
            "instance has enough RAM."
        )

    log_memory("before panel cross-join", config)

    grid = pd.MultiIndex.from_product(
        [stations, all_times], names=["nama_pos", "datetime"]
    ).to_frame(index=False)
    grid["nama_pos"] = grid["nama_pos"].astype("category")

    log_memory("after panel cross-join", config)

    panel = grid.merge(
        environment, on=["nama_pos", "datetime"], how="left", indicator="_environment_merge"
    )
    panel["environment_row_missing"] = panel["_environment_merge"].eq("left_only").astype("int8")
    panel = panel.drop(columns="_environment_merge")

    free(grid)

    environment_numeric = [
        column
        for column in environment.select_dtypes(include="number").columns
        if column in panel.columns
    ]

    for column in environment_numeric:
        if panel[column].isna().any():
            panel[column] = panel.groupby("nama_pos", observed=True)[column].transform(
                lambda values: values.interpolate(limit_direction="both")
            )
            panel[column] = safe_fillna(panel[column], panel[column].median())

    panel = panel.merge(
        train[["datetime", "nama_pos", "tma_mdpl"]],
        on=["datetime", "nama_pos"],
        how="left",
    )

    # FIX: use the `node` parameter that was actually passed in (already
    # standardized in prepare_data), instead of an undefined global.
    panel, embedding_columns = merge_node_embedding(panel, node)

    panel = add_calendar_features(panel)
    panel = panel.sort_values(["nama_pos", "datetime"]).reset_index(drop=True)

    all_times_series = pd.Series(panel["datetime"].sort_values().unique())
    time_to_step = pd.Series(np.arange(len(all_times_series)), index=all_times_series)
    panel["step"] = panel["datetime"].map(time_to_step).astype("int32")

    panel = downcast_numeric(panel)

    log_memory("after build_panel", config)

    return panel, embedding_columns


# ----------------------------------------------------------------------
# Climatology
# ----------------------------------------------------------------------

def fit_climatology(panel, cutoff):
    history = panel[panel["datetime"].le(cutoff) & panel["tma_mdpl"].notna()].copy()

    climatology = (
        history.groupby(["nama_pos", "month", "hour"], as_index=False, observed=True)["tma_mdpl"]
        .median()
        .rename(columns={"tma_mdpl": "climatology"})
    )
    station_median = history.groupby("nama_pos", observed=True)["tma_mdpl"].median()

    free(history)
    return climatology, station_median


def apply_climatology(frame, climatology, station_median):
    frame = frame.drop(columns=["climatology", "anomaly"], errors="ignore").copy()
    frame = frame.merge(climatology, on=["nama_pos", "month", "hour"], how="left")
    frame["climatology"] = safe_fillna(
        frame["climatology"], frame["nama_pos"].map(station_median)
    )
    frame["climatology"] = safe_fillna(frame["climatology"], 0)
    frame["anomaly"] = frame["tma_mdpl"] - frame["climatology"]
    return frame


def get_model_features(panel):
    """FIX: emb_* columns need no special handling — they are plain
    numeric columns on `panel` after merge_node_embedding(), so they flow
    into both catboost_features and neural_features automatically here.
    """
    ignored = {
        "datetime",
        "tma_mdpl",
        "climatology",
        "anomaly",
        "step",
        "landcover_name",
    }

    numeric_features = [
        column
        for column in panel.select_dtypes(include="number").columns
        if column not in ignored
    ]

    catboost_features = ["nama_pos"] + numeric_features
    neural_features = numeric_features

    return catboost_features, neural_features


# ----------------------------------------------------------------------
# Neural feature selection, ranked by CatBoost importance
# ----------------------------------------------------------------------

def select_neural_features(neural_features, config, catboost_model=None, catboost_features=None):
    limit = config.get("neural_max_exog_features")

    if not limit or len(neural_features) <= limit:
        return neural_features

    if catboost_model is not None and catboost_features is not None:
        importances = pd.Series(
            catboost_model.get_feature_importance(), index=catboost_features
        )
        ranked = [
            feature
            for feature in importances.sort_values(ascending=False).index
            if feature in neural_features
        ]
        selected = ranked[:limit]
    else:
        selected = neural_features[:limit]

    print(
        f"Neural exogenous features capped: {len(neural_features)} -> "
        f"{len(selected)} (config['neural_max_exog_features']={limit})"
    )

    return selected


# ----------------------------------------------------------------------
# CatBoost
# ----------------------------------------------------------------------

def train_catboost(panel, cutoff, features, config):
    training = panel[panel["datetime"].le(cutoff) & panel["anomaly"].notna()]

    parameters = {
        "iterations": config["catboost_iterations"],
        "depth": config["catboost_depth"],
        "learning_rate": 0.05,
        "loss_function": "RMSE",
        "l2_leaf_reg": 5,
        "random_strength": 0.5,
        "random_seed": RANDOM_SEED,
        "verbose": 100,
        "allow_writing_files": False,
        "task_type": config["catboost_task_type"],
    }

    if str(config["catboost_task_type"]).upper() == "GPU":
        parameters["devices"] = str(config["gpu_index"])
        parameters["gpu_ram_part"] = config["catboost_gpu_ram_part"]
    else:
        parameters["rsm"] = 0.85

    model = CatBoostRegressor(**parameters)
    model.fit(training[features], training["anomaly"], cat_features=["nama_pos"])
    return model


def predict_catboost(model, future, features):
    future = future.copy()
    future["catboost"] = model.predict(future[features]) + future["climatology"]
    return future[["nama_pos", "datetime", "catboost"]]


# ----------------------------------------------------------------------
# Seasonal model
# ----------------------------------------------------------------------

def predict_anchored_seasonal(panel, cutoff, horizon, config):
    history = panel[panel["datetime"].le(cutoff)].copy()
    future = panel[panel["datetime"].gt(cutoff)].copy()
    future = future.groupby("nama_pos", group_keys=False, observed=True).head(horizon)

    history = history.sort_values(["nama_pos", "hour", "datetime"])
    window = config["seasonal_window_days"]

    history["seasonal_reference"] = (
        history.groupby(["nama_pos", "hour"], observed=True)["tma_mdpl"]
        .rolling(window=window, center=True, min_periods=max(3, window // 3))
        .median()
        .reset_index(level=[0, 1], drop=True)
        .sort_index()
    )

    reference = history[["nama_pos", "datetime", "seasonal_reference"]].rename(
        columns={"datetime": "reference_time"}
    )

    future["reference_time"] = future["datetime"].map(
        lambda timestamp: timestamp - pd.DateOffset(years=1)
    )
    future = future.merge(reference, on=["nama_pos", "reference_time"], how="left")

    last_observed = (
        history.dropna(subset=["tma_mdpl"])
        .sort_values(["nama_pos", "datetime"])
        .groupby("nama_pos", as_index=False, observed=True)
        .tail(1)[["nama_pos", "datetime", "tma_mdpl"]]
        .rename(columns={"datetime": "origin_time", "tma_mdpl": "origin_tma"})
    )
    last_observed["reference_time"] = last_observed["origin_time"].map(
        lambda timestamp: timestamp - pd.DateOffset(years=1)
    )
    last_observed = last_observed.merge(reference, on=["nama_pos", "reference_time"], how="left")
    last_observed["anchor_offset"] = (
        last_observed["origin_tma"] - last_observed["seasonal_reference"]
    ).fillna(0)

    future = future.merge(last_observed[["nama_pos", "anchor_offset"]], on="nama_pos", how="left")
    future["anchor_offset"] = future["anchor_offset"].fillna(0)
    future["horizon_step"] = future.groupby("nama_pos", observed=True).cumcount() + 1
    future["horizon_day"] = np.ceil(future["horizon_step"] / OBSERVATIONS_PER_DAY)

    half_life = config["seasonal_anchor_half_life_days"]
    decay = 0.5 ** (future["horizon_day"] / half_life)
    seasonal_base = safe_fillna(future["seasonal_reference"], future["climatology"])
    future["seasonal"] = seasonal_base + decay * future["anchor_offset"]

    free(history, reference, last_observed)
    return future[["nama_pos", "datetime", "seasonal"]]


# ----------------------------------------------------------------------
# Neural model
# ----------------------------------------------------------------------

def prepare_neural_frames(panel, cutoff, horizon, neural_features, use_static_exog=True):
    """Prepare temporal frames and one-row-per-station static covariates.

    embedding_* columns are static by construction. When use_static_exog=True
    they are passed through NeuralForecast static_df/stat_exog_list instead of
    being repeated at every timestamp as future exogenous variables.
    """
    history = panel[panel["datetime"].le(cutoff)].copy()
    future = panel[panel["datetime"].gt(cutoff)].copy()
    future = future.groupby("nama_pos", group_keys=False, observed=True).head(horizon)

    static_features = [f for f in neural_features if f.startswith("embedding_")] if use_static_exog else []
    dynamic_features = [f for f in neural_features if f not in static_features]

    counts = future.groupby("nama_pos", observed=True).size()
    if counts.empty or not counts.eq(horizon).all():
        raise ValueError("The environmental file does not cover the full horizon.")

    history["y"] = history["anomaly"]
    history["y"] = history.groupby("nama_pos", observed=True)["y"].transform(
        lambda values: values.interpolate(limit=6, limit_direction="both")
    )
    history["y"] = history["y"].fillna(0)

    neural_scale = history.groupby("nama_pos", observed=True)["y"].std()
    neural_scale = neural_scale.replace([np.inf, -np.inf, 0], np.nan).fillna(1)
    neural_scale = neural_scale.clip(lower=0.1)

    history["neural_scale"] = history["nama_pos"].map(neural_scale)
    future["neural_scale"] = future["nama_pos"].map(neural_scale).fillna(1)
    history["y"] = history["y"] / history["neural_scale"]

    history = history.rename(columns={"nama_pos": "unique_id", "step": "ds"})
    future = future.rename(columns={"nama_pos": "unique_id", "step": "ds"})

    history["unique_id"] = history["unique_id"].astype(str)
    future["unique_id"] = future["unique_id"].astype(str)

    train_columns = ["unique_id", "ds", "y"] + dynamic_features
    future_columns = ["unique_id", "ds"] + dynamic_features

    static_df = None
    if static_features:
        static_df = (
            history[["unique_id"] + static_features]
            .drop_duplicates("unique_id")
            .reset_index(drop=True)
        )
        if static_df["unique_id"].duplicated().any():
            raise ValueError("static_df must contain one row per unique_id")

    return history[train_columns], future[future_columns], future, static_df, dynamic_features, static_features


def train_and_predict_neural(panel, cutoff, horizon, neural_features, config):
    history, future_covariates, future_map, static_df, dynamic_features, static_features = prepare_neural_frames(
        panel, cutoff, horizon, neural_features,
        use_static_exog=config.get("neural_use_static_exog", True),
    )

    common = {
        "h": horizon,
        "input_size": config["input_days"] * OBSERVATIONS_PER_DAY,
        "futr_exog_list": dynamic_features or None,
        "stat_exog_list": static_features or None,
        "loss": MAE(),
        "scaler_type": "identity",
        "max_steps": config["tide_max_steps"],
        "batch_size": config["tide_batch_size"],
        "windows_batch_size": config["tide_windows_batch_size"],
        "inference_windows_batch_size": config["tide_inference_batch_size"],
        "start_padding_enabled": True,
        "random_seed": RANDOM_SEED,
        "accelerator": config["accelerator"],
        "devices": config["devices"],
        "precision": config["precision"],
        "alias": "neural",
    }

    if config["neural_model"] == "nhits":
        model = NHITS(
            n_blocks=[1, 1, 1],
            mlp_units=[[128, 128], [128, 128], [128, 128]],
            n_pool_kernel_size=[4, 2, 1],
            n_freq_downsample=[12, 4, 1],
            pooling_mode="AvgPool1d",
            dropout_prob_theta=0.1,
            **common,
        )
    else:
        model = TiDE(
            hidden_size=config["tide_hidden_size"],
            decoder_output_dim=16,
            temporal_decoder_dim=32,
            dropout=0.2,
            **common,
        )

    forecast = NeuralForecast(models=[model], freq=1)

    try:
        log_memory("before neural fit", config)
        forecast.fit(df=history, static_df=static_df)
        prediction = forecast.predict(futr_df=future_covariates).reset_index()
        log_memory("after neural fit", config)
    except (torch.cuda.OutOfMemoryError, RuntimeError) as error:
        is_gpu_oom = (
            config["accelerator"] == "gpu" and "out of memory" in str(error).lower()
        )
        if not is_gpu_oom:
            raise

        print("GPU memory was insufficient. Retrying the neural model on CPU...")
        free(forecast, model)
        release_gpu_memory()

        cpu_config = config.copy()
        cpu_config["accelerator"] = "cpu"
        cpu_config["devices"] = 1
        cpu_config["precision"] = "32-true"

        return train_and_predict_neural(panel, cutoff, horizon, neural_features, cpu_config)

    if "neural" in prediction.columns:
        prediction_column = "neural"
    else:
        prediction_columns = [
            column
            for column in prediction.columns
            if column not in {"index", "unique_id", "ds", "cutoff"}
        ]
        if len(prediction_columns) != 1:
            raise ValueError(
                "Could not identify the neural prediction column. "
                f"Available columns: {prediction.columns.tolist()}"
            )
        prediction_column = prediction_columns[0]

    prediction = prediction.rename(columns={prediction_column: "neural_scaled_anomaly"})

    non_finite = ~np.isfinite(prediction["neural_scaled_anomaly"])
    if non_finite.any():
        print(
            f"Warning: {non_finite.sum()} neural values were non-finite; "
            "using zero anomaly (climatology) for those rows."
        )
        prediction.loc[non_finite, "neural_scaled_anomaly"] = 0

    prediction["unique_id"] = prediction["unique_id"].astype(str)
    future_map["unique_id"] = future_map["unique_id"].astype(str)

    prediction = prediction.sort_values(["unique_id", "ds"]).copy()
    prediction["forecast_order"] = prediction.groupby("unique_id").cumcount()

    future_lookup = future_map.sort_values(["unique_id", "datetime"]).copy()
    future_lookup["forecast_order"] = future_lookup.groupby("unique_id").cumcount()

    prediction_counts = prediction.groupby("unique_id").size()
    future_counts = future_lookup.groupby("unique_id").size()

    if not prediction_counts.equals(future_counts):
        raise ValueError(
            "The neural model returned a different number of rows than requested.\n"
            f"Predicted counts:\n{prediction_counts}\n"
            f"Expected counts:\n{future_counts}"
        )

    prediction = prediction.merge(
        future_lookup[["unique_id", "forecast_order", "datetime", "climatology", "neural_scale"]],
        on=["unique_id", "forecast_order"],
        how="left",
        validate="one_to_one",
    )
    prediction["neural_anomaly"] = prediction["neural_scaled_anomaly"] * prediction["neural_scale"]
    prediction["neural"] = prediction["neural_anomaly"] + prediction["climatology"]
    prediction = prediction.rename(columns={"unique_id": "nama_pos"})
    prediction["nama_pos"] = prediction["nama_pos"].astype("category")

    free(forecast, model, history, future_covariates, future_map, static_df)
    release_gpu_memory()

    return prediction[["nama_pos", "datetime", "neural"]]


# ----------------------------------------------------------------------
# Horizon banding
# ----------------------------------------------------------------------

def add_horizon_band(frame):
    frame = frame.sort_values(["nama_pos", "datetime"]).copy()
    frame["horizon_step"] = frame.groupby("nama_pos", observed=True).cumcount() + 1
    frame["horizon_day"] = np.ceil(frame["horizon_step"] / OBSERVATIONS_PER_DAY).astype(int)
    frame["horizon_band"] = pd.cut(
        frame["horizon_day"],
        bins=[0, 7, 30, 90, np.inf],
        labels=["1-7 days", "8-30 days", "31-90 days", "91+ days"],
    )
    return frame


# ----------------------------------------------------------------------
# Ensemble weighting
# ----------------------------------------------------------------------

def learn_ensemble_weights(validation, config):
    """Gunakan bobot manual per horizon atau optimasi grid sebagai fallback."""
    manual_weights = config.get("manual_horizon_weights")
    weight_mode = str(config.get("weight_mode", "optimize")).lower()

    if weight_mode == "manual" and manual_weights:
        rows = []
        for band, weights in manual_weights.items():
            rows.append({
                "horizon_band": str(band),
                "score": np.nan,
                "catboost_weight": float(weights["catboost"]),
                "neural_weight": float(weights["neural"]),
                "seasonal_weight": float(weights["seasonal"]),
            })
        return pd.DataFrame(rows, columns=[
            "horizon_band", "score", "catboost_weight",
            "neural_weight", "seasonal_weight"
        ])

    return _optimize_ensemble_weights(validation, config)


def _optimize_ensemble_weights(validation, config):
    rows = []
    models = ["catboost", "neural", "seasonal"]
    usable = validation.dropna(subset=["tma_mdpl"] + models)
    step = config["weight_step"]
    candidate_weights = np.arange(0, 1 + step / 2, step)

    for band, group in usable.groupby("horizon_band", observed=True):
        actual = group["tma_mdpl"].to_numpy()

        individual_rmse = []
        individual_mae = []
        for model in models:
            error = group[model].to_numpy() - actual
            individual_rmse.append(np.sqrt(np.mean(error ** 2)))
            individual_mae.append(np.mean(np.abs(error)))

        rmse_scale = max(min(individual_rmse), 1e-8)
        mae_scale = max(min(individual_mae), 1e-8)
        best = None

        for catboost_weight in candidate_weights:
            for neural_weight in candidate_weights:
                seasonal_weight = 1 - catboost_weight - neural_weight
                if seasonal_weight < -1e-9:
                    continue
                seasonal_weight = max(0, seasonal_weight)
                blended = (
                    catboost_weight * group["catboost"].to_numpy()
                    + neural_weight * group["neural"].to_numpy()
                    + seasonal_weight * group["seasonal"].to_numpy()
                )
                error = blended - actual
                rmse = np.sqrt(np.mean(error ** 2))
                mae = np.mean(np.abs(error))

                if config["ensemble_objective"] == "rmse":
                    score = rmse
                elif config["ensemble_objective"] == "mae":
                    score = mae
                else:
                    score = 0.5 * rmse / rmse_scale + 0.5 * mae / mae_scale

                if best is None or score < best["score"]:
                    best = {
                        "score": score,
                        "catboost_weight": float(catboost_weight),
                        "neural_weight": float(neural_weight),
                        "seasonal_weight": float(seasonal_weight),
                    }

        rows.append({"horizon_band": str(band), **best})

    return pd.DataFrame(
        rows,
        columns=["horizon_band", "score", "catboost_weight", "neural_weight", "seasonal_weight"],
    )


def apply_ensemble_weights(predictions, weights):
    predictions = predictions.copy()
    weight_table = weights.set_index("horizon_band")
    band = predictions["horizon_band"].astype(str)

    for model in ["catboost", "neural", "seasonal"]:
        weight_column = model + "_weight"
        predictions[weight_column] = band.map(weight_table[weight_column]).fillna(1 / 3)

    numerator = np.zeros(len(predictions))
    denominator = np.zeros(len(predictions))

    for model in ["catboost", "neural", "seasonal"]:
        weight = predictions[model + "_weight"].to_numpy()
        value = predictions[model].to_numpy()
        available = np.isfinite(value)
        numerator += np.where(available, weight * value, 0)
        denominator += np.where(available, weight, 0)

    predictions["tma_mdpl"] = np.divide(
        numerator, denominator, out=np.full(len(predictions), np.nan), where=denominator > 0
    )

    return predictions


def calculate_validation_metrics(validation, weights, config):
    actual = validation["tma_mdpl"].copy()
    scored = apply_ensemble_weights(validation.drop(columns="tma_mdpl"), weights)
    scored = scored.rename(columns={"tma_mdpl": "ensemble"})
    scored["actual"] = actual.to_numpy()

    groups = [("Overall", scored)]
    groups += [
        (str(band), group) for band, group in scored.groupby("horizon_band", observed=True)
    ]

    rows = []
    for horizon_band, group in groups:
        for model in ["catboost", "neural", "seasonal", "ensemble"]:
            model_group = group.dropna(subset=["actual", model])
            error = model_group[model] - model_group["actual"]
            mse = np.mean(error ** 2)
            rows.append(
                {
                    "horizon_band": horizon_band,
                    "model": model,
                    "RMSE": np.sqrt(mse),
                    "MSE": mse,
                    "MAE": np.mean(np.abs(error)),
                    "rows": len(model_group),
                }
            )

    metrics = pd.DataFrame(rows)

    station_rows = []
    for station, group in scored.groupby("nama_pos", observed=True):
        for model in ["catboost", "neural", "seasonal", "ensemble"]:
            model_group = group.dropna(subset=["actual", model])
            error = model_group[model] - model_group["actual"]
            mse = np.mean(error ** 2)
            station_rows.append(
                {
                    "nama_pos": station,
                    "model": model,
                    "RMSE": np.sqrt(mse),
                    "MAE": np.mean(np.abs(error)),
                    "bias": np.mean(error),
                    "rows": len(model_group),
                }
            )

    station_metrics = pd.DataFrame(station_rows)
    metrics.attrs["station_metrics"] = station_metrics
    scored.attrs["station_metrics"] = station_metrics

    report_count = int(config["report_worst_stations"])
    if report_count > 0:
        worst = (
            station_metrics[station_metrics["model"].eq("ensemble")]
            .sort_values("RMSE", ascending=False)
            .head(report_count)
        )
        print("\nWorst validation stations (ensemble):")
        print(worst.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

    return metrics, scored


# ----------------------------------------------------------------------
# Model orchestration
# ----------------------------------------------------------------------

def run_models(panel, cutoff, horizon, catboost_features, neural_features, config):
    """NOTE: `panel` already contains the embedding_* embedding columns (merged in
    build_panel), and catboost_features/neural_features already include
    them (from get_model_features). No separate `node` argument is needed
    here — that's what caused the TypeError in the version you had."""
    climatology, station_median = fit_climatology(panel, cutoff)
    data = apply_climatology(panel, climatology, station_median)

    future = data[data["datetime"].gt(cutoff)].copy()
    future = future.groupby("nama_pos", group_keys=False, observed=True).head(horizon)

    log_memory("before CatBoost fit", config)
    catboost_model = train_catboost(data, cutoff, catboost_features, config)
    catboost_prediction = predict_catboost(catboost_model, future, catboost_features)
    log_memory("after CatBoost fit", config)

    selected_neural_features = select_neural_features(
        neural_features, config, catboost_model, catboost_features
    )

    free(catboost_model)

    neural_prediction = train_and_predict_neural(
        data, cutoff, horizon, selected_neural_features, config
    )
    seasonal_prediction = predict_anchored_seasonal(data, cutoff, horizon, config)

    prediction = catboost_prediction.merge(
        neural_prediction, on=["nama_pos", "datetime"], how="left", validate="one_to_one"
    )
    prediction = prediction.merge(
        seasonal_prediction, on=["nama_pos", "datetime"], how="left", validate="one_to_one"
    )

    missing_neural = prediction["neural"].isna().sum()
    if missing_neural:
        print(
            f"Warning: {missing_neural} neural rows are missing; "
            "using the other models as fallback."
        )

    prediction = add_horizon_band(prediction)

    free(data, future, catboost_prediction, neural_prediction, seasonal_prediction)
    log_memory("end of run_models", config)

    return prediction


def choose_validation_cutoff(panel, train, test, horizon, config):
    train_end = train["datetime"].max()
    training_times = sorted(panel.loc[panel["datetime"].le(train_end), "datetime"].unique())

    if len(training_times) <= horizon:
        raise ValueError("Training history is shorter than the test horizon.")

    if config["validation_mode"] == "same_season":
        validation_start = test["datetime"].min() - pd.DateOffset(years=1)
        eligible_cutoffs = [time for time in training_times if time < validation_start]

        if eligible_cutoffs:
            cutoff = eligible_cutoffs[-1]
            future_times = [time for time in training_times if time > cutoff][:horizon]
            if len(future_times) == horizon:
                return cutoff, future_times[0], future_times[-1]

        print("Same-season validation was unavailable; using the last block.")

    cutoff = training_times[-horizon - 1]
    future_times = [time for time in training_times if time > cutoff][:horizon]
    return cutoff, future_times[0], future_times[-1]


# ----------------------------------------------------------------------
# Shared setup used by both entry points
# ----------------------------------------------------------------------

def _build_common_state(df_train, df_test, df_lingkungan, node_embedding, config):
    train, test, environment, node = prepare_data(df_train, df_test, df_lingkungan, node_embedding)
    environment = clean_environment(environment)

    if config["add_hydrology_features"]:
        print("Adding causal hydrology features...")
        environment = add_hydrology_features(environment)

    # FIX: `node` is now actually passed into build_panel(), and build_panel
    # merges it in and returns the list of embedding_columns it created.
    panel, embedding_columns = build_panel(train, test, environment, node, config)

    free(environment)

    catboost_features, neural_features = get_model_features(panel)

    print(
        f"Total fitur: {len(catboost_features)} (CatBoost), "
        f"{len(neural_features)} (neural) — termasuk {len(embedding_columns)} "
        "kolom node embedding di masing-masing."
    )

    horizon_counts = test.groupby("nama_pos", observed=True).size()
    if not horizon_counts.eq(horizon_counts.iloc[0]).all():
        raise ValueError("Every station in df_test must have the same horizon.")

    horizon = int(horizon_counts.iloc[0])

    return train, test, panel, catboost_features, neural_features, horizon, embedding_columns


# ----------------------------------------------------------------------
# Entry point 1 of 2 — validation phase only
# ----------------------------------------------------------------------

def run_validation_only(df_train, df_test, df_lingkungan, node_embedding, config=None):
    config = prepare_config(config)

    print("Pipeline version:", PIPELINE_VERSION, "(validation phase)")
    log_memory("start of run_validation_only", config)

    train, test, panel, catboost_features, neural_features, horizon, _ = _build_common_state(
        df_train, df_test, df_lingkungan, node_embedding, config
    )

    validation_cutoff, validation_start, validation_end = choose_validation_cutoff(
        panel, train, test, horizon, config
    )

    print(
        "Validation period:",
        pd.Timestamp(validation_start),
        "to",
        pd.Timestamp(validation_end),
    )

    print("\nFitting validation models...")
    # FIX: run_models() takes no separate node/embedding argument — the
    # embedding columns are already inside `panel`/`catboost_features`/
    # `neural_features`.
    validation_prediction = run_models(
        panel, validation_cutoff, horizon, catboost_features, neural_features, config
    )

    validation_actual = train[["nama_pos", "datetime", "tma_mdpl"]]
    validation_prediction = validation_prediction.merge(
        validation_actual, on=["nama_pos", "datetime"], how="left"
    )

    weights = learn_ensemble_weights(validation_prediction, config)
    metrics, validation_prediction = calculate_validation_metrics(
        validation_prediction, weights, config
    )

    print("\nLearned horizon weights:")
    print(weights.to_string(index=False))
    print("\nValidation metrics (lower is better):")
    print(metrics.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

    weights_path = Path(config["weights_output_file"]).expanduser()
    metrics_path = Path(config["metrics_output_file"]).expanduser()
    weights.to_csv(weights_path, index=False)
    metrics.to_csv(metrics_path, index=False)
    print(f"\nSaved weights to: {weights_path.resolve()}")
    print(f"Saved metrics to: {metrics_path.resolve()}")

    free(train, test, panel)
    release_gpu_memory()
    log_memory("end of run_validation_only", config)

    print(
        "\nIf you are on a tight-RAM Colab instance, this is a good point "
        "to Runtime > Restart runtime before calling run_final_only()."
    )

    return weights, metrics, validation_prediction


# ----------------------------------------------------------------------
# Entry point 2 of 2 — final fit + submission
# ----------------------------------------------------------------------

def run_final_only(
    df_train,
    df_test,
    df_lingkungan,
    node_embedding,
    df_submission=None,
    weights=None,
    weights_path=None,
    config=None,
):
    config = prepare_config(config)

    if weights is None:
        if weights_path is None:
            weights_path = config["weights_output_file"]
        weights = pd.read_csv(weights_path)

    print("Pipeline version:", PIPELINE_VERSION, "(final phase)")
    log_memory("start of run_final_only", config)

    # FIX: consistently use `node_embedding` (no more undefined df_koordinat).
    train, test, panel, catboost_features, neural_features, horizon, _ = _build_common_state(
        df_train, df_test, df_lingkungan, node_embedding, config
    )

    train_end = train["datetime"].max()

    print("\nFitting final models on all training data...")
    test_prediction = run_models(
        panel, train_end, horizon, catboost_features, neural_features, config
    )
    test_prediction = apply_ensemble_weights(test_prediction, weights)

    submission = test.merge(
        test_prediction[["nama_pos", "datetime", "catboost", "neural", "seasonal", "tma_mdpl"]],
        on=["nama_pos", "datetime"],
        how="left",
        indicator="_prediction_merge",
    )

    missing_prediction = submission[
        submission["tma_mdpl"].isna() | submission["_prediction_merge"].ne("both")
    ]

    if not missing_prediction.empty:
        examples = missing_prediction[
            ["nama_pos", "datetime", "_prediction_merge", "catboost", "neural", "seasonal", "tma_mdpl"]
        ].head(5)
        raise ValueError(
            f"{len(missing_prediction)} test rows did not receive a prediction.\n"
            f"First unmatched rows:\n{examples.to_string(index=False)}"
        )

    submission = submission.drop(columns="_prediction_merge")
    prediction_by_id = submission.set_index("id")["tma_mdpl"]

    if df_submission is None:
        final_submission = test.sort_values("original_order")[["id"]].copy()
    else:
        final_submission = df_submission.copy()

    final_submission["tma_mdpl"] = final_submission["id"].map(prediction_by_id)

    if final_submission["tma_mdpl"].isna().any():
        raise ValueError("df_submission contains IDs that were not found in df_test.")

    output_path = Path(config["output_file"]).expanduser()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    final_submission.to_csv(output_path, index=False)

    if not output_path.exists() or output_path.stat().st_size == 0:
        raise OSError(f"Submission file was not written correctly: {output_path}")

    final_submission.attrs["output_file"] = str(output_path.resolve())
    print("\nSaved submission:", output_path.resolve())
    print("Submission rows:", len(final_submission))

    free(train, test, panel, test_prediction, submission)
    release_gpu_memory()
    log_memory("end of run_final_only", config)

    return final_submission


# ----------------------------------------------------------------------
# Convenience wrapper — same interface as the original pipeline
# ----------------------------------------------------------------------

def run_ensemble(df_train, df_test, df_lingkungan, node_embedding, df_submission=None, config=None):
    """
    Train, validate, and forecast with CatBoost + neural + seasonal models,
    with node_embedding (GAT emb_*) included as a feature in BOTH CatBoost
    and the neural model.

    Returns:
        submission             final test predictions
        metrics                RMSE, MSE, and MAE by model and horizon
        weights                learned ensemble weight per horizon band
        validation_prediction  validation predictions for further analysis
    """
    config = prepare_config(config)

    # FIX: node_embedding is now forwarded to both phases.
    weights, metrics, validation_prediction = run_validation_only(
        df_train, df_test, df_lingkungan, node_embedding, config
    )

    gc.collect()
    release_gpu_memory()

    submission = run_final_only(
        df_train,
        df_test,
        df_lingkungan,
        node_embedding,
        df_submission,
        weights=weights,
        config=config,
    )

    return submission, metrics, weights, validation_prediction

In [ ]:
gat_config = RiverGATConfig(
    output_dir='/content/artifacts',
    include_all_upstream=True,
    include_path_to_outlet=True,
    embedding_dim=32,
    device='auto',
)

node_embedding, gat_reports = run_spatial_embedding_pipeline(
    HYDRO, koor_df, gat_config
)

display(node_embedding.head())
print(gat_reports['subgraph'])

submission, metrics, weights, validation = run_ensemble(
    df_train_clean,
    test_df,
    datling_df,
    node_embedding,
    df_submission,
    config=build_experiment_config(
        "experiment_5",
        device="auto",
        neural_model="nhits",
        neural_max_exog_features=40,
        neural_use_static_exog=True,
        output_file="/content/submission_exp5.csv",
    ),
)

display(metrics)
display(weights)
display(submission.head())

## 6. Quantile Forecasting dan Deteksi Anomali Prediksi per Pos

Interval prediksi dibentuk dari distribusi residual validasi per pos dan horizon. Hasilnya berupa batas bawah (`q10`), median (`q50`), dan batas atas (`q90`) yang mengikuti karakteristik error masing-masing stasiun.

Prediksi validasi ditandai anomali ketika nilai aktual berada di luar interval quantile. Interval yang sangat lebar juga ditandai sebagai ketidakpastian tinggi.


In [ ]:
def _resolve_prediction_column(frame):
    for column in ["ensemble", "prediction", "pred", "tma_mdpl_pred", "y_hat"]:
        if column in frame.columns:
            return column
    raise KeyError("Kolom prediksi ensemble tidak ditemukan pada validation.")

def fit_pos_quantile_calibrator(
    validation_df,
    actual_col="tma_mdpl",
    pred_col=None,
    lower_q=0.10,
    upper_q=0.90,
    min_samples_per_pos=20,
):
    frame = validation_df.copy()
    pred_col = pred_col or _resolve_prediction_column(frame)
    if "horizon_band" not in frame.columns:
        frame["horizon_band"] = "all"

    frame = frame.dropna(subset=["nama_pos", actual_col, pred_col]).copy()
    frame["residual"] = frame[actual_col] - frame[pred_col]

    def summarize(group):
        return pd.Series({
            "n_calibration": len(group),
            "residual_q_low": group["residual"].quantile(lower_q),
            "residual_q50": group["residual"].quantile(0.50),
            "residual_q_high": group["residual"].quantile(upper_q),
            "abs_residual_q90": group["residual"].abs().quantile(0.90),
        })

    by_pos_horizon = (
        frame.groupby(["nama_pos", "horizon_band"], observed=True)
        .apply(summarize, include_groups=False)
        .reset_index()
    )
    by_pos = (
        frame.groupby("nama_pos", observed=True)
        .apply(summarize, include_groups=False)
        .reset_index()
    )

    return {
        "pred_col": pred_col,
        "actual_col": actual_col,
        "lower_q": lower_q,
        "upper_q": upper_q,
        "min_samples_per_pos": min_samples_per_pos,
        "by_pos_horizon": by_pos_horizon,
        "by_pos": by_pos,
        "global": summarize(frame).to_dict(),
    }

def apply_pos_quantile_forecast(prediction_df, calibrator, pred_col=None):
    frame = prediction_df.copy()
    pred_col = pred_col or calibrator["pred_col"]
    if "horizon_band" not in frame.columns:
        frame["horizon_band"] = "all"

    ph = calibrator["by_pos_horizon"].copy()
    ps = calibrator["by_pos"].copy()

    frame = frame.merge(ph, on=["nama_pos", "horizon_band"], how="left")
    frame = frame.merge(ps, on="nama_pos", how="left", suffixes=("_ph", "_pos"))

    stats = [
        "n_calibration", "residual_q_low", "residual_q50",
        "residual_q_high", "abs_residual_q90"
    ]
    min_n = calibrator["min_samples_per_pos"]

    for col in stats:
        ph_col = f"{col}_ph"
        pos_col = f"{col}_pos"
        if ph_col not in frame.columns and col in frame.columns:
            frame.rename(columns={col: ph_col}, inplace=True)
        use_specific = frame["n_calibration_ph"].fillna(0).ge(min_n)
        frame[col] = np.where(use_specific, frame[ph_col], frame[pos_col])
        frame[col] = pd.Series(frame[col], index=frame.index).fillna(
            calibrator["global"][col]
        )

    frame["q_low"] = frame[pred_col] + frame["residual_q_low"]
    frame["q50"] = frame[pred_col] + frame["residual_q50"]
    frame["q_high"] = frame[pred_col] + frame["residual_q_high"]

    ordered = np.sort(frame[["q_low", "q50", "q_high"]].to_numpy(float), axis=1)
    frame[["q_low", "q50", "q_high"]] = ordered
    frame["prediction_interval_width"] = frame["q_high"] - frame["q_low"]

    threshold = frame.groupby("nama_pos", observed=True)[
        "prediction_interval_width"
    ].transform(lambda x: x.quantile(0.95))
    frame["is_high_uncertainty"] = frame["prediction_interval_width"] > threshold
    return frame

def flag_validation_anomalies(frame, actual_col="tma_mdpl"):
    result = frame.copy()
    result["is_prediction_anomaly"] = (
        (result[actual_col] < result["q_low"])
        | (result[actual_col] > result["q_high"])
    )
    result["anomaly_direction"] = np.select(
        [
            result[actual_col] < result["q_low"],
            result[actual_col] > result["q_high"],
        ],
        ["below_interval", "above_interval"],
        default="normal",
    )
    return result

def plot_pos_quantile_forecast(
    quantile_df,
    nama_pos,
    datetime_col="datetime",
    actual_col="tma_mdpl",
    max_points=300,
):
    data = (
        quantile_df[quantile_df["nama_pos"].eq(nama_pos)]
        .sort_values(datetime_col)
        .tail(max_points)
        .copy()
    )
    if data.empty:
        raise ValueError(f"Tidak ada data untuk pos: {nama_pos}")

    plt.figure(figsize=(16, 5))
    plt.fill_between(
        data[datetime_col], data["q_low"], data["q_high"],
        alpha=0.25, label="Interval quantile"
    )
    plt.plot(data[datetime_col], data["q50"], label="Median forecast")
    if actual_col in data.columns:
        plt.plot(data[datetime_col], data[actual_col], label="Aktual", linewidth=1)
    if "is_prediction_anomaly" in data.columns:
        anomaly = data[data["is_prediction_anomaly"]]
        plt.scatter(
            anomaly[datetime_col], anomaly[actual_col],
            marker="x", s=55, label="Anomali prediksi"
        )
    plt.title(f"Quantile Forecast dan Anomali — {nama_pos}")
    plt.xlabel("Datetime")
    plt.ylabel("TMA (mdpl)")
    plt.legend()
    plt.tight_layout()
    plt.show()

quantile_calibrator = fit_pos_quantile_calibrator(
    validation,
    lower_q=GLOBAL_MODEL_CONFIG["anomaly_lower_quantile"],
    upper_q=GLOBAL_MODEL_CONFIG["anomaly_upper_quantile"],
)

validation_quantiles = apply_pos_quantile_forecast(validation, quantile_calibrator)
validation_quantiles = flag_validation_anomalies(validation_quantiles)

quantile_summary = (
    validation_quantiles.groupby("nama_pos", observed=True)
    .agg(
        n_observation=("is_prediction_anomaly", "size"),
        n_anomaly=("is_prediction_anomaly", "sum"),
        anomaly_rate=("is_prediction_anomaly", "mean"),
        mean_interval_width=("prediction_interval_width", "mean"),
    )
    .sort_values(["anomaly_rate", "mean_interval_width"], ascending=False)
    .reset_index()
)

display(quantile_summary.head(15))


## 7. Evaluasi Recursive Forecast: Step-Ahead dan Per-Pos 🔍


Simulasi recursive forecast penuh (fungsi `recursive_forecast_eval` sudah
didefinisikan di atas, dipakai ulang di sini) di atas `val_part`/`test_part`
dari split final.


In [ ]:
# Calculate split points for train, validation, and test sets
# Assuming merged_df is already sorted by DATETIME_COL and GROUP_COL
unique_dates = merged_df[DATETIME_COL].dt.normalize().unique()
n_dates = len(unique_dates)

# Determine the number of days for test and validation sets
n_test_dates = int(n_dates * TEST_RATIO)
n_val_dates = int(n_dates * VAL_RATIO)

# Ensure there's enough data for training
if n_dates < (n_test_dates + n_val_dates):
    raise ValueError("Not enough unique dates in merged_df for the specified TEST_RATIO and VAL_RATIO.")

# Get the dates for each split
test_split_date_start = unique_dates[n_dates - n_test_dates]
val_split_date_start = unique_dates[n_dates - n_test_dates - n_val_dates]

# Create the dataframes
train_part = merged_df[merged_df[DATETIME_COL].dt.normalize() < val_split_date_start].copy()
val_part = merged_df[(merged_df[DATETIME_COL].dt.normalize() >= val_split_date_start) &
                     (merged_df[DATETIME_COL].dt.normalize() < test_split_date_start)].copy()
test_part = merged_df[merged_df[DATETIME_COL].dt.normalize() >= test_split_date_start].copy()

# Rentang historis per stasiun (dipakai untuk clipping) -- berbasis train+val,
# supaya tidak "curang" mengintip test_part.
pos_bounds = pd.concat([train_part, val_part])[[GROUP_COL, TARGET]].groupby(GROUP_COL)[TARGET].agg(["min", "max"])

history_for_eval = pd.concat([train_part, val_part], ignore_index=True)

recur_eval_xgb = recursive_forecast_eval(history_for_eval, test_part, koor_df, datling_df,
                                          model, feature_cols, encoders, pos_bounds, is_lgb=False)
recur_eval_xgb["model"] = "XGBoost"

recur_eval_lgb = recursive_forecast_eval(history_for_eval, test_part, koor_df, datling_df,
                                          lgb_model, feature_cols, encoders, pos_bounds, is_lgb=True)
recur_eval_lgb["model"] = "LightGBM"

recur_eval_all = pd.concat([recur_eval_xgb, recur_eval_lgb], ignore_index=True)
recur_eval_all = recur_eval_all.dropna(subset=["y_true"])
print(f"Jumlah baris hasil simulasi recursive forecast: {len(recur_eval_all)}")
recur_eval_all.head()

### Ranking 30 Pos: Dari yang Terbaik ke Terburuk

In [ ]:
pos_eval = {}
for name in ["XGBoost", "LightGBM", "Ensemble (Best-per-Pos)"]:
    pos_eval[name] = rmse_by_pos(recur_eval_all[recur_eval_all["model"] == name]).rename(
        columns={"RMSE": f"RMSE_{name}", "MAE": f"MAE_{name}"}
    ).set_index(GROUP_COL)

pos_compare = pd.concat(pos_eval.values(), axis=1)
display(pos_compare.sort_values("RMSE_Ensemble (Best-per-Pos)"))


## 8. Evaluasi Keseluruhan dan Persiapan Submission 🏁


Bandingkan tiga strategi: XGBoost saja, LightGBM saja, dan Ensemble
Best-per-Pos.


In [ ]:
RMSE_THRESHOLDS = [
    (2.0, "Siap dikirim", "Excellent -- RMSE di bawah 2.0, aman untuk submit final."),
    (3.5, "Cukup baik, masih bisa dioptimalkan", "Layak submit, tapi masih ada ruang perbaikan sebelum deadline."),
    (6.0, "Perlu perbaikan sebelum submit final", "Cek lagi stasiun-stasiun dengan RMSE tertinggi di tabel ranking."),
    (float("inf"), "Belum layak, cek ulang pipeline", "RMSE terlalu tinggi -- curigai bug (join/merge, arah lag, scaling)."),
]

def rmse_status(rmse):
    for limit, label, note in RMSE_THRESHOLDS:
        if rmse < limit:
            return label, note
    return RMSE_THRESHOLDS[-1][1], RMSE_THRESHOLDS[-1][2]


scoreboard = []
for name in ["XGBoost", "LightGBM", "Ensemble (Best-per-Pos)"]:
    sub = recur_eval_all[recur_eval_all["model"] == name]
    rmse = np.sqrt(mean_squared_error(sub["y_true"], sub["y_pred"]))
    mae = mean_absolute_error(sub["y_true"], sub["y_pred"])
    label, note = rmse_status(rmse)
    scoreboard.append({"Model": name, "RMSE": rmse, "MAE": mae, "n_obs": len(sub), "Status": label, "Catatan": note})

scoreboard_df = pd.DataFrame(scoreboard).sort_values("RMSE").reset_index(drop=True)
print("=" * 88)
print("EVALUASI KESELURUHAN (simulasi recursive forecast + clipping guard)")
print("=" * 88)
display(scoreboard_df[["Model", "RMSE", "MAE", "n_obs", "Status"]])

best = scoreboard_df.iloc[0]
best_name = best["Model"]
best_rmse = best["RMSE"]
best_status = best["Status"]
print(f"\nModel/strategi terbaik: {best_name} (RMSE = {best_rmse:.4f}) -> {best_status}")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
bands = [(0, 2.0, "#c8e6c9"), (2.0, 3.5, "#fff9c4"), (3.5, 6.0, "#ffe0b2"), (6.0, 9.0, "#ffcdd2")]
for start, end, color in bands:
    ax.axvspan(start, end, color=color, alpha=0.6)

y_pos = np.arange(len(scoreboard_df))
ax.barh(y_pos, scoreboard_df["RMSE"], color="#37474f", height=0.4, zorder=3)
ax.set_yticks(y_pos)
ax.set_yticklabels(scoreboard_df["Model"])
for y, val in zip(y_pos, scoreboard_df["RMSE"]):
    ax.text(val + 0.05, y, f"{val:.2f}", va="center", fontweight="bold")

x_max = max(9.0, scoreboard_df["RMSE"].max() + 1)
ax.set_xlim(0, x_max)
ax.set_xlabel("RMSE")
ax.set_title("Evaluasi Keseluruhan vs Ambang Batas Submission")
ax.invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Simpan kedua model
joblib.dump(model, "xgboost_model.pkl")
joblib.dump(lgb_model, "lightgbm_model.pkl")


### Persiapan Submission

Strategi final dipilih otomatis berdasarkan `scoreboard_df` (RMSE terendah).
Untuk strategi Ensemble, submission juga memakai pemilihan model per-pos
(`pos_to_model`) — bukan rata-rata.

In [ ]:
def recursive_forecast(train_df_raw, test_df_raw, koor_df, datling_df, model,
                       feature_cols, encoders, pos_bounds, is_lgb=False):
    test_df_raw = test_df_raw.copy()
    test_df_raw[["datetime", "nama_pos"]] = test_df_raw["id"].str.split(" - ", n=1, expand=True)
    test_df_raw["datetime"] = pd.to_datetime(test_df_raw["datetime"])

    max_hist = max(LAG_LIST + ROLLING_WINDOWS) + 5
    tail_per_pos = train_df_raw.sort_values(DATETIME_COL).groupby(GROUP_COL).tail(max_hist)

    combined_raw = pd.concat([
        tail_per_pos[[DATETIME_COL, GROUP_COL, TARGET]],
        test_df_raw[[DATETIME_COL, GROUP_COL]].assign(**{TARGET: np.nan})
    ], ignore_index=True)

    static_df = build_static_features(combined_raw, koor_df, datling_df, encoders)
    static_df = static_df.sort_values([GROUP_COL, DATETIME_COL]).reset_index(drop=True)
    if is_lgb:
        static_df = sanitize_feature_names(static_df)

    predictions = []
    for pos, group in static_df.groupby(GROUP_COL):
        group = group.sort_values(DATETIME_COL).reset_index(drop=True)
        test_dates_pos = set(test_df_raw.loc[test_df_raw[GROUP_COL] == pos, DATETIME_COL])

        history = list(
            tail_per_pos.loc[tail_per_pos[GROUP_COL] == pos].sort_values(DATETIME_COL)[TARGET]
        )

        if pos in pos_bounds.index:
            lo, hi = pos_bounds.loc[pos, "min"], pos_bounds.loc[pos, "max"]
            margin = (hi - lo) * CLIP_MARGIN_PCT
            clip_lo, clip_hi = lo - margin, hi + margin
        else:
            clip_lo, clip_hi = -np.inf, np.inf

        for idx in range(len(group)):
            row = group.iloc[idx]
            if row[DATETIME_COL] not in test_dates_pos:
                continue

            row = row.copy()
            hist_arr = np.array(history[-max_hist:])

            for lag in LAG_LIST:
                row[f"{TARGET}_lag_{lag}"] = hist_arr[-lag] if len(hist_arr) >= lag else np.nan
            for window in ROLLING_WINDOWS:
                w = hist_arr[-window:]
                row[f"{TARGET}_roll_mean_{window}"] = w.mean() if len(w) > 0 else np.nan
                row[f"{TARGET}_roll_std_{window}"] = w.std() if len(w) > 1 else np.nan
            row[f"{TARGET}_diff_1"] = (hist_arr[-1] - hist_arr[-2]) if len(hist_arr) >= 2 else np.nan

            X_row = pd.DataFrame([row])
            if is_lgb:
                for c in cat_cols_lgb:
                    if c in X_row.columns:
                        X_row[c] = X_row[c].astype("category")

            pred = model.predict(X_row[feature_cols])[0]
            pred = float(np.clip(pred, clip_lo, clip_hi))

            history.append(pred)
            predictions.append({DATETIME_COL: row[DATETIME_COL], GROUP_COL: pos, TARGET: pred})

    pred_df = pd.DataFrame(predictions)
    result = test_df_raw.merge(pred_df, on=[DATETIME_COL, GROUP_COL], how="left")

    n_missing = result[TARGET].isna().sum()
    if n_missing > 0:
        print(f"[Peringatan] {n_missing} baris prediksi masih kosong, dilakukan fallback fill.")
        original_order = result.index
        result = result.sort_values([GROUP_COL, DATETIME_COL])
        result[TARGET] = result.groupby(GROUP_COL)[TARGET].transform(lambda s: s.ffill().bfill())
        pos_median = train_df_raw.groupby(GROUP_COL)[TARGET].median()
        still_missing = result[TARGET].isna()
        if still_missing.any():
            result.loc[still_missing, TARGET] = result.loc[still_missing, GROUP_COL].map(pos_median)
        result = result.loc[original_order]

    return result[["id", TARGET]]


In [ ]:
# Rentang historis final untuk clipping: seluruh train.csv yang sudah dibersihkan
pos_bounds_full = train_df.groupby(GROUP_COL)[TARGET].agg(["min", "max"])

submission_xgb = recursive_forecast(train_df, test_df, koor_df, datling_df, model,
                                     feature_cols, encoders, pos_bounds_full, is_lgb=False)
submission_lgb = recursive_forecast(train_df, test_df, koor_df, datling_df, lgb_model,
                                     feature_cols, encoders, pos_bounds_full, is_lgb=True)

# Ensemble Best-per-Pos: pilih prediksi model dengan RMSE recursive terendah
# di stasiun tsb (pos_to_model dihitung dari evaluasi holdout test di atas).
submission_ensemble = submission_xgb.merge(submission_lgb, on="id", suffixes=("_xgb", "_lgb"))
submission_ensemble[["_dt", "_pos"]] = submission_ensemble["id"].str.split(" - ", n=1, expand=True)
submission_ensemble["model_terpilih"] = submission_ensemble["_pos"].map(pos_to_model).fillna("XGBoost")
submission_ensemble[TARGET] = np.where(
    submission_ensemble["model_terpilih"] == "XGBoost",
    submission_ensemble[f"{TARGET}_xgb"], submission_ensemble[f"{TARGET}_lgb"],
)
submission_ensemble = submission_ensemble[["id", TARGET]]

print("Jumlah nilai kosong (XGBoost)  :", submission_xgb[TARGET].isna().sum())
print("Jumlah nilai kosong (LightGBM) :", submission_lgb[TARGET].isna().sum())
print("Jumlah nilai kosong (Ensemble) :", submission_ensemble[TARGET].isna().sum())


In [ ]:
# Pilih strategi final otomatis berdasarkan scoreboard_df (RMSE recursive terendah)
best_strategy = scoreboard_df.iloc[0]["Model"]
strategy_map = {"XGBoost": submission_xgb, "LightGBM": submission_lgb, "Ensemble": submission_ensemble}

submission_xgb.to_csv("submission_xgb.csv", index=False)
submission_lgb.to_csv("submission_lgb.csv", index=False)
submission_ensemble.to_csv("submission_ensemble.csv", index=False)
print("Semua file submission berhasil dibuat (submission.csv = strategi terbaik).")


### Sanity Check Submission

In [ ]:
final_submission[["datetime", "nama_pos"]] = final_submission["id"].str.split(" - ", n=1, expand=True)

sanity = final_submission.groupby("nama_pos")[TARGET].agg(["min", "max", "mean"]).join(
    pos_bounds_full, rsuffix="_train"
)
sanity["out_of_range"] = (sanity["min"] < sanity["min_train"] - 1e-6) | (sanity["max"] > sanity["max_train"] + 1e-6)
display(sanity.sort_values("out_of_range", ascending=False))

print(f"\nJumlah pos dengan prediksi di luar rentang historis (setelah clipping): "
      f"{sanity['out_of_range'].sum()} / {len(sanity)}")
